# ViTPose-base + RT-DETR — DIMER E2E keypoint fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/vitpose-keypoint-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/vitpose-keypoint-pipeline/blob/main/tutorials/vitpose_keypoint_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-usyd--community%2Fvitpose--base-ffcc4d?style=flat)](https://huggingface.co/usyd-community/vitpose-base) [![Upstream](https://img.shields.io/badge/Upstream-ViTAE--Transformer%2FViTPose-181717?style=flat&logo=github&logoColor=white)](https://github.com/ViTAE-Transformer/ViTPose) [![arXiv](https://img.shields.io/badge/arXiv-2204.12484-b31b1b.svg)](https://arxiv.org/abs/2204.12484)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.2 — **standalone** (§4)  
**Capability:** two-stage human pose estimation (`person` boxes from the pinned `PekingU/rtdetr_r50vd` detector or boxes you supply, then 17 COCO keypoints per person from the pinned `usyd-community/vitpose-base` weights) and bounded supervised fine-tuning of the ViTPose heatmap head and last encoder blocks on labelled persons

**This notebook is standalone.** It carries the repository's package (3 modules under `src/vitpose_keypoint_pipeline/`, at revision `e98384143a7a`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `95be2991424e646950d656bb7fc15ec9be119700` (~532 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2.2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime builds an isolated environment from the hash-locked pins (nothing is installed into the notebook's own Python, so no restart is needed and Run all completes in one pass), stages and digest-verifies the pinned `usyd-community/vitpose-base` and `PekingU/rtdetr_r50vd` snapshots (a 360 MB and a 172 MB `model.safetensors`; no pickle is opened anywhere), fetches the 300 pinned COCO val2017 photographs from the COCO image host (about 52 MB, each refused on any byte-size or SHA-256 mismatch), turns their 498 labelled persons into the stated small-person input (the whole image downscaled so the person box is 40 px tall, then JPEG at quality 30) with the boxes and joints scaled to match, splits them by image into 180 / 45 / 75 photographs (290 / 72 / 136 persons), estimates a synthetic drawing through the two-stage inference contract with an input manifest and a rejection probe, measures the frozen model's PCK and mean OKS over the 136 held-out persons beside the box-centre and mean-pose baselines and against the same persons at full resolution, runs a bounded fine-tuning of the heatmap head and the last two encoder blocks with the heatmap mean-squared error and epoch selection on the mean validation PCK and OKS, scores the held-out persons again per category, re-estimates the drawing and four held-out persons with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify keypoint parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On an RTX 5070 Ti the whole path took under three minutes after the downloads; a CUDA runtime is used automatically when present, and on a 2-vCPU hosted runtime expect the eight epochs to take half an hour or more.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and either set `BYOD_PATH` to a zip or folder in the runtime (Colab, Kaggle or Jupyter) or leave it empty to upload one zip in Colab, then choose **Run after** from that cell (it first puts the model back to the pinned base) to supply photographs plus a `keypoints.csv` (`file`, `person`, `x0`, `y0`, `x1`, `y1`, optional `category`, and one `<joint>_x` / `<joint>_y` pair per labelled COCO joint, blank when unlabelled — at least 12 of the 17 per person, at at least **12 persons on 12 single-person images**, or more persons per image with at least 8 on the training images — `min_byod_records()`). Your persons are used as labelled — the notebook does not degrade them — and pass through the same validation, image-disjoint split, baselines, fine-tuning, held-out evaluation, artifact export and reload-parity cells as the COCO sample. Uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

ViTPose is a **top-down** pose estimator: it needs a person box first. Stage 1 runs the RT-DETR R50-VD detector (the same checkpoint the sibling `rtdetr-detection-pipeline` wraps) over the whole image and keeps the boxes labelled `person` above a threshold; stage 2 affine-warps each box to a 192×256 crop, runs a plain ViT-B backbone with a heatmap decoder (89,994,513 parameters, of which the head is 4,199,697; published under the **Apache-2.0** licence), and reads 17 COCO keypoints — nose, eyes, ears, shoulders, elbows, wrists, hips, knees, ankles — back in input-pixel coordinates, each with the maximum of its 64×48 heatmap as its score (a ranking signal, not a calibrated probability).

What this notebook adds to inference is **adaptation with labelled persons** under a stated input shift. ViTPose was trained on COCO persons at the crop resolution a 256-px box gives it; on the same photographs the frozen model reaches a PCK of **0.957** (the build record's figure on the 136 held-out persons), so there is nothing honest to gain there. The input the adaptation is about is the **distant, low-quality person** a surveillance camera or a wide shot hands a pose estimator: each photograph is downscaled so the person box is **40 px tall** and re-encoded as JPEG at quality 30, and on that input the frozen model drops to **0.598** — not far above the 0.472 that placing every joint at its mean position in the box achieves with no model at all. So the honest question is narrow: does a bounded fine-tuning of the heatmap head and the last two encoder blocks on 290 such persons move the held-out **PCK** and **mean OKS** on an image-disjoint test split, per category, past the two **box-only baselines** — while the same model still scores the clean, full-resolution persons as before? Nothing here is a claim about your cameras: it is one seeded split of one sample under one stated degradation.

**Snapshot note:** both pinned revisions ship `model.safetensors` (4-file manifests) — no pickle is opened anywhere in this notebook. Section 3 stages and digest-verifies both before either processor or model is constructed.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; stage and digest-verify both immutable upstream snapshots; fetch a digest-pinned photograph set with its keypoint labels, degrade it under a stated recipe, validate it and split it by image without leakage; estimate a synthetic drawing through the public two-stage API and read the output contract correctly (17 joints for any box, heatmap maxima as scores, PCK against a self-drawn figure is not a benchmark); measure the frozen model's PCK and mean OKS beside two box-only baselines and against the clean counterparts, and read the per-category breakdown; run a bounded fine-tuning with the heatmap MSE, explicit hyperparameters and validation-based epoch selection; evaluate on an image-disjoint test split; look at the adapted skeletons next to the frozen ones and the labels; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** bottom-up or multi-person association without boxes (every person needs a box, from the detector or from you), 3-D pose, tracking across frames, hand, face or whole-body keypoints beyond the 17 COCO joints, fine-tuning of the detector or of the patch embedding and earlier encoder blocks, COCO OKS-based average precision proper (only PCK and the per-joint OKS kernel averaged over labelled joints are computed here — no AP over thresholds), evaluation on the full COCO validation set or any benchmark (only one seeded 498-person sample from 300 of its images is scored), and any claim that a 40-px JPEG person stands in for your camera's degradation. The repository exposes none of these.

**Who this notebook is for.** A learner who knows basic Python, has used Colab or Jupyter, and wants to see how a pretrained pose estimator behaves under an input shift — small, low-quality persons — and what a bounded fine-tune recovers, read honestly against box-only baselines and the clean input. No prior experience with pose estimation or fine-tuning is assumed; each term is explained where it first matters and again in the **Glossary** at the end. A GPU is strongly recommended: on CPU the fine-tune takes half an hour or more.

**Input → Model → Output.**

| | Two-stage pose estimation | Bounded fine-tuning |
|---|---|---|
| Input | one photograph (16..4,096 px); person boxes from the detector or from you | labelled persons: 290 training and 72 validation in the sample, each person downscaled to a 40-px box and JPEG-30, split by photograph |
| Model | RT-DETR finds `person` boxes; ViTPose warps each box to 192 × 256 and reads 17 COCO joints from 64 × 48 heatmaps | the heatmap head and the last two encoder blocks trained on the joint-weighted heatmap MSE; validation PCK + OKS chooses the epoch |
| Output | 17 joints per box with heatmap-maximum scores — a ranking signal, not a probability | a safetensors adapter, and held-out PCK and mean OKS on the small persons and on their clean counterparts |

**How to use this notebook.** Choose a runtime (a GPU is strongly recommended), then **Runtime → Run all**. Run all completes in one pass: Section 1 installs nothing into the notebook's own Python, so no restart is needed. Sections 1–3 are **infrastructure** — the isolated environment, the carried package and the two model snapshots — and their cells are collapsed; you may run them without studying them. The learning path starts in Section 4. Form fields (`# @param`) are the only values meant to be edited, and the defaults reproduce the recorded run. Before each principal result the notebook asks you to **Predict**; after it come **What to notice** and a collapsible **Check your reasoning** with a worked answer that names the run it quotes — the Kaggle T4 release run of 20 September 2026 or the build workstation's record (RTX 5070 Ti). Section 10 is a **change-one-thing experiment**, off by default. **Troubleshooting**, a **Glossary** and a **Conclusion** template are at the end. Writing your predictions down is optional.

**Roadmap:** 1–3 infrastructure → 4 COCO persons, the small-person degradation and an image-level split *(evaluation practice)* → 5 the two-stage inference contract on a drawing *(core concept: top-down pose)* → 6 box-only baselines and the frozen model, small and clean *(evaluation practice)* → 7 bounded fine-tuning *(core concept)* → 8 held-out evaluation → 9 skeletons, export and reload *(engineering)* → 10 change one thing (optional) → conclude.

## Prerequisites

- **Learner:** basic Python and Colab or Jupyter familiarity; no prior experience with pose estimation or fine-tuning. The notebook explains top-down pose, heatmaps, PCK, OKS, the box-only baselines, the clean counterparts and the adapter where they are first used; the Glossary repeats them.
- **Runtime:** a fresh supported **Linux x86_64** runtime (Google Colab, Kaggle or Linux Jupyter); a GPU is strongly recommended for Section 7. Section 1 builds its own Python 3.12.12 environment from a hash-locked list of manylinux wheels, so the kernel's own Python version does not matter and nothing is installed into it. The default path runs on CPU (float32) and uses CUDA automatically when available. Stage 2 costs about 0.15 s per person on the build workstation's CPU and a few milliseconds on a GPU; the build record measured 1.1 s to prepare the 290 training crops and 31.5 s for the eight epochs with per-epoch validation scoring on an RTX 5070 Ti, and the whole default path took about two and a half minutes there with both snapshots and the photographs already cached. A 2-vCPU hosted runtime will take much longer (ViT-B forward and backward on 290 crops per epoch). The pinned `torch==2.14.0` install and the two checkpoints (360 MB ViTPose, 172 MB RT-DETR) are the large downloads of the run; the photographs are about 52 MB.
- **Knowledge:** basic Python, NumPy and PIL; what a bounding box in xyxy pixel coordinates is; what a keypoint heatmap is and why its maximum is not a probability; what PCK and OKS measure and why a self-drawn figure is a plumbing check while a held-out split under a stated degradation is a measurement of that degradation only.
- **Data contract:** records are `{id, image, box, keypoints, category?}` — `image` a PIL image (or a file decodable by Pillow) with sides within 16..4096 px, `box` one `[x0, y0, x1, y1]` person box inside it, `keypoints` a mapping from COCO joint name to `[x, y]` for every labelled joint (at least 12 of the 17), an optional `category` of at most 32 characters. Ids match `[A-Za-z0-9_.:-]{1,64}` and are unique; a training split needs 8..5,000 records and validation and test at least one each, so with the default 15 % + 20 % image hold-out the effective BYOD minimum is **12 persons on 12 single-person images** (`min_byod_records()`); splitting keeps every image (by decoded pixels) in one split and every person with its image. BYOD accepts one zip (or directory) of photographs plus a `keypoints.csv` in the layout named above.
- **Validation is structural, not semantic:** every image is decoded, every box and joint checked to lie inside it, but nothing checks that a joint is where its name says — a mislabelled set is fine-tuned on without complaint.
- **Privacy:** Do not upload confidential or restricted data (photographs of identifiable people you have no consent to process) to a hosted runtime unless you are authorized to process it there. The default path uploads nothing.
- **External access (data):** besides the two model snapshots, the default path fetches 300 JPEG files from `http://images.cocodataset.org/val2017/<id>.jpg` (about 52 MB in total), each pinned by byte size and SHA-256 in the carried `samples.py` and refused on any mismatch; every photograph's Flickr page and licence (CC BY 2.0 or CC BY-SA 2.0 — COCO licence ids 4 and 5 only) are kept in its record, and the 2017 keypoint annotations it carries are CC BY 4.0 (COCO Consortium). Nothing is redistributed by this repository.
- **External access:** the Hugging Face Hub, to fetch the pinned `usyd-community/vitpose-base` snapshot and the pinned detector snapshot (~532 MB together) at their pinned revisions (`95be2991424e…` for the pose model), and `images.cocodataset.org`, for the 300 pinned photographs named in the data bullet above — over plain HTTP, so integrity rests on the per-file byte size and SHA-256, which every file is checked against. No credentials are required; nothing is installed from this repository. The isolated environment also needs PyPI (`pypi.org`, `files.pythonhosted.org`) for the pinned `uv` wheel and the 48 hash-locked packages, and the managed CPython 3.12.12 build (python-build-standalone) that `uv` downloads.

## 1. Install the pinned runtime (isolated environment)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

Hosted runtimes such as Colab and Kaggle import some packages, NumPy and often PyTorch among them, before the first cell runs, and Python cannot swap a module that is already loaded. Installing the pins into the notebook's own Python would therefore leave mixed versions or require a manual restart. Instead, the next cell builds a separate environment (`dimer_isolated_env_<lock digest>/`) and leaves the kernel's packages untouched: it downloads the pinned `uv` 0.12.15 wheel and refuses it unless its size and SHA-256 match, has `uv` install the managed CPython **3.12.12** (whatever Python the kernel itself runs), and installs the 48 packages of the carried hash-locked requirements (`tutorials/requirements-colab.lock.txt`, compiled from the pins in the cell) with `--require-hashes --only-binary :all:`, so every package, direct or transitive, is the exact file that was locked. No repository code is installed. The lock holds manylinux x86_64 wheels, so the notebook supports **Linux x86_64 runtimes only** (Google Colab, Kaggle or Linux Jupyter); on any other platform the cell stops with that message.

The same cell then starts one Python process in that environment and routes **every later code cell** to it. Printed output comes back to the notebook as usual, variables persist from cell to cell, and an error stops **Run all** as it would in the kernel. It is the only cell that runs in the kernel (it is marked `# dimer: kernel cell`), and it is safe to re-run: a complete environment built from this exact lock is reused rather than rebuilt, and a live worker is kept with every variable created so far, so the cells after it keep working. To start over, restart the session and choose **Run all**. `DIMER_NOTEBOOK_CI_PREINSTALLED=1` lets an executor that has already installed exactly these pins run every cell in its own kernel instead.

In [ ]:
# @title Infrastructure: build (or reuse) the isolated environment and route later cells to it
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
import hashlib
import io
import os
import platform
import signal
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile
from multiprocessing.connection import Connection
from pathlib import Path

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
    'scipy==1.18.1',
]
MANAGED_PYTHON = '3.12.12'
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_BYTES = 20081404
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
LOCK_NAME = 'requirements-colab.lock.txt'
LOCK_SHA256 = '82f7302e59fffbb5c7d2d0480b8112153afbf22e6b05b24a585c484c69fe0841'
LOCKED_PACKAGES = 48
# The hash-locked requirements, compiled from the PINS above with `uv pip compile --generate-hashes` for manylinux x86_64.
LOCK_TEXT = r'''# This file was autogenerated by uv via the following command:
#    uv pip compile pyproject.toml --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: -o tutorials/requirements-colab.lock.txt
certifi==2026.7.22 \
    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \
    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55
    # via requests
charset-normalizer==3.5.2 \
    --hash=sha256:01077390b03f7988f11d700a2194e69b119741a86b1a638b1db88891e3eced8e \
    --hash=sha256:01b0c0d2262a9e28e8484a278c7e1b5d650e3ac8cf2683d2967e25899f208bdf \
    --hash=sha256:04851f73ae72b8413dddadb16a49dfee95263553741fd42d546f7d66907e6be5 \
    --hash=sha256:0521c5665880b33d603717defa76c094048900010897909952397feb3039da56 \
    --hash=sha256:0774bf9bf620249fee3e0b8b9fd3065de213be30f3aa94ce2494b3b638949e26 \
    --hash=sha256:0891b9d3903c5571c03771ca669a4b0ec5618ca722a5c957d3d29cd4e5062848 \
    --hash=sha256:0c951d5e6dd9c2ff60609476752bee49da4206adde960ebc247766937f72e718 \
    --hash=sha256:0fed1d06615f022ee3b13caf5e8b180cfea32bb2c5aded8a9d44277afc040f93 \
    --hash=sha256:114e4d0c92d618409ed82a99e22b5c5e768fe995f2973f78265f4524f49d4640 \
    --hash=sha256:11912e4bb14baae7c5d8791aa55ba0a3a03ec6729073307b0f57270abaa713d3 \
    --hash=sha256:11a4d68a6ecda3292cb1e50239e111543ba5d709bb62a6b4ea1afcfa729d8875 \
    --hash=sha256:124fbf1a8ff966d87ae05bb8bd45a71f966055ed8bba320d0c7cf450bc5f4d0e \
    --hash=sha256:1461ac396c4fdb983a675f20aa555624f0ee18ac83d832b9244ffff3d8055275 \
    --hash=sha256:1503bccbeb36d5527790c3930327704c39af22de3112f1b1666a9f3ce15ee204 \
    --hash=sha256:15bb4005af6320d259dc7593ca84a38d7fe06a421dbcf7b910ae23979101e787 \
    --hash=sha256:15c44f7edfd477b06f517a5cc317fc1707edb9de2c865f43d4b6513907473234 \
    --hash=sha256:16fa0eccf81304b79c5cd87f9271c3b85dd9dd99245e4422ae9c0dd45e0f99d3 \
    --hash=sha256:183b88127acdb4fabe59d951ab424faf1af7b63cdbb5f776186c1ea2ffcaed98 \
    --hash=sha256:195c26fb65950f8fce54e26349852b7bdd7c5f120aeefbcc440b8a20faaed4a3 \
    --hash=sha256:1afb975bd5d68d5ce9f6b6d44fdf2f7e34b895a35e95708a7a91b20a3b51d187 \
    --hash=sha256:1b4cbc7c3491ccb4aa17fcd8165649d01cf39f76de1696da8631b5f71b85401d \
    --hash=sha256:1bc0baf5ef96b6ede57d47f4b8fe4d9d84019c3bfcbeb20a41edc6a6ee341f1f \
    --hash=sha256:1c50fe28bbc2ced33386f298650d91218076c05420e6cbd790b913adc41659e7 \
    --hash=sha256:1db38f4c5496827c1a501846d64d14c3b80c7e6714e406cd7dc36a9899fa1011 \
    --hash=sha256:211d5a3eb6af8f513b8d4ca19a8c1b7accab1b5f0d3175f9826b03c1a920dc1f \
    --hash=sha256:23851fb4e1b85ed3f6c2a27b777cdfe2e19fb5b38429a8faf38c7542b7665869 \
    --hash=sha256:254eb48b9fa5ee9898a3c445825a1f340fe53712a098904b39b0bddba8ea3cb1 \
    --hash=sha256:2625388c6c754520c37abaf3b41eb34d1cc4a373f457898f08606c8e362b891d \
    --hash=sha256:281cb91036248400f4cc957495cccd44c275c2e0c5854f7e45ac5cf7dc193847 \
    --hash=sha256:28a15fdad492a99b6eccfaaed66ef3f74050680545ea61ec8b2f4c538f1f1320 \
    --hash=sha256:28b4f0d66fb834ff90f28209ac7bce77868c45d8c93e26f906709d9b7c2e1af9 \
    --hash=sha256:2a925889534b3748302dae5dead07cc13480de1dac3aea80a941b729b471ef93 \
    --hash=sha256:2b7b3bbfb4fe8ef40600792d762fbaa9057559f9d3fad209525b7a22b99e91fd \
    --hash=sha256:2c9ad19a6cfcd5ea5c0d41161d22f9df1dcc277e9bef2751391334546a314c00 \
    --hash=sha256:2cc961b171b3f3440f410489ab3573e86aea8736134ebbb40ea1338b7f0831bc \
    --hash=sha256:2ce45c6627b22c47e390bc91a41c3d13032192e699fa0bea96e9671b373d69b0 \
    --hash=sha256:2e06a3a98f916dd41d27f3105e02e7a40181c98c94b9158733d03a6f80506c09 \
    --hash=sha256:304d5463e65a35d7bb0850550e0780395395f6fcf452f04db7d5ca7cecc425ac \
    --hash=sha256:304d8e4d493af723536393eee0c689eb7813f4a474c8b479dee63f1fdd98f621 \
    --hash=sha256:30fcd120b732aa79317f08dee04d7de0847822e4cf7ee0e9f445bb958832252c \
    --hash=sha256:31f3930700408d211f13378ccbe1c40845d8da54bd0681fac3a9b5aae81c7aa8 \
    --hash=sha256:34276fd796040bf0993ab33a369aa572e6979c7aab225a88893667ad8eac8f7a \
    --hash=sha256:355ad8011081dec5412240c087a9a0c9d4d5039f3ed11a3f13e18c2b29b56c51 \
    --hash=sha256:38a873987f3be698494da8b2e3085e29da02da7b633dce73e79c699a113d7bf0 \
    --hash=sha256:39de2a259fc954455c57274dc94c79d5842774e1247a016aff30bc0efed0f4ef \
    --hash=sha256:3d14b50de6bf4d0edf857a9386836846f982b8f524e188e2e68b96d702bcf4aa \
    --hash=sha256:3d21b8b13c7592db2ac5e544a6d83187b995257472b0c9e8351b6d507ae37ed6 \
    --hash=sha256:3d31298449090ab8d47b7b1b2a555ff73cac7ed438a08b7ac160980c7ebed649 \
    --hash=sha256:3ddacd27458c45bdacd6bd6db644bfb730efbf9e830310186e3045c9c5be8fb2 \
    --hash=sha256:3df041de8887954562c9b261cba85ca0e9ded74048daf125f45edcfaa4832229 \
    --hash=sha256:40ab6bffa02ae10a0581e6c198be7d2d8ca5c2a0c64e4ed3465d766df457573e \
    --hash=sha256:4275811936e2f06feff5e598fb42a1b7ae852da8e39605211892b56b81a34efd \
    --hash=sha256:443eae2bf318abeaf6f15d785138f71fd6de770e99a92158b8b814265e079115 \
    --hash=sha256:447441e76ec720b15e64418d32e092297340387053047c7c694f579efb0ee1d9 \
    --hash=sha256:4495c5002a7b28557e7e222e77e0b661183e432b7d6d2e788101e3f240e05b8c \
    --hash=sha256:44bd4fbb29dfbeba60e7d2bd000c59e4b21ddb3cc53912b14048d37092706d7c \
    --hash=sha256:4685902cf26edf013ed7a3da0f426ebba7a00ebb9541386d835afbf002c11cab \
    --hash=sha256:498dc3188ca05a68231ac3fdbfc7f57eb67e1343c30e0fea17f8218c1599b253 \
    --hash=sha256:4c2b5031f63e331e3839b40aed2dd6f191e9c07edbde303e7876846ea1946995 \
    --hash=sha256:4d48f2d08b9de5864e2c8744d4461b862fb149a18274abc8b698c45975573438 \
    --hash=sha256:4f87960d57feabfb618e4e0af6e7371645fa26a277860739d6e5d6e0012c92f0 \
    --hash=sha256:50e3adfb96fc189eb27b1cf62d3b598b89b4bb0420d93a3d3e42e137409011be \
    --hash=sha256:51cf45226a9b588d0d2b4880c62d686934b63ab0bd79ca23ab0e9762eb27441b \
    --hash=sha256:52aa6992700996af31f375de0c6bacd402b0097fe40b53c426b9f51a90ebabc7 \
    --hash=sha256:55ea99acb17b9325618de155a0cd6a2e8f5d10be008113e1d433bbb58db543b2 \
    --hash=sha256:56bc200a365efb37383b7852e4cc5898d3b2da5987289b543956cf8cad71018a \
    --hash=sha256:588461c2e8384d309bd63e5826019b6977bc66d629b99ac8737bb795d7b2cb5a \
    --hash=sha256:58ca3755ee7ff7f59b57789ec9833c9de9ea275405cdd240eda1f193112e398a \
    --hash=sha256:58f361dcbab699cf8f42db3f47c8e7fd1036f138c23a5d08de9fde5f425a730c \
    --hash=sha256:598a11a2c7ebaa5334bf698bf29568c9c390abac6a154d8170fedecd1cea38c5 \
    --hash=sha256:59f63901b0031c3136cf64704dcb21de0bbae62ce2c9529bc39d27665463de37 \
    --hash=sha256:5cde776b7cc66e4f6c99612cea4aa7269aa65863f7a15841b2c264f103822f4e \
    --hash=sha256:5e2b6b57e9733d39f0c9fd3185efa6b8e29652c4cd8fe94180272cf6ed9a78c4 \
    --hash=sha256:5fb29fb8cd1a46c27a1bf9613ad5ec2599310d46b4025d9556404a6b6a292800 \
    --hash=sha256:6045373d5a89a5ec71afde535db987ca28e76dfa276c2d4c818265b375d4b055 \
    --hash=sha256:619799369eeef6366ed3e8755a5670f4f2f0fb6b30a0fd7264dc0fdc2357058e \
    --hash=sha256:62588a277bfb59def052abd940703fa35107152bf479781a878617d60faf8fb5 \
    --hash=sha256:62603db9a7caa0802eaa28c1c46fecd7b3a263a774069c24c3c28c302448721c \
    --hash=sha256:65cd72beeeca9d3aaea1201e5923859f308f952f9c71de93f06063c79f0f7a3b \
    --hash=sha256:68eb192d85ab8e5f6ec69c2bc6ac0179fbf04a5ac1569d12fbef74883fe102d0 \
    --hash=sha256:6bd128f206a7752ae1f2ab6c61bf8a24ba28913a10df8b14c2637b973ff97a80 \
    --hash=sha256:6be488a102b8cf28d0391d8c4ba7748938ae28b78ad901f8585520fca33ead1a \
    --hash=sha256:7218e8f32b0956cfcd048fd42d9d5779809745ca1d86113ca56f66e7ae1549c4 \
    --hash=sha256:7441d755b7ab94f8d4eb3e43ec05482d760842fd263d003a99102d742cd835e2 \
    --hash=sha256:749e97e1b32313717a565abbe321bc2190bc8b35f1a67e4cdbc7c56c8d8ffe58 \
    --hash=sha256:75a3ceed0724d625d64b86ca20aba182e4df462e04c2414fc941c0f523f06aac \
    --hash=sha256:780fbe7cab297b81dad9fb8dc5eb003c0468ffb0d9e5f65068c53a34661a96bc \
    --hash=sha256:78456a747de8dc58360ffa581f30a002baf5aa28cb262536545e91f113ed7639 \
    --hash=sha256:7967d08cf06dee78443b874f98c98036f624f3a4e73e11f9f64f5be4d25393cf \
    --hash=sha256:7a881931aa470808df94a8c380eed2bbbc76cd9dc622310f99665658c821eb6d \
    --hash=sha256:7dcd882da75ef9adf94903b1e3b9419e8aa8fb4c7396822b834b9ef7fb96954f \
    --hash=sha256:7e841fb9010836c992c9f12fcbd43a831de93a5f726fc1ccd8ca1d0268c5014c \
    --hash=sha256:7fdde2c9fd9e3eca40631e024664cf2584272cc8f96308cbe5fdfc930f51d8bc \
    --hash=sha256:8024d00c3faf3fc0c16e07a69f4405e8eac7cc0ab15f65fe6cf43827c4cf72b4 \
    --hash=sha256:80d02b6f04e92601a081dd97b23d3128033098bff5d35d392ddcc0476ea11253 \
    --hash=sha256:838dcc90063569a0448120554591a1d6c4a4ffe11babf048908793154ab86ade \
    --hash=sha256:849df64e889b2e17230d58410a03dba311a65b163508fd33679b2b737d4b7858 \
    --hash=sha256:87475fabc8d9996fd9c27debb395e642e8c838d78a00b6e932227a0e06b81e26 \
    --hash=sha256:87e50a3e7cb90af586b6c5faf23e302a970415ac73bd7bd90a515a04b427ef96 \
    --hash=sha256:89b53f3cda69831909888e0494f4fa0bcd3537e3e138dabeb620bd6ad946bae8 \
    --hash=sha256:8a893cc101149f80a653f82062ebc95b34525a2614382e1da5458fe7c6997249 \
    --hash=sha256:8b2bfab86aa71ae13aa41a6a26aab338e0db2b8bc75434b05aea89e011ff35a4 \
    --hash=sha256:8d86d6fc60743dc916eb79e2eb1ec4818e21e427731543af40a3021851174a13 \
    --hash=sha256:915563965d418f986e7e145accc592eae9e1a1be3566ff98a05d7a9ec42a76e1 \
    --hash=sha256:92888bb3187c5ba50500b00b3b310c9f2c651709d28036077680cb5255450a03 \
    --hash=sha256:93223adc95033dd47133a46ccfc316a0139176fd79085762e27202ec56018f03 \
    --hash=sha256:9373ad13ef0d2c0fb761e04e55bfdee5a08b52cef2c882c8fbe9935b1517152e \
    --hash=sha256:9409a8bf35cf78353942504b24a57de3d75b708997a1e4bd8db71ac8633ce364 \
    --hash=sha256:9b7f416ff0978e2f2249330527f0ad6fa02f4932e6199692d3b52da2048c19e4 \
    --hash=sha256:9bde855991b7e362c146535e3136a50bfaffc0487d38b33ca7e5edefc6e23849 \
    --hash=sha256:9cae88599c7219005d879f98e5ed53341e9a122af585e1091200358a3003d2a0 \
    --hash=sha256:9cf9b1a857e25c4baceeb3624e92a56df3668f398c4acba74e174d81fb4d1d3a \
    --hash=sha256:9f56f72050826f63dcee7a7f55b0a77168cb3bfc553fd405e7f8f9ece75a4036 \
    --hash=sha256:a090bb2c68df85450502e3e20d665e3a5af9c65a84d6508ed477badd49166fd3 \
    --hash=sha256:a192e2c40070d92c3ccf777e3a5c4ff515573cd2bb7ed0c537fdadbbec5bbf21 \
    --hash=sha256:a19a731138fc27d5682277d3b9df22855cea1239bce7fcec5f78f42ef2d1f3c3 \
    --hash=sha256:a66c3bc5ab1f0ff2164fc9965ddd611ff0802173f4b9d24554c563f6ab7e1d6e \
    --hash=sha256:a815775b6c38d4e0ff7bcffbeba67feded90202bb6a226b8dd35f1c855217413 \
    --hash=sha256:a89012d6d5476ee112d20d998570ed58df2260a852afb1758809cd6900411d21 \
    --hash=sha256:ae4f5fea5b8b8ccff88238cc8569303e5ee95efae67fa62922a311397a71f346 \
    --hash=sha256:b6856554c4f44d79fc2307d5768854310a8f0096e501c75637542c82292b0429 \
    --hash=sha256:b6b751274acb69d77b3323d6b7dbaa3c7fdfc1eb829b7eb61d262f32e1af9685 \
    --hash=sha256:b736353c0a625bbd5fcec108576e2385db3496f4f771f785ff32e108d3c3bc45 \
    --hash=sha256:b7fd005a73d9e657273b7a10dc71a9e03c8fb9ee6999798d6918ce095b81ac7f \
    --hash=sha256:b91363207bd9dc966a691e959bb47f64b30f7ac4b072be9968b366982f7db77c \
    --hash=sha256:ba0b1d2620edf869789c3879223f52bf2afc5d31b3cb47cc57b3a12c05e2aa9d \
    --hash=sha256:bbbfc8e28816f19d7c0f1816664980c0a9875d01b27cdf8eedddb639d9e108ad \
    --hash=sha256:bd16aabe4a02a297c23417aa17ac6299dbd8c49f673bcd645b4929b11f5a4400 \
    --hash=sha256:c0afc6800ba57ccc350374c5bd6150419915d95ce93cdbab2d783d75eaf30ecb \
    --hash=sha256:c6708715abcf3c73b99508253e961a9967f02fe536532834149574eda6de0d1c \
    --hash=sha256:c7c9ab723cde841fefb34efbad91e87f00a674b1fe1cd0784fde742bf2c154dc \
    --hash=sha256:c8f3d67aeaf55f017982b73683f0e7342ba2f6635a78f69ce89ebb26aa411e5c \
    --hash=sha256:c9790464842f85f437dbbb54417eda1e0e6bfc52dd8d22d6fd1c994b73b2dc74 \
    --hash=sha256:ca403d7e4798f525fdfc78e258820419cbbd0f0ecbab9de7840e3c017cf6b8cf \
    --hash=sha256:d008d90a7f2471519aef0c90dfbe73b3e6e4d5e66ac48e19154c17e89e98b604 \
    --hash=sha256:d19fbd981a488e22cd04883659ca6b08f50b5974f9fd7c95655ef6a043e5893f \
    --hash=sha256:d1befeed746d247c81127bb14de9dc3d30edb6e5976d34f83f86ed262b1d9105 \
    --hash=sha256:d2374b62878abb00cd8309b32af6c0b715cd02dec0ca74ef12e5069bdc64144a \
    --hash=sha256:d376bbd28b3a8999db1a103b3b388aee6f1ddeb3e51bc2172993efdcd86e064d \
    --hash=sha256:d4a7319f304a774bed22115bc891618e45f85065ab44ea6acd07d274e750519a \
    --hash=sha256:d6734d2ef8a50fbf8445c139477da401f50d62a0606bf00e20ec6d87773fefb1 \
    --hash=sha256:d760fe2a4d7c3b226cb9026d6a842868d52a7901bd98420e1baf14e80da85cf5 \
    --hash=sha256:d913de495d90407cd859d263bee2e5d1a4ed3eb6573c04e70d9ec619a7cbed7f \
    --hash=sha256:db19d07e2e0129e974a0e65d0064fc222a446cd5122c2fd4184d2af9fc734a9e \
    --hash=sha256:dca9ab98072a5a54ebacebdc45f53e645336b320c667410b061be1ca588ae709 \
    --hash=sha256:ddc7dacc8ece3a182e7f15cb862d1fd616b46d076cb1ae9dd232b2c38b655874 \
    --hash=sha256:ddf19c062bea7a0cc80f519243d2c01dd091be0cf952a0750d4ad576709559f5 \
    --hash=sha256:def79fa35ef0cef8d2accec024f4fdc7ead3012ff02f5215c783f39f03ef8cfc \
    --hash=sha256:df29a0a7107f7011e77f4eebdddec4c7331e24d787a0b21a46d63bdf7445da95 \
    --hash=sha256:e09a3942ecbdee5cce73ea9d42da82b81b72ac1bf031ce069b93b5adf4eac8cd \
    --hash=sha256:e242bb1c5e76e97dfa9e7f209a71e93a01d7f19ffdd5cfbb2e2d55b4f08f8ab0 \
    --hash=sha256:e243bd13217235fc7290c621941c3f5cc8b66e4872495be821d7436ba2fb838d \
    --hash=sha256:e2af3aad578aa6bd1384bcf4750fc285e5a9de53f40b7d41e5a0bf748edeb2b3 \
    --hash=sha256:e4e81e09c1578b8df602e3db08b0b3ea0a6947ad612f52bf8dc5ea8d47691f0c \
    --hash=sha256:e54da4baf05720032d527874d40b65fa4d7e5c6c6a43d0c3adbeffcaf275a2b3 \
    --hash=sha256:e80e6c2f55656b4824d72065abb4ddd6a525c74bd78a0aab5d9fc2cf4fb5af50 \
    --hash=sha256:ed2a239c0ea213acc1908150a3037257083c7c083128f1a4cec2ec4b97dca491 \
    --hash=sha256:ed905975ab14056a2e5eb1c376cb2e1ebc5396baf84163939c518556fccde9f5 \
    --hash=sha256:ee21e28f0430bd6dc9086c6e525d5e818a44a5ad19720c8a0ef766792f3eb5e5 \
    --hash=sha256:ee43c17b173d46a3212baa6ead3ae258eeabdae48c263a01ccf0218c366dd655 \
    --hash=sha256:ef4fcbf3327382cd4c9f540babd61248208af7b93eec4de397b4d5f58a09e288 \
    --hash=sha256:eff0ac9dbe711a4aee69bf04a83896aa9b85f19641264053a9f6d48573abb7dd \
    --hash=sha256:f0aa869112ef88429ae17820d99c3dd9504c9e9c671d3c246f3d7442cb051084 \
    --hash=sha256:f3c96f633825733f735c5a9cf21d21a257d8e1edf0b1cee0a064b9c424ca0f7d \
    --hash=sha256:f5833ad231be5eb6553de524a70f48d71b2c8563101750531e0b80184e175cd4 \
    --hash=sha256:f5ec61164adcec446f8969a3358ec3f9b26bbda3b9213e5586d219afa8df2915 \
    --hash=sha256:f7d486c83842422badd511868fd8a9a20e9407ace71564b6af47ce7e60a336c1 \
    --hash=sha256:fb9e68df06293761f9fe66ade60a9bc6d0f5e42b8acf2939a9158af86ab0e5bd \
    --hash=sha256:fc14a032f813bf5fe624d991960ea83e9715adc27e4c1830a2361eb1d02ac341 \
    --hash=sha256:fcff63213e8e6e47770541a4607175404f47cbb3ebea7b6058cc82d524a0e424 \
    --hash=sha256:fd1fbe0f116b6e55da77aca2c6ddcddcfac2186cbf78bdebf40fc156efca389d \
    --hash=sha256:fe9753dfee015c570d73df76f899f18444d41388bffcde097deba51c4fadbb9f
    # via requests
cuda-bindings==13.4.3 \
    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \
    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \
    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \
    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \
    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \
    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \
    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \
    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \
    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \
    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \
    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \
    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \
    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \
    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \
    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \
    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \
    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \
    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \
    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \
    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \
    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \
    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \
    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368
    # via torch
cuda-pathfinder==1.8.3 \
    --hash=sha256:e29e59829c297a7a5233bd9cc71094fc5bddbd076951482670178f9eade39b1f
    # via cuda-bindings
cuda-toolkit==13.0.3.0 \
    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f
    # via torch
filelock==4.0.12 \
    --hash=sha256:5f17ee83ecee8a6f3e389c75822fb70a1c2f0506b99438a6dffa1d56793588c8 \
    --hash=sha256:cf42711a7ac791818b299fab0332a088c65aeeefa36290de98db92c434303b0c
    # via
    #   huggingface-hub
    #   torch
    #   transformers
fsspec==2026.9.0 \
    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \
    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f
    # via
    #   huggingface-hub
    #   torch
hf-xet==1.6.0 \
    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \
    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \
    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \
    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \
    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \
    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \
    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \
    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \
    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \
    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \
    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \
    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \
    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \
    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \
    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \
    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \
    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b
    # via huggingface-hub
huggingface-hub==0.36.2 \
    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \
    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270
    # via
    #   vitpose-keypoint-pipeline (pyproject.toml)
    #   tokenizers
    #   transformers
idna==3.20 \
    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \
    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c
    # via requests
jinja2==3.1.6 \
    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \
    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67
    # via torch
markupsafe==3.0.4 \
    --hash=sha256:007e1ffd9bf65bb6ee96df7b258fc632a4868dd5566037986c64781f35a36e98 \
    --hash=sha256:02fa4acbc6a3fc5c693c34d4dd8c1130b7fe99cc915181b0ddd6f72aeb296002 \
    --hash=sha256:03470d1a8268e692ecf79ecd565593e59d44219377a7ead61f1f1b94c1f7ff6b \
    --hash=sha256:04e7902ba80ee4bac1d50a549606527a1dcf0476cd81403db41099d3b60ec653 \
    --hash=sha256:051417f74bcaaefa316276e0ff723f541616ca51043d070da00249d9bddd3e3c \
    --hash=sha256:05295589e619b9bed252a86b532b8e27350abc372d18ba89b59375325e91ec1e \
    --hash=sha256:06de8ef6331f6e822c28d577dc8bf43fe398800477c49498f38fc38b67ff33fc \
    --hash=sha256:0764a13d34cae40db7bbf3a09b7e9b491bf4603e20b263a7a9d6b8e324975d0a \
    --hash=sha256:077293e425f28ec737dbcad442a71752e28f8ae27cde3d68acd1fb212091cd92 \
    --hash=sha256:0930db9bdc62d22944e10b066448bb65dc9abe9112880c7cab8da54db4284d5f \
    --hash=sha256:0cee7cb0f9a1b6892ea482237d9403b3d1b4603aee057d0ff01f0fac2d019a97 \
    --hash=sha256:0d9c47709875fdb321452056622e930c52afbc07a7d780762fbb8b4d91ce6fa4 \
    --hash=sha256:11935df9bf455ed0c04eb87bcd720f02b1fe5e02128a9430f23aed6f93336fc7 \
    --hash=sha256:12a606a492de952afcb43b59a14aaaaad120e708d3663dd0fdf2d738d427a691 \
    --hash=sha256:14bd2d845d62ab678eaf81da89d7b621b51756c72346745c1a594c09d49207a2 \
    --hash=sha256:15ba9e28640feef770374b116a6f019c21f52404aeabe516aa7f800587b98cfc \
    --hash=sha256:18a801868a884f216e784d7d14db2a4077143ce7610440aee2ce8f734e7cfcde \
    --hash=sha256:1c0df495a977d10460a94941799c72d5b5ab03d3858d949b55b5a66c8f371c99 \
    --hash=sha256:1caa2fa5a6184fb233153b35f654e6687bd555476f6170f29d8ee9be1a8b0af9 \
    --hash=sha256:1e1451fab512d1bcc3dc26988ec1edb0b82c2db909132872cd9356070a6b63df \
    --hash=sha256:1f1f9477e174582b0a1b583d60b66e1f2cf5d3fe12cee985e4aedf44766600e5 \
    --hash=sha256:2628d3a8cb648ecebb3c5d6b0a1052d400e4d8b7ac0fb786be8d285b50040d17 \
    --hash=sha256:26e9867520db70d37f7fb421a7f0d8adb40171011fb84ce869afa1a83370dfa8 \
    --hash=sha256:2a6ef68ae94aed8721934072b27a3b654ea2100b97e4ab864cf1489c90926fbc \
    --hash=sha256:2b2b1e18af909b448bb3cf9e3433366f7a8726271fc214e8b10e0f62a78c724b \
    --hash=sha256:2cb3dd71fc6be918ad4264346a8ed69485f9b7ed7bf35495d8e22807cd6b8bea \
    --hash=sha256:2d1b7d9308288661f56672b1b157d75fc536714d3638487bbea17b6318a78248 \
    --hash=sha256:2dad610540cb2e6272855c178f08ae9a1c7ac258a7fb71660553a5f104b42741 \
    --hash=sha256:2e5a7cd7fdd14fcb1ae5d7d8bf23d24fbd1daefd1fbca2580132e1ea75f098b5 \
    --hash=sha256:2e9ad7dd851bf45fab9f75cbff4cb493fee9979e8d8c7c9c3ee119022518edd6 \
    --hash=sha256:340cbb1957ba99929cbf19a75626d36ba1ae21d1730b287d1cf7f824a20c4fc7 \
    --hash=sha256:34bdde374c5932765d7dc685c4a1d191a3207852d67e8e0a9eb6ea85156181f1 \
    --hash=sha256:353bd63081912ab8cfa6a0c7d185934cdf8426f04c618bba6bc4b394f2069b67 \
    --hash=sha256:387d8cd30e69b3f0a72877b9ae717033396404e19095b17fe89753a981fda44f \
    --hash=sha256:3882fb412298575bae3b9c46868251f15cc69307359f87bb1b382e53d6e5a2c9 \
    --hash=sha256:38fc55594dab834470b6733dead2ee9e3f657fb0608c769dcafa0ba5ab52f45c \
    --hash=sha256:396ec4e65cc889f69786b3b89478b471cee5a3bcf468b9d9bb03e1a30fb291fc \
    --hash=sha256:39dbacefc411633db5b4378b066a9aca70a3d7e2922c9e578d825f844026eeba \
    --hash=sha256:3a93d9616ddecfb393727a0041a562cf0b15a244e20f2bd25efc7949be4c4f17 \
    --hash=sha256:3d23795802fc8bd72534836d64489bbf0f67c088959091bdb22e10735a5107bf \
    --hash=sha256:434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6 \
    --hash=sha256:436e3ffc6310d3c41878c601db29098102fe5d8a467c49da4a4125254e0980f2 \
    --hash=sha256:489505b03f692c3f376394e49194fa7a7f9e8558d6e293a7056a0032b0c38163 \
    --hash=sha256:4a540e2d3192792fc84eced57bef37851ccb2b41f73291bb17408eea77bcd278 \
    --hash=sha256:4a7cdc2a420ca01058182da4253329764d4bfa055564d1eced90e6ba1e8b1d3d \
    --hash=sha256:4bced6e2a6dba6a28f7dd3c6ce14df1b2dd495923f16ea484cad03decd463b2b \
    --hash=sha256:4cf3468d5ec187ffffcaca8e61929a37448f215dafc1386a12c750a72fe53634 \
    --hash=sha256:4e2c4809c14559aa7ef426f27fb35afbb38104c349a903bf8f3600456764bb38 \
    --hash=sha256:4ed644d75aa94a2baf7ec3a96eaa160ea58c742eb9d27c6506053c5c40fc84ed \
    --hash=sha256:4f6e0852a0283b1b1fd776eeb7b766a5f440b3e2bd31ab51af3b400585f3965c \
    --hash=sha256:5066b244f576f91afc8ee3ba029a89f99d39c79b1853fe9d39bea9f0afbec148 \
    --hash=sha256:5086f9975abb1ab531ee6afca1761e4b59a19b446f3f6522ed776963228cfe5a \
    --hash=sha256:50b5bedc9ed8a94fc8857a42ef4f84a81ea88f8d4f05dc8705fb23ee6d8dcca7 \
    --hash=sha256:52704c5d36eb6dda8866493decd61111fff86244c9b1ad225ca01b9e91e5970f \
    --hash=sha256:55ffd6ce583d97dc71dc92e930324c8c0d25aea7e3ade6ae54ef77cedb096811 \
    --hash=sha256:569d65055d367e3dcdf30c3f41119467b73d9ee9faf332bdf40402644f5ac08e \
    --hash=sha256:57f9947a7e57a081c1e3e0a2dd0d2dcf290a4531450e6f611e30084c222a7295 \
    --hash=sha256:5989cb26b2e1efc6a42216a9f6b5ee495ce5ace2e5b352a9af489976b32d1ee2 \
    --hash=sha256:5c22873ad1f0532ba40fa1727f3c0fc1bbbaab6d373d4cbe3f0dc74b2e2521c7 \
    --hash=sha256:5e8b3d0b18fd623afa12ecb2ce8d8becef69f9b5440c6330c7972200e0bb84b0 \
    --hash=sha256:61631e08084be9e21a8967ec3139c7616ed7c5e9368e05c86d1b39562c8a57b6 \
    --hash=sha256:64511c54db4e4987aef4c41923235927428729e8174c5dba488429be70a998ed \
    --hash=sha256:6669c1bf34080161ce49c589cc512ef24d4c704ac9d2b2d3667f519c60418378 \
    --hash=sha256:672d207103e6b16ca098611b0f9efad6bc00afd47c03d6ef62186495ca677dc0 \
    --hash=sha256:6768d67d1bce64270e0fdc2e69309d68b9b18ae56ddf6c711d168e9d051c2cac \
    --hash=sha256:6a45c3d514f2436064db00d7fc8778d888f0236ebfed649b53d13a59e69ad51b \
    --hash=sha256:6bd9e1788e15bfcf6a9082de42e30387e7b85d211ab21e57a939bb8cfaaf8d96 \
    --hash=sha256:6d2a9efe686f9de00d0d1ea32a4a5a86d558a2277501bd78d964214eab625e59 \
    --hash=sha256:6da83a088f8ef93b2d483a8232a4dbf4d69d3d8496b568a03c56becac43e1808 \
    --hash=sha256:7018d4af1cd272e847aa5917983ab5e83e4f6579f9dbfecd4a79c0ca80b144c2 \
    --hash=sha256:71f88e749ea29f67f21f3b36433c1dc54c7729ed2a6d9e2da2e0d9e0d7b224eb \
    --hash=sha256:737c9c3981998eba27f11786f84fddcbabc74068b72a4a1f454ea02094b57b65 \
    --hash=sha256:73e77980c7207854f00fc4e71fb1626868d5740ab4012623d55c7a99ad122a72 \
    --hash=sha256:799c39bdf5e2f1292fedd3009f7b3c9e760f10b2420cb9638d56920840ff6db8 \
    --hash=sha256:7a83aa6e4805df46fed18e989d3d16f86ef60cb50bbc8d9ce3a6be89165fbf6e \
    --hash=sha256:7d3391b2188d18737cb2fa147028b1096236eaa7e156446c650a489fa2cadc91 \
    --hash=sha256:7e1636da3d8dfc220b6dd10264db5f2b165e4888c4518594898fbe381049af8a \
    --hash=sha256:805c8b84534fa10891890f0e4be39f3a99e94615d93e8836bf9fa1fdca2feeb2 \
    --hash=sha256:811d02d5122171c1941357efd8f9bf4ffe907b7f0a1a4e729a880e4be3f46e3e \
    --hash=sha256:8138eb83940ec7299024d92d4dee45f601b9e6c5ffde9d25f4e35e326203c707 \
    --hash=sha256:83b3944fea42a8400edf92fd1770fb8d0d4f7de651353bd2d8525a92dba69a21 \
    --hash=sha256:849dd2bb0e5e4ab2b71c7191726a4a8d5aa8a610daa584728cbee0b710ddc4ef \
    --hash=sha256:8698d70a8081ee8c090dbb394768b5789a1da8b131b5499f89d071dd3cfaf6be \
    --hash=sha256:8781a792a070cf2bd1b86d3aa943894115faaba6e88122a7bf32d62072742453 \
    --hash=sha256:88d59b473bfb03259722600839af9bbd7fa13a2eb514beefeedb95997882f69a \
    --hash=sha256:8909c2f1c6dd65e054ac4b573a91c8384d1492281e55d82d159d653f7a13adf6 \
    --hash=sha256:8965520ac587c94a4ac48b729be3d8b8de00af39699b17585dfb599babe77977 \
    --hash=sha256:8b5d563170ff8ba3181caa967c99a3c804d1dedb702c7cb93a6a7c32247da978 \
    --hash=sha256:8e124f974786f831d6043728e38296969d3579db8896fe004682f5758e613581 \
    --hash=sha256:8f0fac8b13d14bb06c68195f849371924ae53dd7b1c00fed24650f704383b692 \
    --hash=sha256:9240187afb63d2f9ddc3e032c670356fe941f6e20662ea168a5dc3f1f317e1b3 \
    --hash=sha256:925f929d6b59a8b3f8b8c6ac363cd0af7eecc81efb3071770b3c6717c450a369 \
    --hash=sha256:9348cbb300d224fe3b89793262cb093504d4ae927004468463f745188a193e4a \
    --hash=sha256:9388003072b95f2f1e3fd908604194d653ba21330d811961a78b7da1a77e9e36 \
    --hash=sha256:9438a2648b2195980cb2dd8e53ed7b8df91319e2d0b70ae61a9e1d1bc8d3bec9 \
    --hash=sha256:94e4c421742086aeee4c32a506eec8859d7634aad943f7e6aacf70f813478768 \
    --hash=sha256:94f5407f7bc64fa6463906b896f9904beeeb7dd8dc116ee8e9056c8714ff9916 \
    --hash=sha256:971a3bbb75d97ae4e2e8f7d4834236f86f85f0c85e04ab2e191db1123b04f80b \
    --hash=sha256:9e227f3dbe6bde7491cf0a9965d00b88c6b1a4a95d11480ddf88bb96d397c19f \
    --hash=sha256:9e25feb9e330b63edb0278a0acdf85e50d0cb0fbf49c3084abbe4e24ae195346 \
    --hash=sha256:9f098115c247e11d138ab83a28fa0323c77015007ea2df73ba5fd714dfefd67c \
    --hash=sha256:a18f38cafc329bac5e3c2b96c765b4c96d3d103421ed22ab7988c1e3fce27464 \
    --hash=sha256:a4bbd2d87dd233b9fc5812160c3d0ffbe42edc22a26ce0469f58479ede633fe9 \
    --hash=sha256:a5fcffb37e602b0b3c1638a97746b9b96125caa9bcf6fa41d337a9261de231ee \
    --hash=sha256:a8e9f292fcda89b324f2f5c91d13f1424a153e40fc2756f38ee23b15835ff300 \
    --hash=sha256:a9f54054101545a9a9cccefddf54316aa6e4491611fcbef9e91b3b6bebec04f6 \
    --hash=sha256:aa2c838cc024642cc04c6854232f32b43e5e22833dd11119c1766c7873b8370d \
    --hash=sha256:ac0c7c9f1609b0c4c114feb1d7a3409564c7fb77e360bed9e97e5d25dfeaf868 \
    --hash=sha256:add96447a86d205ab616665d53b2950ee81083757f56e6ea833c8b2917646b46 \
    --hash=sha256:ae9dcb8fbe244cb82f8a6458b455b927a03685e383d9bacf1ea5ce180b96dc97 \
    --hash=sha256:b4a635a0487774f841cb1fb62e907e7195cc95bc761e053184b8acc3ceb20733 \
    --hash=sha256:b4d12837e0203bbace818ff4a7461afdcd78bcd782351cea148139180d7bcffe \
    --hash=sha256:b61687d0828e72bf5cda24a2690188f37170bd31c9359ac97e4e66569f120a16 \
    --hash=sha256:b807e598953730f82e4eae3bd30f6a122cf6b31c398c6b504c0e04c13c170429 \
    --hash=sha256:b8cd1f918b26fd7b1832ece557cc18f2d8747309ff8b3f0ef9d4250c5ad67a39 \
    --hash=sha256:b91cc9d336957239ff200f30097e6fea2dc6d6fb3c81e853eaa09eac904fd894 \
    --hash=sha256:bd3ce56ae2cbae3ba82b683bc425cd7e48d2ed8b10f3e818186b6f5646d9271c \
    --hash=sha256:be6cb0c799abb0e2ba3e618e6d28ddddf7e485f6c2ce938dfa237daf3905072c \
    --hash=sha256:befb4158af32106b9a93db8d6d1d1cbbd418c0d5aca0cabb7b1780abf0c89169 \
    --hash=sha256:bf053da3c97a4bc5ecfbb218cdd2983febd91c617be8367d139882aa11e490aa \
    --hash=sha256:c02e8f18bdedba082cef725942ac823b9b60656db07f7e265cb31618dfd00d77 \
    --hash=sha256:c1bc67752d5f21013cfe430df4062441714eab79f65a6a05e01505957e9c35fe \
    --hash=sha256:c61750fadcd119d0825bcb7d7d675dd264dcc89cc05292aab5be68ebdbb374ad \
    --hash=sha256:c90d5b3d4e944e065a301d741b3c1d784f6bd1f503aa68b4967e32b2ba313d85 \
    --hash=sha256:c9a7f43c0b202b334cc9184af09bb8f21d3a209e038efaf106936fb69e6b026e \
    --hash=sha256:cb96e6e088d6cf71c1ea977510948320234824cf226e32f6f6e044f7a9c82b34 \
    --hash=sha256:cf63c214fe879a65e69a386f915e36104fc84254ab141240f8854602d8e0be2a \
    --hash=sha256:d1aca03ede943eb80ab3d63bb082c84b7aab85ea83bd0fd0c200260945fb49d9 \
    --hash=sha256:d2e56fd3b00222722abfb3f5f0759ddbae4b90811b5ad4343c64030ad1bde70c \
    --hash=sha256:d5f93ebbeb8032d47e349328ec8662d973d9b05a70b3c35df1f91fe419b84749 \
    --hash=sha256:d882a373d8093c2941e01291b7ced96e9cbe4781da9a7751ca7e6c70385e5214 \
    --hash=sha256:d920abdfa61279ba1a2ef9484aab07bf03331f8c08a10120fa332353d06e6932 \
    --hash=sha256:da2af0d7aebfc2074080d72efa6ab8317c62481ef1f896f65d9999c1c01f4494 \
    --hash=sha256:dd8ea6ebee7aedbf7c749fa80521d9ccf1ba473e0d1e14805caafbaad281c889 \
    --hash=sha256:de8b364c423ef0a4bad9069657d617f9a5d2b2062457a89b1fa16ee199c399c1 \
    --hash=sha256:df1ae86ff54725a01fa1a0510b914ca53a161b7050be74f6204e24aded5971d0 \
    --hash=sha256:dff05cb7016dff1e9fd68f4122c127b65dfc59de5306cfb7ad92f956f230bee2 \
    --hash=sha256:e1a622f13970d81f95d0c72f9dc090dce9085fccfa4c9f2174377ee32bd15786 \
    --hash=sha256:e49fb0d1ce92cfa0cb198cc5b1b11cdf9d0638658e2a2db2687e39db7c87fc78 \
    --hash=sha256:e5c802729725bd07e2bc3ab7b76dc7e0bbfc53129d8f1eb1c002c24cf774717e \
    --hash=sha256:e841068dc0be4cb6dfb5c890eb88cbdcff2f4a332393c7ec94e8e618bd32c1a8 \
    --hash=sha256:e916035e3e9930cbdfdd10abf48861340221857f45509565898e012263f7b289 \
    --hash=sha256:eba154571c16e032112afac0dc2dfe9e63c2ceb7aedd07bb7eecf2ce26d4dd4c \
    --hash=sha256:f03460ff076f70ab595bb45a0205ccea1971443575b6920c52e755dec2b3fbfe \
    --hash=sha256:f0ec3b750b59375eab5b0fb2b9254810c00a3375be6d789899f1055a1d556237 \
    --hash=sha256:f291bcf42ae98eb5107edb162c3c998b4a89648fd8e99ed4cbd12705292788cd \
    --hash=sha256:f61efe1d2fe0de16158a5fe1d1cf3c14bdb6aecd54d8938fd26512c525c1f624 \
    --hash=sha256:f68edfc67aabac33708941f26f22a7b8e9f81429bc0cf249fcf7d66b23af8d19 \
    --hash=sha256:fa95848c929b6a75f6848d3c9793e59db365ee436776e57db835cdbfa79ba977 \
    --hash=sha256:fd9f8797427910198f95bced71ddfed61130d7e349213bfb8466c9c99e2c46a8 \
    --hash=sha256:fdb4ca07ab75ffadab4a8b135ad59cdbb3156b99310f3d565370da74a15d6bd3
    # via jinja2
mpmath==1.3.0 \
    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \
    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c
    # via sympy
networkx==3.7 \
    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \
    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a
    # via torch
numpy==2.5.3 \
    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \
    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \
    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \
    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \
    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \
    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \
    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \
    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \
    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \
    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \
    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \
    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \
    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \
    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \
    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \
    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \
    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \
    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \
    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \
    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \
    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \
    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \
    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \
    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \
    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \
    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \
    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \
    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \
    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \
    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \
    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \
    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \
    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \
    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \
    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \
    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \
    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \
    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \
    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \
    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \
    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \
    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \
    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \
    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \
    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \
    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \
    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \
    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \
    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \
    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \
    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \
    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \
    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \
    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \
    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \
    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \
    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \
    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \
    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \
    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \
    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \
    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \
    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \
    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \
    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \
    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab
    # via
    #   vitpose-keypoint-pipeline (pyproject.toml)
    #   scipy
    #   torchvision
    #   transformers
nvidia-cublas==13.1.1.3 \
    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \
    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \
    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5
    # via
    #   cuda-toolkit
    #   nvidia-cudnn-cu13
    #   nvidia-cusolver
nvidia-cuda-cupti==13.0.85 \
    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \
    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \
    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151
    # via cuda-toolkit
nvidia-cuda-nvrtc==13.0.88 \
    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \
    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \
    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b
    # via
    #   cuda-toolkit
    #   nvidia-cublas
nvidia-cuda-runtime==13.0.96 \
    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \
    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \
    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492
    # via cuda-toolkit
nvidia-cudnn-cu13==9.24.0.43 \
    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \
    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \
    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f
    # via torch
nvidia-cufft==12.0.0.61 \
    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \
    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \
    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3
    # via cuda-toolkit
nvidia-cufile==1.15.1.6 \
    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \
    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1
    # via cuda-toolkit
nvidia-curand==10.4.0.35 \
    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \
    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \
    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f
    # via cuda-toolkit
nvidia-cusolver==12.0.4.66 \
    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \
    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \
    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65
    # via cuda-toolkit
nvidia-cusparse==12.6.3.3 \
    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \
    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \
    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79
    # via
    #   cuda-toolkit
    #   nvidia-cusolver
nvidia-cusparselt-cu13==0.8.1 \
    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \
    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \
    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215
    # via torch
nvidia-nccl-cu13==2.30.7 \
    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \
    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50
    # via torch
nvidia-nvjitlink==13.4.92 \
    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \
    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \
    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \
    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323
    # via
    #   cuda-toolkit
    #   nvidia-cufft
    #   nvidia-cusolver
    #   nvidia-cusparse
nvidia-nvshmem-cu13==3.4.5 \
    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \
    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9
    # via torch
nvidia-nvtx==13.0.85 \
    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \
    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \
    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519
    # via cuda-toolkit
packaging==26.3 \
    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \
    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c
    # via
    #   huggingface-hub
    #   transformers
pillow==11.3.0 \
    --hash=sha256:023f6d2d11784a465f09fd09a34b150ea4672e85fb3d05931d89f373ab14abb2 \
    --hash=sha256:02a723e6bf909e7cea0dac1b0e0310be9d7650cd66222a5f1c571455c0a45214 \
    --hash=sha256:040a5b691b0713e1f6cbe222e0f4f74cd233421e105850ae3b3c0ceda520f42e \
    --hash=sha256:05f6ecbeff5005399bb48d198f098a9b4b6bdf27b8487c7f38ca16eeb070cd59 \
    --hash=sha256:068d9c39a2d1b358eb9f245ce7ab1b5c3246c7c8c7d9ba58cfa5b43146c06e50 \
    --hash=sha256:0743841cabd3dba6a83f38a92672cccbd69af56e3e91777b0ee7f4dba4385632 \
    --hash=sha256:092c80c76635f5ecb10f3f83d76716165c96f5229addbd1ec2bdbbda7d496e06 \
    --hash=sha256:0b275ff9b04df7b640c59ec5a3cb113eefd3795a8df80bac69646ef699c6981a \
    --hash=sha256:0bce5c4fd0921f99d2e858dc4d4d64193407e1b99478bc5cacecba2311abde51 \
    --hash=sha256:1019b04af07fc0163e2810167918cb5add8d74674b6267616021ab558dc98ced \
    --hash=sha256:106064daa23a745510dabce1d84f29137a37224831d88eb4ce94bb187b1d7e5f \
    --hash=sha256:118ca10c0d60b06d006be10a501fd6bbdfef559251ed31b794668ed569c87e12 \
    --hash=sha256:13f87d581e71d9189ab21fe0efb5a23e9f28552d5be6979e84001d3b8505abe8 \
    --hash=sha256:155658efb5e044669c08896c0c44231c5e9abcaadbc5cd3648df2f7c0b96b9a6 \
    --hash=sha256:1904e1264881f682f02b7f8167935cce37bc97db457f8e7849dc3a6a52b99580 \
    --hash=sha256:19d2ff547c75b8e3ff46f4d9ef969a06c30ab2d4263a9e287733aa8b2429ce8f \
    --hash=sha256:1a992e86b0dd7aeb1f053cd506508c0999d710a8f07b4c791c63843fc6a807ac \
    --hash=sha256:1b9c17fd4ace828b3003dfd1e30bff24863e0eb59b535e8f80194d9cc7ecf860 \
    --hash=sha256:1c627742b539bba4309df89171356fcb3cc5a9178355b2727d1b74a6cf155fbd \
    --hash=sha256:1cd110edf822773368b396281a2293aeb91c90a2db00d78ea43e7e861631b722 \
    --hash=sha256:1f85acb69adf2aaee8b7da124efebbdb959a104db34d3a2cb0f3793dbae422a8 \
    --hash=sha256:23cff760a9049c502721bdb743a7cb3e03365fafcdfc2ef9784610714166e5a4 \
    --hash=sha256:2465a69cf967b8b49ee1b96d76718cd98c4e925414ead59fdf75cf0fd07df673 \
    --hash=sha256:2a3117c06b8fb646639dce83694f2f9eac405472713fcb1ae887469c0d4f6788 \
    --hash=sha256:2aceea54f957dd4448264f9bf40875da0415c83eb85f55069d89c0ed436e3542 \
    --hash=sha256:2d6fcc902a24ac74495df63faad1884282239265c6839a0a6416d33faedfae7e \
    --hash=sha256:30807c931ff7c095620fe04448e2c2fc673fcbb1ffe2a7da3fb39613489b1ddd \
    --hash=sha256:30b7c02f3899d10f13d7a48163c8969e4e653f8b43416d23d13d1bbfdc93b9f8 \
    --hash=sha256:3828ee7586cd0b2091b6209e5ad53e20d0649bbe87164a459d0676e035e8f523 \
    --hash=sha256:3cee80663f29e3843b68199b9d6f4f54bd1d4a6b59bdd91bceefc51238bcb967 \
    --hash=sha256:3e184b2f26ff146363dd07bde8b711833d7b0202e27d13540bfe2e35a323a809 \
    --hash=sha256:41342b64afeba938edb034d122b2dda5db2139b9a4af999729ba8818e0056477 \
    --hash=sha256:41742638139424703b4d01665b807c6468e23e699e8e90cffefe291c5832b027 \
    --hash=sha256:4445fa62e15936a028672fd48c4c11a66d641d2c05726c7ec1f8ba6a572036ae \
    --hash=sha256:45dfc51ac5975b938e9809451c51734124e73b04d0f0ac621649821a63852e7b \
    --hash=sha256:465b9e8844e3c3519a983d58b80be3f668e2a7a5db97f2784e7079fbc9f9822c \
    --hash=sha256:48d254f8a4c776de343051023eb61ffe818299eeac478da55227d96e241de53f \
    --hash=sha256:4c834a3921375c48ee6b9624061076bc0a32a60b5532b322cc0ea64e639dd50e \
    --hash=sha256:4c96f993ab8c98460cd0c001447bff6194403e8b1d7e149ade5f00594918128b \
    --hash=sha256:504b6f59505f08ae014f724b6207ff6222662aab5cc9542577fb084ed0676ac7 \
    --hash=sha256:527b37216b6ac3a12d7838dc3bd75208ec57c1c6d11ef01902266a5a0c14fc27 \
    --hash=sha256:5418b53c0d59b3824d05e029669efa023bbef0f3e92e75ec8428f3799487f361 \
    --hash=sha256:59a03cdf019efbfeeed910bf79c7c93255c3d54bc45898ac2a4140071b02b4ae \
    --hash=sha256:5e05688ccef30ea69b9317a9ead994b93975104a677a36a8ed8106be9260aa6d \
    --hash=sha256:6359a3bc43f57d5b375d1ad54a0074318a0844d11b76abccf478c37c986d3cfc \
    --hash=sha256:643f189248837533073c405ec2f0bb250ba54598cf80e8c1e043381a60632f58 \
    --hash=sha256:65dc69160114cdd0ca0f35cb434633c75e8e7fad4cf855177a05bf38678f73ad \
    --hash=sha256:67172f2944ebba3d4a7b54f2e95c786a3a50c21b88456329314caaa28cda70f6 \
    --hash=sha256:676b2815362456b5b3216b4fd5bd89d362100dc6f4945154ff172e206a22c024 \
    --hash=sha256:6a418691000f2a418c9135a7cf0d797c1bb7d9a485e61fe8e7722845b95ef978 \
    --hash=sha256:6abdbfd3aea42be05702a8dd98832329c167ee84400a1d1f61ab11437f1717eb \
    --hash=sha256:6be31e3fc9a621e071bc17bb7de63b85cbe0bfae91bb0363c893cbe67247780d \
    --hash=sha256:7107195ddc914f656c7fc8e4a5e1c25f32e9236ea3ea860f257b0436011fddd0 \
    --hash=sha256:71f511f6b3b91dd543282477be45a033e4845a40278fa8dcdbfdb07109bf18f9 \
    --hash=sha256:7859a4cc7c9295f5838015d8cc0a9c215b77e43d07a25e460f35cf516df8626f \
    --hash=sha256:7966e38dcd0fa11ca390aed7c6f20454443581d758242023cf36fcb319b1a874 \
    --hash=sha256:79ea0d14d3ebad43ec77ad5272e6ff9bba5b679ef73375ea760261207fa8e0aa \
    --hash=sha256:7aee118e30a4cf54fdd873bd3a29de51e29105ab11f9aad8c32123f58c8f8081 \
    --hash=sha256:7b161756381f0918e05e7cb8a371fff367e807770f8fe92ecb20d905d0e1c149 \
    --hash=sha256:7c8ec7a017ad1bd562f93dbd8505763e688d388cde6e4a010ae1486916e713e6 \
    --hash=sha256:7d1aa4de119a0ecac0a34a9c8bde33f34022e2e8f99104e47a3ca392fd60e37d \
    --hash=sha256:7db51d222548ccfd274e4572fdbf3e810a5e66b00608862f947b163e613b67dd \
    --hash=sha256:819931d25e57b513242859ce1876c58c59dc31587847bf74cfe06b2e0cb22d2f \
    --hash=sha256:83e1b0161c9d148125083a35c1c5a89db5b7054834fd4387499e06552035236c \
    --hash=sha256:857844335c95bea93fb39e0fa2726b4d9d758850b34075a7e3ff4f4fa3aa3b31 \
    --hash=sha256:8797edc41f3e8536ae4b10897ee2f637235c94f27404cac7297f7b607dd0716e \
    --hash=sha256:8924748b688aa210d79883357d102cd64690e56b923a186f35a82cbc10f997db \
    --hash=sha256:89bd777bc6624fe4115e9fac3352c79ed60f3bb18651420635f26e643e3dd1f6 \
    --hash=sha256:8dc70ca24c110503e16918a658b869019126ecfe03109b754c402daff12b3d9f \
    --hash=sha256:91da1d88226663594e3f6b4b8c3c8d85bd504117d043740a8e0ec449087cc494 \
    --hash=sha256:921bd305b10e82b4d1f5e802b6850677f965d8394203d182f078873851dada69 \
    --hash=sha256:932c754c2d51ad2b2271fd01c3d121daaa35e27efae2a616f77bf164bc0b3e94 \
    --hash=sha256:93efb0b4de7e340d99057415c749175e24c8864302369e05914682ba642e5d77 \
    --hash=sha256:97afb3a00b65cc0804d1c7abddbf090a81eaac02768af58cbdcaaa0a931e0b6d \
    --hash=sha256:97f07ed9f56a3b9b5f49d3661dc9607484e85c67e27f3e8be2c7d28ca032fec7 \
    --hash=sha256:98a9afa7b9007c67ed84c57c9e0ad86a6000da96eaa638e4f8abe5b65ff83f0a \
    --hash=sha256:9ab6ae226de48019caa8074894544af5b53a117ccb9d3b3dcb2871464c829438 \
    --hash=sha256:9c412fddd1b77a75aa904615ebaa6001f169b26fd467b4be93aded278266b288 \
    --hash=sha256:a1bc6ba083b145187f648b667e05a2534ecc4b9f2784c2cbe3089e44868f2b9b \
    --hash=sha256:a418486160228f64dd9e9efcd132679b7a02a5f22c982c78b6fc7dab3fefb635 \
    --hash=sha256:a4d336baed65d50d37b88ca5b60c0fa9d81e3a87d4a7930d3880d1624d5b31f3 \
    --hash=sha256:a6444696fce635783440b7f7a9fc24b3ad10a9ea3f0ab66c5905be1c19ccf17d \
    --hash=sha256:a7bc6e6fd0395bc052f16b1a8670859964dbd7003bd0af2ff08342eb6e442cfe \
    --hash=sha256:b4b8f3efc8d530a1544e5962bd6b403d5f7fe8b9e08227c6b255f98ad82b4ba0 \
    --hash=sha256:b5f56c3f344f2ccaf0dd875d3e180f631dc60a51b314295a3e681fe8cf851fbe \
    --hash=sha256:be5463ac478b623b9dd3937afd7fb7ab3d79dd290a28e2b6df292dc75063eb8a \
    --hash=sha256:c37d8ba9411d6003bba9e518db0db0c58a680ab9fe5179f040b0463644bc9805 \
    --hash=sha256:c84d689db21a1c397d001aa08241044aa2069e7587b398c8cc63020390b1c1b8 \
    --hash=sha256:c96d333dcf42d01f47b37e0979b6bd73ec91eae18614864622d9b87bbd5bbf36 \
    --hash=sha256:cadc9e0ea0a2431124cde7e1697106471fc4c1da01530e679b2391c37d3fbb3a \
    --hash=sha256:cc3e831b563b3114baac7ec2ee86819eb03caa1a2cef0b481a5675b59c4fe23b \
    --hash=sha256:cd8ff254faf15591e724dc7c4ddb6bf4793efcbe13802a4ae3e863cd300b493e \
    --hash=sha256:d000f46e2917c705e9fb93a3606ee4a819d1e3aa7a9b442f6444f07e77cf5e25 \
    --hash=sha256:d9da3df5f9ea2a89b81bb6087177fb1f4d1c7146d583a3fe5c672c0d94e55e12 \
    --hash=sha256:e5c5858ad8ec655450a7c7df532e9842cf8df7cc349df7225c60d5d348c8aada \
    --hash=sha256:e67d793d180c9df62f1f40aee3accca4829d3794c95098887edc18af4b8b780c \
    --hash=sha256:ea944117a7974ae78059fcc1800e5d3295172bb97035c0c1d9345fca1419da71 \
    --hash=sha256:eb76541cba2f958032d79d143b98a3a6b3ea87f0959bbe256c0b5e416599fd5d \
    --hash=sha256:ec1ee50470b0d050984394423d96325b744d55c701a439d2bd66089bff963d3c \
    --hash=sha256:ee92f2fd10f4adc4b43d07ec5e779932b4eb3dbfbc34790ada5a6669bc095aa6 \
    --hash=sha256:f0f5d8f4a08090c6d6d578351a2b91acf519a54986c055af27e7a93feae6d3f1 \
    --hash=sha256:f1f182ebd2303acf8c380a54f615ec883322593320a9b00438eb842c1f37ae50 \
    --hash=sha256:f8a5827f84d973d8636e9dc5764af4f0cf2318d26744b3d902931701b0d46653 \
    --hash=sha256:f944255db153ebb2b19c51fe85dd99ef0ce494123f21b9db4877ffdfc5590c7c \
    --hash=sha256:fdae223722da47b024b867c1ea0be64e0df702c5e0a60e27daad39bf960dd1e4 \
    --hash=sha256:fe27fb049cdcca11f11a7bfda64043c37b30e6b91f10cb5bab275806c32f6ab3
    # via
    #   vitpose-keypoint-pipeline (pyproject.toml)
    #   torchvision
pyyaml==6.0.3 \
    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \
    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \
    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \
    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \
    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \
    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \
    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \
    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \
    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \
    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \
    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \
    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \
    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \
    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \
    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \
    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \
    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \
    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \
    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \
    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \
    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \
    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \
    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \
    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \
    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \
    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \
    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \
    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \
    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \
    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \
    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \
    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \
    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \
    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \
    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \
    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \
    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \
    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \
    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \
    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \
    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \
    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \
    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \
    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \
    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \
    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \
    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \
    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \
    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \
    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \
    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \
    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \
    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \
    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \
    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \
    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \
    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \
    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \
    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \
    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \
    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \
    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \
    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \
    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \
    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \
    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \
    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \
    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \
    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \
    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \
    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \
    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \
    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0
    # via
    #   huggingface-hub
    #   transformers
regex==2026.9.29 \
    --hash=sha256:01000ddf0e3ffef97f2413ceb514f6313040106b6d18a03ee00a4fe35c1eb1db \
    --hash=sha256:0166844493626c5015c6088ee15c9ca2fd060ca15b7641d1657da6a58432ae33 \
    --hash=sha256:044265d77d94f5e3cb2fd72c76723807c429cb8c533e9d4672d0334a6f14f588 \
    --hash=sha256:0476e5bcbe6e1ba3d1c4cc7bbb1c3ba78e3b979b5c8a88d0a6a8cdd4992b8c84 \
    --hash=sha256:066d0e3dbfdd739bce2bf8c2a41dd16f73e3d8adc2eb06dd803a36a307f56075 \
    --hash=sha256:0b65c72739f981377c9c22e0c5c3cd7f42da7bd8a3c9209330fac772c7d893ed \
    --hash=sha256:0c992c19cd45058a4b92f68f139c93db168b48fb1f322c9a7cd620806afb6b51 \
    --hash=sha256:0cc63b5e47c12a48d90c7e9d7de6a035dd14f62868aaedbb4e0ff8ba2b8bfe7b \
    --hash=sha256:0dd8af32e9f7b56b7f95cc1fd79b23054c3bdc172392ae560acc24d57b7ffe71 \
    --hash=sha256:0def9fb6abac55492d6d51cddb7225d07d6f279e774e0adc08569a54a5fc8d46 \
    --hash=sha256:0fd2c901cc307a745ad4bc87f20060d7a0825a3371d1e93488af22e7a387f78f \
    --hash=sha256:1043aedf5917caa861bcb25a9c11460049656bdf0017a90a309fa8f255467725 \
    --hash=sha256:121a76a0985db80ceae9e171c337f8c927868e37d01b54e3ce87bc87f9c6a208 \
    --hash=sha256:143533cc4b6fbc5b95aca0a5b8d541088d374831593def000ec89322c220221d \
    --hash=sha256:14e953ff3607c92d7675bf79c4d4509ef6782aa8c08509f179f9b3d6d0679e86 \
    --hash=sha256:18ae8eed4526e35bdb754d61562b90bf5c00a67fdcf3cc1380dd59597486631b \
    --hash=sha256:19959129885356df0e97556856f77eb2888380dac18bed075a7c05c5128c618d \
    --hash=sha256:1ba8c6a416569ce0d37e83e28a254a61dc99a419084dfb6476cea02d997f74fa \
    --hash=sha256:1c2a0026062abcc321a53db4a185ceba0b59a66b5d37b0808917a88b55a5257f \
    --hash=sha256:1d9fe8091b2e89d470df68a9331111ed008ae8aae6bf1e8e1fba4086a495c84e \
    --hash=sha256:2089fe39c406784d90101c726755ffa1497bb74638fd434300d2b88006186de8 \
    --hash=sha256:23ae6fdad9e63e54038f5ef78aba2933faca61e24d432786589e737bc5522ebb \
    --hash=sha256:26ec4ccce55aa533fbd603d08911b01101a8fcfec987845ac3ae2c7087b2bde3 \
    --hash=sha256:2f7f7aa47b229f2b39a2ae2596d2ad5625d77b5eb9856fac2dab3eb506cdd0a0 \
    --hash=sha256:31b003f9a070335e2a8233ee9b14a3ca8e6d792012ae011f741bf0aaf11744c5 \
    --hash=sha256:32ab11df9677ca80bcbb5fe4eb1da9109a5019239a054836efc6fa1c64e683cf \
    --hash=sha256:33026515aebc0e70d1c89978e53e8d695d35d9e472f8d5b34465ba3c74028650 \
    --hash=sha256:34b6925af9853bf461950e6508910f179fd6e9b1a7ec8548e069606b7e51a26b \
    --hash=sha256:352cf115a810b357caa35193ab656ecf5ef41056855e82f292c99e8514f8d954 \
    --hash=sha256:39ab5894d971f9ac68baa6eca5c50387db579cfcacf36ae8df3feceb1815e6d0 \
    --hash=sha256:3a21a9509d0ee88e7a70e1ad228cd2f0e0fd1e187458db132e8a8d18c97daf9d \
    --hash=sha256:3c5c2ef13797466aa64170cbb66ad98a32351dd4127694cea7199f80f213750d \
    --hash=sha256:3e778bfccd63075167709136afbc251c1f683758d5bf49c803c60ac3f894ce6b \
    --hash=sha256:3f1e6cb402a89457582cd696f982559217d13484a193202c394015297968c86d \
    --hash=sha256:42e82e578c904445d4c8a35b8f28052cf567593215fa5db06266fbc6f77aaa2e \
    --hash=sha256:4408b2b27a95ca8cc48b7411945753773353b5c93b307754781086c99d3a576f \
    --hash=sha256:446654b29bfaa30500d80947eda42cef1449dc8a87f4e3cf061cc8485d3a1f0b \
    --hash=sha256:45010bcfe66df41522d56c9b6114e87ecc597a08970ff6a2ced24415c141ae5f \
    --hash=sha256:49ee178ca31c94621294bf9b8b676a92a2e6bba8af0529591753719e57edb621 \
    --hash=sha256:4d7d93613b01b0199961330e49cfc52d479b3d5776c56c691db31130c0a07d91 \
    --hash=sha256:4fb41211d2333eb930a51e0546a65999761cf1f572a4da56ef9b8a62966c06f2 \
    --hash=sha256:4fe97894d1b306c919b4e50def1e6f6c522f4d03a7283811f4d108f1ce5d3ac2 \
    --hash=sha256:554bffadcbcb6d5f4e5fb10a61cc52084b9a63d1dab5f10bcd2c4343972e8e2c \
    --hash=sha256:59b49507f47479e299a9e1bc41b5cb83a7afda0540625f1dbae886615978acbf \
    --hash=sha256:5eeb8edc6110d9194a4d0d54610f64c37a31c605b5dbb7e407fc6ec7fa34a4a1 \
    --hash=sha256:612b709381c0355b70d89cdb51b7f670591ed5cbbc0e3b5337488019dc667b65 \
    --hash=sha256:61956f074ecd123f55adca68ee3eab46e6a07ad3f8e64e6db95dfacb444f55c4 \
    --hash=sha256:6398d5145689503412cc1748895242598d8846b8967b851133b20dc2ed1e21e8 \
    --hash=sha256:65b408d8fcb273e3499e7ef2ce796810da1becd208c7fb4373692a242d79d461 \
    --hash=sha256:686ac5350fceae63830bb98805fcb8039325bf4c06d9f6f048ff65229d5bffa5 \
    --hash=sha256:6a1a824fbed817e0a891103886b68f063b1e83cc51bc97192a90a60195a9291f \
    --hash=sha256:6abb75ab16bc3281714a5b99548a2225db70dba1f995f6d7f7419b76eb5a8fbe \
    --hash=sha256:6f7121a8914ed13fcfe2099f895341bfb789f004d4c5a0bdece8fa667da10849 \
    --hash=sha256:7020ed44df30b3aa492c00ee3b52d0548c1f30c2c6c5bb13ae897680900d3413 \
    --hash=sha256:720537c7ea6f80dc61913184edb0ce2497a306b39ef19f28505b322553d52bdb \
    --hash=sha256:724184b4aafed865e4f13ca313fdcb43024300c028ec67319cfa16847d84685e \
    --hash=sha256:7c03031610e3e6ed1768a2b7a8fc84637c1257b50c5eacaf094c6e17a84fc563 \
    --hash=sha256:80a5ea3b4fd9d6a5b9a44f7976a9acaaab35aa3c1f6b29e5bd857dfabaded223 \
    --hash=sha256:80c7cadd3fd2bfde5df8aa0787e315812cad0c313a753095d02f4c2b6c01677b \
    --hash=sha256:80ea96f5c1a30bf09007d48466521d9c294bebe197c708c3359096e3e3691632 \
    --hash=sha256:864e9b87ac33c3fb9fb4ad48166d4fdb579c351d5c77deb0d34bccb36a775cd9 \
    --hash=sha256:87fb80cbe3557e27e7b28b995c2b2eedf689b8886f941ab93e0e288f0976518a \
    --hash=sha256:8873c4a11c50b9989168881aeb3f08859f469d809941866aa1feefd8be5431f6 \
    --hash=sha256:888d60953908dcf761aa320c3e390ab8556efbdb551ace63921de90f6ae0848d \
    --hash=sha256:8b5fcc4771732191b2b7d1dd68d8f0353f47f8d90b6150f6dce58bf1112442cb \
    --hash=sha256:8f39588af4731c8923c26810eb3b33f76f17633985e40f59c3cd45a33805a895 \
    --hash=sha256:9173db3be74a35cb6731701094b98120f7ee4876a287882a59cdea1fa7da342f \
    --hash=sha256:92f05c9c42bde5785dc48770bc2194d9f7442544156f951e19cd31b096cec562 \
    --hash=sha256:951733b1bbdb71e377cec567b409f1a7881b47cfcad84121aa74cb575fa425ea \
    --hash=sha256:957bb708e8057ab1649ba566456429d691ec9b90d1c9ad1af1ba7ffbbeaf05f2 \
    --hash=sha256:9916fda742cd4eede63b286f58c06718324265d727ce0856eb1aac86d0d150d6 \
    --hash=sha256:9e1d3a4cb7993b708f0ada8d0c84590efd853f169e7147d2202c9da503180242 \
    --hash=sha256:9e4482589065c8ecd761cff522dcd85f2d39e62f551e37e025d1c7d54772def3 \
    --hash=sha256:a5300757f8a68f5b6cc33f57338d72a0e3589c5cc9ad5f8504ea06f028be582a \
    --hash=sha256:a540abfab208e1b7ef2df231c40ef3b6cbb30a0aad6204e9b6a81c10a6794628 \
    --hash=sha256:a5758353650079898dc1b2b0e95aa51fa23a30d020e06f62c430dd08ee56cdd8 \
    --hash=sha256:a64b85a4760337cfefdb27d42da6ed8b58e8cde3f2d57b6ef43e76ef6ea9ef47 \
    --hash=sha256:a655d34b2a6943af32401f3d94f72e9d731f6ad16285815550bf2b4ee69d420a \
    --hash=sha256:a714befaacbd10092ffe4cea0d3c5f008fb9efe9bc322c715bcdfdee414b9a3d \
    --hash=sha256:a760da040b47767b4b873adfb7c3b691e9ba2fc60f113f9d0b88f1a62f323e85 \
    --hash=sha256:addd736a0547d553283adaf4e05d7104e7f2c7b0b092e9b4d28756825f14531f \
    --hash=sha256:ae4613d7d9dda60fcba95f846cc6f808017f1843f392cf9daad14a6534493d71 \
    --hash=sha256:b11b589e00095ec69cf79841a76360f9b079e95b0368a25b5ebb951ab0c157ff \
    --hash=sha256:b3e445b66c80b4eb4234e855ce94d9adc183eedbd632816228d89930b91b2c5b \
    --hash=sha256:b7b893976e7fe42053da64f2aa27239c24252fd2ec6df471e1be197c0addc3b1 \
    --hash=sha256:b84f186a7f0536fe4ff9a9fa12d06d007b9b71d4b5352ddcc41f59ad6522a312 \
    --hash=sha256:b89efc38431793d28b7cd91227e2f952ad7c48df19132b17f43a5fec3c14143b \
    --hash=sha256:b97a38fb4c732b6832db6bf108963adbcd82ef1268ba2025dce390f45af75efa \
    --hash=sha256:b9d74e4eee9ddb64c2e92d5d61472c59c21684c059eb7b68767be9628e977859 \
    --hash=sha256:bb90e7177944b6684738c1fc36aabd2dd00d1de3be7dbe09f91e196f1bc0dc81 \
    --hash=sha256:bec37990e3d6121f29ecfb594bd8f1bf009e9f7926daba2e50e3b27d3892a783 \
    --hash=sha256:bf3c49863c23a1ad6da9c30351aed6cff8d5ddbeb63c5c8420ae54e98c7d0138 \
    --hash=sha256:bf48516e35cf848390ea68850aba53e7c333720d2945b4d2c25b69fc5171723f \
    --hash=sha256:bfc71e6d970419c1309b3640305298643e2a734cad3f7cfb6d2ddee4175ab53d \
    --hash=sha256:c0094897d7d01f184b2d7fe8c56c66d64efe01b31f4b7d34205b391387df1111 \
    --hash=sha256:c03c6eb6ece86dfdcbb34799efaa339b093132e1aceed491ba5e08fe06cdf699 \
    --hash=sha256:c1a9a6651197fbed6f0212591418b9def774fc3f8324f78d1bf0e6a63e5f8aa1 \
    --hash=sha256:c3589f40749acce747510bf5d589d54e376cb0930ea58b35effac97e5312b0c1 \
    --hash=sha256:c4e38dd8f39c43a91d2410ad2b85610701b0979342c3df1d69eaf8e838c757d8 \
    --hash=sha256:c6c8fabf1dafc1f1ddcbb67896d3f93efb092e8c4b6322d7389b944e76a484e5 \
    --hash=sha256:c90fcf7804ea0a54b896ce0f2b9565350220b8d4890fd0db461a476a4c687963 \
    --hash=sha256:c9b602fae1e00b7c035d661ce85575365719192a7b46784bd71cf64c68053aa0 \
    --hash=sha256:ccb64d887a9db1cd76dbc0f92051a1a478a2a67e7f56c62d915cb881d7734704 \
    --hash=sha256:d06fcdecc10fc7954d7c8f27a03c96055fe525274dc84a7b0dbdc3d6b9e03dab \
    --hash=sha256:d0c3082bf79bcd6a614d55916590ad4b8f93200e10b97f463ea5d9d07c9b5f23 \
    --hash=sha256:d49c18f1ea294cf4adde2e5ac256e98c82ea9d708462ce4bf799dffa7cfe8a2c \
    --hash=sha256:d60030baaa7bfbb02d650c126cdcddcb6e33dbff14d819434c8fa2fdcaeeeba5 \
    --hash=sha256:d7cab119d0df0b9413f106b4d7fc34f2872d3574ed3806fb48959c830b1537da \
    --hash=sha256:d9b77b25b4f395f92de6099ab08e8ae2bc7e51dfe157f22900902243a5cc90c7 \
    --hash=sha256:dabee8f4935e731fb46b2a3091bdda0d3d94b3bbfb907d2b4f12eefce4009619 \
    --hash=sha256:db5e82ba15c142425b8406690032df89e39cca4a2e8afbbb9a3d84edc2373ac3 \
    --hash=sha256:dc79d36d0618752265f0d575915bdc5c5130ecb9c9f6b3bcefeae32e4bdfafcf \
    --hash=sha256:ddfa987262763c3c22a8367d2a49c244b018a74c3a8e3ab1a864119ad45c5633 \
    --hash=sha256:e1172147d28d8fbcf8cb8d26c41506169f5ad8fe9ec969cb116835a19d4d8eca \
    --hash=sha256:e11edba5bc344a32b029a7af9d4b3173982dd79eeafa0b9dbd787364414b0509 \
    --hash=sha256:e2c89e9b762c57f59d5e99ee8b20202adb892e35f8d3485741340999ca55058e \
    --hash=sha256:e31f72490b7c12f7790e1e25c3afffd20503ee1bfb43461d7838b871ff244b19 \
    --hash=sha256:e8c65ef3862a8ad6e86492b6ed9327805dd66904c012bd3649dc67d822ed6c34 \
    --hash=sha256:ebb8912f565b8cdbbf27debfe00df04202c20e2f651b9e32767930c5eace3621 \
    --hash=sha256:ed511a0708e2297e1d6431e7fb217e3402791e491e02da800658ace4973df1bb \
    --hash=sha256:edf06545875f3efa31560d94121e95c7fd70d98b1dfedc0157097d79b13b52ea \
    --hash=sha256:f0fe9834e5aeccaf19a0d8feb296d66a24be1a7c9922002f842a682cd5abb787 \
    --hash=sha256:f1a0d5117230dd46b399a30a38afa44f79c99f3168988fdc4f425c3f928b39df \
    --hash=sha256:f37964e4a5e993d2fd45147741e9dff7f34a2d8c00ab94c4ea0514a4677f959e \
    --hash=sha256:f57dc6b8fef170f105d2cf5cdce254f47b137d7755086cf7050f47e16582abba \
    --hash=sha256:f93bc1c3486ef3747e07c9d7c1d0a147b8fbaab975f80e348aed6f71309dfaca \
    --hash=sha256:fb00027a09a8f9f08028b40dce4c933cf73e4833240ed356583fdc9cfa721566 \
    --hash=sha256:fb99cc9d45f48895d9d67f6a0b8a57f08d39c174d9f25ad97a313e0470267b1c \
    --hash=sha256:fdd88ed5e20b1bcdd234421e454962c971aa44b653bdb7f1ea9ef683e90fb649 \
    --hash=sha256:fe3fa1dd453ed5c7f5ea23a26218329790ed7197a99b90e94330e313959a7f52
    # via transformers
requests==2.34.2 \
    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \
    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed
    # via
    #   huggingface-hub
    #   transformers
safetensors==0.8.0 \
    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \
    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \
    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \
    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \
    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \
    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \
    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \
    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \
    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \
    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \
    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \
    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \
    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \
    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \
    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \
    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \
    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774
    # via
    #   vitpose-keypoint-pipeline (pyproject.toml)
    #   transformers
scipy==1.18.1 \
    --hash=sha256:011413b7426b75012840e35649e00fe0a2c3bae89fed433876e3a99251572efc \
    --hash=sha256:0ac49ea97594532dd44b7136094d35f5440fa06e6d9c6384a74c01764df388c5 \
    --hash=sha256:0e82073ecc7acc6436fac4b31674109c7e1d3e596789767eda01258a8c9e8123 \
    --hash=sha256:0fcb3c93519f27bb4f0c4b0f7802cdcaca7fcf93267b75edda2e9f4e8a55cbd7 \
    --hash=sha256:10ac20c69d880f77f375db44c22e3e6a644f9fefa291d4cd2fb9790a89fc99fd \
    --hash=sha256:11c423f1049c5755ad4409af52a9ada1cff96fe9b50795d4af3619f292901239 \
    --hash=sha256:179ce34a8d0fe273d8883ba59e17e052247d08973dfcb743ca52bb1cce2d60b0 \
    --hash=sha256:1bca3b943fc2567ea49cd02c99abde49da4d5178ec46f624bd8255cda8755beb \
    --hash=sha256:1d73131e358976663dd969e1fb4ed1404b815cd977eaaedc3b3a133ba2d81c35 \
    --hash=sha256:2a0b02f9fc46f8520330c23d45e6560db7e3a0d927232139427637f98943e11d \
    --hash=sha256:2d3ab0e8c69a17dd3559eab8cbb88f258e285c94d572c2719033f90f83290c89 \
    --hash=sha256:30f464bee641fa8e282577c7dce027308403213c6ca8270bba73285c91024bc5 \
    --hash=sha256:33a834464fdabc0f26a45508df31b3cc5d028e04dbf6c5ed398541418e0a12fe \
    --hash=sha256:3ab3523da44749156e1f68b464dc56af11ae4cbc5c739a49d05f32b982eca9f3 \
    --hash=sha256:3c085faa2cfa879c5141df483f836f4d691045a078224a670fa570fa01612d89 \
    --hash=sha256:457fd7a2a8edeb044ab6ffbc0aa03ff6cd18491356e5e0c834d76ce621b916d1 \
    --hash=sha256:49023963c193dacee096301452f223ee24d86ec5807f8df93c0f7221d119e305 \
    --hash=sha256:52c4b7422442aba924d03ad4019852b08a92e64ea187b933135687bfe2747307 \
    --hash=sha256:559ed65f60c1af5a03f3912605a1b5114f522c7c32fb23c3376ae8f03219fe28 \
    --hash=sha256:5632e3ae3d09197c446310cd5187de63e28448ce22f0f67b2b93d97503c0c230 \
    --hash=sha256:5e4d44984abc0020154ea81b247adeddcc3ac5527b975ff798bd1ba0adc513c2 \
    --hash=sha256:75b00eb8fb802090aa903f4ea1c7f5a584779f967361e68b7e98e531cc2d7174 \
    --hash=sha256:78a0d7c918e74a232394117160e7e3db503377572a45bcef8826e4ab8a35feba \
    --hash=sha256:78c0665edead396b1abb4897c41a5c1d9bf090c8a637a4c20a61678e0a264e66 \
    --hash=sha256:7bbf207c4453ce1ad2e00b17313852b33310b83090c2311bdaf97f93c0380d12 \
    --hash=sha256:7f4b8bc363b6d65ee2152bec57568e3c52639bb34c46057b09857a307ed5e21d \
    --hash=sha256:82f201b4c878551d48558337aab270d3c6cca5507b8737c8d8a608d234cccde0 \
    --hash=sha256:83de5453a7799afc9048b4616bd085cef126e36412f0ea2f6370c36a2a3a51e7 \
    --hash=sha256:88f0e784020649f88ea48c9f5ddfa403bf9205820667c0914740b392035afb82 \
    --hash=sha256:8bcf3c1ba5d6456e2effd30fcbd3459b044d683fcdac79a2e6830f0bdf7de487 \
    --hash=sha256:911de823097db8b63f034299d12662db93344e6ffa0b881cbb57748974b70168 \
    --hash=sha256:92c14f5bdbfb6216315ce33e78080474082de8b3830122ba97809bfbe65f75c0 \
    --hash=sha256:95298364e251be3e60249facbeeca03631d3bb7584f85879516ec55ac717b81f \
    --hash=sha256:9554bcc6d715ee87a633a3cc8e7703c6628b100dd29cb8a2efc4c0533c7ff729 \
    --hash=sha256:9f2897bf7737392ad0d5213ea7b6add72a4edf5679b3153106aeb88b6507b3b9 \
    --hash=sha256:a1d33a7836f7ddc1993427966a0823468ec41bcbdb1a9f9942d1d7e57f803ba3 \
    --hash=sha256:ac0333bdf38309aa3dcbe7e3fa7ea29e7a2c37c6ea306a757b700ded8e4596ad \
    --hash=sha256:bff0b729edd992766136b34e39cc76bc2fad905aa58897ee72a9cd000a6d8443 \
    --hash=sha256:c24acac1e18912761c4700239bbc1fd32f615af690f1584d49b35859be51324d \
    --hash=sha256:c35d74ce0e193ff740c2f2be2ac913ddc232fe6c1ff40b26cfecb9c670c63314 \
    --hash=sha256:c825cef2f49e46753726a7181a8e199804a912b29519ada542c6ebc654951899 \
    --hash=sha256:c9d18a33309122074ea483dd92dd444189166b8b2ec429fe9ed5ac73c7a0aa23 \
    --hash=sha256:cbf38d043c1aa4ab306e1ada6ab6eddacc3322a20b7af1b30bc93254b366fe09 \
    --hash=sha256:cd479fc04dd9401e3b4f49e76518768ef99c4f517a98c284eb091fd725719adf \
    --hash=sha256:ceb30a00ce7c92d459819443d29ca486d882b83fb6738bdcbb2a1cce94ac5daa \
    --hash=sha256:cfbf154f2ba187f2ed6cce2639efff7d105f1140573642c0161615b6d91d6a87 \
    --hash=sha256:d2924a03db38dc2e848bca2fe9f077dafb891480b91a00a0963a8cf86dfc31c1 \
    --hash=sha256:d416b16cccfd70fbf62400e84d0bb2f4e6af519a45557f1692c749b37f14b315 \
    --hash=sha256:d65d448389b8436493abcf629cc94ad0cf32aecaf06e1acca1de53cc795f2f12 \
    --hash=sha256:d84a09d0dad90ba6525d8ac1c2334b33e64bf3ccfe9e841f02feb867a22681e4 \
    --hash=sha256:ddef79fb382df40104a19bb7151b3b23e57c1778fcf857c71ceecd9bd264513f \
    --hash=sha256:e3b417bf8c2c7c16e8f58ad91db17783ec911ac16e7b50eb6eab6e809b4f5b07 \
    --hash=sha256:e402cf31eb68f453dbb2d36fc6d722b33f24a55d68b2ae1d92fa6305ca71c298 \
    --hash=sha256:e6fb6a55cc0ba97b59a1f288fb86dc6fce8bdfc0fffcbfd015e3a954bf2a2d93 \
    --hash=sha256:e708533e8b2ae2497d65346538a7dcc92814410b25b81432eac66de0f2af8265 \
    --hash=sha256:ea324d9dd34c38bfb9bec8ca4d1b407db97dbb74029f566b8e322b1b6fe56fe6 \
    --hash=sha256:eb0dfcf4e28a99c12c999744a2ff67c9b06200e20401c7c88186e33552a46331 \
    --hash=sha256:eda632a7981f69730d6281f451db9c1c370993a2c0d7ddb43e2a809a2862b83a \
    --hash=sha256:f29633129f9fa7e88a3f0fca835de2d030bfc9643f7799e1a0c46cee24d38fc7 \
    --hash=sha256:f55fa87b6c612ecd6b058f167c53231b1d14e412efe361d3d6e38b3631c73218 \
    --hash=sha256:fdaf5ea890a6183d0565f51a61799d67081bd5b1cf03c5f4b3fd3732108625c9
    # via vitpose-keypoint-pipeline (pyproject.toml)
setuptools==84.0.0 \
    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \
    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73
    # via torch
sympy==1.14.0 \
    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \
    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5
    # via torch
tokenizers==0.22.2 \
    --hash=sha256:143b999bdc46d10febb15cbffb4207ddd1f410e2c755857b5a0797961bbdc113 \
    --hash=sha256:1a62ba2c5faa2dd175aaeed7b15abf18d20266189fb3406c5d0550dd34dd5f37 \
    --hash=sha256:1c774b1276f71e1ef716e5486f21e76333464f47bece56bbd554485982a9e03e \
    --hash=sha256:1e418a55456beedca4621dbab65a318981467a2b188e982a23e117f115ce5001 \
    --hash=sha256:1e50f8554d504f617d9e9d6e4c2c2884a12b388a97c5c77f0bc6cf4cd032feee \
    --hash=sha256:2249487018adec45d6e3554c71d46eb39fa8ea67156c640f7513eb26f318cec7 \
    --hash=sha256:25b85325d0815e86e0bac263506dd114578953b7b53d7de09a6485e4a160a7dd \
    --hash=sha256:29c30b83d8dcd061078b05ae0cb94d3c710555fbb44861139f9f83dcca3dc3e4 \
    --hash=sha256:319f659ee992222f04e58f84cbf407cfa66a65fe3a8de44e8ad2bc53e7d99012 \
    --hash=sha256:369cc9fc8cc10cb24143873a0d95438bb8ee257bb80c71989e3ee290e8d72c67 \
    --hash=sha256:37ae80a28c1d3265bb1f22464c856bd23c02a05bb211e56d0c5301a435be6c1a \
    --hash=sha256:38337540fbbddff8e999d59970f3c6f35a82de10053206a7562f1ea02d046fa5 \
    --hash=sha256:473b83b915e547aa366d1eee11806deaf419e17be16310ac0a14077f1e28f917 \
    --hash=sha256:544dd704ae7238755d790de45ba8da072e9af3eea688f698b137915ae959281c \
    --hash=sha256:64d94e84f6660764e64e7e0b22baa72f6cd942279fdbb21d46abd70d179f0195 \
    --hash=sha256:753d47ebd4542742ef9261d9da92cd545b2cacbb48349a1225466745bb866ec4 \
    --hash=sha256:791135ee325f2336f498590eb2f11dc5c295232f288e75c99a36c5dbce63088a \
    --hash=sha256:9ce725d22864a1e965217204946f830c37876eee3b2ba6fc6255e8e903d5fcbc \
    --hash=sha256:a6bf3f88c554a2b653af81f3204491c818ae2ac6fbc09e76ef4773351292bc92 \
    --hash=sha256:bfb88f22a209ff7b40a576d5324bf8286b519d7358663db21d6246fb17eea2d5 \
    --hash=sha256:c9ea31edff2968b44a88f97d784c2f16dc0729b8b143ed004699ebca91f05c48 \
    --hash=sha256:df6c4265b289083bf710dff49bc51ef252f9d5be33a45ee2bed151114a56207b \
    --hash=sha256:e10bf9113d209be7cd046d40fbabbaf3278ff6d18eb4da4c500443185dc1896c \
    --hash=sha256:f01a9c019878532f98927d2bacb79bbb404b43d3437455522a00a30718cdedb5
    # via transformers
torch==2.14.0 \
    --hash=sha256:0e7cf18cb0d8bd666b6120932e29c7aef3502b61a08b44da4839580c539a7cdb \
    --hash=sha256:2fb30099be1fec9d163a6dd05ace176305ce8cf914b4d7a5f4f6d7a326aa4b60 \
    --hash=sha256:44b044b9f6f633d982839422a57433d6a1da520037fd88e0c8a47efde589b3b8 \
    --hash=sha256:4dbea8a10d80b10dff2be1ed467d2617b704fe6bf76bbfad4194774c8f6d886b \
    --hash=sha256:553aec938d37d77b783bcf801e638cad068870e7643c47eadac21eb180f551ed \
    --hash=sha256:6981872df75eb5409c439050d36b67cb57d88b4e11080516e2aa7410f6730705 \
    --hash=sha256:731784e3914843c6bcc7aba3987ff7610ac57dbbc816a5d6b9b62e04c240a641 \
    --hash=sha256:731b9ebdea402b8b1996d4c2ae613b16660e559b19e47bdc45d970568bc91c53 \
    --hash=sha256:84bf384779a10c02fc3c6bdbab71a9cb66b0dd93c652d1ed5d6dfc0cb37e5962 \
    --hash=sha256:860423e970f2ce02c4476e8e2d1350131b1c5c5a5e4912180e78b50b53241efa \
    --hash=sha256:8d9e232b6376c62f3090237889fb1cb6887c0ece9f73e6c1052e80fc1481f999 \
    --hash=sha256:96383c62423c3f2767023c4a94774b6435e2c006691daf396e9ddb55d412fbc6 \
    --hash=sha256:9d4b1022a5d9b71282ec67ad0d9e7235870096b8a246dc1c32d6ea1fc83dc998 \
    --hash=sha256:ada340e62591d06a2bcc2d68170f20f45f0b0665d372dc510a8ed7eb3b1d609a \
    --hash=sha256:ae530ddd3f3b94248b77f1fd3313c4f3405bd0d17283d505b81574816767133a \
    --hash=sha256:b2cd92bce63d40bf6fc2e5d840fc2f0063bd180a242daa761cb6088cc2f46e27 \
    --hash=sha256:b985c7defeb8d28691b7513ebaecc64946c5f68ec770e9f4ddba39688864f46a \
    --hash=sha256:c1f844f1c750e87df4b68bc3afbc0e2b0c7ef19d7b8f666e48bdcf6a0c4f0056 \
    --hash=sha256:cad84f41bbdf3dcf333ce394aeeaf25237c4d94fd6b659ba5eb813c829978823 \
    --hash=sha256:cd8cd8f714d511ccdca907282d1da3d9be8322d4e1520b9c3bce39a5c1318a4b \
    --hash=sha256:d2526f71e6638133b97b3cd2881ece3df521460a4727030e8ae2a72a7d3ae31d \
    --hash=sha256:d30207cb89e713dfd6afb5d592b30d9ef252acf80407c5543157ea7b07c4d9a0 \
    --hash=sha256:d70c2b41bde81efde59fcefb147b9247fd062497e778677661b7de1bee5c8c99 \
    --hash=sha256:fecffb58f51fd643d213acd68da21cc3fc19bea05a3bc64b4ee55128f47a4963
    # via
    #   vitpose-keypoint-pipeline (pyproject.toml)
    #   torchvision
torchaudio==2.11.0 \
    --hash=sha256:00e9f71ab9c656f0abdb40c515bd65d4658ab0ad380dee27a2efd7d51dabd3d6 \
    --hash=sha256:13cff988697ccbad539987599f9dc672f40c417bed67570b365e4e5002bbd096 \
    --hash=sha256:1424638adb8bb40087bc7b6eb103e8e4fe398210f09076f33b7b5e61501b5d66 \
    --hash=sha256:1a07ec72fd6f26a588c39b5f029e0130d16bb40bc4221635580bf8fb18fcbc80 \
    --hash=sha256:1be3767064364ae82705bdf2b15c1e8b41fea82c4cd04d47428a8684b634b6ed \
    --hash=sha256:1c1101c1243ef0e4063ec63298977e2d3655c15cf88d9eb0a1bd4fe2db9f47ea \
    --hash=sha256:478110f981e5d40a8d82221732c57a56c85a1d5895fb8fe646e86ee15eded3bd \
    --hash=sha256:492dd64645e9d0bb843e94f1d9a4d1e31426262ffc594fafecc1697df9df5eb9 \
    --hash=sha256:5847fe2022b17c6580aeb39c8797a443411cc09edfd9183cd50ac1a3b8ccf97c \
    --hash=sha256:6503c0bdb29daf2e6281bb70ea2dfe2c3553b782b619eb5d73bdadd8a3f7cecf \
    --hash=sha256:67f6edac29ed004652c11db5c19d9debb5d835695930574f564efc8bdd061bba \
    --hash=sha256:6ebb59c694909eccb5d61b7cc199d297692012c43286e36d92983aa7bad7586d \
    --hash=sha256:73dab4841f94d888bc7c2aed7b5547c643edc974306919fe1adfb65d57cccf4b \
    --hash=sha256:79fb3cb99169fd41bd9719647261402a164da0d105a4d81f42a3260844ec5e79 \
    --hash=sha256:7e2da1df4f6fe885c46db350a0dc90a0dff4b54541dff8846faa904d255e2bfe \
    --hash=sha256:88fb5e29f670a33d9bac6aabb1d2734460cf6e461bde5cdc352826035851b16d \
    --hash=sha256:986f4df5ed17b003dc52489468601720090e65f964f8bebccf90eb45bba75744 \
    --hash=sha256:9fe3083c62e035646483a14e180d33561bdc2eed436c9ab1259c137fb7120b4a \
    --hash=sha256:a1cf1acc883bee9cb906a933572fed6a8a933f86ef34e9ea7d803f72317e8c1b \
    --hash=sha256:b034d7672f1c415434f48ef17807f2cce47f29e8795338c751d4e596c9fbe8b5 \
    --hash=sha256:bb59ba4452bbbe95d75ad3ef18df9824955625f36698ce9a5998a4a9f3c1ba1d \
    --hash=sha256:bc653defca1c16154398517a1adc98d0fb7f1dd08e58ced217558d213c2c6e29 \
    --hash=sha256:bda09ea630ae7207384fb0f28c35e4f8c0d82dd6eba020b6b335ad0caa9fed49 \
    --hash=sha256:be7ad472acb16d16e98c005f0219b0db06a47dfe8f7b4d177062e1638f871e3b \
    --hash=sha256:cc09cd1f6015b8549e7fe255fb1be5346b57e7fee06541d3f3dbb012d8c4715f \
    --hash=sha256:da2725e250866da42a12934c9a6552f65a18b7187fd7a6221387f0e605fb3b96 \
    --hash=sha256:e3f9696a9ef1d49acc452159b052370c636406d072e9d8f10895fda87b591ea9 \
    --hash=sha256:ed404c4399ad7f172c86a47c1b25293d322d1d58e26b10b0456a86cf67d37d84
    # via vitpose-keypoint-pipeline (pyproject.toml)
torchvision==0.29.0 \
    --hash=sha256:01a029fa1b2eacac27e1e4f9e0dba24d112bcbd523261b294e4b4a1c7f8330bc \
    --hash=sha256:0e3b99c62f7f095153887330c2c60c9ebc2da84dfbe08eea564a41ec361629cf \
    --hash=sha256:0e631a2f8b24732672d35224d2574ff89a78d56bef5364ef6094a625fa5f1771 \
    --hash=sha256:2428a13706a354ee3901fd63e3f1ef8c2a59f6e20d24cce13210b4e61be08f7a \
    --hash=sha256:28a3964a9e6db34354d4ca440d2c2038deeef24e7219e8d6f49d9d174f44830c \
    --hash=sha256:710df2c4f03ab6da161520c10d9467f4c0201ca314fd1432499a48c9d7a5818f \
    --hash=sha256:784c8b8de9e81e02dea093373bc35040791378df2daec0017e657985183bf862 \
    --hash=sha256:80789f9b50f5277302e7020c8128344e0269b12f6e3b5855034e56bb7081b874 \
    --hash=sha256:83db0299170502b038640444214c99d7b54bad72db447f0554deb8d2c4f02f94 \
    --hash=sha256:85fa54bec1f7d9227b5e4e201ed4bac92247ee10fb19148d92f9836e3e91d5c3 \
    --hash=sha256:942a3b3fb4e981e1abb7846679ee3882dfc733533a30b68df0dadf6c06f238a9 \
    --hash=sha256:994687b818cac0e6d34cb407a41458e2c3262cb7db927465841c9522ba7ebb92 \
    --hash=sha256:9b99a7385da8d706f2dc14cacffb8cbf653d48fc37013a38c41a36659acc3731 \
    --hash=sha256:a292ca7044236a2702ec3d4664de135e01fc9fb3ae9422db3fecc819546e831c \
    --hash=sha256:add88479361b2bf08790338fce4ba11a11fdc1ae508e80a3a013daf1c0d2893d \
    --hash=sha256:b7a736eaa6b2e22476c95ceb62986d195f3e600cdd7d196d7329aa2dfc951994 \
    --hash=sha256:bb74d08d65785b51c61d4d4ec167bfa50a5ea644b57555a33ba2f4435b0d05dd \
    --hash=sha256:bbe15455b59a6c9d822584fd3faaaac50ee972101d7bb7e2887da456e148a3ea \
    --hash=sha256:bd12e152640d1024fca15deaf6f39c53702e12eb398db4b2b038a5a97ab85f7f \
    --hash=sha256:ce3b58631f0b7c8828e8675301bff5cb0623aaed704e3ac91079e89607753205 \
    --hash=sha256:dc5717feef9a0b430b052895db71f1d695939575b74344da981120e9f4da7620 \
    --hash=sha256:e8fdf234d76dca6fc47bc5f4af86ecf1ef57fab6285a7b21681bdcbb494e4c1b \
    --hash=sha256:ebf54f6744556ebd8d7b5f9b6515e2060e5d3156723970dc257bf093f63b2170 \
    --hash=sha256:f9e9627d5036cac8e6de76aeb2f1acc67841ff5ff4b63c4cf78ff954efeb073e
    # via vitpose-keypoint-pipeline (pyproject.toml)
tqdm==4.70.1 \
    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \
    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4
    # via
    #   huggingface-hub
    #   transformers
transformers==4.57.6 \
    --hash=sha256:4c9e9de11333ddfe5114bc872c9f370509198acf0b87a832a0ab9458e2bd0550 \
    --hash=sha256:55e44126ece9dc0a291521b7e5492b572e6ef2766338a610b9ab5afbb70689d3
    # via vitpose-keypoint-pipeline (pyproject.toml)
triton==3.8.0 \
    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \
    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \
    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \
    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \
    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \
    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \
    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \
    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \
    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \
    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \
    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \
    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3
    # via torch
typing-extensions==4.16.0 \
    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \
    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5
    # via
    #   huggingface-hub
    #   torch
urllib3==2.8.0 \
    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \
    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63
    # via requests
'''

SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
# One environment per lock digest: a re-run of this cell, or a second Run all in the same runtime, reuses a complete
# environment built from this exact lock instead of rebuilding it; a different lock gets a different folder.
ISOLATED_ENV = Path(os.environ.get('DIMER_ISOLATED_ENV', 'dimer_isolated_env_' + LOCK_SHA256[:12])).resolve()
ISOLATED_PYTHON = ISOLATED_ENV / 'bin' / 'python'
ISOLATED_TOOLS = ISOLATED_ENV.with_name(ISOLATED_ENV.name + '_tools')
ISOLATED_READY = ISOLATED_ENV / '.dimer-lock-sha256'


def _isolated_environment_ready():
    return ISOLATED_PYTHON.is_file() and ISOLATED_READY.is_file() and ISOLATED_READY.read_text(encoding='utf-8').strip() == LOCK_SHA256


if SKIP_INSTALL:
    print('DIMER_NOTEBOOK_CI_PREINSTALLED=1: the pins are already installed; the notebook runs in this kernel.')
elif _isolated_environment_ready():
    print({'isolated_environment': str(ISOLATED_ENV), 'reused': True, 'lock_sha256': LOCK_SHA256[:16] + '...', 'locked_packages': LOCKED_PACKAGES, 'kernel_python': platform.python_version()})
else:
    if platform.system() != 'Linux' or platform.machine() != 'x86_64':
        raise RuntimeError('This notebook needs a Linux x86_64 runtime (Google Colab, Kaggle or Linux Jupyter): its locked environment is built for manylinux x86_64.')
    setup_started = time.perf_counter()
    if hashlib.sha256(LOCK_TEXT.encode('utf-8')).hexdigest() != LOCK_SHA256:
        raise RuntimeError('The carried lock does not match its digest: regenerate the notebook from the repository instead of editing this cell.')
    ISOLATED_TOOLS.mkdir(parents=True, exist_ok=True)
    lock_path = ISOLATED_TOOLS / LOCK_NAME
    lock_path.write_text(LOCK_TEXT, encoding='utf-8', newline='\n')
    for attempt in range(3):
        try:
            with urllib.request.urlopen(UV_URL, timeout=90) as response:
                wheel = response.read(UV_BYTES + 1)
            break
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == 2:
                raise
            time.sleep(2**attempt)
    if len(wheel) != UV_BYTES or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
        raise RuntimeError('The pinned uv wheel failed its size/SHA-256 check: refusing to run it. Run this cell again; if it repeats, the download is being altered.')
    with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
        member = next(name for name in archive.namelist() if name.endswith('.data/scripts/uv'))
        uv = ISOLATED_TOOLS / 'uv'
        uv.write_bytes(archive.read(member))
    uv.chmod(0o700)
    # uv gets no kernel Python path; the managed interpreter is downloaded once and reused on a re-run.
    uv_env = dict(os.environ, MPLBACKEND='Agg')
    for name in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP'):
        uv_env.pop(name, None)
    if not ISOLATED_PYTHON.is_file():
        subprocess.run([str(uv), 'venv', '--quiet', '--managed-python', '--python', MANAGED_PYTHON, str(ISOLATED_ENV)], env=uv_env, check=True)
    isolated_version = subprocess.run([str(ISOLATED_PYTHON), '-c', 'import platform; print(platform.python_version())'], env=uv_env, check=True, capture_output=True, text=True).stdout.strip()
    if isolated_version != MANAGED_PYTHON:
        raise RuntimeError(f'{ISOLATED_ENV} holds Python {isolated_version}, not {MANAGED_PYTHON}: delete that folder (or start a fresh runtime) and run this cell again.')
    ISOLATED_READY.unlink(missing_ok=True)
    subprocess.run([str(uv), 'pip', 'install', '--quiet', '--python', str(ISOLATED_PYTHON), '--require-hashes', '--only-binary', ':all:', '--index-url', 'https://pypi.org/simple', '-r', str(lock_path)], env=uv_env, check=True)
    ISOLATED_READY.write_text(LOCK_SHA256 + '\n', encoding='utf-8')
    print({'isolated_environment': str(ISOLATED_ENV), 'reused': False, 'isolated_python': isolated_version, 'kernel_python': platform.python_version(), 'locked_packages': LOCKED_PACKAGES, 'setup_seconds': round(time.perf_counter() - setup_started)})

# The worker runs in the isolated environment. It executes each routed cell in one persistent namespace and sends
# back printed text, displayed objects and matplotlib figures, so every cell behaves as it would in the kernel.
_WORKER_SOURCE = r"""
import ast, base64, builtins, io, linecache, os, signal, sys, traceback, types
from multiprocessing.connection import Connection

_send = Connection(int(sys.argv[1]), readable=False)
_recv = Connection(int(sys.argv[2]), writable=False)


class _Stream(io.TextIOBase):
    def __init__(self, name):
        self._name = name

    @property
    def encoding(self):
        return "utf-8"

    def writable(self):
        return True

    def isatty(self):
        return False

    def write(self, text):
        if text:
            _send.send(("stream", self._name, str(text)))
        return len(text)


sys.stdout, sys.stderr = _Stream("stdout"), _Stream("stderr")


def _figure_bundle(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight")
    return {"image/png": base64.b64encode(buffer.getvalue()).decode("ascii"), "text/plain": repr(fig)}


def _flush_figures():
    plt = sys.modules.get("matplotlib.pyplot")
    if plt is None:
        return
    for number in plt.get_fignums():
        _send.send(("display", _figure_bundle(plt.figure(number))))
    plt.close("all")


def _mimebundle(obj):
    if hasattr(obj, "savefig"):
        return _figure_bundle(obj)
    data = {"text/plain": repr(obj)}
    for method, mime in (("_repr_html_", "text/html"), ("_repr_markdown_", "text/markdown"), ("_repr_png_", "image/png")):
        render = getattr(obj, method, None)
        if callable(render):
            try:
                value = render()
            except Exception:
                value = None
            if isinstance(value, bytes):
                value = base64.b64encode(value).decode("ascii")
            if value is not None:
                data[mime] = value
    return data


def display(*objects, **kwargs):
    for obj in objects:
        _send.send(("display", _mimebundle(obj)))


try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot

    matplotlib.pyplot.show = lambda *args, **kwargs: _flush_figures()
except ImportError:
    pass

if os.environ.get("DIMER_KERNEL_IS_COLAB") == "1":
    # google.colab only exists in the kernel; forward the BYOD upload dialog to it.
    def _upload():
        _send.send(("upload",))
        reply = _recv.recv()
        if reply[1] is None:
            raise RuntimeError("The notebook kernel could not open the upload dialog.")
        return reply[1]

    import importlib.machinery

    def _stub(name, package):
        # A spec on every stub: importlib.util.find_spec("google.colab") (accelerate does this) raises on a None __spec__.
        module = types.ModuleType(name)
        module.__spec__ = importlib.machinery.ModuleSpec(name, None, is_package=package)
        if package:
            module.__path__ = []
        return module

    try:
        import google
    except ImportError:
        google = _stub("google", True)
        sys.modules["google"] = google
    _colab = _stub("google.colab", True)
    _files = _stub("google.colab.files", False)
    _files.upload = _upload
    _colab.files = _files
    google.colab = _colab
    sys.modules["google.colab"] = _colab
    sys.modules["google.colab.files"] = _files

_main = types.ModuleType("__main__")
_main.__dict__.update(__builtins__=builtins, display=display)
sys.modules["__main__"] = _main
_count = 0
while True:
    # An interrupt only lands inside a running cell; between cells it is ignored.
    signal.signal(signal.SIGINT, signal.SIG_IGN)
    try:
        message = _recv.recv()
    except EOFError:
        break
    if message[0] != "run":
        continue
    _count += 1
    filename = f"<isolated cell {_count}>"
    source = message[1]
    linecache.cache[filename] = (len(source), None, source.splitlines(True), filename)
    try:
        signal.signal(signal.SIGINT, signal.default_int_handler)
        tree = ast.parse(source, filename)
        tail = ast.Expression(tree.body.pop().value) if tree.body and isinstance(tree.body[-1], ast.Expr) else None
        exec(compile(tree, filename, "exec"), _main.__dict__)
        if tail is not None:
            value = eval(compile(tail, filename, "eval"), _main.__dict__)
            if value is not None:
                display(value)
        _flush_figures()
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        _send.send(("done",))
    except BaseException as exc:
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        frames = exc.__traceback__.tb_next if exc.__traceback__ is not None else None
        _send.send(("error", "".join(traceback.format_exception(type(exc), exc, frames)), f"{type(exc).__name__}: {exc}"))
"""


class IsolatedCellError(RuntimeError):
    """A routed cell raised inside the isolated environment; its traceback is printed above."""


class IsolatedRuntime:
    """One persistent worker process in the isolated environment, fed one cell at a time."""

    def __init__(self, python, display=None):
        self.python = str(python)
        to_kernel_r, to_kernel_w = os.pipe()
        to_worker_r, to_worker_w = os.pipe()
        env = dict(os.environ, MPLBACKEND="Agg", PYTHONUNBUFFERED="1", DIMER_NOTEBOOK_CI_PREINSTALLED="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
        for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            env.pop(name, None)
        env["DIMER_KERNEL_IS_COLAB"] = "1" if "google.colab" in sys.modules else "0"
        self.proc = subprocess.Popen(
            [str(python), "-c", _WORKER_SOURCE, str(to_kernel_w), str(to_worker_r)],
            pass_fds=(to_kernel_w, to_worker_r),
            env=env,
            start_new_session=True,  # interrupts reach the worker only through run(), exactly once
        )
        os.close(to_kernel_w)
        os.close(to_worker_r)
        self._recv = Connection(to_kernel_r, writable=False)
        self._send = Connection(to_worker_w, readable=False)
        if display is None:
            from IPython.display import display
        self._display = display

    def alive(self):
        return self.proc.poll() is None

    def _exited(self):
        return RuntimeError(
            f"The isolated environment's Python process exited (code {self.proc.wait()}); a crash of this kind is "
            "usually running out of memory. Restart the session and choose Run all again."
        )

    def run(self, source):
        try:
            self._send.send(("run", source))
        except OSError:
            raise self._exited() from None
        while True:
            try:
                message = self._recv.recv()
            except EOFError:
                raise self._exited() from None
            except KeyboardInterrupt:
                self.proc.send_signal(signal.SIGINT)
                continue
            kind = message[0]
            if kind == "stream":
                (sys.stdout if message[1] == "stdout" else sys.stderr).write(message[2])
            elif kind == "display":
                self._display(message[1], raw=True)
            elif kind == "upload":
                self._send.send(("upload", self._colab_upload()))
            elif kind == "error":
                sys.stderr.write(message[1])
                raise IsolatedCellError(message[2]) from None
            elif kind == "done":
                return

    @staticmethod
    def _colab_upload():
        try:
            from google.colab import files
        except ImportError:
            return None
        return files.upload()

    def close(self):
        self._send.close()
        self.proc.wait(timeout=30)


def _is_user_cell():
    # ipykernel transforms a cell before executing it; its caller knows whether this is a silent frontend request.
    frame = sys._getframe(2)
    while frame is not None:
        local = frame.f_locals
        if "silent" in local and "store_history" in local:
            return bool(local["store_history"]) and not bool(local["silent"])
        frame = frame.f_back
    return True


def _route_to_isolated_runtime(lines):
    source = "".join(lines)
    if not source.strip() or "# dimer: kernel cell" in source or not _is_user_cell():
        return lines
    return [f"_DIMER_ISOLATED_RUNTIME.run({source!r})\n"]

if SKIP_INSTALL:
    print('Routing disabled: the notebook runs in this kernel.')
else:
    from IPython import get_ipython as _kernel_shell

    _ip = _kernel_shell()
    _ip.input_transformers_cleanup[:] = [t for t in _ip.input_transformers_cleanup if getattr(t, '__name__', '') != '_route_to_isolated_runtime']
    # Idempotent: a live worker on this environment is kept, with every variable the later cells created, so
    # re-running this cell alone never strands the cells after it. A dead or different worker is replaced.
    _previous = globals().get('_DIMER_ISOLATED_RUNTIME')
    _reuse = _previous is not None and getattr(_previous, 'python', None) == str(ISOLATED_PYTHON) and _previous.proc.poll() is None
    if _reuse:
        _DIMER_ISOLATED_RUNTIME = _previous
    else:
        if _previous is not None and _previous.proc.poll() is None:
            _previous.close()
        _DIMER_ISOLATED_RUNTIME = IsolatedRuntime(ISOLATED_PYTHON)
    _ip.input_transformers_cleanup.append(_route_to_isolated_runtime)
    print({'routed_to': str(ISOLATED_PYTHON), 'worker_pid': _DIMER_ISOLATED_RUNTIME.proc.pid, 'worker_reused': _reuse})

{'isolated_environment': '/content/dimer_isolated_env_82f7302e59ff', 'reused': False, 'isolated_python': '3.12.12', 'kernel_python': '3.13.15', 'locked_packages': 48, 'setup_seconds': 69}
{'routed_to': '/content/dimer_isolated_env_82f7302e59ff/bin/python', 'worker_pid': 3154, 'worker_reused': False}


### Record the runtime

This is the first cell that runs in the isolated environment. Nothing is installed here. It records the notebook's source revision and the versions actually imported. Look for a dictionary reporting the notebook's source revision, the isolated Python (3.12.12), `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
# @title Infrastructure: record the runtime
import importlib
import os
import platform
import sys

NOTEBOOK_SOURCE = {
    'repository': 'vitpose-keypoint-pipeline',
    'repository_revision': 'e98384143a7a28255ed0227dd970f94aa81c6acd',
    'embedded_module': 'src/vitpose_keypoint_pipeline/pipeline.py',
    'embedded_modules': ['src/vitpose_keypoint_pipeline/pipeline.py', 'src/vitpose_keypoint_pipeline/metrics.py', 'src/vitpose_keypoint_pipeline/samples.py'],
    'module_sha256': '7f6b81e1e294e4a4dfb98036332bb72aa3899e53d24dae35a27741a7c3baa636',
    'generator': 'build_notebook.py/2.2',
    'notebook_spec': '2.2',
}
import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'executable': sys.executable, 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

{'notebook_source': {'repository': 'vitpose-keypoint-pipeline', 'repository_revision': 'e98384143a7a28255ed0227dd970f94aa81c6acd', 'embedded_module': 'src/vitpose_keypoint_pipeline/pipeline.py', 'embedded_modules': ['src/vitpose_keypoint_pipeline/pipeline.py', 'src/vitpose_keypoint_pipeline/metrics.py', 'src/vitpose_keypoint_pipeline/samples.py'], 'module_sha256': '7f6b81e1e294e4a4dfb98036332bb72aa3899e53d24dae35a27741a7c3baa636', 'generator': 'build_notebook.py/2.2', 'notebook_spec': '2.2'}, 'python': '3.12.12', 'executable': '/content/dimer_isolated_env_82f7302e59ff/bin/python', 'torch': '2.14.0+cu130', 'transformers': '4.57.6', 'cuda': True}


## 2. Pipeline code (carried verbatim from `src/vitpose_keypoint_pipeline/` @ `e98384143a7a`)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/vitpose_keypoint_pipeline/pipeline.py`

In [ ]:
"""Two-stage human pose estimation with the pinned ``usyd-community/vitpose-base`` checkpoint (ViTPose)
and the pinned ``PekingU/rtdetr_r50vd`` person detector (RT-DETR).

Both models load only from digest-verified local snapshots (``weights/<key>/``) or, when explicitly
allowed, from the Hugging Face Hub at their pinned revisions — always with ``trust_remote_code=False``:
the architectures come from the pinned ``transformers`` release, the weights are SafeTensors, and no
model-repository code is executed. ViTPose is top-down: it needs a person box first, either from the
carried detector stage or from the caller.
"""

from __future__ import annotations

# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width
import hashlib
import json
import math
import random
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "usyd-community/vitpose-base"
MODEL_REVISION = "95be2991424e646950d656bb7fc15ec9be119700"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "vitpose-base"
_WEIGHTS_ROOT = Path.cwd() / "weights"  # standalone rewrite (build_notebook.py): working-directory-relative
DEFAULT_WEIGHTS_DIR = _WEIGHTS_ROOT / MODEL_KEY
MANIFEST_NAME = "dimer-base-manifest.json"

# Stage 1 (person detection) is a SECOND pinned snapshot with its own manifest: the RT-DETR R50-VD
# checkpoint the sibling rtdetr-detection-pipeline wraps. Only its `person` class is used here.
DETECTOR_MODEL_ID = "PekingU/rtdetr_r50vd"
DETECTOR_REVISION = "df939e661d8c52e80608d1ec566561aabd25a4e7"
DETECTOR_LICENSE = "apache-2.0"
DETECTOR_KEY = "rtdetr-r50vd"
DETECTOR_WEIGHTS_DIR = _WEIGHTS_ROOT / DETECTOR_KEY
DETECTOR_PERSON_LABEL = "person"
# Detection threshold on the detector's per-class sigmoid for `person`: the value the pinned ViTPose
# README's two-stage example passes (threshold=0.3). Uncalibrated; the deployment owns tuning it.
DETECTION_THRESHOLD = 0.3
# Keypoint threshold on ViTPose's per-keypoint heatmap maximum: the README example's value (0.3).
# Keypoints below it are dropped from `keypoints` but still listed under `all_keypoints`.
KEYPOINT_THRESHOLD = 0.3
# The 17 COCO keypoints in the checkpoint's id2label order (config.json) and its skeleton edges.
KEYPOINT_NAMES = (
    "Nose",
    "L_Eye",
    "R_Eye",
    "L_Ear",
    "R_Ear",
    "L_Shoulder",
    "R_Shoulder",
    "L_Elbow",
    "R_Elbow",
    "L_Wrist",
    "R_Wrist",
    "L_Hip",
    "R_Hip",
    "L_Knee",
    "R_Knee",
    "L_Ankle",
    "R_Ankle",
)
SKELETON_EDGES = (
    (15, 13),
    (13, 11),
    (16, 14),
    (14, 12),
    (11, 12),
    (5, 11),
    (6, 12),
    (5, 6),
    (5, 7),
    (6, 8),
    (7, 9),
    (8, 10),
    (1, 2),
    (0, 1),
    (0, 2),
    (1, 3),
    (2, 4),
    (3, 5),
    (4, 6),
)
# Input ceilings. Each person crop is affine-warped to 192x256 (preprocessor_config.json) so the
# pose stage's cost is per person, not per pixel; the detector resizes the whole image to 640x640.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
MAX_PERSONS = 50
# PCK normaliser for the sanity metric: a keypoint is "correct" when within this fraction of the
# person box's longest side of its reference (a stated convention, not the COCO OKS metric).
PCK_FRACTION = 0.1
PARAMETER_COUNT = 89_994_513  # ViTPose-base: backbone + heatmap head (the detector is not counted)
HEAD_PARAMETERS = 4_199_697
ARTIFACT_FORMAT = f"org.valcorza.{MODEL_KEY}.adapter.v1"
ARTIFACT_VERSION = "1.0"
ADAPTER_WEIGHTS = "adapter.safetensors"
ADAPTER_MANIFEST = "manifest.json"
WEIGHT_FILE = "model.safetensors"
MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample
MAX_EVAL_RECORDS = 5_000
MAX_TRAINABLE_BLOCKS = 12
HEATMAP_SIGMA = 2.0  # heatmap pixels; the ViTPose/mmpose target Gaussian


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _verify_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != model_id:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {model_id!r}")
    if manifest.get("revision") != revision:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {revision!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the ViTPose snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    return _verify_manifest(root, MODEL_ID, MODEL_REVISION)


def verify_detector_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the RT-DETR person-detector snapshot against its DIMER manifest."""
    root = Path(path) if path is not None else DETECTOR_WEIGHTS_DIR
    return _verify_manifest(root, DETECTOR_MODEL_ID, DETECTOR_REVISION)


def _hub_download(relative_path: str, root: Path, model_id: str, revision: str) -> None:
    """Fetch one manifest-listed file at the pinned revision straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(model_id, relative_path, revision=revision, local_dir=str(root))


def _stage_missing(
    root: Path,
    model_id: str,
    revision: str,
    allow_download: bool,
    downloader: Callable[[str, Path], None] | None,
) -> list[str]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != model_id or manifest.get("revision") != revision:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {model_id}@{revision}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; pass allow_download=True to fetch them at {revision}"
        )
    fetch = downloader or (lambda rel, dst: _hub_download(rel, dst, model_id, revision))
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch ViTPose manifest entries that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    return _stage_missing(root, MODEL_ID, MODEL_REVISION, allow_download, downloader)


def stage_missing_detector_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Same as `stage_missing_files` for the RT-DETR person-detector snapshot at DETECTOR_REVISION."""
    root = Path(path) if path is not None else DETECTOR_WEIGHTS_DIR
    return _stage_missing(root, DETECTOR_MODEL_ID, DETECTOR_REVISION, allow_download, downloader)


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def _check_threshold(name: str, value: Any) -> float:
    if isinstance(value, bool) or not isinstance(value, int | float) or not 0.0 <= value <= 1.0:
        raise ValueError(f"{name} must be a number in [0, 1], got {value!r}")
    return float(value)


def _check_boxes(boxes: Any, width: int, height: int) -> list[list[float]]:
    if isinstance(boxes, str) or not isinstance(boxes, Sequence):
        raise TypeError("person_boxes must be a sequence of [x0, y0, x1, y1] boxes")
    if not 1 <= len(boxes) <= MAX_PERSONS:
        raise ValueError(f"person_boxes has {len(boxes)} entries; expected 1..MAX_PERSONS={MAX_PERSONS}")
    checked: list[list[float]] = []
    for index, box in enumerate(boxes):
        if isinstance(box, str) or not isinstance(box, Sequence) or len(box) != 4:
            raise ValueError(f"person_boxes[{index}] must be [x0, y0, x1, y1]")
        try:
            x0, y0, x1, y1 = (float(v) for v in box)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"person_boxes[{index}] must hold numbers") from exc
        if not (0 <= x0 < x1 <= width and 0 <= y0 < y1 <= height):
            raise ValueError(
                f"person_boxes[{index}] {list(box)} is not a non-empty box inside the {width}x{height} image"
            )
        checked.append([x0, y0, x1, y1])
    return checked


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one image as PIL.Image.Image (any mode, converted to RGB) and optionally the person boxes as "
        "pixel xyxy; without boxes the carried RT-DETR stage detects `person` first"
    ),
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "person_boxes": [1, MAX_PERSONS],
    "detection_threshold": [0.0, 1.0],
    "keypoint_threshold": [0.0, 1.0],
    "keypoints": list(KEYPOINT_NAMES),
    "preprocessing": (
        "stage 1 (when no boxes are given): the whole image resized to 640x640 for RT-DETR, `person` boxes "
        "kept above detection_threshold; stage 2: each person box affine-warped to a 192x256 crop "
        "(ImageNet mean/std) for ViTPose, heatmap maxima mapped back to input pixels"
    ),
    "output": "per person: the box, its source, 17 named keypoints with pixel coordinates and heatmap scores",
}


def _check_inputs(
    image: Any, person_boxes: Any, detection_threshold: Any, keypoint_threshold: Any
) -> tuple[Image.Image, list[list[float]] | None, float, float]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``estimate`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    boxes = None if person_boxes is None else _check_boxes(person_boxes, rgb.width, rgb.height)
    return (
        rgb,
        boxes,
        _check_threshold("detection_threshold", detection_threshold),
        _check_threshold("keypoint_threshold", keypoint_threshold),
    )


def validate_inputs(
    image: Image.Image,
    *,
    person_boxes: Sequence[Sequence[float]] | None = None,
    detection_threshold: float = DETECTION_THRESHOLD,
    keypoint_threshold: float = KEYPOINT_THRESHOLD,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``estimate`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, boxes, det_t, kp_t = _check_inputs(image, person_boxes, detection_threshold, keypoint_threshold)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (estimate takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "person_boxes": boxes,
        "box_source": "caller" if boxes is not None else "detector",
        "detection_threshold": det_t,
        "keypoint_threshold": kp_t,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "detector_model_id": DETECTOR_MODEL_ID,
        "detector_revision": DETECTOR_REVISION,
    }


def keypoint_pck(
    predicted: Mapping[str, Sequence[float]],
    reference: Mapping[str, Sequence[float]],
    box: Sequence[float],
    *,
    fraction: float = PCK_FRACTION,
) -> dict[str, Any]:
    """Percentage of correct keypoints: a named reference joint counts as correct when the predicted
    joint of the same name lies within ``fraction`` of the box's longest side; missing joints are misses."""
    if not reference:
        raise ValueError("reference must contain at least one keypoint")
    if len(box) != 4 or box[2] <= box[0] or box[3] <= box[1]:
        raise ValueError("box must be a non-empty [x0, y0, x1, y1]")
    radius = fraction * max(box[2] - box[0], box[3] - box[1])
    errors: dict[str, float | None] = {}
    correct = 0
    for name, (rx, ry) in reference.items():
        if name in predicted:
            px, py = predicted[name][0], predicted[name][1]
            err = math.hypot(px - rx, py - ry)
            errors[name] = err
            correct += err <= radius
        else:
            errors[name] = None
    measured = [e for e in errors.values() if e is not None]
    return {
        "value": correct / len(reference),
        "correct": correct,
        "total": len(reference),
        "radius_px": radius,
        "fraction": fraction,
        "mean_error_px": sum(measured) / len(measured) if measured else None,
        "errors_px": errors,
    }


def evaluation_report(
    result: Mapping[str, Any],
    reference_keypoints: Sequence[Mapping[str, Sequence[float]]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``reference_keypoints`` (one name -> (x, y) mapping per person, in the order of
    ``result['poses']``) the report carries one ``keypoint_pck`` entry per person as sample-sanity
    geometry evidence; without them the verdict is ``not-measurable`` and the report says what
    labelled data would make the task measurable.
    """
    poses = list(result["poses"])
    base = {
        "task": "two-stage human pose estimation: person boxes -> 17 COCO keypoints per person",
        "score_semantics": (
            "each keypoint score is the maximum of its ViTPose heatmap after the crop's affine warp — a "
            "ranking signal per joint, not a calibrated probability that the joint is where it says; the "
            "person score, when the detector stage ran, is RT-DETR's per-class sigmoid for `person`"
        ),
        "box_source": result.get("box_source"),
        "detection_threshold": result.get("detection_threshold", DETECTION_THRESHOLD),
        "keypoint_threshold": result.get("keypoint_threshold", KEYPOINT_THRESHOLD),
        "sample_kind": sample_kind,
        "n_persons": len(poses),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "detector_model_id": DETECTOR_MODEL_ID,
        "detector_revision": DETECTOR_REVISION,
    }
    if not reference_keypoints:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no reference keypoints were supplied for the evaluated image",
            "needs": (
                "images with COCO-style keypoint annotations from the deployment domain scored with "
                "OKS-based average precision (the COCO keypoint metric) or PCK at a stated normaliser; "
                "no such labelled set ships with this repository"
            ),
        }
    if len(reference_keypoints) != len(poses):
        raise ValueError(f"reference_keypoints has {len(reference_keypoints)} entries for {len(poses)} poses")
    metrics = []
    for index, (pose, reference) in enumerate(zip(poses, reference_keypoints, strict=True)):
        unknown = set(reference) - set(KEYPOINT_NAMES)
        if unknown:
            raise ValueError(f"unknown keypoint names in reference {index}: {sorted(unknown)}")
        predicted = {kp["name"]: (kp["x"], kp["y"]) for kp in pose["all_keypoints"]}
        pck = keypoint_pck(predicted, reference, pose["box"])
        metrics.append(
            {
                "id": "keypoint_pck",
                "person": index,
                "value": pck["value"],
                "correct": pck["correct"],
                "total": pck["total"],
                "radius_px": pck["radius_px"],
                "mean_error_px": pck["mean_error_px"],
                "estimation": "one drawn person on a single image, no dispersion estimate",
            }
        )
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(metrics)} person(s) on one tutorial image whose joints you drew yourself; geometry "
            "sanity evidence, not a pose benchmark"
        ),
        "needs": (
            "a keypoint-labelled image set from the deployment domain (cameras, poses, occlusion, clothing) "
            "for any OKS-AP or PCK claim"
        ),
    }


def _weight_digest(root: Path) -> str | None:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        return None
    with open(manifest_path, encoding="utf-8") as handle:
        entries = json.load(handle).get("files", [])
    return next((e["sha256"] for e in entries if e["path"] == WEIGHT_FILE), None)


def _trainable_names(model: Any, trainable_blocks: int) -> list[str]:
    """The heatmap head plus the last `trainable_blocks` encoder blocks and the final backbone LayerNorm; the patch
    embedding and the earlier blocks stay frozen."""
    names = [name for name, _ in model.named_parameters()]
    layers = sorted({int(n.split(".")[3]) for n in names if n.startswith("backbone.encoder.layer.")})
    keep = set(layers[len(layers) - trainable_blocks :]) if trainable_blocks else set()
    out = [n for n in names if n.startswith("head.")]
    out += [n for n in names if n.startswith("backbone.encoder.layer.") and int(n.split(".")[3]) in keep]
    if keep:
        out += [n for n in names if n.startswith("backbone.layernorm")]
    return out


def _check_artifact_manifest(manifest: Mapping[str, Any], artifact_dir: Path, base_sha256: str) -> None:
    """Refuse an adapter that names another base, another format or a file that does not match its digest."""
    if manifest.get("format") != ARTIFACT_FORMAT:
        raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
    base = manifest.get("base", {})
    if base.get("model_id") != MODEL_ID or base.get("revision") != MODEL_REVISION:
        raise ValueError(f"artifact was trained on {base.get('model_id')}@{base.get('revision')}, not {MODEL_ID}@{MODEL_REVISION}")
    if base.get("weight_sha256") != base_sha256:
        raise ValueError("artifact base weight digest does not match the verified snapshot")
    files = manifest.get("files") or []
    if len(files) != 1 or files[0].get("path") != ADAPTER_WEIGHTS:
        raise ValueError(f"artifact manifest must list exactly {ADAPTER_WEIGHTS}")
    weights = artifact_dir / ADAPTER_WEIGHTS
    if not weights.is_file():
        raise FileNotFoundError(f"artifact weights missing: {weights}")
    size = weights.stat().st_size
    if size != files[0].get("bytes"):
        raise ValueError(f"{ADAPTER_WEIGHTS}: size {size} != manifest {files[0].get('bytes')}")
    digest = _sha256(weights)
    if digest != files[0].get("sha256"):
        raise ValueError(f"{ADAPTER_WEIGHTS}: sha256 {digest} != manifest {files[0].get('sha256')}")
    names = manifest.get("tensors") or []
    if not names or any(not str(n).startswith(("head.", "backbone.encoder.layer.", "backbone.layernorm")) for n in names):
        raise ValueError("artifact tensors must all belong to the heatmap head, the encoder blocks or the final LayerNorm")
    blocks = (manifest.get("adapter") or {}).get("trainable_blocks")
    if not isinstance(blocks, int) or isinstance(blocks, bool) or not 0 <= blocks <= MAX_TRAINABLE_BLOCKS:
        raise ValueError(f"artifact adapter.trainable_blocks must be an int in 0..{MAX_TRAINABLE_BLOCKS}")


def _heatmap_targets(record: Mapping[str, Any], crop: tuple[int, int], heatmap: tuple[int, int]) -> tuple[Any, Any]:
    """Gaussian heatmap targets (17, H, W) and per-joint weights (17,) for a record in the processor's crop frame:
    the same centre/scale/warp the processor applies to the image, then the heatmap stride."""
    import numpy as np
    from transformers.models.vitpose.image_processing_vitpose import box_to_center_and_scale, get_warp_matrix

    crop_h, crop_w = crop
    hm_h, hm_w = heatmap
    x0, y0, x1, y1 = record["box"]
    center, scale = box_to_center_and_scale(np.array([x0, y0, x1 - x0, y1 - y0], dtype=np.float32), image_width=crop_w, image_height=crop_h)
    matrix = get_warp_matrix(0.0, center * 2.0, np.array([crop_w, crop_h]) - 1.0, scale * 200.0)
    stride_x, stride_y = crop_w / hm_w, crop_h / hm_h
    targets = np.zeros((len(KEYPOINT_NAMES), hm_h, hm_w), dtype=np.float32)
    weights = np.zeros(len(KEYPOINT_NAMES), dtype=np.float32)
    ys, xs = np.mgrid[0:hm_h, 0:hm_w]
    for index, name in enumerate(KEYPOINT_NAMES):
        if name not in record["keypoints"]:
            continue
        px, py = record["keypoints"][name]
        cx = (matrix[0, 0] * px + matrix[0, 1] * py + matrix[0, 2]) / stride_x
        cy = (matrix[1, 0] * px + matrix[1, 1] * py + matrix[1, 2]) / stride_y
        if not (0 <= cx < hm_w and 0 <= cy < hm_h):
            continue  # a labelled joint outside the padded crop is unweighted, as in the upstream target encoder
        targets[index] = np.exp(-((xs - cx) ** 2 + (ys - cy) ** 2) / (2.0 * HEATMAP_SIGMA**2))
        weights[index] = 1.0
    return targets, weights


@dataclass
class VitPoseKeypointPipeline:
    """``_detect(image, threshold)`` -> [{"box", "score"}] of persons; ``_pose(image, boxes)`` ->
    per box a list of 17 {"name", "x", "y", "score"} in KEYPOINT_NAMES order."""

    _detect: Callable[[Image.Image, float], list[dict[str, Any]]]
    _pose: Callable[[Image.Image, list[list[float]]], list[list[dict[str, Any]]]]
    device: str = "cpu"
    _model: Any = field(default=None, repr=False)
    _processor: Any = field(default=None, repr=False)
    weight_sha256: str | None = None
    adapter: dict[str, Any] | None = None
    # Pinned-base values of every tensor adapt() or load_artifact() has changed, kept the first time each is about
    # to change: every adaptation starts from the verified base, never from a previous run (review finding VTP-M3).
    _base_state: dict[str, Any] = field(default_factory=dict, repr=False)

    def _remember_base(self, names: Sequence[str]) -> None:
        model, _processor = self._require_model()
        state = model.state_dict()
        for name in names:
            if name not in self._base_state:
                self._base_state[name] = state[name].detach().clone()

    def restore_base(self) -> list[str]:
        """Put the pipeline back to the pinned base: copy the base values into every tensor an earlier adapt() or
        load_artifact() changed and drop the adapter record, so `estimate`, `evaluate` and a new adapt() read the
        untouched checkpoint. Returns the names of the restored tensors."""
        model, _processor = self._require_model()
        state = model.state_dict()
        # Only tensors whose live value differs from the base count as restored (as t5-base-text2text
        # 93a578f), so a repeat restore, or an adapt after one, does not claim an earlier run changed
        # anything.
        restored = sorted(n for n, base in self._base_state.items() if not bool((state[n] == base).all()))
        if restored:
            model.load_state_dict({name: self._base_state[name] for name in restored}, strict=False)
            model.eval()
        self.adapter = None
        return restored

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        detector_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> VitPoseKeypointPipeline:
        roots = {
            "pose": (
                Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR,
                MODEL_ID,
                MODEL_REVISION,
            ),
            "detector": (
                Path(detector_dir) if detector_dir is not None else DETECTOR_WEIGHTS_DIR,
                DETECTOR_MODEL_ID,
                DETECTOR_REVISION,
            ),
        }
        sources: dict[str, tuple[str, dict[str, Any]]] = {}
        for name, (root, model_id, revision) in roots.items():
            if (root / MANIFEST_NAME).is_file():
                _stage_missing(root, model_id, revision, allow_download, None)
                _verify_manifest(root, model_id, revision)
                sources[name] = (str(root), {"local_files_only": True})
            elif allow_download:
                sources[name] = (model_id, {"revision": revision})
            else:
                raise FileNotFoundError(
                    f"no verified {name} snapshot at {root} and allow_download=False; "
                    f"stage {model_id}@{revision} under {root}"
                )
        # Refuse invalid snapshots before importing model libraries.
        import numpy as np
        import torch
        from transformers import AutoImageProcessor, RTDetrForObjectDetection, VitPoseForPoseEstimation

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        det_src, det_kwargs = sources["detector"]
        det_processor = AutoImageProcessor.from_pretrained(det_src, trust_remote_code=False, **det_kwargs)
        detector = (
            RTDetrForObjectDetection.from_pretrained(
                det_src, trust_remote_code=False, use_pretrained_backbone=False, **det_kwargs
            )
            .to(resolved_device)
            .eval()
        )
        det_id2label = {int(k): v for k, v in detector.config.id2label.items()}
        pose_src, pose_kwargs = sources["pose"]
        pose_processor = AutoImageProcessor.from_pretrained(pose_src, trust_remote_code=False, **pose_kwargs)
        pose_model = (
            VitPoseForPoseEstimation.from_pretrained(pose_src, trust_remote_code=False, **pose_kwargs)
            .to(resolved_device)
            .eval()
        )
        for param in pose_model.parameters():
            param.requires_grad_(False)
        pose_id2label = {int(k): v for k, v in pose_model.config.id2label.items()}
        if tuple(pose_id2label[i] for i in range(len(pose_id2label))) != KEYPOINT_NAMES:
            raise RuntimeError("snapshot id2label does not match KEYPOINT_NAMES")

        def detect(image: Image.Image, threshold: float) -> list[dict[str, Any]]:
            inputs = det_processor(images=image, return_tensors="pt").to(resolved_device)
            with torch.inference_mode():
                outputs = detector(**inputs)
            result = det_processor.post_process_object_detection(
                outputs, threshold=threshold, target_sizes=[image.size[::-1]]
            )[0]
            return [
                {"box": [float(v) for v in box.tolist()], "score": float(score)}
                for box, label, score in zip(result["boxes"], result["labels"], result["scores"], strict=True)
                if det_id2label[int(label)] == DETECTOR_PERSON_LABEL
            ]

        def pose(image: Image.Image, boxes: list[list[float]]) -> list[list[dict[str, Any]]]:
            # The ViTPose processor takes boxes as xywh (the README converts xyxy -> xywh the same way).
            xywh = np.array([[b[0], b[1], b[2] - b[0], b[3] - b[1]] for b in boxes], dtype=np.float32)
            inputs = pose_processor(image, boxes=[xywh], return_tensors="pt").to(resolved_device)
            with torch.inference_mode():
                outputs = pose_model(**inputs)
            # threshold=0.0 keeps every joint; the pipeline applies keypoint_threshold itself so both
            # the kept and the dropped joints can be reported.
            persons = pose_processor.post_process_pose_estimation(outputs, boxes=[xywh], threshold=0.0)[0]
            out: list[list[dict[str, Any]]] = []
            for person in persons:
                joints = [
                    {
                        "name": pose_id2label[int(label)],
                        "x": float(kp[0]),
                        "y": float(kp[1]),
                        "score": float(score),
                    }
                    for kp, label, score in zip(
                        person["keypoints"], person["labels"], person["scores"], strict=True
                    )
                ]
                out.append(joints)
            return out

        return cls(detect, pose, resolved_device, pose_model, pose_processor, _weight_digest(roots["pose"][0]))

    def detect_people(self, image: Image.Image, *, threshold: float = DETECTION_THRESHOLD) -> dict[str, Any]:
        """Stage 1 alone: `person` boxes from the carried RT-DETR, sorted by score."""
        rgb, _boxes, det_t, _kp_t = _check_inputs(image, None, threshold, KEYPOINT_THRESHOLD)
        found = sorted(self._detect(rgb, det_t), key=lambda d: -d["score"])
        for det in found:
            if set(det) != {"box", "score"} or len(det["box"]) != 4:
                raise RuntimeError(f"detector returned a malformed detection: {det!r}")
        return {
            "persons": found[:MAX_PERSONS],
            "n_persons": min(len(found), MAX_PERSONS),
            "threshold": det_t,
            "width": rgb.width,
            "height": rgb.height,
            "detector_model_id": DETECTOR_MODEL_ID,
            "detector_revision": DETECTOR_REVISION,
        }

    def estimate(
        self,
        image: Image.Image,
        *,
        person_boxes: Sequence[Sequence[float]] | None = None,
        detection_threshold: float = DETECTION_THRESHOLD,
        keypoint_threshold: float = KEYPOINT_THRESHOLD,
    ) -> dict[str, Any]:
        """Keypoints for every person: boxes from the caller, or from stage 1 when none are given."""
        rgb, boxes, det_t, kp_t = _check_inputs(image, person_boxes, detection_threshold, keypoint_threshold)
        if boxes is None:
            detected = self.detect_people(rgb, threshold=det_t)["persons"]
            boxes = [d["box"] for d in detected]
            person_scores: list[float | None] = [d["score"] for d in detected]
            source = "detector"
        else:
            person_scores = [None] * len(boxes)
            source = "caller"
        poses: list[dict[str, Any]] = []
        if boxes:
            per_person = self._pose(rgb, boxes)
            if len(per_person) != len(boxes):
                raise RuntimeError(f"pose stage returned {len(per_person)} results for {len(boxes)} boxes")
            for box, score, joints in zip(boxes, person_scores, per_person, strict=True):
                if [j["name"] for j in joints] != list(KEYPOINT_NAMES):
                    raise RuntimeError("pose stage returned keypoints out of KEYPOINT_NAMES order")
                kept = [j for j in joints if j["score"] >= kp_t]
                poses.append(
                    {
                        "box": [float(v) for v in box],
                        "person_score": score,
                        "keypoints": kept,
                        "all_keypoints": joints,
                        "n_keypoints": len(kept),
                        "mean_keypoint_score": sum(j["score"] for j in joints) / len(joints),
                    }
                )
        return {
            "poses": poses,
            "n_persons": len(poses),
            "box_source": source,
            "detection_threshold": det_t,
            "keypoint_threshold": kp_t,
            "width": rgb.width,
            "height": rgb.height,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "detector_model_id": DETECTOR_MODEL_ID,
            "detector_revision": DETECTOR_REVISION,
        }

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._processor is None:
            raise RuntimeError("this pipeline has no loaded model (injected runner); use from_pretrained for evaluate/adapt")
        return self._model, self._processor


    def predict_keypoints(self, record: Mapping[str, Any]) -> dict[str, list[float]]:
        """All 17 joints (name -> [x, y]) for a record's person box, through `estimate` with the box as a caller box —
        the single-person policy the corpus measures use; heatmap scores are not thresholded."""
        result = self.estimate(record["image"], person_boxes=[record["box"]], keypoint_threshold=0.0)
        return {kp["name"]: [kp["x"], kp["y"]] for kp in result["poses"][0]["all_keypoints"]}


    def evaluate(self, records: Sequence[Mapping[str, Any]], *, progress: Callable[[int, int], None] | None = None) -> dict[str, Any]:
        """Predict every validated record's joints from its box and score them with `metrics.pose_metrics` (PCK and mean
        OKS, overall and per category)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import pose_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        predictions = []
        for i, record in enumerate(checked):
            predictions.append(self.predict_keypoints(record))
            if progress is not None:
                progress(i + 1, len(checked))
        metrics = pose_metrics(predictions, checked)
        metrics.update(
            {
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics


    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 8,
        lr: float = 5e-5,
        batch_size: int = 8,
        trainable_blocks: int = 2,
        seed: int = 0,
        progress: Callable[[Mapping[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning of the heatmap head and the last `trainable_blocks` encoder blocks (plus the final
        LayerNorm) on labelled persons: every record's box is affine-warped to the processor's crop exactly as at
        inference, the labelled joints become Gaussian heatmap targets (sigma `HEATMAP_SIGMA`) in the same frame, and the
        head's heatmaps are trained with the joint-weighted mean-squared error ViTPose was trained with. AdamW (no weight
        decay), gradient clipping at 1.0, seeded shuffling, no scheduler, no augmentation; the head's BatchNorm statistics
        stay frozen so the adapter is parameters-only. Epoch 0 records the frozen
        model's validation metrics; the epoch with the highest mean of validation PCK and OKS is kept (the final one
        without a validation split). Every call starts from the pinned base: tensors an earlier adapt() or
        load_artifact() changed are restored first, so epoch 0 is always the frozen model whatever ran before. On any
        exception the weights and the adapter record this call found are put back. The detector is untouched."""
        model, processor = self._require_model()  # refuse before importing torch
        import numpy as np
        import torch

        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 50:
            raise ValueError("epochs must be an int in 1..50")
        if not isinstance(lr, int | float) or not 0.0 < float(lr) <= 1e-2:
            raise ValueError("lr must be in (0, 1e-2]")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 1..64")
        if isinstance(trainable_blocks, bool) or not isinstance(trainable_blocks, int) or not 0 <= trainable_blocks <= MAX_TRAINABLE_BLOCKS:
            raise ValueError(f"trainable_blocks must be an int in 0..{MAX_TRAINABLE_BLOCKS}")
        train_checked = validate_dataset(train)["records"]
        val_checked = validate_dataset(val, min_records=1)["records"] if val is not None else None
        names = _trainable_names(model, trainable_blocks)
        name_set = set(names)
        device = torch.device(self.device)
        # The weights and adapter as this call found them: a failed call puts them back (the transactional
        # contract), while a successful one starts from the pinned base.
        previous_state = {
            k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set or k in self._base_state
        }
        previous_adapter = self.adapter
        restored = self.restore_base()
        self._remember_base(names)
        frozen_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
        cudnn_flags = (torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark)
        torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False  # repeatable on one device
        history: list[dict[str, Any]] = []
        started = time.perf_counter()

        def _val() -> dict[str, Any] | None:
            if val_checked is None:
                return None
            result = self.evaluate(val_checked)
            return {"pck": result["pck"], "oks": result["oks"], "score": (result["pck"] + result["oks"]) / 2.0, "n": result["n"]}

        try:
            for param in model.parameters():
                param.requires_grad_(False)
            params = []
            for name, param in model.named_parameters():
                if name in name_set:
                    param.requires_grad_(True)
                    params.append(param)
            n_trainable = sum(p.numel() for p in params)
            crop = (int(processor.size["height"]), int(processor.size["width"]))
            prepared = []
            with torch.no_grad():
                probe_record = train_checked[0]
                x0, y0, x1, y1 = probe_record["box"]
                probe = processor(probe_record["image"], boxes=[np.array([[x0, y0, x1 - x0, y1 - y0]], dtype=np.float32)], return_tensors="pt")
                heatmap = tuple(model(pixel_values=probe["pixel_values"].to(device)).heatmaps.shape[-2:])
            for record in train_checked:
                x0, y0, x1, y1 = record["box"]
                pixel_values = processor(record["image"], boxes=[np.array([[x0, y0, x1 - x0, y1 - y0]], dtype=np.float32)], return_tensors="pt")["pixel_values"][0]
                targets, weights = _heatmap_targets(record, crop, heatmap)
                prepared.append((pixel_values, torch.from_numpy(targets), torch.from_numpy(weights)))
            prep_seconds = round(time.perf_counter() - started, 3)
            entry = {"epoch": 0, "train_loss": None, "val": _val(), "note": "frozen model"}
            history.append(entry)
            if progress is not None:
                progress(entry)
            best_epoch, best_score = 0, (history[0]["val"] or {}).get("score", -1.0)
            best_state = frozen_state
            optimizer = torch.optim.AdamW(params, lr=float(lr), weight_decay=0.0)
            rng = random.Random(seed)
            torch.manual_seed(seed)
            for epoch in range(1, epochs + 1):
                model.train()
                for module in model.modules():  # BatchNorm statistics stay frozen: small batches, parameters-only adapter
                    if isinstance(module, torch.nn.modules.batchnorm._BatchNorm):
                        module.eval()
                order = list(range(len(prepared)))
                rng.shuffle(order)
                losses = []
                for start in range(0, len(order), batch_size):
                    batch = [prepared[k] for k in order[start : start + batch_size]]
                    pixel_values = torch.stack([b[0] for b in batch]).to(device)
                    targets = torch.stack([b[1] for b in batch]).to(device)
                    weights = torch.stack([b[2] for b in batch]).to(device)
                    heatmaps = model(pixel_values=pixel_values).heatmaps
                    loss = (((heatmaps - targets) ** 2).mean(dim=(2, 3)) * weights).sum() / weights.sum().clamp(min=1.0)
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    optimizer.step()
                    losses.append(float(loss.detach()))
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": _val()}
                history.append(entry)
                if progress is not None:
                    progress(entry)
                if val_checked is None or entry["val"]["score"] > best_score:
                    best_epoch, best_score = epoch, (entry["val"] or {}).get("score", -1.0)
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
            model.load_state_dict(best_state, strict=False)
            for param in model.parameters():
                param.requires_grad_(False)
            model.eval()
        except BaseException:
            model.load_state_dict(previous_state, strict=False)
            for param in model.parameters():
                param.requires_grad_(False)
            model.eval()
            self.adapter = previous_adapter
            raise
        finally:
            torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = cudnn_flags
        self.adapter = {
            "trainable_blocks": trainable_blocks,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "batch_size": batch_size,
            "best_epoch": best_epoch,
            "selection": "highest mean of validation PCK and OKS" if val_checked is not None else "final epoch (no validation split)",
            "loss": f"joint-weighted mean-squared error against Gaussian heatmap targets (sigma {HEATMAP_SIGMA}) in the {heatmap[0]}x{heatmap[1]} heatmap frame",
            "lr": float(lr),
            "seed": seed,
            "started_from": "pinned base"
            + (f" (restored {len(restored)} tensors changed by an earlier run)" if restored else ""),
            "n_train": len(train_checked),
            "n_val": len(val_checked) if val_checked is not None else 0,
            "preparation_seconds": prep_seconds,
            "history": history,
            "seconds": round(time.perf_counter() - started, 3),
        }
        return dict(self.adapter)


    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the trained tensors as safetensors plus a manifest naming the base, the digests and the training
        configuration. Requires a prior `adapt`."""
        model, _processor = self._require_model()  # refuse before importing torch
        import torch
        from safetensors.torch import save_file

        if self.adapter is None:
            raise RuntimeError("nothing to save: call adapt() first")
        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = list(self.adapter["trainable_names"])
        state = model.state_dict()
        tensors = {name: state[name].detach().cpu().contiguous() for name in names}
        weights = out / ADAPTER_WEIGHTS
        save_file(tensors, str(weights), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "version": ARTIFACT_VERSION,
            "base": {"model_id": MODEL_ID, "revision": MODEL_REVISION, "weight_file": WEIGHT_FILE, "weight_sha256": self.weight_sha256},
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": names,
            "files": [{"path": ADAPTER_WEIGHTS, "bytes": weights.stat().st_size, "sha256": _sha256(weights)}],
            "torch": torch.__version__,
            "metadata": dict(metadata or {}),
        }
        with open(out / ADAPTER_MANIFEST, "w", encoding="utf-8") as handle:
            json.dump(manifest, handle, indent=2, ensure_ascii=False)
        return out


    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Overlay a saved adapter onto this (freshly loaded) pipeline after checking its manifest, digest and exact
        tensor set. Refuses tensors outside the recorded head + block scope."""
        model, _processor = self._require_model()  # refuse before importing safetensors
        from safetensors.torch import load_file

        artifact = Path(artifact_dir)
        manifest_path = artifact / ADAPTER_MANIFEST
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest missing: {manifest_path}")
        with open(manifest_path, encoding="utf-8") as handle:
            manifest = json.load(handle)
        _check_artifact_manifest(manifest, artifact, self.weight_sha256 or "")
        expected = _trainable_names(model, int(manifest["adapter"]["trainable_blocks"]))
        if sorted(manifest["tensors"]) != sorted(expected):
            raise ValueError("artifact tensor set does not match its recorded configuration")
        tensors = load_file(str(artifact / ADAPTER_WEIGHTS))
        if sorted(tensors) != sorted(expected):
            raise ValueError("artifact tensor names differ from the manifest")
        state = model.state_dict()
        for name, tensor in tensors.items():
            if tuple(tensor.shape) != tuple(state[name].shape):
                raise ValueError(f"artifact tensor {name} has shape {tuple(tensor.shape)}, base has {tuple(state[name].shape)}")
        self.restore_base()
        self._remember_base(sorted(tensors))
        model.load_state_dict({k: v.to(state[k].device, state[k].dtype) for k, v in tensors.items()}, strict=False)
        model.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": expected, "history": manifest.get("history", [])}
        return dict(self.adapter)


    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        detector_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> VitPoseKeypointPipeline:
        """Load the verified base snapshots, then overlay the adapter (verified before deserialising)."""
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, detector_dir=detector_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe

**Module 2/3:** `src/vitpose_keypoint_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Corpus-level keypoint measures and two non-neural baselines, in numpy.

``pose_metrics`` scores one predicted joint set per record against ``record['keypoints']``: **PCK** — the share of
labelled joints within ``PCK_FRACTION`` (0.1) of the box's longest side of their reference, the package's own
``keypoint_pck`` convention — and **mean OKS** — the COCO object-keypoint-similarity kernel averaged over the
labelled joints with the COCO per-joint sigmas and the box area as the object scale (the primitive of the COCO
keypoint AP, without its thresholds) — overall and per ``category``. The baselines answer from the box alone: every
joint at the box centre, or every joint at the training split's mean position relative to the box.
"""
# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width

from __future__ import annotations

from collections.abc import Mapping, Sequence
from typing import Any

import numpy as np

# standalone rewrite (build_notebook.py): `from .pipeline import KEYPOINT_NAMES, PCK_FRACTION` removed — names are kernel globals defined by the carried modules

METRIC_DEFINITIONS = {
    "pck": f"share of labelled joints whose prediction lies within PCK_FRACTION ({PCK_FRACTION}) of the person box's longest side of the reference; in 0..1, higher is better",
    "oks": "mean over labelled joints of exp(-d^2 / (2 * area * (2 * sigma_joint)^2)) with the COCO keypoint sigmas and the box area — the COCO OKS kernel per joint; in 0..1, higher is better",
}
# COCO per-joint sigmas (pycocotools), in KEYPOINT_NAMES order.
OKS_SIGMAS = {
    name: value
    for name, value in zip(
        KEYPOINT_NAMES,
        (0.026, 0.025, 0.025, 0.035, 0.035, 0.079, 0.079, 0.072, 0.072, 0.062, 0.062, 0.107, 0.107, 0.087, 0.087, 0.089, 0.089),
        strict=True,
    )
}


def _joint_errors(predicted: Mapping[str, Sequence[float]], record: Mapping[str, Any]) -> tuple[np.ndarray, list[str]]:
    names = list(record["keypoints"])
    errors = []
    for name in names:
        rx, ry = record["keypoints"][name]
        if name not in predicted:
            raise ValueError(f"prediction for {record['id']!r} lacks joint {name!r}")
        px, py = predicted[name][0], predicted[name][1]
        errors.append(float(np.hypot(px - rx, py - ry)))
    return np.asarray(errors, dtype=np.float64), names


def pose_metrics(predictions: Sequence[Mapping[str, Sequence[float]]], records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Score one joint mapping (name -> [x, y], all 17 joints) per record against `record['keypoints']`; per-record
    rows, means overall and per category. Raises when the lengths differ or nothing is scored."""
    if len(predictions) != len(records) or not predictions:
        raise ValueError("predictions and records must be non-empty and the same length")
    rows = []
    for predicted, record in zip(predictions, records, strict=True):
        errors, names = _joint_errors(predicted, record)
        x0, y0, x1, y1 = record["box"]
        radius = PCK_FRACTION * max(x1 - x0, y1 - y0)
        area = max(1.0, (x1 - x0) * (y1 - y0))
        sigmas = np.asarray([OKS_SIGMAS[n] for n in names])
        oks = float(np.mean(np.exp(-(errors**2) / (2.0 * area * (2.0 * sigmas) ** 2))))
        rows.append(
            {
                "id": record["id"],
                "category": record.get("category"),
                "pck": float(np.mean(errors <= radius)),
                "oks": oks,
                "mean_error_px": float(errors.mean()),
                "n_joints": len(names),
            }
        )

    def _mean(items: Sequence[Mapping[str, Any]]) -> dict[str, float | int]:
        return {"n": len(items), "pck": float(np.mean([r["pck"] for r in items])), "oks": float(np.mean([r["oks"] for r in items]))}

    categories = sorted({r["category"] for r in rows if r["category"] is not None})
    return {
        **_mean(rows),
        "per_category": {c: _mean([r for r in rows if r["category"] == c]) for c in categories},
        "per_record": rows,
        "definitions": dict(METRIC_DEFINITIONS),
    }


def box_centre_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Every joint at the centre of the person box — the floor."""
    predictions = []
    for record in records:
        x0, y0, x1, y1 = record["box"]
        centre = [(x0 + x1) / 2.0, (y0 + y1) / 2.0]
        predictions.append({name: list(centre) for name in KEYPOINT_NAMES})
    return {**pose_metrics(predictions, records), "baseline": "every joint at the box centre"}


def mean_pose_baseline(train: Sequence[Mapping[str, Any]], records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Every joint at its mean position relative to the box over the training split — the layout prior a box alone
    gives away."""
    if not train:
        raise ValueError("train must not be empty")
    sums: dict[str, list[float]] = {name: [0.0, 0.0, 0.0] for name in KEYPOINT_NAMES}
    for record in train:
        x0, y0, x1, y1 = record["box"]
        w, h = max(1e-6, x1 - x0), max(1e-6, y1 - y0)
        for name, (px, py) in record["keypoints"].items():
            sums[name][0] += (px - x0) / w
            sums[name][1] += (py - y0) / h
            sums[name][2] += 1.0
    mean = {name: ((s[0] / s[2], s[1] / s[2]) if s[2] else (0.5, 0.5)) for name, s in sums.items()}
    predictions = []
    for record in records:
        x0, y0, x1, y1 = record["box"]
        predictions.append({name: [x0 + mean[name][0] * (x1 - x0), y0 + mean[name][1] * (y1 - y0)] for name in KEYPOINT_NAMES})
    return {**pose_metrics(predictions, records), "baseline": "every joint at the training split's mean position relative to the box"}

**Module 3/3:** `src/vitpose_keypoint_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Labelled keypoint datasets for the adaptation contract: the digest-pinned COCO val2017 person sample, a stated
low-resolution degradation, the record contract and its structural validation, image-disjoint splitting, and the
BYOD loader.

A record is ``{id, image, box, keypoints}`` where ``image`` is a PIL image (sides within the pipeline's ceilings),
``box`` the person's ``[x0, y0, x1, y1]`` box inside it and ``keypoints`` a mapping from COCO joint name
(``KEYPOINT_NAMES``) to ``[x, y]`` for every labelled joint (at least ``MIN_LABELLED`` of the 17). An optional
``category`` (free text, at most 32 characters) is carried into the per-category breakdown; the sample sets it from
the number of labelled joints (``full`` 16-17, ``partial`` 14-15, ``sparse`` 12-13).

The default sample is 300 COCO val2017 photographs (Lin et al. 2014) whose Flickr licence is CC BY 2.0 or
CC BY-SA 2.0 (COCO licence ids 4 and 5), each holding at least one non-crowd person with >= 12 labelled joints and a
box at least 120 px tall — 498 persons in all. Every photograph is pinned by byte size and SHA-256 and fetched from
the COCO image host at run time; its Flickr page and licence are kept in the record. The 2017 keypoint annotations
(CC BY 4.0) travel inline in ``SAMPLE_IMAGES``. ``degrade`` turns each photograph into the input the adaptation is
about: the whole image is downscaled so the person's box is ``TARGET_HEIGHT`` px tall and re-encoded as JPEG at
``JPEG_QUALITY`` — a distant, low-quality person — with the box and joints scaled to match.
"""
# ruff: noqa: E501  -- record and pin literals are kept on single lines

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import re
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

from PIL import Image

# standalone rewrite (build_notebook.py): `from .pipeline import KEYPOINT_NAMES, MODEL_ID, _check_boxes, validate_image` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "COCO 2017 validation persons (Flickr CC BY 2.0 / CC BY-SA 2.0 photographs; keypoint annotations CC BY 4.0)"
CORPUS_RELEASE = "COCO 2017 (annotations_trainval2017 person_keypoints_val2017.json; images val2017)"
CORPUS_BASE_URL = "http://images.cocodataset.org/val2017/"
CORPUS_LICENSE = "photographs: each record's Flickr licence (CC BY 2.0 or CC BY-SA 2.0, ids 4 and 5 of the COCO licence table); annotations: CC BY 4.0 (COCO Consortium)"
CORPUS_LICENSES = {4: ("Attribution License", "http://creativecommons.org/licenses/by/2.0/"), 5: ("Attribution-ShareAlike License", "http://creativecommons.org/licenses/by-sa/2.0/")}
CORPUS_BYTES = 52_141_813
CORPUS_IMAGES = 300
CORPUS_PERSONS = 498
DEFAULT_CACHE_DIR = Path("weights") / "coco-val-persons"

TARGET_HEIGHT = 40  # px; the person's box height after the stated downscaling
JPEG_QUALITY = 30  # the JPEG re-encoding after downscaling
MIN_LABELLED = 12  # labelled joints a record needs
MIN_BOX_HEIGHT = 120  # px in the served photograph (before degradation)
SAMPLE_SPLIT = {"train": 180, "validation": 45, "test": 75}  # images; persons follow their image
SAMPLE_SEED = 42
SAMPLE_DIGEST = "6b4553ddb185c06394bccebf1bd870a1d7b1d63c4f585ef3e40ea12f51205d4f"  # dataset_digest over the three default (degraded) splits together; tests pin it
MIN_RECORDS = 8
MAX_RECORDS = 5_000
MAX_CATEGORY_CHARS = 32
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")
_BUCKETS = ((16, "full"), (14, "partial"), (0, "sparse"))

# (image_id, bytes, sha256, width, height, licence_id, flickr_url, ((annotation_id, "x,y,w,h", "x y v ..." x17), ...))
SAMPLE_IMAGES: tuple[tuple[Any, ...], ...] = (
    (785, 133674, "83981537a7baeafbeb9c8cb67b3484dc26433f574b3685d021fa537e277e4726", 640, 425, 4, "http://farm8.staticflickr.com/7015/6795644157_f019453ae7_z.jpg", ((442619, "280.79,44.73,218.70,346.68", "367 81 2 374 73 2 360 75 2 386 78 2 356 81 2 399 108 2 358 129 2 433 142 2 341 159 2 449 165 2 309 178 2 424 203 2 393 214 2 429 294 2 367 273 2 466 362 2 396 341 2"),)),
    (872, 317669, "c2aa138ee3a59b057a7ba6fc5a6a18e62af531aa7dab78a7bfd33c1cd7e55eb6", 621, 640, 4, "http://farm9.staticflickr.com/8447/7805810128_605424213d_z.jpg", ((559508, "145.26,100.67,291.95,457.35", "367 138 2 0 0 0 360 134 2 0 0 0 338 144 2 0 0 0 336 193 2 0 0 0 391 216 2 0 0 0 423 191 2 349 319 2 326 319 1 347 432 1 277 428 1 313 526 2 187 489 2"), (560228, "163.73,126.42,265.69,480.40", "211 187 1 215 179 2 0 0 0 241 175 2 0 0 0 280 228 2 236 232 2 326 300 2 245 292 2 289 358 2 195 331 2 297 354 2 240 355 1 294 480 2 227 470 2 376 410 2 245 570 2"),)),
    (885, 111441, "1c67b783f78b18ddb5dd20aaba4d4aacaf98e5277051688a96da096afa9cdf1f", 640, 427, 4, "http://farm4.staticflickr.com/3715/9639200419_ee41490b2a_z.jpg", ((439117, "277.31,189.99,140.09,208.22", "374 216 2 379 214 2 370 212 2 0 0 0 359 208 2 369 227 2 340 219 2 372 247 2 348 271 2 392 275 2 383 289 2 331 266 2 303 250 2 343 312 2 287 289 2 322 380 2 296 245 2"), (514577, "281.02,89.81,111.83,169.55", "377 110 2 381 107 2 373 106 2 385 109 2 366 105 1 387 123 2 347 119 2 380 149 2 330 146 2 373 174 2 320 175 2 358 144 2 325 141 2 371 183 2 320 190 2 387 239 1 301 247 1"),)),
    (1000, 321136, "24bb77a31928404e45a0454b06f6a0bd54a8db103590c9d7917288a1e0269f05", 640, 480, 4, "http://farm5.staticflickr.com/4115/4906536419_6113bd7de4_z.jpg", ((1234574, "115.16,152.13,83.23,228.41", "162 174 2 167 170 2 157 170 2 174 173 2 151 174 2 182 198 2 144 197 2 198 220 1 121 219 2 187 240 2 136 244 2 173 276 2 152 275 2 172 315 2 150 314 2 177 354 2 149 352 2"), (1259139, "265.33,95.86,88.92,315.88", "311 127 2 318 120 2 305 120 2 328 123 2 298 124 2 341 162 2 284 160 2 351 210 1 271 204 2 335 218 2 278 212 2 327 251 2 289 250 1 323 318 2 294 312 2 321 387 2 295 379 2"), (1269164, "209.23,174.64,99.63,249.08", "253 199 2 257 193 2 247 195 2 265 196 2 236 203 2 267 230 2 227 229 2 282 260 2 212 258 2 294 233 2 218 263 2 264 301 2 237 299 2 271 349 2 235 346 2 270 398 2 234 395 2"), (1303041, "504.67,191.95,135.33,288.05", "562 239 2 571 232 2 555 229 2 594 236 2 0 0 0 599 284 2 543 278 2 636 335 2 526 321 2 599 362 2 0 0 0 567 397 2 531 387 2 572 467 2 532 445 2 0 0 0 0 0 0"), (1304800, "410.20,208.53,114.90,271.47", "475 257 2 487 248 2 468 245 2 502 252 2 458 243 1 509 304 2 444 290 2 510 365 2 425 332 2 474 401 2 429 384 2 477 425 2 442 415 2 0 0 0 444 467 2 0 0 0 0 0 0"), (1306228, "380.85,159.91,88.44,319.25", "426 200 2 433 192 2 423 192 2 454 194 2 0 0 0 461 236 1 420 233 2 0 0 0 405 288 2 0 0 0 393 338 2 450 329 1 415 325 1 444 393 1 413 393 2 440 470 1 414 467 2"), (1311471, "330.07,154.25,83.05,313.88", "378 184 2 386 178 2 374 178 2 397 182 2 366 181 2 409 222 2 352 218 2 412 284 1 345 268 2 391 324 2 339 317 2 387 311 2 357 306 2 381 371 2 346 375 2 377 445 2 364 433 2"), (1318642, "183.40,121.34,95.77,272.75", "238 148 2 245 143 2 232 142 2 253 146 2 226 141 1 260 172 2 217 171 2 279 199 1 201 221 2 271 217 2 200 261 2 257 262 1 219 259 1 248 322 1 201 317 1 245 367 2 200 368 2"), (1755199, "52.14,185.12,59.26,212.53", "89 211 2 90 204 2 82 207 2 0 0 0 69 213 2 92 232 2 64 234 2 97 260 2 57 266 2 103 285 2 66 293 2 92 290 2 66 291 2 90 328 2 67 332 1 94 366 1 68 373 1"),)),
    (2153, 136034, "8795ff243cf20405bd442631d5d1cb7e60665dd459c8d57f5764333808dd49fd", 640, 480, 5, "http://farm3.staticflickr.com/2220/2535221304_39d9542f76_z.jpg", ((436576, "283.40,274.77,182.91,170.27", "0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 353 309 2 395 307 2 330 337 2 416 337 2 318 341 2 432 367 2 367 361 2 399 361 2 326 386 2 431 387 2 307 432 2 453 427 2"), (538502, "371.62,207.81,62.30,197.28", "0 0 0 0 0 0 0 0 0 397 225 2 0 0 0 416 243 2 396 243 2 408 252 2 375 251 2 393 236 2 376 239 2 414 299 2 400 301 2 408 350 1 402 345 1 418 393 1 411 383 1"), (2166351, "275.40,302.20,103.79,134.55", "0 0 0 0 0 0 0 0 0 317 321 2 0 0 0 299 339 2 349 340 2 284 371 2 359 385 1 0 0 0 358 410 2 308 404 1 338 406 2 291 376 2 360 380 1 296 419 2 354 423 2"),)),
    (2473, 95783, "2f1bfa7634a64f7999ba5488ec904ad1f85bbc63d0fe4b9fe35f52abbf8c5f4f", 640, 427, 4, "http://farm5.staticflickr.com/4027/4377710907_7b97c2464d_z.jpg", ((440079, "243.73,93.08,130.49,150.65", "0 0 0 0 0 0 0 0 0 302 106 2 313 106 2 295 117 2 325 119 2 277 125 2 345 128 2 254 121 2 362 122 2 301 161 2 316 163 2 298 192 2 316 193 2 299 236 2 316 225 2"),)),
    (5001, 211700, "99600c56dc25c72dc1cc1d1f2b09aa33b97142560b54d034fa83a38fcf3c5b0f", 640, 480, 4, "http://farm6.staticflickr.com/5298/5457068434_0929d78491_z.jpg", ((1220394, "225.44,255.64,127.28,224.36", "278 299 2 283 293 2 0 0 0 304 297 2 0 0 0 326 347 2 279 318 2 329 385 1 259 333 2 300 402 2 237 297 2 312 415 2 284 408 2 318 460 2 304 456 2 0 0 0 0 0 0"), (1259834, "77.25,244.56,113.30,235.26", "144 283 2 146 278 2 135 282 2 0 0 0 111 280 2 152 303 2 116 321 2 163 329 2 107 371 1 175 349 2 85 344 2 168 408 2 141 422 2 0 0 0 149 471 2 0 0 0 0 0 0"), (1263773, "119.23,81.44,69.26,188.17", "166 120 2 170 111 2 156 114 2 0 0 0 148 120 2 182 139 1 131 145 2 0 0 0 146 193 2 0 0 0 176 195 2 180 230 2 161 235 2 177 300 1 165 297 1 0 0 0 0 0 0"), (1327525, "211.45,88.41,123.67,193.01", "284 142 2 291 134 2 274 134 2 311 125 2 0 0 0 341 163 1 270 167 2 350 243 1 245 203 2 330 296 1 201 238 1 351 269 1 298 277 1 339 355 1 0 0 0 0 0 0 0 0 0"), (1328232, "354.88,91.69,162.87,378.60", "423 154 2 434 148 2 417 146 2 458 138 2 0 0 0 494 181 2 389 180 2 510 245 2 391 253 2 490 309 2 432 312 2 465 307 2 395 304 2 435 391 2 406 387 1 0 0 0 0 0 0"),)),
    (7281, 137409, "57f3508e600d3370320487dde8665f208f11612a11caac0801c7f5fc19188ac3", 640, 361, 4, "http://farm5.staticflickr.com/4112/5022207571_74aab92fd5_z.jpg", ((183349, "344.85,132.99,128.41,166.36", "442 150 2 443 147 2 438 148 2 0 0 0 0 0 0 447 167 2 421 160 2 461 163 2 400 165 1 466 156 2 401 178 2 420 212 2 406 207 2 421 237 2 381 236 1 427 278 2 358 265 2"),)),
    (7511, 135698, "105e70c573b87d7e3f034e1dc08cedcced39be4e52197139f52616c920dda3d7", 640, 480, 5, "http://farm1.staticflickr.com/51/173980729_349afaca85_z.jpg", ((445593, "386.16,240.54,122.96,231.91", "0 0 0 0 0 0 0 0 0 435 267 2 460 265 2 427 296 2 470 291 2 404 320 2 503 311 2 389 310 1 483 280 2 437 371 2 466 368 2 404 416 2 454 414 2 416 469 2 455 469 2"),)),
    (8021, 180088, "235537407867a7b894e78b3aecb55b33b14b64cbb23ca376a00d34c62a1a63a6", 640, 480, 4, "http://farm7.staticflickr.com/6117/6240308255_77782d0723_z.jpg", ((544180, "244.30,101.17,196.96,266.90", "316 141 2 326 133 2 306 135 2 343 147 2 292 149 2 380 206 2 278 213 2 420 308 2 261 320 2 375 318 2 281 287 2 389 380 1 311 380 1 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (8690, 246519, "feca143d5e249b9fcaca7090c2400125ee19c7ad883c11fffe55e8bb451e7429", 640, 480, 5, "http://farm4.staticflickr.com/3753/9867545575_fa972e6c4a_z.jpg", ((188617, "136.70,18.60,214.82,378.24", "307 125 2 323 110 2 291 104 2 0 0 0 255 94 2 332 166 2 241 167 2 320 218 2 181 271 2 328 282 1 203 346 2 289 292 2 218 284 2 0 0 0 0 0 0 0 0 0 0 0 0"), (200151, "285.84,73.35,230.83,323.59", "443 164 2 460 148 2 423 148 2 475 147 2 393 151 2 484 185 2 392 213 2 490 251 2 337 283 1 465 314 1 298 348 1 482 320 1 421 328 1 0 0 0 0 0 0 0 0 0 0 0 0"), (200974, "74.25,10.38,101.69,291.63", "139 37 2 0 0 0 135 34 2 0 0 0 120 34 1 138 67 2 117 64 2 166 104 2 155 119 2 152 108 2 171 109 2 154 160 2 143 165 2 136 226 1 148 232 1 89 265 2 148 291 1"),)),
    (9483, 116766, "c7e6e15f4db067c3e09ab8605759be9d850e2d5c45ea4dfb9cda8f5d85308ea4", 640, 480, 4, "http://farm4.staticflickr.com/3179/2986591710_d76622fdf0_z.jpg", ((219454, "36.44,62.46,140.70,417.54", "112 102 2 118 93 2 104 95 2 133 99 2 86 102 2 148 154 2 56 154 2 169 209 2 58 238 2 151 263 2 112 273 2 140 289 2 91 289 2 133 394 2 80 400 2 126 469 2 0 0 0"),)),
    (10707, 121031, "a361abb7579193709ef3debbec6298bf0c540bece429c4aadb19be82e04ab6a0", 640, 480, 5, "http://farm1.staticflickr.com/147/434840964_a9c5bc3b58_z.jpg", ((193096, "228.16,147.57,168.97,309.95", "336 204 2 345 195 2 328 194 2 361 189 2 0 0 0 393 223 2 297 213 2 393 280 1 251 258 2 347 299 2 273 268 2 353 298 2 311 286 2 332 322 2 242 311 2 330 409 2 286 395 1"), (221149, "347.01,187.53,125.44,264.78", "409 247 2 419 238 2 405 236 2 444 233 2 0 0 0 465 270 2 405 259 2 453 346 2 380 322 2 396 343 2 366 352 2 463 355 2 410 348 2 447 374 1 380 354 2 448 471 1 398 426 2"), (233159, "422.83,2.16,177.98,471.37", "491 71 2 499 60 2 0 0 0 537 51 2 0 0 0 552 122 2 491 121 2 505 192 2 448 189 2 487 105 2 429 119 2 526 283 2 485 275 2 491 432 2 498 426 2 0 0 0 0 0 0"),)),
    (11511, 188164, "4e24ed1528891621658ff01d23e8a74735cd18720d41f7fd2264044497ad2a0e", 640, 464, 5, "http://farm8.staticflickr.com/7124/7556029168_519acfdf09_z.jpg", ((494713, "35.39,44.50,50.22,145.84", "0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 66 75 2 44 80 2 70 95 2 45 99 2 55 93 2 65 99 2 72 116 2 52 119 2 73 147 2 65 149 2 72 182 2 74 183 2"), (2165380, "1.27,51.73,61.55,162.56", "0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 34 95 2 15 96 2 38 112 2 20 117 2 43 129 2 34 133 2 35 143 2 21 146 2 34 178 2 33 175 2 30 205 2 44 208 2"),)),
    (12280, 169019, "158ef0b15ff392039468263fa18735f81b39de9e8a7753e1e1aba7037262da56", 480, 640, 4, "http://farm9.staticflickr.com/8516/8544258219_95e49e415e_z.jpg", ((426225, "143.13,241.34,50.06,128.62", "156 257 2 158 254 2 0 0 0 162 256 2 0 0 0 171 270 2 165 269 2 172 293 2 0 0 0 172 312 2 0 0 0 169 310 2 164 308 2 172 330 2 158 329 2 185 353 2 163 359 2"),)),
    (13291, 124885, "1e46187ec49947bfd8dd22f4d774bddfe3e29ca863fe09530516598998b8e640", 500, 335, 4, "http://farm4.staticflickr.com/3394/3491087208_8efdfeb6d4_z.jpg", ((442908, "288.33,138.57,210.78,161.85", "339 162 2 346 159 2 335 158 2 354 167 2 329 164 1 366 185 2 329 188 2 338 205 2 301 198 2 308 193 2 281 197 1 384 239 2 361 240 2 425 283 2 330 226 2 480 269 2 338 276 2"), (475509, "238.23,135.43,80.81,147.12", "249 153 2 252 148 2 245 151 2 261 150 2 0 0 0 271 160 2 252 177 2 288 165 2 252 211 2 292 186 1 255 198 2 306 205 1 282 209 2 284 218 2 261 223 2 292 261 2 286 248 2"), (547120, "41.40,66.25,124.97,232.62", "107 93 2 112 91 2 102 88 2 0 0 0 87 86 2 125 113 2 70 101 2 143 141 2 52 129 2 141 163 2 73 157 2 117 179 2 79 177 2 151 205 2 69 235 2 145 257 2 83 275 2"), (2155598, "159.23,136.61,85.28,130.52", "223 158 2 229 155 2 222 151 2 234 159 2 210 151 1 230 179 2 198 163 2 0 0 0 175 179 2 0 0 0 179 204 2 215 216 2 192 210 2 241 215 1 165 226 2 225 247 2 186 247 2"),)),
    (13659, 215292, "b357d19dee467587a8788ac4e5b45677f4fde1b62726c690795e188ae75d822b", 640, 480, 4, "http://farm7.staticflickr.com/6144/5987177856_785814a990_z.jpg", ((201643, "406.65,9.71,85.22,141.30", "460 41 2 464 36 2 455 37 2 0 0 0 447 37 2 472 59 2 431 60 2 482 87 2 413 85 2 477 110 2 429 80 2 463 125 2 436 124 2 0 0 0 0 0 0 0 0 0 0 0 0"), (212964, "331.36,94.09,130.34,171.29", "406 123 2 405 118 2 0 0 0 396 119 2 0 0 0 385 130 2 364 119 2 414 136 2 366 158 2 438 150 2 368 189 2 364 175 2 345 184 2 414 188 2 409 196 2 395 240 2 414 250 2"),)),
    (14439, 195229, "fc23e512cfe33e0b7f5e3445dc4189eb053fed6a16964b2454775a6df3fa5eba", 640, 404, 4, "http://farm4.staticflickr.com/3544/3506151605_2a3530b2f9_z.jpg", ((1213473, "413.08,89.88,94.42,198.82", "454 107 2 0 0 0 450 103 2 0 0 0 446 105 2 424 131 2 449 133 2 0 0 0 473 147 2 0 0 0 493 129 2 436 185 1 438 186 1 457 224 2 443 226 2 464 271 2 427 268 2"), (1733380, "505.50,114.08,92.38,157.64", "534 128 2 537 126 2 531 127 2 544 130 2 526 130 2 546 145 2 524 149 2 537 162 2 519 172 2 522 152 2 513 153 2 549 195 2 530 197 2 560 226 2 525 231 2 587 252 2 524 262 2"),)),
    (18491, 109544, "195369c44ad1810c524d5523a556356b2ee384310d7493310ccb8165c42767ed", 640, 427, 5, "http://farm3.staticflickr.com/2134/5814907667_496ae24446_z.jpg", ((1235753, "243.01,184.65,125.80,120.60", "0 0 0 0 0 0 284 210 2 0 0 0 276 206 2 270 214 2 259 214 2 278 224 2 259 242 2 284 228 2 284 254 2 279 257 2 270 265 2 314 267 2 310 261 2 344 289 2 328 291 2"), (1247297, "301.30,186.15,120.90,126.66", "334 207 2 337 203 2 333 204 2 347 204 2 0 0 0 351 213 2 360 226 2 343 214 2 0 0 0 323 222 2 0 0 0 391 246 2 393 247 2 350 255 2 369 274 2 362 296 1 404 293 2"), (1250410, "330.11,69.96,60.81,137.59", "353 85 2 357 81 2 348 82 2 0 0 0 0 0 0 366 99 2 341 99 1 370 123 2 337 122 2 369 144 2 337 140 2 360 140 2 346 140 2 373 170 2 339 165 2 383 196 2 336 199 1"), (1266544, "274.91,68.91,56.59,147.50", "295 84 2 298 81 2 293 82 2 303 82 2 0 0 0 313 97 2 288 99 2 322 119 2 285 120 2 327 139 2 279 130 2 314 139 2 294 140 2 316 176 2 292 175 2 321 203 2 289 204 1"), (1277912, "69.70,95.37,53.10,122.10", "91 108 2 93 106 2 88 106 2 0 0 0 84 108 2 101 121 2 81 123 2 103 140 2 74 139 2 105 153 2 74 156 2 100 152 2 87 153 2 107 178 2 85 179 2 110 201 2 80 205 2"),)),
    (19432, 223434, "fa54de5a592c892ed3aceb11ebacff27abad01dc3bcef308943b097c1d65c0f0", 640, 480, 5, "http://farm6.staticflickr.com/5131/5533564723_6f1633b85e_z.jpg", ((428692, "55.26,26.82,153.86,404.15", "145 74 2 147 66 2 137 69 2 0 0 0 115 68 2 138 92 2 82 116 2 151 128 2 101 180 2 181 178 2 135 233 2 129 200 2 94 213 2 146 294 2 105 302 2 116 390 2 74 406 2"),)),
    (21167, 49463, "b77df96f91ed9a0842559ea8a238d602d4a09069b9129c43a9d62a27361df156", 426, 640, 4, "http://farm4.staticflickr.com/3469/3397730036_36219f4c6e_z.jpg", ((480060, "21.52,114.73,202.34,510.85", "125 186 2 139 177 2 114 173 2 171 181 2 104 177 2 185 252 2 79 249 2 0 0 0 58 344 2 228 290 1 46 456 2 166 433 2 90 431 2 166 584 2 96 580 2 0 0 0 0 0 0"), (1708034, "187.98,38.89,238.02,601.11", "265 117 2 281 98 2 262 100 2 324 98 2 0 0 0 382 189 2 231 195 2 409 335 2 206 296 2 375 436 2 228 331 2 345 431 2 246 424 2 350 603 2 230 595 2 0 0 0 0 0 0"),)),
    (22479, 97567, "9f09af60f0a4f214a8b34ff6badbd25111cd8438c39097197a629255408cdb75", 640, 605, 5, "http://farm4.staticflickr.com/3447/3880630600_2c837dbf23_z.jpg", ((1716695, "34.96,1.30,403.00,455.95", "305 95 2 312 81 2 293 78 2 0 0 0 252 63 2 316 147 2 184 79 2 358 243 2 114 104 2 404 323 2 64 142 2 212 287 2 143 268 2 348 238 2 224 149 2 379 372 1 287 321 1"),)),
    (30504, 226756, "6a07fb2882010ba47140c43ffd7cd4a51331f197666ea9837b43bc077e88f552", 480, 640, 4, "http://farm6.staticflickr.com/5088/5320611177_13273d2c90_z.jpg", ((441780, "189.03,109.81,205.80,381.17", "0 0 0 313 139 2 0 0 0 326 134 2 0 0 0 306 150 2 358 167 2 267 202 2 0 0 0 229 245 2 0 0 0 294 275 2 329 287 2 249 338 2 285 364 2 224 419 2 264 457 2"),)),
    (32887, 190682, "f85c17ab33d54631410d6335a5eaa1de780904b777f1da15f9c06ce6e61d3520", 640, 480, 5, "http://farm6.staticflickr.com/5126/5218142854_bca4459220_z.jpg", ((473235, "373.29,119.19,125.84,345.34", "433 146 2 436 139 2 426 141 2 448 149 2 417 156 2 471 186 2 413 193 2 487 237 2 407 245 2 483 291 2 390 237 2 467 284 2 427 282 2 473 367 2 436 365 2 474 439 2 440 435 2"), (473865, "96.78,48.23,62.44,147.21", "131 64 2 133 62 2 131 62 2 140 62 2 0 0 0 155 75 2 129 74 2 142 87 1 119 99 2 141 75 2 105 100 2 147 123 1 129 120 2 152 160 2 130 160 2 150 187 1 133 188 2"),)),
    (33368, 70626, "8744e00ee252788e713def22b9d93710db7676dd5bda496241d14fe03661187c", 427, 640, 4, "http://farm5.staticflickr.com/4066/4287990820_0a3c8096ec_z.jpg", ((430983, "158.20,41.71,100.68,392.63", "210 93 2 215 86 2 202 87 2 0 0 0 191 77 2 248 99 2 178 100 2 242 156 2 174 154 2 206 119 2 167 211 2 233 230 2 184 227 2 237 300 2 175 306 2 232 387 2 185 389 2"),)),
    (45070, 284565, "0048f03ce376c1c3de6143ea8462c609cf0ff15847e20a20d042f0c4d968eeb8", 640, 596, 4, "http://farm4.staticflickr.com/3819/9272204163_a30ebf1d58_z.jpg", ((198826, "109.58,40.02,326.06,446.33", "291 159 2 301 139 2 0 0 0 346 154 2 0 0 0 346 225 2 332 230 2 278 322 2 261 320 2 179 369 2 177 370 2 346 403 2 324 400 2 256 389 2 246 405 2 163 444 2 157 422 2"),)),
    (47571, 289436, "0398da07fb61f7dd096005253c01a669358c81507ac913c2d86fdd8ca396853a", 639, 640, 5, "http://farm4.staticflickr.com/3771/10008328626_e50e829f0b_z.jpg", ((183778, "179.65,115.37,75.01,163.78", "212 137 2 0 0 0 207 133 2 0 0 0 198 137 2 211 157 2 190 154 2 223 175 2 209 178 2 242 175 2 237 181 2 210 204 2 191 201 2 232 230 2 222 230 2 244 263 2 225 273 2"),)),
    (50638, 243662, "7bddfb9dcb6fced3a5d40fa18ba6bd6ee2f769b37d980989620007b29765ff84", 640, 393, 5, "http://farm4.staticflickr.com/3085/5708760061_15445422e8_z.jpg", ((438455, "344.43,65.35,128.94,317.05", "0 0 0 0 0 0 0 0 0 407 113 2 442 114 2 380 148 2 441 159 2 368 194 2 463 199 2 365 231 2 452 219 2 387 251 2 418 254 2 411 303 2 405 303 2 428 358 2 389 357 2"),)),
    (54164, 103352, "32ce2bb44c2650a805d6282b852f1d5c312cec7df0f8296835bf71005acbe29f", 640, 427, 4, "http://farm3.staticflickr.com/2516/3973938540_7aa9b27812_z.jpg", ((494036, "182.04,105.70,148.63,238.75", "263 132 2 269 129 2 262 127 2 283 131 2 0 0 0 290 152 2 239 161 2 304 161 2 237 201 2 317 158 2 244 241 2 248 216 2 220 208 2 269 285 2 237 270 2 242 325 2 198 321 2"),)),
    (54931, 183503, "a49694d5dc982701f33a1d449a3ff7ebf953d1947ca141352097fb2528b10d1b", 427, 640, 4, "http://farm4.staticflickr.com/3707/9707884793_36195de2b5_z.jpg", ((185090, "197.03,154.89,185.53,468.85", "308 204 2 315 193 2 295 194 2 323 202 2 280 200 2 315 247 2 267 248 2 307 319 2 266 330 2 340 336 2 286 292 2 301 371 2 276 369 2 278 480 2 313 472 2 231 577 2 333 584 2"),)),
    (60899, 139277, "57c62f60ea71b9d40d33fa94635153fc02b1eec74b167324e874ee2ffb12ecab", 360, 640, 4, "http://farm6.staticflickr.com/5004/5212692994_bcf4194494_z.jpg", ((462207, "99.24,143.82,139.50,375.37", "170 249 2 184 238 2 161 232 2 201 236 2 0 0 0 196 272 2 137 265 2 228 237 2 129 226 2 216 195 2 161 182 2 176 384 2 134 379 2 184 435 2 123 430 2 183 480 2 124 465 2"),)),
    (61584, 183572, "e86bc3f63f6196bf2bad02d494af460ea8ae183cc6893af23d3c9888b5c4adbe", 500, 375, 4, "http://farm3.staticflickr.com/2101/2493497923_9d43331a97_z.jpg", ((185146, "234.37,96.54,80.83,185.16", "267 113 2 270 111 2 265 111 2 0 0 0 259 113 2 281 139 2 254 140 1 291 164 2 245 164 2 280 172 2 267 179 2 281 182 2 256 184 2 0 0 0 251 217 2 0 0 0 244 267 2"),)),
    (65350, 191231, "963a129714e3675b99b6a405a2cb46eca5678241358bb6b6bbefc3c3f1943ae0", 640, 427, 4, "http://farm5.staticflickr.com/4139/4869131880_ae3de10447_z.jpg", ((220110, "462.71,30.61,169.35,235.73", "607 73 2 0 0 0 608 69 2 0 0 0 600 54 2 587 68 2 557 54 2 584 90 2 517 66 2 590 124 2 494 102 2 553 117 2 517 113 2 571 176 2 505 191 2 577 246 2 478 244 2"),)),
    (66231, 163610, "9f4fe7d747ae0d163c865e6478afc0a39822817b650b11094bbcb4fe37b46680", 640, 427, 4, "http://farm9.staticflickr.com/8352/8375313873_80051bc3e2_z.jpg", ((495704, "333.71,137.52,111.56,288.52", "365 184 2 367 179 2 0 0 0 383 173 2 0 0 0 403 201 2 386 195 2 391 249 2 378 237 2 359 264 2 352 242 2 421 270 2 405 267 2 411 344 2 419 343 2 410 413 1 434 405 2"), (504367, "451.33,157.44,47.86,150.20", "0 0 0 0 0 0 0 0 0 468 178 2 0 0 0 477 194 2 484 186 2 481 216 2 0 0 0 464 216 2 473 208 2 487 228 2 490 228 2 487 264 2 491 264 2 487 296 2 489 296 2"),)),
    (76547, 247670, "f4f730a011b99aaa7b9ee851b6ba4c77fe2e3c4c81d12d58b7221d0f1b199c01", 640, 480, 4, "http://farm9.staticflickr.com/8557/8771723390_13dc9d18fa_z.jpg", ((188853, "415.11,204.57,172.01,258.54", "482 245 2 487 238 2 0 0 0 506 236 2 0 0 0 534 247 2 518 242 2 508 308 2 490 267 2 494 299 2 456 280 2 546 348 2 531 341 2 458 370 2 452 352 2 516 440 1 455 438 2"),)),
    (79031, 233589, "b82c8d3cc12b0bd3de1481db37e53932ed2d116805d2f9eff726bd2a01370123", 640, 427, 4, "http://farm9.staticflickr.com/8147/7630088322_e0bd2184fa_z.jpg", ((509683, "61.51,0.36,269.80,369.69", "317 27 2 321 19 2 309 12 2 0 0 0 272 3 2 223 15 2 261 33 2 0 0 0 243 91 2 260 67 2 284 157 2 89 107 2 134 114 2 140 197 2 250 197 2 80 306 2 268 342 2"),)),
    (80932, 205633, "c5bdb620680ad74c2bb788cabbf3374d68285bec3a2023fadf0ffc7eb257ba72", 480, 640, 4, "http://farm5.staticflickr.com/4075/5487804660_f06d600b35_z.jpg", ((201839, "247.93,35.32,177.30,350.27", "360 94 2 0 0 0 358 81 2 0 0 0 335 72 2 326 119 2 309 108 2 353 157 2 328 183 2 356 191 2 388 169 2 0 0 0 286 250 1 0 0 0 389 267 2 0 0 0 380 362 2"),)),
    (81988, 112043, "d0504a1ed78d6d875de3b0af307cc3349cafb4ff51f0bc9f4237f0a7a2cf36ac", 640, 427, 4, "http://farm7.staticflickr.com/6080/6038911779_36e4613d84_z.jpg", ((477031, "268.67,129.54,248.53,237.01", "0 0 0 0 0 0 0 0 0 388 149 2 411 149 2 379 177 2 424 178 2 338 191 2 461 193 2 298 193 2 493 185 2 389 260 2 414 258 2 383 317 1 408 319 2 389 371 1 403 382 1"), (494488, "320.18,274.65,87.85,122.27", "360 299 2 365 295 2 355 295 2 370 298 2 349 298 2 384 319 2 339 324 2 383 351 2 335 351 2 370 359 2 335 365 2 376 368 2 354 369 2 389 363 2 326 371 2 340 382 2 0 0 0"),)),
    (85772, 155556, "c21e2a798c8463fa01aace0312ddfb55bd6898565c8c3f6c22e87ea0d803128d", 640, 427, 4, "http://farm6.staticflickr.com/5206/5724188680_6c2980c45e_z.jpg", ((429906, "50.63,68.87,327.65,352.49", "218 116 2 225 106 2 205 112 2 0 0 0 186 120 2 263 143 2 191 165 2 339 195 2 154 217 2 350 259 2 75 222 2 309 266 2 248 273 2 266 413 2 214 389 2 0 0 0 0 0 0"),)),
    (86956, 101836, "468cd2502e79a5ac8973fd4a709c620437879feeb0510d0f2dc9e96c29604453", 480, 640, 5, "http://farm4.staticflickr.com/3096/3125314369_32cb27ef9d_z.jpg", ((508729, "40.93,141.10,186.06,374.63", "85 212 2 96 195 2 78 199 2 143 196 2 0 0 0 200 277 2 74 276 2 197 390 2 69 380 2 86 394 2 158 370 2 179 487 2 100 488 2 0 0 0 0 0 0 0 0 0 0 0 0"), (2154575, "247.20,131.62,232.80,507.70", "359 189 2 368 175 2 345 187 2 399 178 2 0 0 0 447 244 2 359 273 2 448 321 2 338 365 2 351 354 2 282 339 1 0 0 0 406 464 2 0 0 0 379 573 1 0 0 0 0 0 0"),)),
    (88485, 115620, "07cab32f1eac576ec11720ba77b4c2756a5d3f6cdd0352043c4c3a98b1dc362f", 640, 427, 5, "http://farm3.staticflickr.com/2593/5721763173_16433bab1c_z.jpg", ((422400, "188.07,0.00,233.17,422.20", "260 33 2 279 25 2 257 25 2 320 47 2 0 0 0 376 132 2 251 151 2 386 270 2 234 249 2 308 342 2 214 271 2 367 393 2 262 399 2 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (89078, 203134, "cb6ba6d871f582c4f9342032d05f86248c52d5741816ad501d647619ea618a16", 479, 640, 5, "http://farm9.staticflickr.com/8087/8353842968_0fb221e792_z.jpg", ((425691, "146.70,14.38,221.48,573.84", "273 84 2 286 73 2 262 74 2 0 0 0 243 77 2 277 152 2 216 131 2 289 224 2 181 199 2 325 291 2 182 265 2 244 312 2 208 303 2 273 420 2 236 416 2 263 544 2 236 552 2"),)),
    (90284, 128974, "6337308cc3e54d38c875788f1906158697fc25fe48b05bf4b8f8ef1850469f18", 640, 639, 4, "http://farm4.staticflickr.com/3595/3591078425_be6b5caa3d_z.jpg", ((459611, "67.19,143.09,518.92,421.71", "539 177 2 0 0 0 532 168 2 0 0 0 508 174 2 485 228 2 502 207 2 404 271 2 534 272 2 469 295 2 568 323 2 364 283 2 395 317 2 242 318 2 439 420 2 108 312 2 463 541 2"),)),
    (90891, 421559, "d3ca274ad2a7c3620c54f1024a65325fe7bfa0d6bb91a818edfc9fc5b1ea2be6", 640, 431, 4, "http://farm7.staticflickr.com/6145/5938772497_5a89d3ec7c_z.jpg", ((450981, "244.07,181.36,124.94,230.51", "298 209 2 306 205 2 293 203 2 313 209 2 278 200 2 319 249 2 259 248 2 336 295 2 258 296 2 342 315 2 281 322 2 312 315 2 263 314 2 351 329 2 271 331 2 285 348 2 272 420 1"), (499098, "425.19,184.99,113.32,184.02", "0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 527 219 2 477 220 2 509 270 2 447 286 2 504 315 2 453 332 2 484 237 2 453 242 2 487 330 2 447 332 2 484 376 1 447 372 1"), (530474, "517.67,141.46,93.92,235.81", "543 190 2 546 182 2 538 186 2 554 168 2 0 0 0 576 175 2 527 206 1 600 218 2 523 251 1 591 250 2 531 311 1 578 246 2 549 249 2 589 313 2 543 323 2 588 404 1 540 407 1"), (1742736, "143.80,187.86,91.25,183.77", "182 220 2 189 214 2 180 214 2 199 218 2 0 0 0 211 248 2 187 238 2 205 291 2 175 279 2 170 310 2 161 305 2 208 310 2 191 303 2 166 331 2 153 325 2 139 384 1 150 384 1"), (1755358, "51.08,105.36,112.81,320.32", "139 146 2 147 139 2 132 136 2 0 0 0 115 138 2 131 191 2 94 183 2 0 0 0 88 263 2 0 0 0 103 316 2 131 327 2 106 326 2 128 413 2 108 417 2 0 0 0 0 0 0"),)),
    (91406, 177059, "a6019381432c4b2517a5d1fecb8c9b2bf59a50c431d69fd3078ff82fe26ca1e9", 640, 424, 4, "http://farm8.staticflickr.com/7170/6843637155_9379f537f6_z.jpg", ((197055, "245.42,48.23,105.05,252.10", "313 82 2 322 76 2 303 76 2 326 83 2 292 81 2 339 134 2 271 130 2 0 0 0 255 180 2 0 0 0 283 198 2 323 235 2 284 235 2 323 303 1 287 303 1 0 0 0 0 0 0"), (202280, "315.62,76.62,116.94,243.38", "376 108 2 386 102 2 372 100 2 395 110 2 361 106 2 411 150 2 342 150 2 420 208 2 332 204 2 410 261 2 328 247 2 392 259 2 343 251 2 381 340 1 341 334 1 0 0 0 0 0 0"), (212196, "402.09,29.78,96.23,300.13", "443 66 2 450 55 2 432 59 2 463 60 2 423 63 2 472 112 2 411 118 2 486 175 2 0 0 0 448 218 2 0 0 0 467 218 2 421 217 2 453 306 2 429 304 2 0 0 0 0 0 0"), (234594, "470.54,61.20,114.93,336.64", "521 109 2 531 98 2 515 98 2 544 107 2 504 106 2 561 157 2 494 151 2 561 246 2 0 0 0 537 302 2 0 0 0 536 286 2 499 280 2 525 378 2 505 357 2 0 0 0 0 0 0"),)),
    (94871, 243675, "568808faf0e3ca50a66434ac7b053c6f493e48a45bf986e8bdcc7c48e3ee74eb", 480, 640, 4, "http://farm9.staticflickr.com/8503/8282824777_4495950c61_z.jpg", ((216213, "280.48,0.51,96.02,142.00", "289 9 2 292 1 1 0 0 0 310 3 2 0 0 0 346 33 2 295 42 2 371 81 2 287 87 2 363 80 2 291 127 2 338 137 1 302 138 1 336 189 1 301 184 1 0 0 0 0 0 0"), (253084, "123.08,0.00,162.46,391.38", "0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 264 38 2 181 41 2 261 117 2 173 116 2 236 166 2 155 95 2 244 175 2 190 173 1 234 262 2 185 258 2 229 346 2 180 345 2"),)),
    (94944, 105271, "a3fdb98ece1ececeadca3a00c1dc4ef0cb609bb2efd164d58907fbe65a0fb3cf", 640, 427, 5, "http://farm5.staticflickr.com/4008/4414517057_541fe5e979_z.jpg", ((435114, "358.07,224.03,109.14,180.95", "424 250 2 429 245 2 421 245 2 0 0 0 412 243 2 440 273 2 399 271 2 460 284 2 370 282 2 444 308 2 380 312 2 426 334 2 396 333 2 424 389 2 383 385 2 445 362 2 0 0 0"),)),
    (97278, 105630, "dd79077cea11398cd456ee48f7f1124265881a75eb3459c1f527681d1d859590", 457, 640, 5, "http://farm4.staticflickr.com/3255/3161956651_8c8631d721_z.jpg", ((500170, "84.86,77.43,245.47,499.11", "237 143 2 247 133 2 229 132 2 0 0 0 208 136 2 279 197 2 159 198 2 295 263 2 122 265 2 313 329 2 113 336 2 254 339 2 178 339 2 271 449 2 153 459 2 242 535 2 107 539 2"),)),
    (97924, 114215, "015ca73dcfa846a1d24d87f7676fbb4e095fb044f378ebe9dfe3bf2ebab5a798", 600, 400, 4, "http://farm9.staticflickr.com/8527/8546652580_94dbb43241_z.jpg", ((203611, "106.07,124.94,127.64,230.12", "160 150 2 0 0 0 159 145 2 0 0 0 150 145 2 130 172 2 157 167 2 0 0 0 177 196 2 0 0 0 201 197 2 148 238 2 167 238 2 140 278 2 189 284 2 124 332 2 209 343 2"), (206166, "298.88,163.84,123.49,180.77", "0 0 0 0 0 0 0 0 0 313 183 2 325 180 2 317 202 2 350 191 2 299 217 2 394 216 2 305 195 2 406 221 1 346 259 2 369 252 2 343 292 2 391 293 2 363 318 2 413 331 2"),)),
    (100238, 229027, "aeebb3301d652e291323df9567bed787a7966fefbefeb0629634424b1567769b", 640, 480, 4, "http://farm4.staticflickr.com/3192/5846630152_e082bbc6ea_z.jpg", ((515074, "194.80,79.94,230.42,400.06", "278 120 2 294 110 2 268 116 2 324 123 2 0 0 0 366 178 2 257 184 2 415 247 2 211 248 2 380 270 2 231 176 2 356 366 2 253 367 2 380 473 2 0 0 0 0 0 0 0 0 0"), (516913, "347.81,0.81,182.03,473.50", "409 59 2 422 47 2 398 49 2 442 47 2 0 0 0 481 103 2 382 92 2 516 208 2 354 183 1 494 177 2 368 166 1 446 318 2 381 311 1 476 426 2 374 426 1 0 0 0 0 0 0"), (533980, "1.08,33.32,213.46,438.82", "118 78 2 129 68 2 104 68 2 145 81 2 87 74 2 176 146 2 56 144 2 199 242 1 18 222 2 193 311 2 45 246 2 155 339 2 69 337 2 166 466 2 56 470 2 0 0 0 0 0 0"),)),
    (100510, 141572, "a2e2766e27122461d4770e240c9f9fd3e443fb369fd892703df1d8b2171dabe4", 600, 400, 4, "http://farm6.staticflickr.com/5547/9059818049_548550bd8f_z.jpg", ((215456, "180.25,24.68,84.93,120.03", "260 50 2 0 0 0 256 47 2 0 0 0 244 48 2 245 72 2 237 65 2 0 0 0 215 91 2 0 0 0 242 94 2 207 109 2 192 107 2 218 125 1 209 141 1 203 182 1 200 187 1"),)),
    (111207, 203003, "78d7b7eff999cee5eaf64be560eaa42c562b1105b275188d93fced89f01a417a", 478, 640, 4, "http://farm5.staticflickr.com/4143/4786676935_209abc2a42_z.jpg", ((436909, "156.24,350.92,96.20,214.16", "175 392 2 175 385 2 0 0 0 184 378 2 0 0 0 197 384 2 207 377 2 196 405 2 0 0 0 182 422 2 0 0 0 231 449 2 211 449 2 209 486 2 187 489 2 224 540 2 177 539 2"),)),
    (114907, 209467, "1ef6feb0a8d8886de8dc4db6e6c379cdd8fad15111a8edc570eeda66bcc17dba", 640, 425, 4, "http://farm9.staticflickr.com/8303/7825237468_9345ce9a8e_z.jpg", ((1212248, "315.69,74.98,115.96,209.32", "373 106 2 381 100 2 372 99 2 395 103 2 0 0 0 406 129 2 369 128 2 423 183 2 321 116 2 386 162 1 340 110 2 394 200 2 363 202 2 389 274 1 347 275 1 0 0 0 0 0 0"),)),
    (117425, 93718, "3ce22b1b2e55db6f118b1b057ddb7563970d09bce66b4b95d1f896df88605b6c", 640, 426, 5, "http://farm5.staticflickr.com/4037/4395733509_2c1aa888b2_z.jpg", ((462274, "276.04,7.45,297.05,418.55", "432 147 2 456 130 2 423 123 2 522 137 2 0 0 0 527 236 2 436 184 2 488 325 2 377 220 2 428 310 2 334 253 2 517 413 2 460 382 1 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (119445, 149162, "ef9c63a81758fdcbc4351eac736014c63e6677387051351771ed7183b55daa5d", 640, 426, 4, "http://farm4.staticflickr.com/3098/3225259967_8e1c2723c4_z.jpg", ((494213, "184.22,177.02,164.06,209.16", "0 0 0 0 0 0 0 0 0 256 193 2 0 0 0 261 210 2 243 238 2 227 236 2 232 231 2 197 244 2 200 241 2 267 271 2 291 284 2 241 317 2 285 339 2 220 375 2 333 357 2"),)),
    (122046, 227293, "4bd45688ba9e814bcb787e71c389f971aa4b99ad2ff621361054d880ce4a7947", 427, 640, 5, "http://farm5.staticflickr.com/4110/4976101457_b80e3f622f_z.jpg", ((469277, "82.16,191.71,194.60,415.14", "100 248 2 104 235 2 97 242 2 130 229 2 0 0 0 171 268 2 113 291 2 218 339 2 120 362 2 192 286 2 121 425 2 204 394 2 147 404 2 255 429 2 217 425 2 223 532 2 236 550 2"),)),
    (127270, 112096, "37348eec08d518e1e6837e90ea3e0879bba5cc73647b1179c4ce9659f19fe0df", 328, 500, 5, "http://farm4.staticflickr.com/3071/2828903887_8157eb46f0_z.jpg", ((2204768, "39.11,48.82,247.26,443.74", "126 122 2 129 117 2 115 116 2 0 0 0 96 131 2 143 159 2 89 173 2 163 128 2 52 164 2 169 91 2 70 117 2 172 287 2 121 300 2 205 371 2 112 394 2 255 461 2 137 439 2"),)),
    (128372, 171887, "6076863986b65a05fc9fd1edd1558cacee52db9b1e0c263ae3a02b1047f2617f", 640, 427, 5, "http://farm9.staticflickr.com/8479/8172792873_5d083ef147_z.jpg", ((185837, "379.02,202.47,67.17,178.47", "0 0 0 0 0 0 0 0 0 406 218 1 423 218 1 394 235 2 431 236 2 384 257 2 437 269 2 396 281 2 440 292 2 402 292 2 423 293 2 401 330 2 423 329 2 399 370 2 421 371 2"),)),
    (130586, 126287, "3214da76b09571a07b5a4f9e5d4ac5afd802117110cc3bbfd797008d0af536e9", 427, 640, 4, "http://farm4.staticflickr.com/3587/3392836274_5d866f582b_z.jpg", ((422535, "103.36,220.68,222.35,419.32", "241 280 2 253 268 2 229 268 2 275 279 2 218 280 2 298 351 2 206 329 2 307 406 2 173 372 2 257 433 2 134 346 2 265 467 2 219 459 2 261 557 2 218 553 2 253 625 2 0 0 0"),)),
    (132796, 263703, "689329fe1f003664d40a9330421fef02e16c26d5e64d25ac8ecf0c5d5ad6db23", 640, 426, 4, "http://farm6.staticflickr.com/5130/5382853203_aa9df7df0e_z.jpg", ((430174, "290.82,68.16,112.46,331.71", "318 109 2 316 100 2 0 0 0 332 88 2 0 0 0 366 111 2 338 135 2 356 154 2 315 175 2 327 197 2 286 197 1 377 199 2 338 203 2 353 263 2 295 235 1 369 359 2 304 356 1"), (443283, "401.11,61.33,147.42,184.76", "432 101 2 432 94 2 0 0 0 445 91 2 0 0 0 479 112 2 442 121 2 504 135 2 422 161 2 527 150 2 403 179 2 473 175 2 446 178 2 443 172 2 403 187 2 455 219 2 395 229 1"), (526621, "0.00,145.65,147.24,184.65", "109 169 2 116 165 2 104 165 2 122 171 2 98 172 2 129 193 2 113 195 2 123 226 2 0 0 0 99 233 2 0 0 0 120 261 2 115 261 2 49 252 2 52 255 2 18 312 2 23 314 2"),)),
    (136915, 100876, "52e676f1c7ad54796f249b175caf31b93c22348914b6f94d279d298dc2e6d4d7", 640, 427, 4, "http://farm4.staticflickr.com/3467/3402056386_bac84cf09b_z.jpg", ((451460, "75.01,77.90,254.86,344.29", "258 161 2 261 150 2 253 150 2 0 0 0 220 132 2 227 171 2 141 173 2 212 276 2 100 273 2 263 281 2 155 317 2 210 371 2 153 382 2 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (140203, 108471, "5c3a04dada4c36aba47e5c65a25da677451e66c4a7344ebb602bb4fe4cda55d1", 640, 480, 4, "http://farm1.staticflickr.com/47/151349771_c85a058c7d_z.jpg", ((205158, "508.80,20.30,80.69,226.08", "548 42 2 551 40 2 543 37 2 0 0 0 535 35 2 555 61 2 527 61 2 583 86 2 517 94 2 563 105 2 533 70 2 555 113 2 528 113 2 546 171 2 534 170 2 537 226 2 536 223 2"),)),
    (140556, 220902, "132750613623f8796699dc0e4c6f70aea6c9e0fa09dadd37d72b9df729a01a1f", 640, 457, 4, "http://farm9.staticflickr.com/8009/7421122526_0eec2dd48b_z.jpg", ((433532, "466.22,16.51,152.68,434.46", "491 65 2 501 56 2 0 0 0 528 57 2 0 0 0 567 104 2 493 137 2 585 189 2 484 212 2 572 255 2 481 264 2 547 240 2 491 247 2 565 385 2 519 378 2 0 0 0 0 0 0"), (457900, "56.48,70.86,116.05,378.95", "154 116 2 159 110 1 146 107 2 0 0 0 122 110 2 159 158 1 97 147 2 0 0 0 69 206 2 0 0 0 74 269 2 147 284 2 99 284 2 137 386 2 103 388 2 119 451 1 0 0 0"),)),
    (142620, 267317, "0f3dd713a8c98734879b3108ac6a160877b031831048330a0fc09cedfdf1dd80", 640, 426, 4, "http://farm8.staticflickr.com/7375/9756491185_311b0b2b7d_z.jpg", ((1225044, "269.96,5.74,359.95,414.52", "385 110 2 400 93 2 0 0 0 450 81 2 0 0 0 509 140 2 442 125 2 478 310 2 424 232 2 350 297 2 350 263 2 572 309 2 499 278 2 404 351 2 370 289 2 0 0 0 0 0 0"),)),
    (142971, 291301, "837a1b9788dc54393ec3cc7e8522e12e4202b83abc0e7f66b9d3f9954e3707bd", 640, 427, 4, "http://farm4.staticflickr.com/3316/5826779045_d3fb84f09b_z.jpg", ((441882, "176.33,73.42,133.42,154.43", "203 116 2 207 109 2 198 111 2 222 99 2 0 0 0 244 114 2 197 118 2 292 152 2 173 148 1 272 190 2 0 0 0 278 149 2 241 148 2 251 203 2 207 200 1 302 171 2 262 169 2"), (452023, "93.90,108.88,146.70,148.77", "166 143 2 172 137 2 162 137 2 186 137 2 156 138 1 194 157 2 158 155 2 198 186 2 155 182 2 188 220 2 140 215 2 190 210 2 167 208 2 201 216 2 135 208 2 228 248 2 105 225 1"), (1737923, "307.86,189.78,123.36,128.62", "0 0 0 0 0 0 0 0 0 359 216 1 391 216 2 357 227 2 390 228 2 333 250 1 411 253 2 334 271 2 417 276 2 353 256 2 384 262 2 320 290 2 384 292 2 326 311 2 388 305 1"),)),
    (143572, 289590, "fe3203e5b883c416d43c1b8edfcb052767e07a0e0f7576c9479f1994dccb9799", 640, 424, 4, "http://farm7.staticflickr.com/6078/6107512609_48fe33ae34_z.jpg", ((446088, "230.37,170.04,182.49,253.66", "308 195 1 312 189 2 0 0 0 325 188 2 0 0 0 350 216 2 312 222 2 382 246 2 287 238 2 367 251 2 253 240 2 363 294 2 350 296 2 344 356 2 373 365 2 346 421 2 412 401 2"),)),
    (147740, 180965, "0384b4bff79f53a0f2bcc54c99c0f7dc24660dd33f37c9b6dfa6adaa1e4a88ce", 640, 480, 4, "http://farm9.staticflickr.com/8042/8026451116_b1bd0c0c0a_z.jpg", ((210437, "4.35,145.12,92.52,205.72", "29 174 2 33 167 2 24 169 2 44 170 2 18 173 2 61 190 2 9 200 2 73 215 2 2 232 1 77 232 2 0 0 0 57 257 2 19 260 2 79 282 2 4 292 1 81 334 2 6 339 1"),)),
    (148739, 219698, "70adfe4defc746c3309c9ea076c65207924fc295b8cf4772b445d2ec96486a85", 640, 426, 4, "http://farm7.staticflickr.com/6230/7037849457_9d3c2dd395_z.jpg", ((439576, "74.11,120.38,91.34,120.36", "97 138 2 99 135 2 96 135 2 108 134 2 0 0 0 122 143 2 106 151 2 140 155 2 91 170 2 145 173 2 80 188 2 152 163 2 140 174 2 138 200 2 134 208 2 156 230 2 160 225 2"),)),
    (150224, 71131, "fba9c21e16adb27446e532e7ec8f6cc9b0065fc45c8e688a66fed7a3808d3b8f", 640, 427, 4, "http://farm1.staticflickr.com/202/478747964_7c8a47a4fe_z.jpg", ((472843, "346.03,213.74,78.08,186.03", "392 255 2 396 251 2 387 251 2 401 243 2 378 242 2 415 257 2 377 269 2 439 286 1 364 306 2 0 0 0 355 345 2 425 331 1 390 335 2 0 0 0 384 382 1 0 0 0 0 0 0"),)),
    (151480, 192518, "139f4857667bb09c0f197bf65644e9640ad9e9111fa56e37c057e455f9ef8d6b", 640, 480, 4, "http://farm7.staticflickr.com/6061/6054578485_f6d7810ee6_z.jpg", ((1206310, "169.60,175.10,53.78,144.46", "0 0 0 0 0 0 0 0 0 194 188 2 206 187 2 188 203 2 209 205 2 180 226 2 218 227 2 176 238 2 219 245 2 189 245 2 204 246 2 189 273 2 204 274 2 192 307 2 203 306 2"), (1233670, "384.13,188.17,50.54,128.92", "0 0 0 0 0 0 0 0 0 409 198 2 419 199 2 396 212 2 427 213 2 390 230 2 431 232 2 388 246 2 0 0 0 402 250 2 418 250 2 399 277 2 416 280 2 401 295 2 415 311 2"), (1297602, "293.75,181.37,42.25,133.01", "314 189 2 0 0 0 313 188 2 0 0 0 310 192 2 291 206 1 320 208 2 0 0 0 327 230 2 0 0 0 332 243 2 295 242 1 313 244 2 295 275 1 310 274 2 296 305 2 307 309 2"),)),
    (153229, 119187, "6fa2b9017bab8ceb1635e1eff3ffd43225127d2149f0d465fdc5e30d41174003", 640, 457, 4, "http://farm3.staticflickr.com/2498/4015454078_09a71cb072_z.jpg", ((495199, "339.40,147.26,155.77,297.10", "0 0 0 0 0 0 0 0 0 371 175 2 399 168 2 358 197 2 412 195 2 341 239 1 432 235 2 0 0 0 424 248 2 375 291 2 409 285 2 384 353 2 455 323 2 390 422 2 456 381 2"), (517559, "518.76,177.55,61.75,211.00", "0 0 0 0 0 0 0 0 0 543 195 2 561 195 2 540 217 2 574 218 2 540 239 2 586 250 1 528 242 2 566 252 2 546 278 2 570 282 1 527 324 1 0 0 0 538 371 2 0 0 0"), (524602, "233.62,30.74,103.49,361.71", "274 110 2 0 0 0 267 107 1 0 0 0 262 114 1 285 128 2 260 148 1 311 99 2 271 181 2 325 64 2 308 172 2 296 225 2 268 234 2 295 297 2 255 288 2 253 356 2 237 346 2"), (2152975, "168.56,33.24,131.75,371.53", "251 122 2 0 0 0 246 121 2 0 0 0 237 133 2 254 128 2 220 165 2 271 97 2 192 193 2 291 62 2 173 226 2 243 228 2 237 229 2 217 301 2 228 306 2 191 370 2 224 379 2"),)),
    (157046, 149418, "cadd4ed7b812dd75964015ba1879c6c94c5de1c32e077b9fe4498bc16aa007f4", 500, 375, 4, "http://farm1.staticflickr.com/155/371334465_59e28f5469_z.jpg", ((1264215, "1.69,67.42,88.48,261.23", "0 0 0 0 0 0 0 0 0 0 0 0 9 97 2 0 0 0 25 135 2 68 141 2 64 139 2 55 106 2 55 105 2 17 247 2 25 249 2 31 322 2 46 328 1 77 344 1 18 347 1"),)),
    (158956, 66286, "74b8025dfee410ebeb229b7bc355745eb0013b3c199d87ba70c369d3abed4b2c", 640, 425, 5, "http://farm4.staticflickr.com/3362/3594057164_dd6b4f2b62_z.jpg", ((445285, "169.04,39.16,323.77,379.15", "282 178 2 295 168 2 275 163 2 340 150 2 0 0 0 398 199 2 312 176 2 437 316 2 189 122 2 330 314 2 223 171 2 429 399 2 378 392 2 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (161875, 369995, "76dace8890923f2712afb6211331f1a73173ad0e03c1394b4a04342f11daedbd", 640, 422, 4, "http://farm9.staticflickr.com/8095/8424253224_c62e5479c8_z.jpg", ((468075, "174.49,72.73,107.16,312.94", "229 102 2 233 98 2 219 100 2 243 99 2 215 104 2 253 135 2 215 136 2 266 185 2 187 177 2 271 237 1 207 121 2 242 229 2 200 227 2 229 300 2 214 301 2 238 365 2 220 361 2"),)),
    (161879, 185567, "899a765ba61e9f1252055941a01b2202e991d7e04153c18ac7fbb1cd17391bce", 640, 569, 4, "http://farm2.staticflickr.com/1198/5143977073_da69928877_z.jpg", ((452906, "54.10,1.30,182.44,341.18", "196 96 2 203 86 2 188 86 2 0 0 0 174 66 2 203 53 2 137 116 2 165 27 2 86 153 2 0 0 0 77 206 2 168 172 2 122 164 2 180 239 2 93 232 2 161 328 2 80 317 2"),)),
    (163118, 120324, "3567db9db76a5bba0b606d7225fb49e6d92089eb5cdb8cc30f39aa2a7314c1de", 640, 426, 4, "http://farm7.staticflickr.com/6034/6295016809_63d19a29c6_z.jpg", ((446446, "264.78,71.02,111.08,354.98", "0 0 0 0 0 0 0 0 0 322 183 2 0 0 0 318 209 2 352 193 2 306 222 1 349 136 2 349 200 2 333 99 2 328 273 2 359 264 2 304 326 2 349 337 2 285 377 2 348 402 2"), (488697, "150.70,75.51,145.58,300.04", "227 132 2 0 0 0 224 137 2 0 0 0 223 149 2 217 173 2 241 158 2 221 208 2 260 131 2 240 238 2 269 102 2 209 225 2 229 232 2 190 282 2 200 292 2 172 340 2 166 338 2"),)),
    (165518, 106750, "733bdae7be5bccea44a0e4b6e6f079d9da9439f94d721db53694e3c01eb70f47", 612, 612, 4, "http://farm8.staticflickr.com/7216/7243537732_10dda0da2a_z.jpg", ((209396, "248.33,173.92,174.98,286.79", "0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 363 241 2 295 241 2 394 283 2 268 270 1 402 298 2 265 288 2 346 317 1 310 317 2 387 348 2 291 334 2 380 430 2 277 429 2"),)),
    (166165, 273247, "9af70d781b5b0326a360917a64c13df5155f54252918b650d3c24049abc34078", 561, 640, 4, "http://farm5.staticflickr.com/4149/5018726736_c0b5939690_z.jpg", ((444904, "265.82,22.75,225.38,291.83", "350 103 2 356 88 2 340 92 2 391 74 2 0 0 0 442 133 2 336 141 2 482 239 2 273 134 2 425 226 2 364 121 2 422 311 2 346 311 2 0 0 0 0 0 0 0 0 0 0 0 0"), (452210, "89.09,333.34,99.89,234.94", "168 360 1 169 355 1 165 356 1 0 0 0 153 357 2 159 378 2 133 375 2 174 411 2 128 415 2 176 416 2 158 415 2 136 438 2 110 438 2 168 494 2 138 496 2 159 549 2 104 544 2"),)),
    (168883, 156569, "f3247a56b457942d6a1237338f64949f410361471634b5610f348dfbc6641be6", 479, 640, 4, "http://farm4.staticflickr.com/3477/4031027612_1547340e64_z.jpg", ((1751634, "0.00,26.86,175.41,605.24", "67 91 2 80 77 2 51 81 2 100 90 2 23 100 2 151 185 2 0 0 0 186 323 1 0 0 0 147 401 1 0 0 0 128 478 1 5 479 2 122 625 1 16 621 2 0 0 0 0 0 0"),)),
    (168974, 103888, "e994ae206640b01c19f9eb73c9f9231e73057fb09292bb152de9a884bd1837a7", 375, 500, 4, "http://farm3.staticflickr.com/2360/2063838083_64f7514c79_z.jpg", ((519484, "0.00,86.52,321.35,407.86", "177 241 2 194 234 2 165 233 2 247 196 2 0 0 0 274 268 2 161 246 2 302 337 2 143 291 2 220 348 2 139 317 2 222 400 2 146 375 2 162 448 2 49 401 2 98 489 2 0 0 0"),)),
    (170099, 130431, "f2a5b87c4a5249e6b7c27fac24fb1e2994ebe22fe2a2f205f33893965c64b2b9", 640, 478, 4, "http://farm3.staticflickr.com/2280/1812763231_06381c802c_z.jpg", ((469280, "13.96,37.42,349.11,435.03", "179 112 2 202 100 2 169 94 2 229 133 2 152 123 2 273 243 2 91 229 2 315 372 2 42 349 2 327 421 2 70 386 2 222 464 2 115 457 2 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (171382, 222294, "f96faf8870853b0f11e2746eb86cb0dbc7e34c1b4903e3d912aa857002ef9bf3", 640, 426, 5, "http://farm4.staticflickr.com/3401/4572932047_8619a2bfac_z.jpg", ((422098, "106.33,12.54,198.32,245.06", "248 64 2 255 60 2 245 57 2 0 0 0 0 0 0 237 41 2 185 42 2 261 47 2 134 36 2 282 60 2 114 71 2 194 110 2 154 105 2 234 153 2 192 133 2 154 201 2 166 235 2"), (508492, "441.32,150.55,45.74,132.34", "453 162 2 454 160 2 0 0 0 460 160 2 0 0 0 450 177 2 473 175 2 447 194 2 483 190 2 446 211 2 0 0 0 453 211 2 469 208 1 453 243 2 474 242 2 453 270 2 475 270 2"), (508516, "486.31,125.41,70.84,184.76", "522 142 2 526 140 2 520 140 2 535 142 2 0 0 0 542 161 2 517 162 2 539 174 2 508 189 2 533 154 2 496 209 2 529 212 2 511 211 2 531 257 2 519 258 2 548 282 2 523 295 2"), (556171, "113.22,151.73,47.69,132.75", "138 162 2 139 161 2 0 0 0 143 162 2 0 0 0 142 177 2 156 178 2 142 197 1 0 0 0 139 212 1 0 0 0 141 215 1 153 218 1 134 241 2 154 243 1 126 269 2 153 269 2"),)),
    (172877, 74021, "079e9dfb3eb9067760e402cb6d52b7d48037dba9ed1f81cd85929ec4b1c43ea5", 640, 480, 4, "http://farm1.staticflickr.com/47/140251613_8273657d16_z.jpg", ((445068, "105.71,67.96,401.26,406.65", "315 137 2 328 123 2 298 118 2 346 138 2 273 125 2 367 205 2 238 194 2 468 275 2 126 268 2 366 301 2 234 313 2 344 389 2 266 386 2 453 365 2 176 374 2 0 0 0 0 0 0"),)),
    (173830, 149179, "46580410572c0e00f7810b350f1a6c7e75fe70203af2a4deef352b2ab3f441aa", 593, 640, 4, "http://farm6.staticflickr.com/5178/5486149069_ef317e598c_z.jpg", ((521392, "187.03,185.70,230.47,384.34", "371 277 2 380 262 2 359 263 2 0 0 0 327 252 2 393 288 2 286 272 2 407 371 1 291 391 2 411 401 1 341 394 2 325 335 2 239 321 2 373 389 2 275 392 2 335 505 2 246 508 2"),)),
    (177015, 176192, "13bc4db7ed2ac2612fd03cbb25a597712f99cf1e44784d4abde5aca47f6999c7", 640, 480, 4, "http://farm1.staticflickr.com/131/355302776_1d1215b7c1_z.jpg", ((1752244, "2.38,5.35,637.62,469.30", "479 115 2 510 99 2 474 98 2 570 100 2 0 0 0 609 205 2 439 181 2 572 365 1 360 269 1 363 368 2 250 320 2 448 430 1 321 381 2 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (177065, 378061, "443418dd6b1bc5a650c53d427c93538dd0f48f3c9cd9083c54ce3af0138821e0", 640, 564, 4, "http://farm8.staticflickr.com/7325/9899385706_ba38416932_z.jpg", ((424319, "348.84,208.48,106.94,197.33", "362 249 2 364 243 2 357 244 2 380 234 2 0 0 0 388 254 2 391 241 1 414 280 2 411 257 2 435 304 2 0 0 0 416 301 2 400 302 2 393 338 2 388 331 2 398 387 2 388 378 2"), (509292, "172.76,208.32,77.48,295.98", "243 237 2 0 0 0 237 230 2 0 0 0 218 238 2 226 267 2 200 267 2 0 0 0 237 301 2 0 0 0 248 276 2 211 342 2 193 348 2 192 400 2 224 400 2 183 467 2 197 453 2"), (1709501, "38.45,8.63,142.52,545.60", "136 53 2 0 0 0 131 45 2 0 0 0 104 49 2 72 90 2 91 99 2 0 0 0 73 191 2 0 0 0 133 262 2 54 250 2 83 256 2 106 375 2 150 387 2 83 501 2 113 507 2"),)),
    (182021, 73744, "f0ede0ab2c80add01911d7ce81d4462c9693fd311ee933529fe13316fe78d365", 640, 360, 4, "http://farm8.staticflickr.com/7233/7377647736_5e32424fc3_z.jpg", ((2163263, "149.06,69.38,154.69,285.00", "250 99 2 0 0 0 242 96 2 0 0 0 220 101 2 251 129 2 198 132 2 241 170 2 158 176 2 204 163 2 179 150 2 253 220 2 212 222 2 278 300 2 210 303 2 291 357 1 0 0 0"),)),
    (182805, 129812, "e502d762b1a264156f1e6349702d742d6cdddd40a2b54a3390acbd8e5059d1dd", 640, 360, 5, "http://farm7.staticflickr.com/6218/6309322560_cf4708d8dc_z.jpg", ((447701, "95.46,0.00,327.64,352.72", "301 106 2 0 0 0 0 0 0 0 0 0 274 69 2 209 80 2 198 127 2 306 145 1 219 277 2 365 169 2 335 255 2 178 304 2 201 330 2 293 164 1 325 199 1 0 0 0 473 246 1"),)),
    (183391, 245975, "f239b42753cfcbe508bfe3aaaec3397228311fbe398be1a2f59537696e446c28", 632, 640, 5, "http://farm7.staticflickr.com/6049/6233885701_126a5c1821_z.jpg", ((2161236, "210.53,112.34,277.78,481.52", "406 148 2 418 149 2 0 0 0 436 164 2 0 0 0 464 231 2 383 195 2 471 298 2 302 164 2 432 310 2 249 134 2 428 354 2 382 354 2 447 455 2 365 457 2 456 558 2 362 556 2"),)),
    (186624, 202377, "b041e6c1451614e6a7a117291e6519729d38e292cf448cfbe5dab313f3aaa587", 640, 553, 4, "http://farm6.staticflickr.com/5535/9272267604_d034ab2f0d_z.jpg", ((192309, "40.45,297.83,57.46,180.08", "81 302 2 82 298 2 77 298 2 0 0 0 67 301 1 74 320 2 66 320 2 83 349 2 47 351 2 83 330 2 69 341 2 81 381 2 70 380 1 76 419 2 70 420 2 77 460 2 71 467 2"),)),
    (187362, 121792, "631cd7b6bddc0c29b4f8c18b96283544146b6817052e2fb722a438ea8d516659", 640, 425, 4, "http://farm9.staticflickr.com/8090/8521689531_c2d54d64ce_z.jpg", ((424209, "257.29,170.19,85.76,219.17", "0 0 0 0 0 0 0 0 0 299 190 2 0 0 0 315 216 2 267 221 2 329 253 2 262 261 2 338 280 2 264 288 2 309 285 1 278 285 2 309 343 2 277 340 2 307 389 1 279 389 1"),)),
    (188592, 227078, "d91e1e7121a0f877856108f6244212ef0b34cfcbea0b6563f22a0f618eeafc27", 480, 640, 4, "http://farm9.staticflickr.com/8061/8185288357_5c7d3840ae_z.jpg", ((430662, "73.18,127.88,269.78,512.12", "246 296 2 270 286 2 226 273 2 0 0 0 194 242 2 305 343 2 163 318 2 322 420 2 89 368 2 282 409 2 155 364 2 252 525 2 153 510 2 258 633 2 103 597 2 293 545 2 111 544 2"),)),
    (189775, 162649, "931ab2db3edab06d9790a9a9b947550b005aa7dc42759e4188c6b5fa810d701b", 640, 480, 4, "http://farm9.staticflickr.com/8098/8534907929_2574ecdeab_z.jpg", ((1237862, "153.55,109.34,135.04,236.93", "247 128 2 0 0 0 242 126 2 0 0 0 229 130 2 257 150 2 218 162 2 278 165 2 195 193 2 0 0 0 171 209 2 258 222 2 232 229 2 272 271 2 223 276 2 262 324 2 210 326 2"), (1305221, "85.62,90.51,44.69,167.89", "110 109 2 112 104 2 107 106 2 0 0 0 99 109 2 116 122 2 95 127 2 0 0 0 92 150 2 0 0 0 91 168 2 113 168 2 101 168 2 108 200 2 99 203 2 106 232 2 97 233 2"),)),
    (190140, 193724, "abade429d12b97abe8d8e437d38fa2beda1058343ddbc46fa5bdd80000aa6c32", 640, 428, 4, "http://farm7.staticflickr.com/6118/6216901971_786976ff66_z.jpg", ((206071, "158.59,141.97,137.86,161.49", "236 169 2 241 163 2 230 163 2 249 167 2 220 167 2 260 198 2 200 201 2 274 236 1 166 219 2 285 236 1 177 254 2 259 275 1 218 278 1 0 0 0 183 269 2 0 0 0 208 329 1"),)),
    (190753, 239936, "44e80dccef157b3c8db7096d10e78386c29d96bc901564a6fb9f43ed915c3b4e", 640, 478, 4, "http://farm9.staticflickr.com/8459/7887530962_c31e9f1474_z.jpg", ((448217, "167.02,245.59,53.35,166.07", "207 273 2 208 268 2 203 269 2 0 0 0 192 268 2 212 288 2 176 295 2 211 316 2 179 325 2 209 294 2 207 325 1 212 330 1 191 333 1 0 0 0 175 343 2 0 0 0 189 392 2"),)),
    (191288, 143498, "a5cc3e797d789950cd1cd8479d582a56ebfd37001afff0a146126792934f2b0a", 600, 400, 4, "http://farm4.staticflickr.com/3733/9192472019_8099077709_z.jpg", ((214018, "258.29,0.00,132.72,203.15", "282 33 2 288 28 2 281 28 2 304 27 2 0 0 0 321 45 2 286 50 2 317 92 2 283 85 2 277 97 2 258 100 1 350 95 2 324 104 1 333 132 2 300 132 1 378 189 2 302 191 1"),)),
    (192607, 266559, "df1cfbabe293d62e1ad29ebf0e79699de8c1cef2a09177c5192255846a756974", 640, 425, 5, "http://farm6.staticflickr.com/5529/9298728158_2ff276583a_z.jpg", ((474613, "456.56,223.90,62.06,159.01", "498 237 2 499 235 2 496 235 2 0 0 0 485 236 2 506 254 2 470 251 2 513 275 2 461 269 2 497 279 2 468 280 2 495 297 2 477 297 2 491 337 2 482 336 2 490 377 2 486 374 2"),)),
    (192964, 107509, "867d92b86b04b27d8aec542c97d060d822a93911450283bc54cc34d2065989e5", 428, 640, 4, "http://farm6.staticflickr.com/5234/5798076351_87c086196d_z.jpg", ((2152500, "92.30,60.54,194.03,292.25", "144 140 2 146 132 2 0 0 0 168 119 2 0 0 0 203 127 2 168 171 2 182 97 2 157 236 2 125 77 2 131 280 2 250 235 2 255 253 2 181 193 2 149 241 2 182 296 2 152 334 2"),)),
    (193674, 174104, "fe94d775b39e9a7ad325404842bdc5ce2758e5cb3edaeb66d9be772ac5448020", 640, 480, 4, "http://farm6.staticflickr.com/5170/5240208640_839ede4a38_z.jpg", ((219116, "426.65,120.74,147.67,162.11", "439 159 2 440 152 2 435 156 2 454 143 2 0 0 0 482 162 2 458 176 2 0 0 0 454 216 1 0 0 0 444 246 1 558 175 2 518 179 2 523 240 2 487 229 2 528 287 1 489 279 1"),)),
    (199055, 244930, "b1dda03e61f7c502aaf30959faf6dd86e522372106a7389e5934f0e8b931c604", 480, 640, 4, "http://farm2.staticflickr.com/1114/5134639056_ce7fe0524b_z.jpg", ((424921, "143.50,34.61,248.25,565.38", "271 112 2 278 99 2 253 99 2 0 0 0 226 103 2 295 160 2 183 164 2 371 200 2 153 242 2 328 126 2 278 251 2 268 327 2 203 332 2 309 462 2 179 462 2 319 560 2 165 559 2"),)),
    (199771, 201077, "060235d75ac8a669d705e7685f73f53a09da291583f45faf45ac4ac1dbab4388", 640, 425, 4, "http://farm6.staticflickr.com/5035/7115744429_7d4deeb996_z.jpg", ((1277539, "370.45,66.72,195.18,297.42", "410 130 2 419 124 2 414 116 2 454 130 2 0 0 0 510 150 2 419 137 2 522 232 2 400 207 2 487 291 1 402 274 1 494 299 1 432 284 2 0 0 0 0 0 0 0 0 0 0 0 0"), (2156989, "36.52,55.34,193.69,360.81", "162 123 1 0 0 0 158 111 2 0 0 0 136 111 2 70 156 2 118 157 2 103 246 2 143 264 2 170 248 2 200 251 2 65 279 2 106 292 2 69 379 1 127 395 2 0 0 0 0 0 0"),)),
    (201072, 177691, "23bef955722c66bcf17440a52e54aad67b44ada4f5e20a7ee433eeeaad8fdcbe", 411, 640, 5, "http://farm3.staticflickr.com/2824/9374308711_bf5074c37e_z.jpg", ((503515, "80.36,20.09,327.17,602.69", "157 74 2 167 69 2 153 67 2 204 71 2 0 0 0 222 145 2 155 152 2 228 248 2 151 235 2 143 285 2 143 308 2 261 318 1 193 320 1 263 425 2 129 428 2 363 564 2 156 538 2"),)),
    (202339, 195767, "9b1d63dff953ffb99953a9c7073ba01772810928f3b22a205b267f9ae0fa7ed5", 480, 640, 4, "http://farm9.staticflickr.com/8070/8204757796_ce9fd630a2_z.jpg", ((437327, "102.57,35.76,173.37,400.18", "147 69 2 159 59 2 141 64 2 173 65 2 0 0 0 198 119 1 130 127 2 223 176 2 127 183 2 254 162 2 112 240 2 186 235 2 142 233 2 188 317 2 142 326 2 212 370 2 158 404 2"),)),
    (203389, 196414, "eb7fba75eaa1e49014f1fe81dfef9561478502347d8fc60c90a131d0cb8815bc", 640, 480, 4, "http://farm6.staticflickr.com/5257/5429775787_db93d0b30f_z.jpg", ((1691631, "329.33,149.60,278.74,259.37", "461 211 2 478 194 2 449 198 2 489 201 2 433 202 2 520 252 2 415 250 2 575 288 2 376 317 2 581 317 2 341 360 2 484 343 1 424 349 2 0 0 0 0 0 0 0 0 0 0 0 0"), (1701241, "182.03,198.98,184.02,281.02", "281 233 2 287 225 2 271 225 2 299 233 2 257 237 2 308 273 2 235 274 2 343 315 2 199 347 2 0 0 0 224 378 2 294 393 1 245 394 1 334 458 2 219 464 2 0 0 0 0 0 0"),)),
    (203639, 403707, "6bfc106bf02f49054b9514ac8c084a4ca93f44458725f6b5ca700c6cea73d4b0", 486, 640, 4, "http://farm9.staticflickr.com/8211/8423150137_74f6020b26_z.jpg", ((459777, "80.72,38.20,242.16,579.46", "183 138 2 193 128 2 175 129 2 217 115 2 0 0 0 248 148 2 157 164 2 273 237 2 139 248 2 279 292 2 109 226 2 248 296 2 192 297 2 222 441 2 184 447 2 188 561 2 160 570 2"),)),
    (203864, 329379, "a78628c598807687921f4a4b5642ccb45d92aa43d17ce8fe7e61cca735a01e54", 640, 491, 4, "http://farm5.staticflickr.com/4104/5038658829_5baea9b70a_z.jpg", ((494225, "251.57,100.41,180.95,338.73", "338 138 2 344 132 2 329 133 2 359 136 2 0 0 0 382 177 2 316 178 2 385 232 2 320 236 2 376 283 2 330 290 2 376 267 2 340 267 2 395 332 2 301 327 2 419 408 2 285 404 2"), (504454, "108.62,82.85,108.62,254.92", "174 103 2 179 99 2 172 97 2 190 106 2 0 0 0 193 125 2 171 135 2 166 150 2 147 163 2 133 158 2 121 183 2 197 204 2 180 203 2 173 257 2 192 257 2 174 320 2 195 320 2"),)),
    (203931, 301026, "0eeda3f8c5e385ac8d5a55737d5a8a8a05ee90814321be69f47beb361e6f2414", 428, 640, 4, "http://farm6.staticflickr.com/5160/7072495487_28288f842c_z.jpg", ((1286745, "115.06,182.65,189.84,412.77", "251 237 2 0 0 0 244 231 2 0 0 0 221 236 2 257 264 2 176 256 2 228 272 2 129 227 2 196 223 2 178 203 2 229 385 2 180 382 2 257 468 2 176 465 2 291 571 2 129 565 2"),)),
    (205105, 78352, "63126096550f5f9d35c81261d7f8ba341896a1ec8f958200c4fe20c6417462bc", 640, 428, 4, "http://farm4.staticflickr.com/3257/2654468936_be7cbac775_z.jpg", ((444711, "142.26,127.89,161.20,300.01", "222 197 2 0 0 0 213 196 2 0 0 0 203 208 2 243 219 2 199 244 2 265 192 2 171 241 2 285 153 2 154 206 2 252 322 2 221 325 2 265 386 2 208 384 2 0 0 0 0 0 0"),)),
    (206487, 253864, "3df6eaf2b80be894a9909549b5177958ab932776d4e8a5843c51464710284b5f", 640, 480, 4, "http://farm9.staticflickr.com/8010/7686978996_27b7fdfdbf_z.jpg", ((185568, "475.69,197.51,86.29,177.98", "0 0 0 0 0 0 0 0 0 497 219 2 518 216 2 494 240 2 540 239 2 486 271 2 544 268 2 494 285 2 550 287 2 493 290 2 528 290 2 482 301 2 558 303 1 479 342 2 547 343 1"),)),
    (209757, 177953, "e1ae8a5bb12ad20edfca2082821e4028a5bf27a9528ad2aa6d3a6b493a4418ad", 426, 640, 4, "http://farm3.staticflickr.com/2459/5726247637_7eb9302561_z.jpg", ((504226, "125.12,25.45,222.92,303.46", "296 89 2 301 86 2 298 80 2 0 0 0 288 59 2 268 103 2 254 65 2 279 125 2 205 53 2 320 129 2 165 40 2 175 143 2 211 147 2 203 228 2 245 219 2 161 293 2 260 303 2"),)),
    (209829, 117509, "cab95f31de6134c7c35f0e57b421215e818cca083f9dafb90e490e4322273259", 640, 480, 4, "http://farm8.staticflickr.com/7036/6968265053_493a3206a2_z.jpg", ((469105, "339.62,141.10,58.94,154.54", "355 159 2 356 157 2 0 0 0 363 155 2 0 0 0 373 170 2 362 170 2 388 194 2 359 186 2 389 220 2 349 208 2 385 209 2 378 208 2 377 243 2 372 244 2 381 281 2 380 282 2"),)),
    (210230, 202641, "33e35d38f41dd9c94ee87616903b722a0148860613ea02572ce7e481542e8935", 640, 457, 5, "http://farm8.staticflickr.com/7061/6965678017_7ff31e43fc_z.jpg", ((545656, "190.42,53.01,434.35,399.36", "440 145 2 457 124 2 430 130 2 518 120 2 0 0 0 547 207 2 444 212 2 484 336 2 372 274 2 312 349 2 321 234 2 549 444 2 468 435 1 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (210299, 236945, "dbdcbb432c918d11747928a3eb0d80498b9cc52087b5fe515e17b3cffb7e8240", 640, 480, 4, "http://farm7.staticflickr.com/6199/6207968507_effbfb5a1f_z.jpg", ((218627, "126.93,124.16,128.28,201.54", "179 184 2 186 178 2 174 178 2 202 174 2 0 0 0 221 175 2 162 175 2 248 196 2 147 193 2 239 188 2 142 177 2 220 206 2 189 207 2 216 252 2 178 251 2 225 307 2 183 286 1"),)),
    (211825, 99206, "9d33bcc07c852c2ed2c87f8873aa101474b21777adac7f4ce0a5590e0af8adba", 640, 480, 4, "http://farm1.staticflickr.com/33/35522808_8bba7bf555_z.jpg", ((426323, "0.00,0.00,488.67,295.53", "305 91 2 329 58 2 281 57 2 0 0 0 227 26 2 356 81 2 138 56 2 391 158 2 47 88 2 413 242 2 12 181 2 256 278 1 174 276 1 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (213033, 171821, "73a97cb8eceb57e2b1872b7470a11c6e5abb2c03173bd4ce94a45c0d096ae5ea", 480, 640, 4, "http://farm8.staticflickr.com/7432/9152238734_d0dd381aa5_z.jpg", ((194291, "122.25,182.65,218.60,450.16", "245 249 2 252 237 2 229 239 2 0 0 0 209 245 1 286 305 2 179 303 2 313 390 2 132 358 2 316 354 2 190 352 2 263 469 2 187 462 2 264 588 2 172 581 2 0 0 0 0 0 0"),)),
    (213171, 169653, "ad3b48655f5f185d6fd312e018ed1451dbcc063a8e270aa0fe587987f9bc0503", 375, 500, 5, "http://farm1.staticflickr.com/176/464806941_4657ce252f_z.jpg", ((432019, "138.67,145.33,118.66,298.67", "181 180 2 186 172 2 175 173 2 199 172 2 0 0 0 223 204 2 163 208 2 231 245 2 145 239 2 229 287 2 158 268 2 217 289 2 177 293 2 216 352 2 181 355 2 217 413 2 182 414 2"),)),
    (213547, 112871, "61b9b72eba0c02967cfab34df829cdd10e41c2c00e5a43be8db39d97bf43747b", 480, 640, 4, "http://farm3.staticflickr.com/2062/2243132921_779f9cdc24_z.jpg", ((185026, "123.69,139.51,231.55,467.41", "246 225 2 256 214 2 235 213 2 273 218 2 216 218 2 299 271 2 190 268 2 316 373 2 159 338 2 306 359 2 182 383 2 266 410 2 208 407 2 326 473 2 142 474 2 306 558 2 165 546 2"),)),
    (225405, 307375, "5a4ca5fc4209d4708e893acbcff19276f51408cf2fd781875f8671c2d734918f", 640, 471, 5, "http://farm8.staticflickr.com/7255/7502864702_c70cb43f70_z.jpg", ((2204771, "107.24,45.31,177.60,406.63", "0 0 0 0 0 0 0 0 0 135 86 2 171 87 2 117 146 2 183 130 2 124 215 2 215 194 2 0 0 0 0 0 0 144 240 2 184 232 2 176 329 2 247 292 2 136 415 2 247 367 2"), (2206450, "338.42,100.01,168.63,339.59", "406 159 2 412 152 2 0 0 0 436 145 2 0 0 0 468 169 2 409 171 2 493 209 2 408 212 2 499 256 2 415 233 2 451 273 2 428 272 2 417 347 2 432 358 2 396 422 2 487 407 2"),)),
    (229216, 139928, "627bc997b0d21da6ac5ea5b715a456282e8b5f0f62cd30b6fec45ad1341aaabd", 640, 426, 5, "http://farm1.staticflickr.com/57/180948869_0bad2f3167_z.jpg", ((484075, "143.60,9.57,442.27,412.60", "290 173 2 295 149 2 0 0 0 351 124 2 0 0 0 449 211 2 395 179 2 418 418 2 350 334 2 281 410 2 231 353 2 0 0 0 386 407 2 213 419 1 134 350 1 0 0 0 0 0 0"), (518161, "49.78,5.74,307.29,414.52", "207 128 2 231 106 2 189 103 2 0 0 0 0 0 0 327 148 1 144 110 2 203 314 2 125 285 2 207 205 2 152 195 2 311 308 2 196 291 2 161 366 1 82 352 2 0 0 0 0 0 0"),)),
    (229553, 203335, "bff35262976dc1c3c08b79c5347299671f25f2d000177447b3acb08585bdd975", 640, 488, 4, "http://farm5.staticflickr.com/4062/4653205441_96e8943d6f_z.jpg", ((431999, "58.38,55.91,446.14,330.47", "383 129 2 395 116 2 376 114 2 413 120 2 356 116 2 416 193 2 316 157 2 365 251 2 219 178 2 388 324 2 118 185 2 385 315 2 320 304 2 479 198 2 400 218 2 443 340 2 310 347 2"),)),
    (229849, 253182, "3841b7c43e7eed3b0e214312f6ef90bf5d4918fc7a3d921f09a85c44a5f9174c", 427, 640, 4, "http://farm5.staticflickr.com/4098/4943602569_06eebef70f_z.jpg", ((1278462, "151.01,152.45,257.44,376.81", "229 229 2 238 228 2 0 0 0 254 232 2 0 0 0 269 263 2 261 221 2 261 324 2 212 197 2 233 291 2 171 174 2 317 318 2 337 311 2 302 410 2 362 401 2 300 493 2 386 470 2"),)),
    (231508, 150098, "e0af7f51a9a89f1c5d80a1d53a8196dc2e7bfdcde170947f4830822d533f9bb6", 335, 500, 4, "http://farm3.staticflickr.com/2518/3693441995_2118879151_z.jpg", ((1205016, "62.92,215.73,155.06,178.65", "208 234 2 209 232 2 205 231 2 0 0 0 0 0 0 208 255 2 175 246 2 179 276 2 156 267 2 152 265 2 159 305 2 164 293 2 149 276 2 126 316 2 146 322 2 91 282 2 148 381 2"),)),
    (234607, 114588, "d7c0ea1fd165bb9f10a7b2795d6b3f6701446cc30ac5b59d611e33c9ed560653", 375, 500, 4, "http://farm2.staticflickr.com/1437/559863781_93c23c76d0_z.jpg", ((487890, "1.21,85.73,188.97,367.97", "104 179 2 115 178 2 102 170 2 130 190 2 0 0 0 123 240 2 74 195 2 102 300 2 102 129 2 133 282 2 159 110 2 77 307 2 24 309 2 148 333 2 42 411 2 146 428 2 11 382 2"), (518157, "165.92,74.22,209.08,389.01", "208 174 2 204 162 2 198 176 2 218 136 2 0 0 0 268 137 2 242 220 2 203 87 2 257 270 2 171 109 2 198 274 2 351 279 2 286 286 2 0 0 0 220 348 2 0 0 0 218 436 2"),)),
    (236426, 108786, "477054c151b504918e9e395a45b2cbc1ea8f0272ff61f68cc242da1609f11359", 640, 429, 4, "http://farm4.staticflickr.com/3432/3897473363_7f0d22165f_z.jpg", ((1222877, "260.69,64.20,144.88,217.04", "311 84 2 315 79 2 309 79 2 325 82 2 0 0 0 335 101 2 305 96 2 338 127 2 336 113 2 359 144 2 357 109 2 329 165 2 303 167 2 359 182 2 294 212 2 384 229 2 271 249 2"), (1227606, "21.35,95.60,101.08,133.00", "64 112 2 66 109 2 59 109 2 72 109 1 54 110 2 79 120 2 50 120 2 89 148 2 44 146 2 94 168 2 41 167 2 77 152 2 54 153 2 97 182 2 39 180 2 108 211 2 33 211 2"),)),
    (237071, 171407, "c977609c2c685eef309ff9140d94fc4ae47dd66a8d8624f6af6a63fb5a5a166c", 640, 427, 4, "http://farm6.staticflickr.com/5058/5723625041_762048f6f3_z.jpg", ((421607, "181.35,103.65,286.48,323.07", "305 160 2 314 147 2 296 149 2 330 148 2 284 156 2 369 188 2 271 198 2 431 238 2 241 253 2 446 259 2 212 293 2 387 274 2 355 300 2 318 369 2 0 0 0 0 0 0 0 0 0"),)),
    (239347, 159857, "6cc2205b78ab13769a09c779341f60053529fe8506ca579ef27f3e3ee9f22048", 640, 480, 5, "http://farm8.staticflickr.com/7084/7311947216_1eb6416561_z.jpg", ((469619, "206.49,157.30,323.24,135.13", "249 196 2 253 186 2 238 191 2 264 187 2 227 199 2 293 205 2 222 250 2 348 253 2 204 305 1 308 285 1 251 311 1 360 256 2 339 288 1 446 252 2 452 281 1 532 251 1 534 287 1"),)),
    (239537, 137834, "3352c5495d3b9981101135d4a03fa90becd1dec3da753a86874942ee85446e5a", 640, 436, 4, "http://farm1.staticflickr.com/104/297638599_e2ac523aff_z.jpg", ((501149, "263.26,25.25,259.12,380.88", "359 79 2 366 70 2 358 70 2 389 64 2 0 0 0 435 85 2 340 89 2 466 120 2 309 144 2 497 175 2 281 202 2 429 194 2 370 183 2 457 286 2 346 262 2 499 342 2 348 380 2"),)),
    (239773, 118937, "be75e4667ebbabacce06e80a2df46e01f0c813a3a4b54b587b22007584e10267", 500, 344, 4, "http://farm3.staticflickr.com/2458/3883298746_df92113101_z.jpg", ((500446, "100.49,63.39,200.22,276.74", "0 0 0 0 0 0 244 111 2 207 101 2 237 112 2 218 141 2 193 124 2 253 174 2 201 163 2 272 164 2 247 149 2 154 235 2 142 232 2 160 327 2 190 322 2 0 0 0 0 0 0"),)),
    (239857, 136972, "320ed8332ef28f2b53f85d01e509ec684da2765d6b79685a774509f3ca088f48", 500, 333, 4, "http://farm4.staticflickr.com/3493/3923418369_0b5abe5987_z.jpg", ((454038, "293.16,121.59,69.91,151.09", "322 132 2 325 129 2 319 129 1 333 130 2 0 0 0 343 147 2 313 147 2 348 162 2 304 151 2 354 171 2 291 152 1 344 188 2 318 189 2 352 225 2 319 226 2 355 254 2 319 262 2"),)),
    (242946, 141071, "79a2d1d40ed9e58bc559fcc15b0028c1aa91d22711787934e5a3f8c67109df37", 640, 480, 4, "http://farm4.staticflickr.com/3093/2650118269_48decfbb16_z.jpg", ((439897, "207.10,0.00,152.09,174.74", "296 57 2 305 46 2 284 48 2 0 0 0 256 59 2 325 97 2 240 105 2 344 181 1 233 183 1 329 146 2 262 159 2 313 205 1 266 209 1 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (246454, 176221, "b3f955e290bd58ccd622aad1a75465f74b1c382eb3c8641901b4d49213dbf5ec", 640, 640, 4, "http://farm4.staticflickr.com/3067/5758262175_6b35988fbd_z.jpg", ((2155447, "116.61,15.51,437.20,543.59", "367 139 2 400 127 2 363 109 2 437 146 2 0 0 0 507 216 2 384 196 2 486 366 2 273 259 1 353 346 2 118 301 1 498 397 2 415 374 2 338 489 2 304 382 2 467 549 1 369 521 2"),)),
    (247917, 196714, "3d19e31ae4de033abffb9586dffd9663e646f040d1ac5293bc4c7c467150c1c1", 612, 612, 4, "http://farm9.staticflickr.com/8164/7205570594_5aba222a6f_z.jpg", ((511965, "456.81,222.76,141.48,335.13", "498 264 2 504 258 2 491 258 2 0 0 0 473 258 2 528 291 2 467 301 2 548 342 2 466 360 2 561 384 2 477 405 2 539 381 2 503 389 2 552 449 2 505 456 2 562 521 2 511 520 2"), (535534, "386.63,214.52,96.45,342.90", "425 257 2 433 249 2 417 249 2 440 255 2 405 254 2 452 293 2 393 296 2 462 350 1 393 360 1 473 403 1 397 403 1 452 388 2 410 391 2 451 452 2 413 455 2 448 522 2 415 521 2"), (550324, "250.25,240.78,87.57,335.90", "292 280 2 301 274 2 283 274 2 309 275 2 270 277 2 316 319 2 265 318 2 325 372 2 257 374 1 324 414 2 262 411 2 309 407 2 274 407 2 310 474 2 268 476 2 311 532 2 268 529 2"), (550721, "149.20,238.65,113.28,349.52", "216 272 2 222 268 2 210 268 2 238 271 2 200 272 2 247 321 2 193 314 2 258 363 1 163 353 2 209 385 2 145 366 1 229 400 2 191 397 2 224 482 2 180 485 2 227 548 2 170 552 2"), (551278, "34.38,204.92,118.28,391.95", "107 246 2 116 240 2 98 240 2 0 0 0 80 244 2 134 295 2 60 299 2 137 347 2 51 354 2 137 416 2 51 407 2 118 397 2 75 397 2 110 480 2 75 481 2 110 556 2 75 561 2"), (564228, "335.57,97.64,148.53,203.55", "405 136 2 413 126 2 397 127 2 423 134 2 383 134 2 453 182 2 360 182 2 484 252 1 320 253 1 504 302 1 0 0 0 426 309 1 380 311 1 451 400 1 386 402 1 0 0 0 0 0 0"), (1696074, "328.20,241.14,86.20,331.18", "361 276 2 370 269 2 352 269 2 378 275 2 341 276 1 389 314 2 340 314 2 400 377 2 329 374 1 401 415 2 329 418 1 381 389 2 356 393 2 375 470 2 349 474 2 373 532 2 346 534 2"), (1735248, "52.03,55.62,169.32,499.16", "160 98 2 166 92 2 148 90 2 182 94 2 140 90 2 194 161 2 90 150 2 208 228 2 67 224 2 0 0 0 81 281 1 174 300 1 98 296 1 157 398 2 91 388 1 0 0 0 0 0 0"),)),
    (248284, 180959, "9f3308ac7ee40f7e983b1f36adb8a65244063a1cebcd609407951d5457bd260a", 427, 640, 5, "http://farm8.staticflickr.com/7305/8718260325_0ea4d34d5f_z.jpg", ((477700, "255.17,150.82,127.66,260.24", "337 179 2 341 176 2 333 174 2 0 0 0 323 174 2 353 205 2 309 201 2 366 250 2 289 237 2 372 242 1 311 228 2 347 280 2 317 279 2 344 331 2 317 329 2 346 401 2 320 393 2"),)),
    (250137, 173485, "00b60aa56c8552f04a59da5ec1a6ba555991fa22cd8ff932f31d72a159595e8b", 480, 640, 4, "http://farm5.staticflickr.com/4084/5028973928_910bfd2613_z.jpg", ((1224150, "366.00,235.99,114.00,239.91", "419 304 2 428 294 2 409 290 2 0 0 0 386 288 2 432 337 2 381 336 2 474 432 1 394 398 2 474 414 1 464 362 2 465 478 1 407 493 1 0 0 0 0 0 0 0 0 0 0 0 0"), (1274221, "56.09,198.47,107.87,355.24", "124 225 2 0 0 0 118 221 2 0 0 0 105 223 1 111 265 2 90 259 2 0 0 0 95 294 2 0 0 0 127 303 2 101 369 2 89 369 2 98 457 2 80 456 1 87 532 1 61 530 2"),)),
    (252219, 143225, "1cf48bddb1bcab80d9f4d18514ae438c3e4f57d18e80c4fbea9d5d22396f514a", 640, 428, 4, "http://farm4.staticflickr.com/3446/3232237447_13d84bd0a1_z.jpg", ((481918, "326.28,174.56,71.24,197.25", "356 198 2 358 193 2 351 194 2 364 192 2 346 194 2 375 207 2 341 211 2 388 236 2 336 238 2 392 263 2 343 242 2 373 271 2 347 272 2 372 316 2 348 318 2 372 353 2 355 354 2"), (489768, "9.79,167.06,121.94,226.45", "100 190 2 0 0 0 96 185 2 0 0 0 86 188 2 84 208 2 71 208 2 84 245 2 59 240 2 115 263 2 66 271 2 64 268 2 71 264 2 59 324 2 99 322 2 18 363 2 101 377 2"), (495624, "510.44,171.27,123.66,215.76", "536 192 1 538 188 2 0 0 0 552 190 2 0 0 0 568 207 2 555 208 2 559 243 2 554 246 2 542 270 2 550 277 2 573 274 2 559 274 2 589 323 2 541 322 2 617 365 2 530 361 2"),)),
    (253819, 238233, "982f4ce355ba872123b8d13939dbbb1a81a0761726fbf5b334f5a8ac25f2b251", 640, 427, 5, "http://farm4.staticflickr.com/3302/3335797090_7e4fd195a5_z.jpg", ((423444, "196.71,5.49,275.39,315.69", "312 107 2 0 0 0 0 0 0 0 0 0 0 0 0 355 73 2 289 78 2 386 52 2 263 59 2 434 21 2 215 39 2 356 153 2 313 150 2 368 213 2 283 156 2 400 293 2 307 224 2"),)),
    (253835, 195092, "e9e8947e76d5fc5d49b1b94a7f96bb88f873d3bd75362b3e5cde0f885962d9dd", 640, 480, 4, "http://farm4.staticflickr.com/3772/9683835627_634705b4c4_z.jpg", ((229912, "482.15,107.49,95.57,329.66", "499 138 2 504 130 2 0 0 0 521 133 2 0 0 0 556 165 2 523 169 2 570 214 2 526 216 2 532 228 2 506 256 2 554 272 2 524 274 2 554 339 2 519 340 2 558 418 2 546 404 2"),)),
    (260106, 130574, "8e649f03ba9d80d708ace7955a7db6dbf693d83998078975d67139bb558372b6", 400, 600, 4, "http://farm8.staticflickr.com/7355/9062004031_5775136a94_z.jpg", ((2164355, "168.31,90.39,160.52,205.71", "178 126 2 182 118 2 175 120 2 199 116 2 0 0 0 223 133 2 182 147 2 244 174 2 0 0 0 213 177 2 0 0 0 242 185 2 214 189 1 266 229 2 0 0 0 297 280 2 0 0 0"),)),
    (261061, 118480, "886f87e9e2cea4ea6855ae03b8707b99ef321dc3ad409f7ff9a712b94c569cd9", 500, 334, 4, "http://farm3.staticflickr.com/2579/3661648054_10cef105c9_z.jpg", ((484547, "260.16,41.70,111.51,292.30", "324 85 2 332 77 2 314 77 2 344 83 2 300 82 2 334 120 2 288 118 2 343 141 2 272 138 2 321 115 2 314 105 2 313 218 2 274 215 2 305 300 2 280 278 2 0 0 0 290 308 2"),)),
    (261732, 185194, "ddbfc26792efff1d7f2e675390c57303f274df9f8131997505aff801b85b4e6a", 640, 438, 4, "http://farm4.staticflickr.com/3262/3179843291_985d31a5f6_z.jpg", ((428410, "145.67,43.37,166.34,312.02", "238 71 2 244 69 2 232 69 2 252 79 2 223 76 2 250 117 2 216 87 2 239 175 2 233 90 2 263 144 2 278 93 2 232 201 2 195 196 2 260 262 2 180 267 2 281 328 2 155 291 2"),)),
    (261888, 150713, "02df0f1cd1c0997aacb71636e96f06a033cb2c1adee465744802e7cd691ac4de", 640, 426, 4, "http://farm5.staticflickr.com/4079/4918743472_0b684750c4_z.jpg", ((204595, "361.58,143.49,97.18,250.13", "0 0 0 0 0 0 0 0 0 400 175 2 0 0 0 432 194 2 410 201 2 437 241 2 397 229 2 415 267 2 376 257 2 441 265 2 430 263 2 443 318 2 410 307 1 447 376 2 437 346 1"),)),
    (263969, 244864, "77e51b2628fd424c716785c56aead155174b8fcd8e5c8a83d77ac535e5fccc8f", 640, 610, 4, "http://farm8.staticflickr.com/7307/8735597371_053c077264_z.jpg", ((444839, "220.20,145.14,237.98,454.08", "328 228 2 348 208 2 314 212 2 376 211 2 0 0 0 406 283 2 307 280 2 386 422 1 292 376 2 281 447 2 255 434 1 389 518 2 315 501 2 0 0 0 0 0 0 0 0 0 0 0 0"), (451357, "224.81,57.57,413.98,542.83", "326 226 1 348 210 1 311 211 1 373 212 1 0 0 0 389 283 1 302 284 2 390 413 2 283 370 1 291 439 1 262 421 2 375 473 1 305 476 1 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (266768, 238439, "9bf52870172cba87d8917809be3ddd2432862adfd5c8aff310fb4851f50c1ac4", 612, 612, 5, "http://farm4.staticflickr.com/3712/9631334222_f91803a60d_z.jpg", ((441282, "126.53,257.18,63.26,171.91", "170 294 2 172 287 2 162 289 2 0 0 0 142 293 2 156 327 2 140 334 1 155 373 2 147 392 2 171 390 2 189 402 1 0 0 0 131 407 2 0 0 0 202 425 1 0 0 0 210 509 1"), (476971, "61.52,286.80,200.09,282.38", "155 316 2 0 0 0 148 309 2 0 0 0 132 314 2 99 343 2 121 349 2 0 0 0 96 420 2 0 0 0 84 482 2 71 448 2 79 460 2 191 464 2 197 445 1 173 550 2 228 507 2"), (489177, "343.93,275.59,161.50,302.09", "402 310 1 406 303 2 0 0 0 422 313 2 0 0 0 424 345 2 476 344 2 380 415 2 0 0 0 361 402 2 0 0 0 447 464 2 478 461 2 383 459 2 384 421 2 399 560 2 341 510 1"),)),
    (270474, 122842, "27f34f6b32b96a5b236821c8c80afbc4fa170bd42aa43950af5c9e7a62b4b45b", 500, 375, 5, "http://farm1.staticflickr.com/89/214479871_e33f79b98f_z.jpg", ((473521, "190.03,11.82,137.67,358.96", "250 68 2 259 58 2 238 61 2 275 65 2 221 74 2 254 113 2 230 125 2 263 183 2 273 168 2 295 208 2 273 112 2 263 214 2 237 214 2 278 327 2 247 326 2 0 0 0 0 0 0"),)),
    (270908, 106413, "65bb8aa5f5d9824ec94e17911f26d6fd13bf70ff1c5756cc61fe09cfcd308618", 500, 299, 4, "http://farm4.staticflickr.com/3474/3875628760_a9c353f71a_z.jpg", ((448079, "31.58,30.03,335.28,176.71", "206 162 2 198 171 2 212 169 2 188 177 1 223 170 2 168 156 2 237 150 2 114 165 2 286 150 2 75 173 2 335 158 2 174 85 2 220 83 2 162 63 2 236 60 2 157 48 2 250 48 2"),)),
    (273715, 115657, "d67892fbc83b2d8eb38b740fe2804c902b44d7fcc03d76d0952d38be08cc553a", 500, 373, 4, "http://farm4.staticflickr.com/3604/3441464864_01a70b2b99_z.jpg", ((533014, "438.23,153.88,61.77,147.20", "0 0 0 0 0 0 0 0 0 475 170 2 0 0 0 482 184 2 476 184 2 469 205 2 468 201 2 450 209 2 449 198 2 483 226 2 475 221 2 478 257 2 473 257 2 482 291 2 468 289 2"),)),
    (273760, 237862, "535751041007728ef08fac924635e2b76ae7026058d81edca903b4f9785a9d6c", 467, 640, 5, "http://farm6.staticflickr.com/5156/5872238085_6ef7145483_z.jpg", ((476757, "126.56,34.52,176.90,510.56", "240 96 2 249 89 2 231 86 2 263 86 2 223 81 2 272 131 2 183 138 2 266 192 2 151 198 2 281 209 2 143 270 2 246 279 2 194 280 2 203 382 2 229 376 2 182 486 2 204 492 2"),)),
    (274460, 306668, "e8328d1efa9ab49d921fb9eb37723f20e93140cc1391119e4121325de347d2d0", 640, 454, 4, "http://farm7.staticflickr.com/6236/6267475434_0c28897158_z.jpg", ((1306109, "204.97,179.81,66.64,150.11", "242 195 2 246 193 2 239 193 2 250 195 2 234 195 2 254 213 2 230 210 2 262 234 2 216 231 2 265 249 2 211 249 2 249 254 2 229 252 2 247 288 2 227 286 2 242 313 1 227 313 2"), (1306853, "260.33,169.63,56.88,156.90", "293 184 2 296 183 2 290 182 2 299 185 2 286 183 2 308 203 2 274 198 2 310 227 2 267 220 2 313 249 2 274 226 2 297 251 2 278 250 2 299 281 2 272 281 2 298 313 2 268 313 2"), (1315661, "309.88,188.21,49.86,131.17", "336 200 2 339 198 2 334 198 2 345 198 2 330 199 2 353 212 2 320 211 2 0 0 0 310 222 1 0 0 0 294 222 1 341 255 2 322 254 2 343 284 1 319 284 2 346 311 1 317 312 2"), (1328825, "154.17,174.47,52.10,140.35", "190 191 2 191 188 2 186 188 2 0 0 0 177 188 2 195 198 2 169 205 2 197 218 2 166 231 2 199 245 2 186 241 2 190 256 2 174 257 2 189 285 2 167 285 2 193 320 1 170 320 1"), (1329558, "438.43,184.62,57.29,147.87", "467 200 2 470 196 2 464 197 2 474 197 2 461 198 2 484 213 2 451 213 2 492 235 2 446 234 2 478 247 2 461 243 2 476 253 2 458 254 2 481 287 1 451 290 2 483 319 1 450 320 1"),)),
    (275198, 260200, "c34482da7ff6fc53d5e78dc433cc673ab601757761e07948f15450cc8214a483", 640, 480, 4, "http://farm8.staticflickr.com/7262/7488569650_9e0faaf072_z.jpg", ((216854, "298.99,114.87,124.46,135.78", "362 155 2 372 150 2 359 145 2 383 153 2 353 144 2 391 173 2 336 171 2 403 219 2 307 216 2 411 186 2 322 208 2 387 260 1 341 262 1 0 0 0 0 0 0 0 0 0 0 0 0"), (2154764, "0.00,44.32,184.77,435.68", "96 118 1 0 0 0 99 109 2 0 0 0 75 96 2 8 115 2 59 142 2 0 0 0 98 233 2 0 0 0 159 209 2 5 280 2 38 291 2 27 402 2 74 390 2 51 473 2 0 0 0"),)),
    (278705, 151666, "d01c906a85c3f77702bdbf9dc7cedf11310e9aa47c6730927f51d9651dece310", 640, 480, 4, "http://farm4.staticflickr.com/3282/2875114693_2caf3d7481_z.jpg", ((511685, "93.95,115.20,101.89,174.19", "122 141 2 123 137 2 117 139 2 129 132 2 0 0 0 150 137 2 112 148 2 167 163 1 107 173 2 174 187 2 102 200 2 159 196 2 136 201 2 174 230 2 135 237 2 181 257 2 141 272 2"), (1718334, "360.62,117.73,150.51,217.81", "377 147 2 378 141 2 0 0 0 387 136 2 0 0 0 411 154 2 420 146 2 435 190 2 425 190 2 412 224 2 401 172 1 431 221 2 440 221 2 388 256 2 450 277 2 387 315 2 497 326 2"),)),
    (279927, 323477, "7b93a53ec3c3b64f60e8fc54bdf918e43ef75abfc72083756184f9ae083d90a5", 640, 442, 5, "http://farm8.staticflickr.com/7214/7392604632_6e5e3447a9_z.jpg", ((1316136, "158.59,216.16,42.57,131.85", "192 229 2 194 228 2 189 226 2 0 0 0 182 225 2 192 246 2 171 238 2 0 0 0 173 260 2 0 0 0 181 257 2 181 275 2 170 273 2 183 310 2 165 312 2 179 339 2 149 336 1"), (1742669, "189.71,236.82,68.40,141.99", "0 0 0 0 0 0 0 0 0 221 229 1 233 232 1 216 247 2 224 250 2 196 267 1 226 274 2 187 285 1 242 260 1 215 293 2 226 294 2 210 332 2 232 325 2 211 368 2 242 365 2"),)),
    (280930, 206172, "57a58decd3d841b44ab611fda39b45f69815586899d28e0db8a58b8472213639", 640, 425, 5, "http://farm6.staticflickr.com/5289/5227407234_31a3eb81cc_z.jpg", ((421933, "255.96,1.91,273.14,418.31", "419 57 2 441 48 2 407 42 2 464 57 2 381 46 2 467 139 2 372 144 2 497 224 2 350 225 2 503 284 2 294 297 2 475 336 2 399 334 1 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (281759, 187443, "5b5861ca8955f8ff906abe78f2b32bc49deee2832f4518ffe4bb584653f3c9e9", 640, 427, 4, "http://farm8.staticflickr.com/7398/9546010205_60818f8424_z.jpg", ((480462, "206.27,131.81,78.08,274.71", "254 147 2 256 142 2 247 144 2 0 0 0 236 148 2 264 183 2 230 182 2 279 228 2 227 226 1 276 198 2 262 199 2 265 240 2 240 239 2 261 314 2 233 316 2 251 386 2 228 382 2"), (490210, "95.96,97.87,103.63,305.14", "176 121 2 0 0 0 169 115 2 0 0 0 148 121 2 180 157 2 125 150 2 187 199 2 103 194 2 173 196 2 131 184 2 162 235 2 120 230 2 148 305 2 127 299 2 147 380 2 121 376 2"), (502624, "381.90,139.13,84.44,279.23", "400 155 2 406 150 2 0 0 0 420 155 2 0 0 0 439 188 2 398 191 2 454 225 2 383 234 1 454 261 2 387 211 2 430 264 2 402 265 1 422 321 2 413 324 2 451 395 2 433 394 2"), (511921, "299.52,143.98,77.31,271.60", "348 159 2 350 156 2 341 155 2 0 0 0 328 163 2 365 187 2 313 189 2 372 225 2 305 226 2 348 248 1 330 249 1 356 272 1 330 273 1 357 337 1 330 336 1 354 395 2 338 395 2"), (1724839, "458.67,139.20,73.88,283.07", "475 161 2 480 156 2 472 159 2 494 159 2 0 0 0 511 196 2 466 197 2 523 234 2 459 234 2 511 203 2 482 213 2 508 263 2 474 266 2 514 331 2 481 329 2 520 399 2 489 400 2"),)),
    (282298, 279497, "4a5ea7d1f08a112ff876cc121812e1f156f7a5e9d0f283749128fa0279433213", 640, 480, 4, "http://farm6.staticflickr.com/5259/5495196957_335d3637c7_z.jpg", ((1213276, "178.74,259.92,59.78,218.70", "225 285 2 227 280 2 220 280 2 0 0 0 209 281 2 211 297 2 199 296 2 0 0 0 223 322 2 0 0 0 228 299 2 204 362 2 197 364 2 195 417 2 204 419 2 169 461 1 201 473 2"), (1243725, "252.91,272.39,53.58,169.40", "267 290 2 271 287 2 0 0 0 280 289 2 0 0 0 286 305 2 264 304 2 293 325 2 0 0 0 271 314 2 0 0 0 285 347 2 263 345 2 266 389 2 270 386 2 266 431 2 289 407 2"), (1683890, "1.00,268.76,62.22,155.10", "38 284 2 41 282 2 35 283 2 0 0 0 27 283 2 49 299 2 13 298 2 57 323 2 0 316 1 54 341 2 18 322 2 38 353 2 19 352 2 41 379 2 19 382 2 42 407 2 19 412 2"), (1687289, "217.26,263.00,45.33,146.28", "240 277 2 242 275 2 238 275 2 249 276 2 0 0 0 255 295 2 228 293 1 0 0 0 0 0 0 0 0 0 0 0 0 249 336 2 232 335 2 253 368 2 228 364 1 255 396 2 223 391 2"), (1695700, "98.59,272.28,52.68,162.42", "120 289 1 124 287 2 0 0 0 132 289 2 0 0 0 143 312 2 122 303 2 126 334 2 103 321 2 105 336 2 110 317 2 133 351 2 123 345 2 130 386 2 119 384 2 129 424 2 123 420 2"),)),
    (286553, 126326, "f37863b16e8789da5ed5f768a16f651883ac31cdcd8a965d1795e0d5456b2314", 640, 480, 4, "http://farm3.staticflickr.com/2710/4252543190_2e6b88828f_z.jpg", ((445938, "52.85,11.94,334.39,353.79", "165 108 2 181 85 2 144 90 2 201 92 2 119 105 2 252 170 2 94 193 2 348 260 2 72 318 2 466 225 1 119 407 1 256 390 1 143 402 1 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (288042, 214663, "e914c0385ad155f468011797f4203ae3d43add6793154a3258f3efc2c87792d7", 640, 480, 4, "http://farm7.staticflickr.com/6024/5961682633_69797dc630_z.jpg", ((477002, "550.97,196.71,52.63,163.14", "576 194 1 0 0 0 571 192 1 0 0 0 564 196 1 581 220 2 557 222 2 590 242 2 555 246 2 590 228 2 581 254 2 584 268 2 569 269 2 595 307 2 575 314 2 579 337 2 575 348 2"),)),
    (289417, 81398, "a098e0cfd6702bf227c3b5d9e98d4321bc803c3cf8ffab4d867e86c9b5d7e13d", 500, 375, 4, "http://farm5.staticflickr.com/4057/4431669174_e67a2421a5_z.jpg", ((482278, "271.58,26.04,111.10,194.43", "336 56 2 341 52 2 332 52 2 348 52 2 326 49 2 348 75 2 314 73 2 362 98 2 299 92 2 373 121 2 286 105 2 331 126 2 310 124 2 332 160 2 303 157 2 342 191 2 293 185 2"),)),
    (301718, 187495, "54a5a90b270b5ae95672c8ed2947867efabe7acf977d89c985cd1e909a58b1be", 480, 640, 4, "http://farm4.staticflickr.com/3763/9132300748_f22266f91b_z.jpg", ((219162, "4.31,22.77,208.54,257.44", "74 128 2 87 107 2 53 113 2 111 85 2 33 109 2 161 158 2 9 167 2 181 271 1 0 0 0 169 232 1 19 270 2 165 333 1 44 346 1 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (303713, 151849, "dd1ed70486f92a68145f8e1cfaf45e2a3359e81e8953e6f9165da78f74ded2b7", 427, 640, 4, "http://farm4.staticflickr.com/3148/2315297307_c1eb20fdb2_z.jpg", ((1228908, "89.17,326.47,169.71,306.34", "162 378 2 0 0 0 153 378 2 0 0 0 144 389 2 180 391 2 139 419 2 218 371 2 98 448 2 245 348 2 102 399 2 216 472 2 183 478 2 244 550 2 194 561 2 188 603 2 125 602 2"),)),
    (303893, 241253, "e809831e253ff07b4f8f810445c39cab27eab15e1d2a86f90189e35084538468", 640, 427, 5, "http://farm8.staticflickr.com/7378/9383802677_0dc943b387_z.jpg", ((495245, "87.52,46.94,241.39,363.53", "201 119 2 212 107 2 185 111 2 232 117 2 171 123 2 257 167 2 157 178 2 280 213 2 150 233 2 288 240 2 133 286 2 249 256 2 186 263 2 260 303 2 211 310 2 273 356 2 234 368 2"), (1230051, "306.79,90.88,228.88,282.75", "415 185 2 423 172 2 398 175 2 0 0 0 368 165 2 422 199 2 342 199 2 442 252 1 323 246 2 434 231 2 353 271 2 430 278 1 381 279 1 461 302 2 399 310 2 501 344 2 454 344 2"),)),
    (306139, 153453, "42a07a3348e24d370e2b1191eb52126de536b726051ff03a508b79c4d4e90308", 640, 426, 5, "http://farm5.staticflickr.com/4092/5003303411_fd1d0e52ec_z.jpg", ((501536, "338.54,37.18,58.15,134.68", "348 65 2 347 61 2 0 0 0 351 53 2 0 0 0 363 66 2 375 61 2 366 98 2 378 90 2 364 127 2 352 94 2 368 128 2 0 0 0 368 169 1 0 0 0 371 205 1 0 0 0"),)),
    (309655, 217088, "3e641c01b7abb3ce7f00c844cea36d71431c54fce70a23c1940e3a7bd07ce0a2", 640, 480, 4, "http://farm6.staticflickr.com/5011/5389083366_fdf13f2ee6_z.jpg", ((1220804, "241.62,203.87,116.49,268.58", "299 239 2 307 232 2 291 232 2 0 0 0 0 0 0 331 275 2 266 277 2 343 319 2 254 320 2 350 360 2 262 357 2 330 365 2 288 367 2 330 439 2 292 439 2 0 0 0 0 0 0"),)),
    (313588, 233858, "4635596b77228a330bebea2dba69c6c96f33febfeb4047f5e74f34a85692387b", 480, 640, 4, "http://farm9.staticflickr.com/8209/8253725562_84e76ed5e9_z.jpg", ((495970, "340.85,224.12,139.15,352.36", "392 255 2 401 249 2 387 250 2 414 254 2 0 0 0 433 293 2 374 294 2 459 323 2 358 347 2 430 307 2 350 390 2 432 399 2 388 400 2 430 466 2 401 467 2 442 552 2 411 542 2"),)),
    (319369, 263356, "bf8b4c6c9d2028f4209779a1acef68125854fbdfac1d3d609ac0a752d23d030e", 640, 432, 4, "http://farm6.staticflickr.com/5193/5802453698_0cc91183ef_z.jpg", ((464561, "478.79,205.47,66.15,121.40", "512 224 2 515 221 2 510 221 2 520 222 2 502 223 2 528 245 2 494 246 2 540 269 2 481 269 2 535 287 2 500 288 2 523 297 2 503 297 2 534 337 1 501 338 1 0 0 0 0 0 0"), (495421, "272.49,207.93,70.26,162.36", "0 0 0 0 0 0 0 0 0 307 226 1 325 228 2 302 247 2 335 251 2 296 280 2 338 281 2 286 306 2 337 311 2 305 310 2 330 313 2 299 353 2 320 356 2 295 397 1 315 396 1"), (502907, "339.86,213.69,43.29,161.25", "0 0 0 0 0 0 0 0 0 347 228 2 361 227 2 341 250 2 372 249 2 337 280 1 375 281 2 0 0 0 0 0 0 346 290 2 367 291 2 349 340 2 364 340 2 352 382 1 365 382 1"),)),
    (323355, 199542, "3f3cf2287d73bc198fb7c1503cf2ccac77b33d772ff89318b8b83546add5924b", 480, 640, 4, "http://farm6.staticflickr.com/5165/5241297935_1dcda049cf_z.jpg", ((2229452, "100.64,33.17,344.37,597.55", "325 122 2 343 111 2 313 112 2 370 129 2 306 126 2 394 207 2 281 206 2 420 316 2 251 300 2 326 351 2 168 323 2 345 419 2 262 413 1 346 596 2 281 587 2 0 0 0 0 0 0"),)),
    (325838, 101302, "2753bdbf9c6240d1b6a640aa79276132fb3e52a20df86091813377df38cb788b", 640, 480, 4, "http://farm3.staticflickr.com/2247/3539996132_28696cbbf5_z.jpg", ((519905, "196.31,63.46,175.82,313.89", "282 134 2 286 118 2 267 124 2 0 0 0 241 125 2 300 169 2 216 190 2 309 237 2 237 279 2 337 262 2 290 322 2 299 289 2 252 310 2 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (325991, 123207, "0bf88cf023fbbb63af3e33479710dc9faf8a65b5938f353ca7d5600d3b5bbb6f", 500, 334, 5, "http://farm4.staticflickr.com/3639/3419012138_cfdf55b431_z.jpg", ((422442, "145.94,43.25,196.33,290.75", "245 101 2 250 92 2 234 92 2 0 0 0 214 92 2 230 120 2 197 118 2 227 142 2 236 211 2 260 152 2 295 233 2 211 235 2 172 244 2 227 324 2 219 327 2 0 0 0 0 0 0"),)),
    (326128, 114458, "16fc08c185eeec30bf11d3c0944157279885f938838159e13773e4cfde879618", 480, 640, 4, "http://farm3.staticflickr.com/2176/2277537216_e2ddde599b_z.jpg", ((496288, "239.82,184.56,128.58,408.85", "296 229 2 302 218 2 286 219 2 316 223 2 277 229 1 340 272 2 267 280 2 348 328 2 257 331 2 294 351 2 262 371 2 318 403 2 280 400 2 322 486 2 287 468 1 326 556 2 305 524 2"),)),
    (329456, 122100, "07b0642433a6de72e38f49a94337d11b9b34f0345c866c1563fc74fef36f923b", 640, 427, 4, "http://farm4.staticflickr.com/3112/2561836272_6200d49083_z.jpg", ((513973, "226.96,21.64,197.15,401.03", "290 65 2 295 57 2 278 58 2 0 0 0 259 70 2 286 110 2 257 120 2 296 181 2 246 209 2 335 211 2 245 272 1 277 239 2 303 238 2 273 325 2 341 323 2 268 416 1 367 410 2"), (530019, "391.17,2.58,114.59,261.40", "458 24 2 461 20 2 452 21 2 0 0 0 439 25 2 432 45 2 441 50 2 418 88 2 443 97 2 413 130 2 450 128 2 427 128 2 439 128 2 462 180 2 436 190 2 482 222 2 422 247 2"), (534480, "518.37,11.61,101.61,284.12", "0 0 0 0 0 0 0 0 0 0 0 0 576 37 2 535 61 2 574 70 2 524 112 2 592 117 2 523 147 2 616 143 2 538 154 2 561 153 2 573 210 2 561 211 2 573 253 2 551 275 2"),)),
    (338219, 322978, "4b6160bbe6f2e3d95429ac866bcf225ce04dc90f35665ff21b29cc836e3a4a92", 640, 566, 4, "http://farm8.staticflickr.com/7283/8999990657_da131bf6a9_z.jpg", ((202133, "163.97,8.68,96.42,160.97", "218 53 2 225 46 2 211 46 2 234 41 2 201 39 2 247 78 2 187 75 2 251 123 2 170 115 1 0 0 0 189 141 2 234 158 1 194 155 2 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (340451, 229935, "566aa2a499299e354e2127a702783ca64efa9e2e8ed77c9ffcea10d7ad3cb8e4", 640, 424, 5, "http://farm9.staticflickr.com/8513/8399079993_cd99846f90_z.jpg", ((2164049, "238.09,45.69,156.87,336.19", "330 101 2 338 94 2 324 94 2 350 96 2 316 99 1 370 128 2 298 128 2 377 175 2 258 131 2 336 200 2 263 81 2 347 212 2 300 212 2 331 225 2 291 223 2 347 340 2 290 340 2"),)),
    (341469, 218277, "9539a05baa654711352936679b920ba72c69a6b870d6bf12148d9ce5afe68516", 457, 640, 4, "http://farm9.staticflickr.com/8123/8659803549_02c7baa8a3_z.jpg", ((187996, "188.15,4.05,219.75,629.01", "291 62 2 301 46 2 279 51 2 324 47 2 264 63 2 367 131 2 271 138 2 389 238 2 281 201 2 358 337 2 231 218 2 320 326 2 272 317 2 333 442 2 292 441 2 342 606 2 309 533 2"),)),
    (341921, 261909, "40ec6e7a9b261979db6c8294b0cd9c0228541ed6647d3aaf129a6e6915bfe267", 640, 478, 4, "http://farm6.staticflickr.com/5184/5615933755_b24f4bdc42_z.jpg", ((424270, "288.14,244.86,117.33,142.13", "333 288 1 331 279 2 0 0 0 342 271 2 0 0 0 359 295 2 385 282 2 352 354 2 372 317 2 329 349 2 341 328 2 382 374 1 395 358 1 296 364 2 313 357 2 318 441 1 332 434 1"),)),
    (342128, 295808, "850c27fb3df1abd24188fa5d93c9c1338edc46ab544f1ac34d8f116e6a0dd564", 488, 640, 4, "http://farm8.staticflickr.com/7345/9296437874_b852f470d6_z.jpg", ((1219220, "94.52,28.22,308.55,584.86", "175 89 2 185 72 2 165 81 2 214 75 2 0 0 0 254 144 2 196 153 2 233 257 2 197 220 2 171 285 2 137 215 1 290 301 2 242 306 2 349 457 2 192 417 2 372 553 2 184 523 2"),)),
    (342367, 157865, "97537c2e8b3fb21beae758ee7644b8bd79a0d9066796b7099696406e458912ca", 640, 480, 4, "http://farm8.staticflickr.com/7029/6602321969_c79b3d4ac9_z.jpg", ((189717, "190.97,128.07,154.84,257.03", "257 181 2 266 171 2 249 173 2 284 176 2 0 0 0 304 232 2 230 227 2 328 304 2 203 281 2 268 324 2 215 267 2 281 360 2 239 358 2 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (343149, 194367, "8d98267ce9b4c72ec8a37b19110d2e1e67607e59ad9704fb9f6d196784e32c89", 480, 640, 4, "http://farm9.staticflickr.com/8528/8451405956_8c88c54606_z.jpg", ((1727935, "428.57,446.73,50.87,146.44", "450 461 2 452 458 2 447 457 2 460 459 1 0 0 0 468 474 2 443 474 1 474 500 2 446 490 2 0 0 0 458 479 2 471 520 2 451 523 2 0 0 0 445 552 2 0 0 0 433 577 2"), (1736509, "50.15,474.77,51.26,149.03", "76 489 2 78 486 2 0 0 0 87 488 2 0 0 0 91 504 2 88 505 2 88 528 2 77 520 2 81 548 2 74 505 2 84 542 2 81 543 2 76 579 2 87 577 2 69 613 2 91 608 2"),)),
    (343218, 78134, "ff45d2490e8dfa3d775ddf5ea378d835cdec56eed9c33190c9a00cfd584e7871", 640, 428, 4, "http://farm4.staticflickr.com/3169/2653643649_665418b9df_z.jpg", ((439637, "244.85,107.00,139.27,281.18", "373 134 2 0 0 0 369 128 2 0 0 0 355 130 2 361 162 2 318 154 2 367 199 2 279 183 2 374 218 2 260 187 2 341 238 2 315 233 2 325 303 2 337 299 2 298 361 2 320 371 2"), (2163489, "397.72,101.60,44.69,138.89", "431 113 1 0 0 0 427 111 2 0 0 0 419 113 2 431 130 2 406 130 2 434 152 2 401 150 2 437 168 2 400 167 2 426 172 2 411 171 2 425 198 2 411 197 2 425 227 2 411 226 2"),)),
    (343937, 52720, "8695e43af989afc4b1b590c0174d944cd454752fdccf9ba648adb45d4922d20b", 640, 428, 4, "http://farm5.staticflickr.com/4072/4320314557_bfcc972543_z.jpg", ((425370, "275.07,15.23,134.66,394.34", "290 69 2 300 57 2 285 57 2 315 62 2 0 0 0 333 97 2 298 106 2 350 143 2 0 0 0 353 198 2 0 0 0 336 205 2 317 209 2 344 291 2 340 278 2 378 371 2 374 363 2"),)),
    (345466, 149006, "1d43f9258fdf6f974f30d14f383fe0f2bc3c412d4ee807d807b5cb90f3a45da0", 500, 375, 4, "http://farm2.staticflickr.com/1418/1356309053_2247aa7699_z.jpg", ((557961, "110.15,45.91,293.44,269.90", "0 0 0 0 0 0 0 0 0 218 76 2 240 74 2 217 113 2 273 90 2 192 128 2 310 93 2 148 118 2 315 108 2 238 201 2 274 192 2 201 249 2 334 231 2 144 283 2 360 300 2"),)),
    (349860, 104053, "f07d8b177d8f2b0a9fa4dd06eaec034e5c52adec952caeb504eb794edf963da7", 640, 426, 5, "http://farm5.staticflickr.com/4025/4590246533_c351aefccd_z.jpg", ((488820, "279.93,13.34,185.72,152.22", "363 37 2 367 31 2 357 32 2 374 37 2 348 40 2 380 71 2 330 69 2 407 98 2 317 104 2 442 95 2 304 138 2 369 147 1 333 146 2 404 102 2 360 75 2 425 173 1 369 128 1"),)),
    (351530, 199594, "c0dd4c7912f49996c6db70407078f13642da6239661294dc27bf190b2e1669f7", 640, 378, 5, "http://farm9.staticflickr.com/8334/8125752591_b7f7b8da3b_z.jpg", ((437772, "135.60,111.72,162.96,166.57", "182 141 1 186 136 1 0 0 0 198 133 1 0 0 0 208 157 1 211 155 1 190 185 2 186 187 2 160 199 1 160 195 1 238 204 2 240 202 2 178 209 2 181 209 2 180 264 1 182 262 1"),)),
    (352684, 141020, "da8028056a1503782c966bfb4643cd9ec67a926be49f45016b5f968fbc5579e9", 424, 640, 5, "http://farm8.staticflickr.com/7150/6442776257_d51029651f_z.jpg", ((1229759, "167.09,223.15,123.39,412.57", "177 262 2 185 253 2 174 256 2 204 247 2 0 0 0 232 280 2 205 283 2 276 323 2 0 0 0 249 379 2 202 371 2 230 406 2 204 405 2 228 500 2 202 483 2 249 601 2 221 558 2"), (1282910, "300.58,240.18,79.11,286.20", "0 0 0 0 0 0 0 0 0 333 262 2 357 266 2 318 300 2 366 304 2 309 345 2 370 349 2 0 0 0 0 0 0 324 381 2 357 380 2 329 432 2 345 442 2 342 493 2 334 509 2"), (1315629, "1.44,271.39,40.80,141.38", "0 0 0 0 0 0 0 0 0 0 0 0 24 285 2 25 298 2 15 300 2 0 0 0 19 324 2 40 336 2 32 339 2 28 343 2 17 346 2 31 372 2 24 373 2 32 397 2 21 402 2"), (1317704, "374.13,282.78,45.76,144.68", "404 295 2 406 293 2 400 293 2 0 0 0 0 0 0 411 312 2 382 313 2 414 337 2 381 334 2 416 331 1 404 332 2 409 355 2 392 355 2 407 385 2 386 385 2 405 416 2 383 417 2"),)),
    (352760, 146347, "e8e628840f8275433d429abede2e426da7092545c692d418a12cf58e4fa2ee4d", 544, 640, 4, "http://farm9.staticflickr.com/8290/7885970872_9b4c310757_z.jpg", ((473214, "142.38,80.54,235.87,348.04", "225 131 2 233 118 2 214 120 2 245 118 2 205 125 2 269 133 2 191 144 2 310 165 2 175 188 2 339 212 2 169 227 2 276 221 2 234 228 2 311 300 2 225 317 2 337 374 2 226 394 2"),)),
    (356424, 160969, "c669af906be21b4c9113b05a7a9dc489d4f893b25641ffafad19ef6327275740", 480, 640, 4, "http://farm9.staticflickr.com/8116/8607070980_b791deaeb4_z.jpg", ((461392, "35.24,68.73,302.06,388.88", "178 208 2 194 178 2 145 185 2 0 0 0 89 179 2 244 218 2 68 248 2 277 418 2 68 418 2 124 412 2 133 298 2 252 463 1 148 472 1 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (356505, 194567, "2551b5ea629e74fe93644bbc16f80dc0d908e71485e4a0f1cf9e4d4df872c492", 640, 425, 4, "http://farm5.staticflickr.com/4079/4879236570_540fbb5b7a_z.jpg", ((1242078, "222.64,85.72,175.98,269.79", "312 122 2 316 116 2 306 116 2 323 116 2 294 118 2 335 144 2 289 146 2 349 192 2 271 194 2 336 225 2 247 221 2 370 206 2 334 212 2 357 269 2 302 275 2 386 337 2 328 335 2"),)),
    (357742, 148562, "592f3b383bde4be94eb70469301905d2ffbaf5bab77e59a08ed6d11c3d25a7ce", 429, 640, 4, "http://farm6.staticflickr.com/5247/5219132350_14c13c7939_z.jpg", ((421636, "143.53,437.73,82.32,181.79", "0 0 0 180 448 2 0 0 0 0 0 0 192 450 2 168 462 2 192 464 2 147 473 2 203 487 2 0 0 0 218 481 2 171 501 2 187 501 2 167 551 2 198 540 2 159 589 2 199 564 2"),)),
    (357888, 128916, "a3e2453b0293f04a582274239f4a00290e968ef615480314c2cf00cff986f134", 640, 480, 5, "http://farm4.staticflickr.com/3577/3853391167_2ff6070b1a_z.jpg", ((195050, "301.74,159.03,259.87,309.26", "444 205 2 455 195 2 432 196 2 469 206 2 414 206 2 497 274 2 378 271 2 518 353 2 318 303 2 541 425 2 342 241 1 476 444 2 387 442 1 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (358195, 169443, "6130b647946095659a441922ce1a47e32cfb0d8db65fc00ed67ce1acac83524d", 483, 640, 4, "http://farm3.staticflickr.com/2749/4222767499_919d266f71_z.jpg", ((447901, "202.79,106.43,138.06,438.65", "248 170 2 257 162 2 239 167 2 277 155 2 229 167 2 298 195 2 228 202 2 321 255 2 223 255 2 329 303 2 218 308 2 304 312 2 253 318 2 297 403 2 254 406 2 320 488 2 258 499 2"),)),
    (363666, 229262, "a9fddd22efc52bc382cfb88adc507134ce80d1ba09bff7d24debbaa1b26e6d78", 640, 419, 4, "http://farm3.staticflickr.com/2854/9178936197_56a8862bd5_z.jpg", ((441360, "420.87,65.71,217.47,344.94", "469 155 2 475 143 2 0 0 0 523 144 2 0 0 0 573 248 2 571 177 2 567 386 2 544 266 2 478 297 2 485 219 2 536 396 2 528 346 2 400 373 1 408 323 1 0 0 0 0 0 0"), (496586, "267.40,5.07,160.02,176.52", "323 62 2 336 53 2 322 50 2 361 53 2 0 0 0 397 96 2 327 81 2 405 167 2 315 127 2 345 154 2 296 129 2 386 199 1 330 178 1 0 0 0 269 248 1 0 0 0 0 0 0"), (543109, "16.95,39.55,202.44,374.74", "123 96 2 0 0 0 114 88 2 0 0 0 84 90 2 107 126 2 47 161 2 128 170 2 103 218 2 180 150 2 169 167 2 133 253 1 79 281 2 206 300 1 151 375 1 162 405 1 65 394 2"),)),
    (365208, 183824, "a1a58588a52869234419bdf068722001a86e3cc4dbcd1f2da1665921ce46370b", 603, 640, 4, "http://farm9.staticflickr.com/8188/8111728980_6cec69a31f_z.jpg", ((458040, "175.07,176.50,172.19,426.19", "242 73 1 0 0 0 236 63 1 0 0 0 210 73 1 167 151 1 233 151 1 0 0 0 224 229 1 0 0 0 275 230 2 167 313 1 229 310 2 203 446 1 289 424 2 0 0 0 300 561 2"),)),
    (365521, 117494, "d57fb884deed6688bb2e8fc64998f5015fc3766d2c376caaea1950b10c4c91bd", 428, 640, 5, "http://farm9.staticflickr.com/8029/7920294234_223ebd5544_z.jpg", ((439617, "214.76,37.69,188.87,216.15", "334 70 2 341 67 2 335 63 2 0 0 0 327 56 2 345 85 2 313 69 2 374 110 2 280 80 2 387 105 2 278 91 2 301 143 2 281 141 2 335 173 2 268 180 2 301 228 2 226 237 2"), (542658, "105.41,164.24,57.01,124.77", "0 0 0 0 0 0 0 0 0 139 173 2 155 177 2 127 192 2 153 195 2 115 213 2 155 217 2 0 0 0 159 230 2 122 242 2 141 243 2 116 275 2 137 276 1 108 301 1 133 304 1"),)),
    (366711, 288147, "0b143541e290144f4b515de9d81d238df12b438896e3817584d27a37b161d623", 427, 640, 4, "http://farm9.staticflickr.com/8463/8406266140_d29926b155_z.jpg", ((426629, "28.76,102.11,212.86,225.80", "192 160 2 207 151 2 184 154 2 228 157 2 0 0 0 241 218 1 139 217 2 260 290 1 82 251 2 154 306 1 58 289 2 229 384 1 162 385 1 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (369323, 98018, "237736cd6cfbb668e9b5e99cd08a826f63b79e53f76a7beb57dfd562a0914551", 377, 500, 5, "http://farm4.staticflickr.com/3159/2876155563_b7f860ac4b_z.jpg", ((467509, "42.88,66.68,257.21,395.46", "181 109 2 189 104 2 179 102 2 215 105 2 0 0 0 244 123 2 167 141 2 250 168 2 202 203 2 269 167 2 267 180 2 210 233 2 154 229 2 236 335 2 118 323 2 245 434 2 67 416 2"), (474673, "104.34,98.85,119.61,258.55", "123 125 1 127 121 2 0 0 0 145 126 2 0 0 0 167 152 1 118 152 2 167 193 1 112 190 2 144 242 1 116 234 2 164 217 1 131 215 2 178 280 2 121 278 1 213 333 2 124 341 1"),)),
    (370375, 135244, "620bf23ce72f95d868618f4c414aa9037fa6cba93daa465cfed30d03b0f55dec", 427, 640, 4, "http://farm3.staticflickr.com/2183/2435864370_901e470541_z.jpg", ((1242999, "186.97,87.73,189.84,545.08", "0 0 0 0 0 0 0 0 0 0 0 0 223 165 2 327 229 2 222 235 2 351 290 2 216 331 2 311 288 2 217 305 2 315 400 2 243 401 2 315 501 2 241 506 2 320 592 2 237 604 2"),)),
    (370486, 235294, "30df28745b0b675a6b7d00556dd516a4ad5afacb9a7a92bbb4fa638ea9753274", 421, 640, 5, "http://farm4.staticflickr.com/3823/9173218034_5fef21a1e0_z.jpg", ((435158, "348.42,127.28,72.58,380.00", "381 166 2 382 158 2 0 0 0 395 159 2 0 0 0 383 192 2 0 0 0 357 258 2 0 0 0 351 304 2 0 0 0 385 280 2 420 279 1 378 386 2 414 391 2 372 474 2 404 483 2"), (514102, "158.59,109.51,118.04,495.07", "240 147 2 0 0 0 233 141 2 0 0 0 210 151 2 240 204 2 174 207 2 260 277 2 191 281 2 201 278 2 218 239 2 249 359 2 186 363 2 252 466 2 189 456 2 231 564 2 192 573 2"), (1286407, "245.28,135.17,62.59,357.84", "271 165 2 273 156 2 262 155 2 0 0 0 249 158 2 285 208 2 237 212 1 287 269 2 0 0 0 295 233 2 0 0 0 283 311 2 243 311 1 286 372 2 251 378 1 266 454 1 252 458 1"),)),
    (371529, 234250, "1db0c839e7060314fa73cefcf19d616a4ced1e295b1b6f891c61f122fb448484", 512, 640, 4, "http://farm3.staticflickr.com/2689/4352374320_c024d09e1a_z.jpg", ((2161893, "33.28,52.49,309.99,522.51", "196 208 2 0 0 0 187 193 2 0 0 0 164 193 2 88 220 2 111 304 2 177 321 2 189 430 2 201 352 2 265 487 2 184 375 2 156 444 2 0 0 0 164 506 2 0 0 0 167 540 2"),)),
    (372819, 203698, "ba959429e80e0291609b81caa25e1a31abb24a3de0df6c7f1f68dd1649349d47", 640, 426, 4, "http://farm3.staticflickr.com/2046/2516944023_d00345997d_z.jpg", ((201291, "442.29,33.57,83.05,129.80", "500 47 2 504 45 2 497 44 2 509 48 1 492 47 1 517 65 2 485 65 2 513 88 2 480 89 2 493 87 2 497 95 2 505 109 2 490 109 2 484 98 2 483 118 2 460 136 2 477 155 2"), (205810, "543.22,22.01,67.67,151.53", "568 45 2 572 41 2 565 43 2 580 43 2 563 46 1 593 61 2 555 62 2 599 92 2 550 87 2 581 104 2 566 99 2 586 110 2 566 111 2 572 120 2 558 99 2 564 161 2 593 122 2"),)),
    (374369, 88591, "96c613c64bc275eabd73b24f2ffccc4b022c85df8c9384054d54d3af4e7f11a5", 640, 480, 4, "http://farm9.staticflickr.com/8228/8349316842_0d6dd5fdbf_z.jpg", ((519236, "227.13,33.10,156.46,332.37", "284 83 2 286 78 2 0 0 0 298 81 2 0 0 0 299 115 2 336 109 2 289 164 2 345 154 2 251 163 2 0 0 0 307 206 2 338 203 2 288 276 2 324 254 2 285 340 2 352 316 2"),)),
    (375493, 99860, "00d649a45082e0a1fada27c1c603f0ca4a70a63f34c5f1bc349ef9777bff6a70", 640, 480, 5, "http://farm1.staticflickr.com/33/38514047_b57c0ced5e_z.jpg", ((186499, "252.35,24.80,128.86,448.86", "338 81 2 343 75 2 329 75 2 0 0 0 308 72 2 0 0 0 302 106 2 0 0 0 269 157 2 329 239 2 275 220 2 273 230 2 297 220 2 294 311 2 349 326 2 296 404 2 310 439 2"),)),
    (378116, 185180, "1a3b7ce5613ce6c417d12d141b7d8c5a98cccf4b592c279bb783318e29847a11", 640, 454, 4, "http://farm6.staticflickr.com/5150/5619719330_f8c8934184_z.jpg", ((514927, "320.58,99.01,171.23,174.13", "352 128 2 355 122 2 349 123 2 365 117 2 0 0 0 380 138 2 361 130 2 394 145 2 345 129 1 415 163 2 323 152 2 425 179 2 414 185 2 430 224 2 415 225 2 478 252 2 473 252 2"),)),
    (378244, 181029, "c1fc0edbc1e6f3e3c7825a7d7e7206eeb020f417f7aeeeaeed7418f6f0353d50", 427, 640, 4, "http://farm5.staticflickr.com/4049/4632437600_9d863c8a11_z.jpg", ((451428, "135.70,261.29,113.28,139.79", "211 287 2 218 283 2 208 280 2 0 0 0 200 277 2 216 309 2 178 293 2 228 335 2 152 298 2 238 353 2 154 310 2 175 356 2 152 349 2 205 352 2 0 0 0 173 396 2 0 0 0"),)),
    (378454, 81103, "1ae76b4a2a9b1f4cb20cbc63977de7326eba87405aac9e16d0e958619ebccd98", 427, 640, 4, "http://farm1.staticflickr.com/192/462809186_70b4d6ea95_z.jpg", ((431114, "47.68,222.85,248.58,141.21", "0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 90 251 2 124 231 2 59 266 2 170 297 2 54 304 2 174 340 1 112 281 2 136 255 2 152 301 2 202 304 2 212 337 2 269 348 2"),)),
    (383842, 182606, "de9e05f37d8b3dd2c025ce80be273e364c6ad95dd337dacd7cd811f17db91e7b", 640, 434, 4, "http://farm5.staticflickr.com/4023/4536776986_0e75095fcb_z.jpg", ((436670, "255.52,50.71,324.77,383.29", "465 135 2 483 121 2 464 115 2 530 122 2 0 0 0 560 212 2 453 168 2 542 296 2 402 199 2 541 324 2 325 231 2 489 379 2 417 352 2 0 0 0 0 0 0 0 0 0 0 0 0"), (555546, "296.09,103.83,104.87,283.25", "368 135 2 374 130 2 364 128 2 387 134 2 0 0 0 398 174 2 338 154 2 0 0 0 0 0 0 0 0 0 0 0 0 364 248 1 336 243 1 352 315 2 320 307 2 344 369 2 309 361 2"), (569699, "228.22,34.95,101.42,362.80", "292 83 2 300 75 2 286 73 2 311 76 2 273 72 2 316 120 2 262 106 2 325 178 2 254 171 2 313 203 2 300 214 2 307 214 2 268 209 2 285 295 2 263 303 2 277 344 2 260 372 2"),)),
    (386352, 151173, "7bb02fb58747feee0f5464428b8a48076aa15a2e4dec1e4cab489c74742b950d", 640, 427, 5, "http://farm8.staticflickr.com/7054/7105320959_32647b8dc0_z.jpg", ((1281114, "244.83,3.25,150.36,181.21", "311 24 2 320 18 2 301 20 2 330 23 2 279 31 2 356 66 2 264 84 2 380 130 2 306 150 2 0 0 0 354 138 2 0 0 0 294 187 1 0 0 0 343 174 1 0 0 0 338 263 1"), (1304831, "489.37,75.51,150.63,332.01", "565 118 2 571 109 2 556 108 2 0 0 0 528 116 1 599 172 2 515 176 2 634 205 2 532 232 2 631 238 2 602 228 2 628 279 2 580 289 1 0 0 0 618 360 2 0 0 0 0 0 0"), (1755265, "365.85,86.45,267.80,333.17", "433 136 1 439 125 1 421 125 1 0 0 0 398 132 2 468 175 2 391 181 2 507 237 2 405 243 2 429 237 2 486 231 2 505 307 2 453 322 2 563 301 2 554 346 2 590 419 1 0 0 0"),)),
    (388056, 167914, "dae09ddbc116e2ffde1150640de08298620b6bef5e39e2caddbf16d6dadf4a2a", 500, 332, 4, "http://farm4.staticflickr.com/3218/2443485512_533ba95111_z.jpg", ((1294252, "393.92,94.75,106.08,206.66", "401 120 2 404 117 2 0 0 0 416 117 2 0 0 0 423 128 2 446 122 2 440 164 2 0 0 0 442 203 2 0 0 0 488 180 2 0 0 0 444 226 2 476 222 2 446 286 2 488 271 2"),)),
    (389532, 50720, "dbbd71bdca1e21d52a9e0ec34d0ed68141e45ab804e67d5adefa392b00378b98", 640, 421, 4, "http://farm8.staticflickr.com/7288/9496990496_470806b95e_z.jpg", ((183464, "467.36,139.31,58.65,170.29", "0 0 0 0 0 0 0 0 0 0 0 0 492 154 2 474 172 2 495 170 2 0 0 0 521 177 2 506 165 2 520 162 1 480 225 2 498 225 2 479 268 2 500 265 2 479 302 2 501 301 1"),)),
    (392722, 200702, "137bbeb12ff0d4152279be001c8c5e59b03e2aac88a939db1a3667fd4aac72c6", 640, 423, 4, "http://farm3.staticflickr.com/2829/9953419346_8bcbd5480f_z.jpg", ((234292, "580.79,252.56,44.68,130.23", "0 0 0 0 0 0 0 0 0 591 262 2 605 262 2 587 275 2 611 276 2 579 290 1 618 295 1 0 0 0 0 0 0 593 315 2 610 315 2 593 344 2 611 344 2 595 373 2 613 373 2"),)),
    (393226, 204532, "7cfb2fecb604d1f7e5429ab118ecda7fe84c296067ce8038a89e7ea8af836a0f", 640, 480, 4, "http://farm9.staticflickr.com/8003/7321339838_42fe225709_z.jpg", ((471260, "94.30,193.67,59.74,143.75", "137 207 2 0 0 0 134 205 2 0 0 0 126 206 2 126 221 2 114 220 2 132 242 2 110 241 2 144 254 2 122 260 2 123 268 2 119 268 2 111 298 2 131 292 2 101 327 2 138 327 2"),)),
    (393569, 114851, "83f53c85f83977281211de6cd34a7eef01944b02704b17a25cf213e839586256", 640, 480, 4, "http://farm5.staticflickr.com/4014/4222801461_63b8674d2a_z.jpg", ((434474, "375.37,121.56,196.32,220.04", "0 0 0 0 0 0 0 0 0 377 161 1 406 165 2 367 209 1 401 215 2 361 253 1 420 287 2 415 255 2 450 266 2 379 309 2 404 316 2 476 291 2 481 319 2 523 348 1 0 0 0"),)),
    (397133, 200576, "09e1d25c75f7879bdaa69c327fece5cabacd53939c8c2ef9e87f1c97a2e478c4", 640, 427, 4, "http://farm7.staticflickr.com/6116/6255196340_da26cf2c9e_z.jpg", ((200887, "388.66,69.92,109.41,277.62", "433 94 2 434 90 2 0 0 0 443 98 2 0 0 0 420 128 2 474 133 2 396 162 2 489 173 2 0 0 0 0 0 0 419 214 2 458 215 2 411 274 2 458 273 2 402 333 2 465 334 2"),)),
    (397303, 159379, "1c8a18f308b9febf0994c0bb26eed328c8a21cdb36a63a4432695994bb984af2", 640, 480, 5, "http://farm3.staticflickr.com/2433/3830853864_c3216021f5_z.jpg", ((485269, "336.54,65.80,194.16,393.71", "373 156 2 386 130 2 358 139 2 420 116 2 0 0 0 489 183 2 396 192 2 492 312 2 408 294 2 494 423 2 417 393 2 489 357 2 434 344 2 0 0 0 444 476 1 0 0 0 0 0 0"), (532898, "300.41,82.71,41.62,148.39", "330 101 2 0 0 0 330 98 2 318 96 2 326 96 2 312 109 2 322 109 2 0 0 0 325 129 2 0 0 0 331 152 2 311 163 2 318 164 2 315 194 1 329 193 2 318 225 1 331 225 1"),)),
    (397354, 109047, "067fb73c369a8433d1f9d97c2c36f51f487cf3a6d37c1f2fe0368a39f3dc38db", 640, 480, 5, "http://farm1.staticflickr.com/188/410311643_a3c829f3c7_z.jpg", ((205241, "338.84,153.83,62.66,179.55", "350 179 2 354 172 2 0 0 0 365 178 2 0 0 0 361 212 2 380 207 2 354 265 1 0 0 0 340 309 1 0 0 0 361 286 2 376 280 2 361 364 1 374 354 1 362 423 1 373 415 1"), (254502, "474.61,240.54,163.95,230.83", "604 284 2 616 277 2 597 274 2 629 285 2 0 0 0 636 333 2 570 301 2 587 380 2 531 342 1 541 394 2 0 0 0 576 424 2 544 389 2 482 437 2 465 410 1 0 0 0 0 0 0"),)),
    (399205, 201390, "6abf2e8fb8c46cf1e07a39b284846adce7ae28257faca20118fa34656990045b", 640, 480, 5, "http://farm4.staticflickr.com/3622/3567742683_f451d108b7_z.jpg", ((1237455, "73.95,24.59,207.08,455.41", "212 98 2 216 90 2 203 89 2 0 0 0 164 72 2 204 133 2 103 141 2 236 228 2 118 234 2 257 215 2 212 206 2 219 323 2 143 338 2 0 0 0 0 0 0 0 0 0 0 0 0"), (1257466, "313.89,77.66,129.44,395.87", "346 134 2 342 123 2 0 0 0 359 110 2 0 0 0 374 147 2 414 144 2 358 224 2 397 210 2 335 215 2 351 203 2 374 285 2 419 277 2 379 390 2 415 380 2 384 466 2 419 453 2"), (1281295, "392.64,92.66,110.66,348.37", "408 120 2 417 113 2 0 0 0 438 114 2 0 0 0 466 149 2 420 152 1 490 211 2 0 0 0 489 261 1 0 0 0 461 255 1 416 251 1 461 330 2 416 326 1 473 402 2 414 396 1"), (1294657, "508.17,116.36,46.95,121.95", "0 0 0 0 0 0 539 138 2 0 0 0 532 138 2 543 151 2 521 150 2 547 166 2 517 164 2 540 166 2 513 178 2 536 180 2 526 180 2 530 205 2 528 203 2 525 223 2 530 223 2"), (1691756, "237.06,88.65,128.21,344.61", "330 116 2 0 0 0 324 112 2 0 0 0 301 116 2 344 158 2 272 155 2 370 195 1 235 193 1 0 0 0 0 0 0 343 260 1 295 267 2 331 336 2 314 332 2 323 403 2 358 376 1"),)),
    (401862, 244329, "82e64d65eb79042a20684d83a09c4aa90b98dd4616eab25c8bb1d40d45fa4cef", 640, 480, 4, "http://farm9.staticflickr.com/8540/8698627837_6d1fb59ce4_z.jpg", ((1201609, "196.31,203.87,73.35,221.12", "0 0 0 0 0 0 0 0 0 236 218 1 0 0 0 226 244 2 260 245 2 223 279 2 260 279 2 209 282 2 0 0 0 233 309 2 260 307 2 221 364 2 251 362 2 221 416 2 250 413 2"), (1256141, "261.58,243.93,49.10,171.32", "270 259 2 274 257 2 0 0 0 285 258 2 0 0 0 278 278 2 286 271 2 270 297 2 0 0 0 0 0 0 0 0 0 284 318 2 292 309 2 285 363 2 272 359 2 287 402 2 275 398 2"), (1262391, "388.31,231.91,70.12,228.67", "0 0 0 0 0 0 0 0 0 408 252 2 433 252 2 400 278 2 442 278 2 393 307 2 448 307 2 0 0 0 0 0 0 403 342 2 434 342 2 408 395 2 432 396 2 410 441 2 435 445 2"), (1689412, "438.39,226.80,43.39,215.72", "0 0 0 0 0 0 0 0 0 446 246 2 467 243 2 461 271 2 464 261 2 440 292 1 0 0 0 442 263 1 0 0 0 455 331 2 458 328 2 452 372 1 458 374 1 455 430 1 458 413 1"),)),
    (402720, 124083, "94d78b7c6ded336af99a5d48fcd507ddeb7b43e93d1d7e7c1bdc62aa3d0b6fd6", 612, 612, 5, "http://farm8.staticflickr.com/7245/7276665146_20af39bbaf_z.jpg", ((434701, "375.45,242.05,235.17,298.44", "432 332 2 442 305 2 416 317 2 478 292 2 0 0 0 538 388 2 420 361 2 597 517 2 401 432 2 470 501 2 486 477 2 524 570 1 441 545 1 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (402774, 204500, "de2566157fe6dc38cd6a87b557d7b731079b39ea009e835f310f558445b506ca", 425, 640, 5, "http://farm9.staticflickr.com/8201/8158349940_3eb8a41660_z.jpg", ((486282, "127.71,158.01,149.24,464.94", "0 0 0 0 0 0 0 0 0 179 150 1 231 151 1 151 187 2 250 190 2 138 265 2 257 275 2 0 0 0 0 0 0 147 339 2 217 345 2 194 475 2 189 476 2 209 577 2 161 601 2"),)),
    (407002, 230557, "bb7a986f21b54cb4d9f35cdb753e29eb5a029f60ce21260b6103a5a2e3d63a3f", 480, 640, 4, "http://farm8.staticflickr.com/7156/6842071389_bd9154b1e1_z.jpg", ((464947, "210.91,187.27,116.07,385.62", "262 236 2 268 221 2 253 221 2 0 0 0 0 0 0 300 288 2 246 274 2 304 360 2 0 0 0 284 412 2 0 0 0 266 418 2 241 411 2 262 483 2 240 478 2 269 542 2 242 529 2"),)),
    (408696, 120455, "b73523841e0dd790e28cfb7b74fa2081ac156e012f56aa6a63ddcb3a9545a8a1", 640, 480, 4, "http://farm3.staticflickr.com/2339/2240506821_f84d6a9df5_z.jpg", ((453999, "250.81,42.70,241.08,418.38", "360 92 2 368 82 2 350 82 2 0 0 0 325 82 2 373 128 2 299 133 2 410 186 2 289 210 2 428 238 2 326 252 2 361 248 2 311 259 2 437 277 2 389 305 2 438 371 2 386 406 2"),)),
    (409475, 97749, "25f106ec83ccc0525071ab67706096e8e692dd651830982ffb297ebffa0be400", 640, 480, 4, "http://farm4.staticflickr.com/3541/3404640306_5487b904da_z.jpg", ((437799, "127.32,141.75,173.16,332.31", "194 183 2 197 176 2 187 179 2 214 181 2 179 187 2 233 221 2 167 222 2 239 273 2 149 274 2 278 294 2 164 294 2 217 334 2 175 334 2 219 408 2 186 413 2 204 468 2 156 473 2"), (1264001, "352.25,155.61,224.62,324.39", "470 201 2 479 192 2 461 192 2 491 194 2 451 198 2 507 242 2 448 236 2 533 304 2 435 300 2 528 344 2 391 330 2 520 370 2 531 363 2 473 462 2 541 459 2 0 0 0 0 0 0"),)),
    (410456, 243603, "d2dd4d77bad4f711f6d2f1ec9359c2d35cb406030f6a42ab40f8a2d4ec5d4aa2", 640, 480, 4, "http://farm9.staticflickr.com/8336/8438575589_a01a6990e2_z.jpg", ((1217336, "173.08,151.30,95.30,150.96", "238 171 2 0 0 0 235 169 2 0 0 0 226 167 2 238 184 2 207 182 2 245 208 2 199 204 2 241 229 2 197 227 2 231 231 2 212 232 2 250 256 2 191 258 1 256 292 2 189 289 1"), (1287263, "49.85,119.22,128.35,204.27", "129 140 2 0 0 0 127 135 2 0 0 0 119 136 2 122 162 2 102 158 2 128 193 2 86 190 2 135 220 2 84 220 2 123 214 2 102 213 2 141 264 2 85 265 2 151 310 2 61 304 2"), (1297145, "112.94,146.82,86.97,163.77", "167 164 2 0 0 0 164 163 2 0 0 0 153 164 2 171 180 2 139 180 1 177 205 2 134 207 2 179 229 2 0 0 0 166 230 2 145 230 2 187 258 2 125 261 2 189 297 2 119 293 2"),)),
    (411938, 263449, "8d5a4b2e84e045a974113175429619a0a0118f6f0b95124d53057c733f4ff8d1", 640, 427, 5, "http://farm6.staticflickr.com/5301/5694283812_1ba30885f9_z.jpg", ((193398, "264.24,156.17,155.10,265.20", "328 187 2 335 180 2 324 179 2 0 0 0 314 182 2 356 213 2 299 210 1 372 242 2 264 257 1 401 269 2 285 283 1 344 293 1 312 295 1 368 345 1 293 364 2 342 374 1 273 419 2"), (238383, "387.66,141.85,137.21,277.31", "429 170 2 434 167 2 424 167 2 445 169 2 0 0 0 453 199 2 405 205 2 469 240 2 412 248 2 490 264 2 434 266 2 441 281 2 414 287 1 491 293 2 410 363 2 483 363 2 0 0 0"), (1711138, "155.58,130.99,88.97,174.20", "223 157 2 0 0 0 221 153 2 0 0 0 207 153 2 219 182 2 176 186 2 231 215 2 167 224 2 238 245 2 191 257 2 220 266 2 189 268 2 230 316 1 181 320 1 0 0 0 0 0 0"),)),
    (412362, 200836, "f128deccbf3e1043c473351db3d0510e6ecc82329600e41938b31615f30bb4a1", 640, 499, 4, "http://farm7.staticflickr.com/6055/6220306659_5f590c47cf_z.jpg", ((567147, "442.93,71.84,197.07,419.39", "0 0 0 523 142 2 495 136 2 548 161 2 478 142 2 603 237 2 473 233 2 630 346 2 445 316 1 552 406 2 458 349 1 561 481 2 478 467 1 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (413552, 69579, "9551e30c4b72c2c8034424d1da07d5ca23a6f8f27462c2402561e2d83a039069", 426, 640, 4, "http://farm4.staticflickr.com/3339/3672181666_cfe25af65d_z.jpg", ((2203726, "93.09,29.92,332.47,541.92", "183 161 2 205 141 2 153 144 2 244 151 2 112 166 1 255 233 2 143 267 2 0 0 0 130 395 2 354 295 2 200 509 1 318 438 1 295 480 1 0 0 0 0 0 0 0 0 0 0 0 0"), (2203727, "0.00,0.00,426.00,640.00", "187 167 1 204 142 1 157 144 1 244 155 1 114 169 2 251 239 1 139 279 1 289 295 1 140 389 1 346 293 1 189 486 1 340 429 2 265 481 1 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (414510, 248413, "f6f2620707df79c606e0c1cfab462860ece49bee7768f0fb2dbdfac7bdf8346e", 480, 640, 4, "http://farm7.staticflickr.com/6011/5883783041_745340195b_z.jpg", ((2157191, "115.72,247.31,87.10,159.08", "167 270 2 0 0 0 164 266 2 150 264 2 0 0 0 127 288 2 144 285 2 122 319 2 176 306 2 0 0 0 164 282 2 116 359 1 135 361 1 168 379 2 192 346 2 134 383 2 160 389 2"),)),
    (414795, 157034, "8d3945f3c30be9a21aed6e9808f39e74a46bb20198f7bd29e24f4279e716af1e", 640, 480, 4, "http://farm1.staticflickr.com/69/221842206_28bfc63fc8_z.jpg", ((421551, "220.04,345.17,119.74,129.44", "273 371 2 0 0 0 272 366 2 0 0 0 259 367 2 229 381 2 258 403 2 245 398 2 286 448 2 281 407 2 321 428 2 236 455 2 266 466 2 0 0 0 328 431 2 0 0 0 311 478 1"),)),
    (417249, 143442, "a0788f6d609cf3a4e194b317b810b1e806fe95436588355cc76073d783a4f7d2", 480, 640, 4, "http://farm7.staticflickr.com/6081/6073438444_f4535c3cd4_z.jpg", ((467212, "179.78,250.01,133.75,245.93", "254 275 1 255 269 2 0 0 0 265 269 2 0 0 0 256 295 2 290 296 2 242 325 2 303 326 2 212 349 2 285 339 2 264 369 2 288 369 2 255 415 2 280 419 2 266 474 2 290 472 2"),)),
    (418062, 180456, "6878c319a0fcf0cb9a1f24efa3129fefe01b07be2c655746cfa31516f4344c52", 427, 640, 5, "http://farm8.staticflickr.com/7052/7003361838_ae193e2c8a_z.jpg", ((472132, "186.97,176.46,240.03,453.03", "257 243 2 283 225 2 241 228 2 310 239 2 231 245 1 376 325 2 218 336 2 0 0 0 193 429 2 383 543 2 215 486 2 332 579 2 225 568 2 0 0 0 0 0 0 0 0 0 0 0 0"), (512847, "0.00,153.89,195.60,473.17", "131 231 2 152 212 2 113 206 2 169 224 2 86 217 1 185 327 2 23 321 2 169 493 2 0 0 0 168 569 2 55 567 2 129 628 1 51 633 1 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (423123, 113251, "dabaea39eeb74df84ce54dbd486adb1c0d782dc01cca67fa2fa06ed34feb90d8", 640, 426, 5, "http://farm6.staticflickr.com/5328/9358340286_480b6f05db_z.jpg", ((428685, "36.38,88.07,352.29,333.14", "0 0 0 0 0 0 350 171 2 0 0 0 338 159 2 245 110 1 277 140 2 289 189 2 260 271 2 357 223 1 316 257 2 91 185 1 114 208 1 133 294 2 142 341 2 155 371 2 95 422 1"),)),
    (430377, 153785, "77d618c8a700daec616d1827bc0afff3fa5b6423670ccf91003506c02f2ee73e", 439, 640, 5, "http://farm9.staticflickr.com/8451/7903958976_9073ca847f_z.jpg", ((492491, "83.42,46.38,182.65,510.56", "166 96 2 175 87 2 159 86 2 185 96 2 138 96 2 200 157 2 128 158 2 227 237 2 102 236 2 233 262 2 96 269 2 194 294 2 145 292 2 187 416 2 146 415 2 176 521 2 116 527 2"),)),
    (433103, 194671, "103ec46b0058a4f51b5ead2764940cb02af9d3812023ab22dcaa14c1432a4496", 640, 425, 5, "http://farm8.staticflickr.com/7169/6771617469_cb52603210_z.jpg", ((1233577, "0.96,13.54,287.47,406.68", "125 125 2 132 106 2 109 113 2 0 0 0 51 97 2 133 166 2 29 169 2 145 241 2 24 282 2 172 276 2 89 352 2 135 286 2 84 294 2 249 362 1 100 404 2 0 0 0 0 0 0"), (1261053, "141.03,24.46,281.11,393.55", "299 116 2 0 0 0 299 103 2 0 0 0 266 86 2 268 136 2 182 117 2 281 216 2 160 219 2 350 210 2 254 238 2 285 251 2 220 261 1 375 315 2 311 349 2 0 0 0 0 0 0"), (1296539, "282.70,41.07,185.28,378.20", "342 122 2 350 116 2 335 114 2 0 0 0 304 106 1 344 153 2 283 156 2 0 0 0 0 0 0 0 0 0 332 219 2 356 241 2 317 249 1 447 272 2 417 290 2 439 374 2 417 404 2"), (1303521, "401.77,79.28,142.48,238.69", "464 139 2 467 132 2 0 0 0 448 117 2 0 0 0 410 127 2 437 129 2 400 173 1 430 171 2 453 192 1 465 191 1 413 207 1 434 198 1 497 240 1 512 228 2 490 316 1 499 310 1"), (1320081, "538.62,45.97,85.42,227.07", "593 85 2 602 80 2 590 78 2 614 79 2 580 72 1 603 97 2 561 91 2 599 139 2 531 113 1 596 153 2 0 0 0 579 161 2 558 162 1 614 188 2 595 194 2 587 244 2 567 241 2"), (1327326, "569.74,5.92,70.26,213.52", "596 28 2 600 24 2 594 24 2 614 22 2 0 0 0 634 49 2 588 50 2 0 0 0 0 0 0 616 84 2 0 0 0 614 105 2 593 105 1 612 157 2 589 154 1 613 209 1 586 210 1"), (1738456, "344.19,54.90,156.17,329.58", "390 119 2 0 0 0 385 109 2 0 0 0 361 97 1 390 131 2 339 133 1 409 175 2 337 192 1 448 193 1 380 195 2 414 214 1 368 216 1 484 243 2 460 266 2 457 326 2 424 355 1"),)),
    (437351, 150457, "0ab9e4c945f59f9b5da0d64fc45e200a64089b89345b563b83f2433e16cd7853", 399, 500, 5, "http://farm3.staticflickr.com/2106/2243101628_d1a74995f7_z.jpg", ((492988, "16.55,18.31,227.30,475.09", "121 85 2 131 74 2 104 78 2 152 77 2 0 0 0 176 133 2 68 133 2 153 221 2 116 233 2 76 244 2 201 257 2 154 261 2 95 260 2 133 273 2 74 310 2 178 434 2 0 0 0"),)),
    (438304, 269177, "64ab6a5556b022cbd398d98cd5bb243a4ee6e4ea6e3285f433eb78b76b53fd4e", 464, 640, 4, "http://farm6.staticflickr.com/5124/5333311458_e88df6ffe4_z.jpg", ((456037, "107.62,55.01,199.47,401.79", "257 97 2 264 95 2 254 88 2 0 0 0 232 83 2 253 141 2 197 116 2 257 182 2 216 176 2 275 205 2 248 204 2 203 227 2 171 214 2 176 335 2 203 325 2 122 393 2 195 435 2"), (1252849, "41.64,118.28,138.25,251.38", "92 152 2 97 147 2 86 146 2 107 142 2 78 143 2 125 159 2 70 168 2 143 175 2 50 204 2 0 0 0 77 213 2 131 217 2 88 219 2 145 275 1 68 276 2 165 337 1 55 349 2"),)),
    (440336, 102909, "e5c2df473a26427ae57950acec86d1e4d3a49cdf1a18d427cd1a354465408f00", 500, 335, 4, "http://farm4.staticflickr.com/3356/3490271591_e4923583b7_z.jpg", ((431499, "96.79,115.51,130.07,166.37", "183 141 2 184 137 2 177 137 2 0 0 0 166 142 2 176 158 2 157 155 2 183 171 2 179 163 2 187 197 1 203 163 2 144 190 2 134 190 2 137 221 2 156 220 2 107 253 2 162 274 2"), (491285, "189.60,130.19,45.33,128.46", "225 147 1 0 0 0 222 146 1 0 0 0 215 147 2 220 158 1 207 161 1 220 169 2 214 181 2 225 176 2 225 197 2 208 187 2 200 187 2 209 212 2 207 213 2 206 232 2 205 250 2"), (501274, "268.75,109.03,76.79,148.31", "289 128 2 293 125 2 286 126 2 300 127 2 0 0 0 318 143 2 279 154 2 337 157 2 274 181 2 336 172 2 279 201 2 317 187 2 295 194 2 324 224 2 300 225 2 322 244 2 307 238 2"),)),
    (441442, 154542, "2d1d5253fc1c62b68d70407e15a4c0375ab3c972a5b71f2242b23219ae32e1a5", 600, 400, 4, "http://farm8.staticflickr.com/7284/8715724078_725b401d61_z.jpg", ((216406, "251.69,52.13,74.60,177.08", "279 78 2 281 74 2 0 0 0 292 75 2 0 0 0 305 96 2 291 97 2 303 125 2 283 129 2 284 144 2 270 142 2 304 144 2 299 142 2 288 170 2 0 0 0 307 218 2 0 0 0"),)),
    (447187, 109434, "3535201a7c9ec8aab07ad195809c3f2b30e54df8546dfd4adb5d20f8aa366009", 640, 480, 4, "http://farm4.staticflickr.com/3094/2684280938_a5b59c0fac_z.jpg", ((448505, "217.79,196.51,204.78,251.38", "0 0 0 0 0 0 0 0 0 292 236 2 338 237 2 271 290 2 370 287 2 270 338 2 390 335 2 0 0 0 401 384 2 296 399 2 341 397 2 256 352 2 384 346 2 250 417 2 377 417 2"), (498458, "58.51,108.08,170.11,359.73", "0 0 0 0 0 0 0 0 0 158 151 2 199 155 2 131 178 2 200 180 2 66 223 2 179 207 2 102 271 2 141 271 2 86 241 2 120 247 2 141 330 2 106 339 2 130 425 2 83 439 2"), (541792, "492.16,170.95,84.66,132.28", "506 201 2 508 196 2 501 199 2 516 194 2 501 204 2 528 203 2 508 218 2 543 187 2 510 242 2 553 178 2 508 233 2 547 230 2 531 238 2 556 273 2 529 263 2 573 268 2 532 295 2"),)),
    (447314, 83211, "3fb6b82c5c04e6fd280a51bc5a17331c3f1c799531eb294873802f34e85c7c97", 640, 458, 5, "http://farm1.staticflickr.com/42/263967623_cc4ad0bbbb_z.jpg", ((435623, "88.51,227.46,132.77,172.90", "176 243 2 178 241 2 172 240 2 0 0 0 163 241 2 176 262 2 151 257 1 188 283 2 137 272 2 203 276 2 125 281 2 159 308 2 138 305 2 119 337 2 132 346 2 114 382 2 131 388 2"),)),
    (451144, 175772, "eacf66836810f340717892b95d529f160c7c7c663181a96d82584ad364135bbe", 640, 480, 5, "http://farm7.staticflickr.com/6001/5967535194_c90bfa09fe_z.jpg", ((449097, "379.10,193.16,72.80,185.84", "0 0 0 0 0 0 0 0 0 404 209 2 0 0 0 398 229 2 436 225 2 395 258 2 442 262 2 385 280 2 430 286 2 412 283 2 427 282 2 391 320 2 424 322 2 395 366 2 437 365 2"), (449173, "516.92,204.29,71.94,158.17", "0 0 0 534 224 2 0 0 0 539 223 2 547 219 2 539 240 2 561 233 2 533 266 2 0 0 0 0 0 0 0 0 0 551 284 2 561 285 2 535 317 2 559 318 2 530 346 2 572 341 2"),)),
    (457884, 161561, "b88b479e718e62d179773be99c3d0708e62d1c86d2f0adc118b733d0659dff48", 640, 458, 4, "http://farm4.staticflickr.com/3438/3751850733_03b65c54c5_z.jpg", ((487194, "522.11,129.16,91.31,181.20", "573 143 2 578 141 2 571 141 2 587 147 2 0 0 0 591 171 2 557 168 2 593 190 2 545 189 2 587 196 2 533 204 2 589 213 2 561 215 2 592 258 2 561 261 2 598 295 2 556 298 2"), (2207717, "276.10,116.69,128.64,267.86", "344 163 2 351 161 2 0 0 0 362 174 2 0 0 0 355 188 2 369 194 2 336 164 2 349 238 2 309 139 2 313 239 2 366 246 2 387 251 2 336 306 2 376 319 2 354 358 2 389 364 2"),)),
    (458045, 148871, "db6f717462215404a51b56d39d70f0653f773317898fc64758a4573cccc76b7b", 500, 375, 5, "http://farm1.staticflickr.com/186/469842019_a13b928a2f_z.jpg", ((1207264, "366.09,138.17,93.90,133.52", "417 158 2 419 155 2 414 155 2 424 156 2 410 157 2 425 166 2 409 170 2 430 174 2 428 181 2 444 180 2 445 186 2 423 205 2 407 206 2 438 232 2 399 237 2 447 260 2 375 253 2"), (1287219, "195.29,116.12,45.57,120.84", "219 134 2 221 132 2 217 132 2 0 0 0 213 132 2 227 144 2 209 143 2 235 141 2 204 139 2 230 130 2 209 129 2 226 180 2 209 179 2 221 204 2 207 202 2 218 229 2 200 226 2"), (1691080, "64.45,132.93,136.80,134.75", "172 160 2 175 154 2 168 153 2 0 0 0 160 152 2 172 172 2 140 175 2 0 0 0 132 204 2 0 0 0 133 233 2 141 211 2 115 213 2 166 206 2 127 258 2 178 253 2 83 255 2"),)),
    (459272, 91481, "86fcc50b7b87ac6fb66e75e2158d0017730a1f05c5e6351c0d3e87bb760223f9", 480, 640, 4, "http://farm3.staticflickr.com/2448/3675278938_d31af5d707_z.jpg", ((1757755, "401.76,252.23,78.24,170.76", "450 278 2 453 273 2 444 274 2 0 0 0 432 275 2 461 305 2 418 307 2 472 338 2 408 349 1 0 0 0 0 0 0 463 381 1 431 384 1 470 430 1 436 434 1 0 0 0 0 0 0"),)),
    (463199, 128069, "eabc38a2ae22262952cc2c406be88cbc61e7d62b33d613ed040778d60064330f", 640, 480, 5, "http://farm2.staticflickr.com/1055/1399822653_c1d0fbc717_z.jpg", ((219934, "291.44,106.95,196.44,373.05", "356 165 2 367 152 2 345 152 2 392 161 2 333 160 2 441 240 2 316 237 2 505 319 1 302 286 2 459 368 1 313 343 2 417 422 2 351 414 2 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (463690, 141994, "ae3f98e35c69157ae5653541324d3fa8c27680054a4aaf1c7420aae3d03bffff", 640, 427, 4, "http://farm7.staticflickr.com/6224/6272789669_be80f59938_z.jpg", ((424293, "163.86,20.96,80.97,211.09", "207 42 2 211 39 2 203 41 2 219 38 2 199 43 2 232 61 2 198 64 2 235 90 2 193 95 2 216 70 2 191 124 2 226 123 2 201 122 2 227 171 1 201 171 2 230 217 1 203 212 2"), (429221, "49.02,20.40,48.93,191.07", "57 37 2 61 34 2 54 34 2 68 33 2 0 0 0 76 55 2 52 53 1 84 85 2 0 0 0 70 107 2 0 0 0 73 117 2 57 118 1 75 153 2 67 154 2 83 194 2 73 192 2"), (438578, "259.49,141.04,261.36,280.56", "326 306 2 0 0 0 314 305 2 0 0 0 307 326 2 369 344 2 292 353 2 431 365 2 278 419 2 485 379 2 0 0 0 412 397 2 350 406 2 446 294 2 341 273 2 449 412 2 328 164 2"), (501114, "1.19,18.42,45.32,196.61", "9 45 2 14 42 2 6 42 2 19 36 2 0 0 0 27 56 2 0 0 0 44 78 2 0 0 0 27 84 2 0 0 0 22 113 2 5 114 2 24 149 2 6 152 2 29 188 2 12 193 2"),)),
    (473406, 137909, "821a802ea625958f0c8465a313723b265674f891482e9e6dff781eeed3314385", 480, 640, 5, "http://farm4.staticflickr.com/3422/3941022247_9b39c55b5a_z.jpg", ((454237, "84.85,93.48,245.94,526.39", "242 200 2 254 178 2 218 183 2 277 189 2 193 196 2 288 268 2 175 256 2 312 342 2 174 316 2 281 349 2 160 374 2 244 391 2 191 386 2 174 435 2 100 321 2 167 531 2 128 442 2"),)),
    (473869, 109371, "d2f1193dc4c0cd50e0a233810fde1875f7211e936c35d9a3754bf71c2c8da84e", 640, 427, 5, "http://farm4.staticflickr.com/3500/3725168577_4b8a7d72b9_z.jpg", ((475292, "51.70,12.40,273.82,388.70", "242 71 2 263 76 2 231 52 2 0 0 0 199 49 2 214 156 2 131 111 2 188 282 2 133 259 2 278 293 2 269 266 2 179 396 2 92 397 2 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (474028, 246520, "f2c62d024959d0961cd5bee4f78efc015c221b908005c5bbdbeaaede7d8d0cf6", 640, 427, 4, "http://farm8.staticflickr.com/7086/7258228366_b29766be26_z.jpg", ((1231218, "407.39,76.08,189.03,285.95", "452 114 2 457 108 2 449 109 2 477 107 2 0 0 0 501 117 2 444 142 2 531 118 2 424 187 2 570 109 2 419 174 2 508 202 2 474 213 2 549 254 2 506 266 2 554 335 2 485 349 2"), (1317826, "126.66,224.53,191.91,172.72", "135 261 2 141 258 2 0 0 0 164 254 2 0 0 0 190 276 2 158 272 2 242 308 2 185 287 2 199 328 2 163 316 1 239 331 2 209 331 2 217 389 2 185 381 2 291 364 2 251 352 2"),)),
    (474078, 175959, "607abd6bca3220055496c450a547b3cf86bd2494cecad96c9a812902cd084046", 640, 480, 5, "http://farm4.staticflickr.com/3278/2554058095_4a57649215_z.jpg", ((530761, "127.40,273.67,151.44,183.73", "0 0 0 0 0 0 0 0 0 0 0 0 226 297 2 186 304 2 225 317 2 155 325 2 250 349 2 170 356 2 269 346 2 153 353 2 180 362 2 160 384 2 228 390 2 137 426 2 228 435 2"), (1729803, "316.36,216.27,157.65,173.95", "404 240 2 0 0 0 398 238 2 0 0 0 386 241 2 376 252 2 386 276 2 391 264 2 419 293 2 413 275 2 454 287 2 352 302 2 331 306 1 383 317 2 320 347 1 374 378 2 284 380 1"), (2158053, "238.04,291.94,166.49,162.55", "0 0 0 0 0 0 330 316 2 0 0 0 322 318 2 288 333 2 307 337 2 338 335 2 322 368 2 366 332 2 336 352 2 259 392 2 279 390 2 307 384 2 334 381 2 264 425 2 317 442 2"),)),
    (476119, 204732, "b1e77b03b318546e0725250efdfb28a1ce24414750ae8e8c3b5097c5ffddf415", 640, 427, 4, "http://farm8.staticflickr.com/7046/7062224927_570b4f61f7_z.jpg", ((466988, "91.70,16.33,240.73,378.28", "229 60 2 238 55 2 222 54 2 0 0 0 0 0 0 260 86 2 184 87 2 287 132 2 145 132 2 307 146 2 116 183 2 224 214 2 179 211 2 199 296 2 180 287 2 144 343 2 111 361 2"),)),
    (476770, 217181, "d422b950c039d32c351565e096532dd7e13e65301f312493eae8f265f446ddce", 640, 426, 4, "http://farm5.staticflickr.com/4120/4802608215_14b4b44a1e_z.jpg", ((1287039, "235.50,116.79,112.96,185.72", "249 147 2 252 139 2 245 143 2 265 136 2 0 0 0 275 154 2 0 0 0 293 167 2 0 0 0 300 182 2 0 0 0 325 203 2 321 209 2 264 214 2 280 241 2 285 276 2 308 296 2"), (1299337, "362.00,40.12,87.88,233.05", "384 77 2 387 71 2 379 71 2 396 72 2 371 72 1 413 90 2 371 100 2 441 111 2 377 139 2 427 79 2 392 131 2 417 169 2 389 170 2 0 0 0 396 222 2 0 0 0 377 240 2"),)),
    (477288, 163874, "b03abc9ddfb262030d239d7052951a776df42761d7d89cd598af9c434cfce94f", 480, 640, 4, "http://farm1.staticflickr.com/44/145871865_ebd837f546_z.jpg", ((453649, "194.16,69.03,194.15,524.95", "0 0 0 0 0 0 361 115 2 0 0 0 333 118 2 302 158 2 311 173 2 267 259 2 315 234 2 265 313 2 357 168 2 333 321 2 309 311 2 370 437 1 303 440 2 0 0 0 220 530 2"), (492460, "133.01,207.53,130.68,267.51", "225 240 2 0 0 0 223 236 2 0 0 0 212 240 2 210 261 2 197 262 2 0 0 0 214 292 2 234 278 2 244 272 2 215 340 2 200 339 2 227 401 2 195 400 2 215 447 2 147 425 2"), (508221, "0.00,117.48,165.77,448.29", "95 154 2 97 147 2 89 145 2 0 0 0 61 148 2 95 210 2 21 210 2 0 0 0 3 289 2 0 0 0 13 274 2 90 342 2 56 343 2 107 439 2 33 437 2 119 532 2 6 505 2"),)),
    (479126, 194152, "a64293cb5d4bfa0b1e6e1c0603f7a4f9892afd90f353c9ed0b250449818d5d36", 640, 427, 4, "http://farm6.staticflickr.com/5189/5734725144_37742ab51b_z.jpg", ((452193, "336.67,156.87,236.43,257.10", "434 241 2 449 222 2 424 226 2 481 218 2 0 0 0 496 267 2 465 272 2 557 384 2 456 352 2 454 379 2 385 339 2 487 426 1 465 414 1 298 423 1 298 410 1 0 0 0 0 0 0"), (2152991, "108.69,148.29,108.69,268.40", "177 178 2 0 0 0 174 171 2 0 0 0 155 173 2 173 208 2 123 206 2 178 253 2 132 258 2 171 263 2 174 260 2 168 277 2 135 273 2 191 289 2 178 310 2 168 362 2 181 400 2"),)),
    (479953, 185727, "2b098be3cd25cbc1d3d0f38189b26d2b7f956cbeb0023b9fef7dfd23208779c9", 640, 393, 5, "http://farm9.staticflickr.com/8425/7757941410_6452048d5e_z.jpg", ((442946, "86.94,99.14,184.52,159.68", "212 122 2 0 0 0 209 116 2 0 0 0 198 113 2 190 126 2 193 137 2 167 131 2 185 161 2 147 133 2 185 185 2 194 190 2 172 188 2 230 204 2 145 210 2 248 244 2 100 214 2"),)),
    (480936, 116216, "2487034e07fc8d7b581e0d920afae23f801f5005b5cd37b9d7fe3ed47a8a319d", 482, 640, 4, "http://farm2.staticflickr.com/1154/560533087_715b92d751_z.jpg", ((519833, "62.60,8.53,311.29,631.47", "206 73 2 215 56 2 183 66 2 246 64 2 169 85 2 282 148 2 150 151 2 324 258 2 92 254 2 329 193 2 147 309 2 262 301 2 166 306 2 277 412 2 197 420 2 265 552 2 166 504 2"),)),
    (481386, 264715, "f218e3c3a63a1600ad6b696d35f324bf78b0f787459d10793e39b420f80cad27", 427, 640, 4, "http://farm6.staticflickr.com/5328/8789640387_14e10562cb_z.jpg", ((228697, "96.36,96.36,224.36,486.11", "220 159 2 233 148 2 206 147 2 247 166 2 186 154 2 255 204 2 150 206 2 196 267 2 152 267 2 134 256 2 179 303 1 253 291 2 196 310 1 248 408 2 223 406 2 229 524 2 200 493 2"),)),
    (481413, 105634, "2ce2ac7144577c0a6349c272e034295852aa5e1d0e81dcf5451f148e72e98d3d", 640, 427, 4, "http://farm1.staticflickr.com/23/35099655_d7c448ccd7_z.jpg", ((512187, "201.96,101.71,132.82,232.97", "0 0 0 0 0 0 0 0 0 250 135 2 280 132 2 241 158 2 280 159 2 226 189 2 283 181 2 213 217 2 315 194 2 245 222 2 269 221 2 236 269 2 268 267 2 234 315 2 268 320 2"),)),
    (482800, 183646, "06d730d8b8e3175f8b3ff03ec402ff479f0eb5e78d2ca70f02251a38570db006", 640, 457, 4, "http://farm4.staticflickr.com/3214/2734186920_4d1f5b2cff_z.jpg", ((424389, "333.24,81.10,165.77,368.47", "402 121 2 409 113 2 397 113 2 426 114 2 0 0 0 452 160 2 375 163 2 474 210 2 361 219 2 488 262 2 344 270 2 437 258 2 393 259 2 420 346 2 401 334 2 426 409 2 410 403 2"), (528200, "128.37,106.80,66.75,261.88", "157 136 2 162 131 2 153 131 2 173 134 2 0 0 0 177 163 2 149 161 2 184 206 2 142 200 2 173 241 2 134 237 2 176 232 2 153 230 2 179 296 2 136 296 2 185 343 2 159 337 2"), (538003, "303.98,88.39,91.40,243.39", "372 107 2 373 103 2 366 103 2 0 0 0 353 108 2 388 131 2 339 143 2 0 0 0 328 174 2 0 0 0 313 210 2 393 196 1 358 202 1 402 249 1 362 261 2 407 306 1 362 316 2"),)),
    (484351, 217452, "1c1890dbb87c386fdf3cce1b47701aff184c1d995b1a9cec232b5394903fde0c", 640, 426, 4, "http://farm9.staticflickr.com/8232/8528454206_1749cd4960_z.jpg", ((1213878, "388.67,113.92,181.88,307.29", "480 167 2 487 155 2 468 158 2 0 0 0 444 156 2 525 194 2 422 189 2 549 272 2 395 219 1 483 277 2 416 280 1 499 288 2 430 287 1 558 362 2 0 0 0 508 408 1 0 0 0"), (1255944, "122.53,28.72,55.53,155.08", "150 43 2 152 41 2 147 41 2 156 44 2 140 43 2 161 62 2 133 63 2 170 90 2 133 97 2 173 107 2 141 121 2 165 117 2 149 117 2 166 148 2 146 150 2 166 179 1 149 183 1"), (1276280, "329.74,36.98,54.82,182.09", "341 54 1 343 50 2 0 0 0 352 52 2 0 0 0 369 68 2 342 66 2 377 99 2 336 86 2 354 105 2 342 103 2 366 119 2 343 117 2 369 167 2 339 163 1 374 204 2 0 0 0"),)),
    (490515, 54346, "a6654419cd09f606a15c6120ba95f9976d364841d8499048b98675c27149b6d4", 640, 427, 4, "http://farm5.staticflickr.com/4057/4224575445_ebe5ef59b3_z.jpg", ((441107, "190.34,117.02,321.19,187.95", "289 141 2 288 135 2 281 137 2 0 0 0 266 145 2 291 165 2 262 181 2 0 0 0 241 200 2 0 0 0 210 216 2 351 216 2 337 228 1 421 233 2 410 254 2 493 255 2 475 278 2"),)),
    (491366, 110675, "e8b01653f4ef3eace69310ede3a0a31311b2fb4cb5f0d63ef560ac01e91acd26", 500, 375, 5, "http://farm4.staticflickr.com/3031/2548567657_4f81a39e0f_z.jpg", ((190722, "88.09,117.59,129.31,240.84", "116 145 2 117 138 2 107 143 2 0 0 0 97 150 2 142 162 2 103 178 2 169 149 2 0 0 0 135 138 2 0 0 0 165 230 2 135 246 1 182 287 1 164 295 2 0 0 0 186 340 2"), (196779, "0.00,56.88,178.18,318.12", "75 110 2 0 0 0 68 104 2 0 0 0 40 107 2 57 145 2 15 179 2 102 199 2 51 263 2 132 251 2 120 266 2 140 282 2 108 305 2 173 364 1 0 0 0 0 0 0 0 0 0"),)),
    (492282, 168365, "9d3f32bb93adf432173ff294315d7632e8a9947b40650d0e8480e0c95216ab8c", 480, 640, 4, "http://farm4.staticflickr.com/3086/2567250393_eb51096d65_z.jpg", ((211337, "158.93,230.65,73.02,191.86", "199 264 2 204 259 2 196 260 2 214 262 2 0 0 0 223 291 2 194 285 2 208 324 2 191 315 2 187 348 2 179 330 2 219 342 2 202 339 2 210 380 2 0 0 0 214 421 1 0 0 0"), (214784, "380.38,270.51,48.78,137.56", "404 287 2 408 284 2 402 284 2 416 286 2 0 0 0 423 305 2 396 302 2 420 328 2 392 322 2 399 331 2 391 332 2 412 340 2 395 339 2 405 359 2 0 0 0 410 393 2 0 0 0"), (225403, "0.00,262.21,47.67,180.59", "16 277 2 20 274 2 12 273 2 26 275 2 0 0 0 30 295 2 14 294 2 23 323 2 17 328 2 1 318 1 0 0 0 35 348 2 21 347 2 15 379 2 3 366 2 10 428 2 3 401 2"),)),
    (492362, 223908, "f42e9fe48bafa7571d3d862c19a2601e405eba5beb056a694e0e698eb42845a6", 427, 640, 5, "http://farm8.staticflickr.com/7383/9825580104_5ffd1e274c_z.jpg", ((423295, "97.93,104.47,168.85,452.52", "104 159 2 104 149 2 0 0 0 132 145 2 0 0 0 203 180 2 140 200 2 252 251 2 154 259 2 182 233 2 146 269 2 216 328 2 166 327 2 221 430 2 167 437 2 226 512 2 168 524 2"), (429200, "268.93,165.69,85.65,276.28", "325 197 2 325 190 2 319 191 2 0 0 0 0 0 0 330 222 2 286 227 2 0 0 0 278 271 2 343 261 2 315 264 2 335 298 2 304 300 2 340 361 2 312 365 2 330 419 2 287 411 2"),)),
    (493864, 144060, "531fe470e68b60a2954ab5ea2649411b3727197a613a8a413559708cb24f61ce", 480, 640, 4, "http://farm8.staticflickr.com/7344/9691828107_2258c3d127_z.jpg", ((421918, "121.70,84.74,197.58,478.21", "178 140 2 188 128 2 167 131 2 200 127 2 155 134 2 226 169 2 137 190 2 252 235 1 176 255 2 0 0 0 235 228 2 245 321 2 192 326 2 284 393 2 196 417 2 282 487 2 195 522 2"),)),
    (496854, 184111, "e537a5602570eaa29fc40932434113959077e8fb69eb4b63a6c3cce1f3ce6933", 640, 480, 5, "http://farm3.staticflickr.com/2795/5715794037_73f069675d_z.jpg", ((556737, "276.58,156.21,167.79,323.79", "352 204 2 359 197 2 346 196 2 379 194 2 0 0 0 407 239 2 314 237 2 433 293 2 292 288 2 392 301 2 288 322 2 384 369 2 335 370 2 373 475 2 344 475 2 0 0 0 0 0 0"), (563070, "247.39,202.95,51.82,132.50", "0 0 0 0 0 0 0 0 0 266 213 2 276 213 2 259 227 2 284 229 2 251 245 2 290 251 2 0 0 0 293 263 2 261 268 2 280 269 2 263 298 2 280 299 1 263 326 2 279 324 1"), (563757, "85.21,149.12,193.08,330.88", "149 193 2 156 183 2 138 185 2 0 0 0 117 191 2 176 226 1 107 243 2 230 229 2 99 307 2 263 194 2 96 379 2 189 393 2 136 394 2 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (499622, 106691, "b863e07e2ae99859bd5d3713fa30cbd580a24fa6f01e8e6a65394e9967819ee8", 412, 456, 5, "http://farm7.staticflickr.com/6158/6183504440_29f4164214_z.jpg", ((187537, "151.98,41.34,121.23,294.95", "224 79 2 230 73 2 215 74 2 0 0 0 198 80 2 241 112 2 179 117 2 280 140 1 162 159 2 0 0 0 177 175 2 248 187 1 193 192 2 0 0 0 196 217 2 0 0 0 199 312 2"),)),
    (504074, 130545, "20b41a4613f4764272f97bd4522ffeefd52cd0e9a7cb604578e2644a7dce4c83", 640, 427, 4, "http://farm6.staticflickr.com/5266/5734723994_0d5f3754d9_z.jpg", ((484032, "100.02,75.98,103.86,283.70", "139 109 2 146 105 2 133 106 2 155 107 2 124 107 1 165 144 2 113 140 2 170 190 2 123 193 2 149 205 2 174 196 2 153 209 2 115 212 2 180 214 1 174 245 2 154 295 1 174 334 2"), (2154820, "260.29,87.37,293.58,339.63", "412 176 2 429 156 2 406 158 2 467 154 2 0 0 0 487 201 2 460 203 2 531 305 2 444 279 2 455 321 2 367 271 1 475 380 2 452 347 2 281 356 2 282 344 2 0 0 0 0 0 0"),)),
    (506178, 101146, "de41bacebcb85dfdf3f86b66ff169f2d2e2f7151d7b6b22d42b675a3a767ffa6", 375, 500, 4, "http://farm1.staticflickr.com/155/411777080_a4dfa5fef0_z.jpg", ((1722608, "70.63,90.94,229.82,403.59", "201 145 2 206 135 2 186 134 2 0 0 0 152 146 2 217 220 2 120 219 2 231 289 2 86 323 2 259 320 2 136 298 2 206 398 2 144 395 2 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (507015, 130210, "3e1e27b0ffcd3d5673c69ba4914fa68ab44c23716e2435111ea5298648e68a70", 640, 426, 4, "http://farm4.staticflickr.com/3421/3225241751_061cbe8156_z.jpg", ((461717, "270.30,166.59,81.30,131.28", "323 176 2 327 175 2 321 174 2 0 0 0 316 175 2 334 193 2 309 193 2 344 212 2 293 204 2 346 226 2 282 216 2 328 231 2 311 228 2 320 261 2 300 254 2 332 278 2 295 293 2"), (473712, "367.84,163.95,64.65,142.54", "406 177 1 0 0 0 403 175 2 0 0 0 394 179 2 408 193 2 381 194 2 421 205 2 375 217 2 429 196 2 374 235 2 401 231 2 385 231 2 400 263 2 382 259 2 395 299 2 373 294 2"), (501709, "485.35,185.72,66.06,221.13", "543 205 2 0 0 0 541 202 2 0 0 0 529 204 2 535 227 2 512 234 2 0 0 0 514 270 2 0 0 0 522 298 2 538 290 2 519 293 2 528 343 2 513 343 2 519 386 2 497 389 2"), (503329, "82.95,133.91,47.20,142.05", "114 148 2 0 0 0 112 145 2 0 0 0 103 146 2 111 165 2 96 163 2 120 185 2 93 183 2 124 198 2 106 198 2 110 206 2 97 202 2 107 236 2 98 235 2 102 261 2 90 264 2"),)),
    (507223, 166000, "dca48ffaf4a309da07246199c52a8086576a7bffa222c6209998d24123db48ac", 480, 640, 4, "http://farm2.staticflickr.com/1404/1307049889_f157382a16_z.jpg", ((2164598, "109.17,95.29,201.06,531.94", "203 134 2 212 124 2 190 126 2 0 0 0 178 132 2 245 187 2 151 193 2 275 240 2 106 266 1 285 296 2 133 268 2 236 330 2 170 327 2 241 467 2 188 474 2 184 535 2 201 583 2"),)),
    (511076, 202839, "9a159bde5fa9b68ab5f5ed7818091f0bbfc1402d328cf49ee845911cb85ad247", 640, 427, 4, "http://farm8.staticflickr.com/7350/10147993846_a41ff2f060_z.jpg", ((209739, "352.49,72.69,190.10,303.77", "0 0 0 0 0 0 0 0 0 459 104 2 0 0 0 492 137 2 473 135 2 499 203 2 481 194 2 451 191 2 429 213 2 503 229 1 483 227 1 434 255 2 411 250 2 415 343 2 396 326 2"),)),
    (516173, 94845, "e69ad4420e419f68f080a90eeb4eb2e2b01725ef9b71b1bfd656b28a51b12f30", 408, 640, 4, "http://farm5.staticflickr.com/4026/4390072668_db704274f8_z.jpg", ((444129, "58.97,156.52,168.27,428.59", "0 0 0 0 0 0 0 0 0 113 194 2 161 201 2 92 257 2 174 253 2 72 339 2 209 340 2 77 399 2 212 408 2 106 397 2 170 397 2 115 483 2 152 483 2 130 589 1 148 577 2"),)),
    (516677, 258869, "78547387ed604e3065739508b75e28f640efa9db784a6419c4ac44d924f3a599", 640, 480, 4, "http://farm6.staticflickr.com/5186/5789961160_d750638deb_z.jpg", ((1260998, "314.89,161.31,51.85,146.89", "332 181 2 332 178 2 0 0 0 337 176 2 0 0 0 350 185 2 348 186 2 362 206 2 337 205 2 348 221 2 326 203 2 352 227 2 349 227 2 346 258 2 343 256 2 358 294 2 350 288 2"), (1269910, "455.02,151.51,43.71,162.20", "466 170 1 468 167 1 0 0 0 476 167 2 0 0 0 484 185 2 488 184 2 482 207 2 0 0 0 475 220 2 470 200 2 479 227 2 482 227 2 480 271 2 483 270 2 483 301 2 488 298 2"), (1282888, "324.08,257.19,126.02,132.38", "385 286 2 387 284 2 0 0 0 400 278 2 0 0 0 421 287 2 396 286 2 401 317 2 379 306 2 369 334 2 360 328 2 432 326 2 410 322 2 409 348 2 386 336 2 383 376 2 347 353 2"),)),
    (520707, 137326, "2757d460f5dc41dfaf6a8f7d61000663d697af89bb8dd7aa9b7771b2bc1c8776", 640, 480, 4, "http://farm1.staticflickr.com/23/30474880_7ceed20fb0_z.jpg", ((1201866, "326.83,138.97,78.74,255.64", "0 0 0 0 0 0 0 0 0 346 169 2 0 0 0 346 193 2 382 187 2 347 239 2 397 226 2 365 263 2 385 255 2 361 263 2 387 260 2 361 317 2 386 313 2 370 379 2 397 371 2"), (1702757, "402.19,155.99,49.48,136.14", "403 167 2 406 165 2 0 0 0 0 0 0 0 0 0 424 186 2 409 185 1 426 204 2 0 0 0 409 201 2 0 0 0 425 223 2 414 222 2 429 252 2 409 251 2 444 271 2 414 284 2"), (1756536, "99.31,179.82,102.86,234.09", "0 0 0 0 0 0 0 0 0 130 208 2 0 0 0 153 227 2 131 226 2 169 265 2 0 0 0 144 249 2 123 252 2 157 294 2 138 294 2 172 341 2 133 346 2 195 380 2 126 396 2"), (2017004, "180.78,170.81,68.57,194.49", "191 183 1 195 180 1 193 179 1 209 184 2 0 0 0 230 204 2 193 206 1 245 239 2 0 0 0 247 266 2 0 0 0 226 268 2 200 269 2 231 311 2 196 309 2 237 352 2 199 355 2"),)),
    (520910, 101892, "bee8191e06b40c9d08da28f576fe09b8822c96c4477b91476aa9e7cad7aeb773", 427, 640, 5, "http://farm6.staticflickr.com/5345/7066872039_6f780f6537_z.jpg", ((456804, "160.19,162.32,81.80,309.29", "200 187 2 205 181 2 194 181 2 212 186 2 186 186 1 229 220 2 167 220 2 230 264 2 165 263 2 229 305 2 167 305 1 216 291 2 182 290 2 214 376 2 187 374 2 207 435 2 190 434 2"),)),
    (521141, 132221, "51ef3a1cd7d830bdf919810e8dc10cd464bd740f7acfaa8fcd7efcbea7441607", 375, 500, 4, "http://farm1.staticflickr.com/49/121598568_f73fc969ce_z.jpg", ((528376, "48.74,258.04,56.05,146.87", "0 0 0 0 0 0 0 0 0 72 272 2 0 0 0 66 288 2 94 288 2 61 310 2 98 308 2 55 330 2 0 0 0 74 341 2 95 338 2 76 358 2 94 364 2 78 387 2 99 395 2"), (546060, "73.67,249.38,41.57,128.14", "0 0 0 0 0 0 75 258 1 81 256 2 0 0 0 78 271 1 102 270 2 70 288 1 111 288 2 0 0 0 110 302 2 82 311 1 100 310 2 86 335 1 102 335 2 88 362 1 105 360 2"),)),
    (523957, 94207, "163a959d2c41cab3366c5d6f8cd24a63fe981d91c3e018244e42947cb10a2450", 640, 427, 4, "http://farm3.staticflickr.com/2079/2660283696_07138cec04_z.jpg", ((1309217, "55.78,82.23,260.62,272.16", "204 114 2 207 109 2 196 110 2 0 0 0 182 110 2 233 131 2 162 131 2 264 164 2 123 154 2 295 197 2 81 155 2 230 197 2 193 209 2 217 248 2 190 269 2 248 323 2 224 319 2"), (1322137, "343.53,169.04,61.41,170.29", "358 185 2 360 182 2 354 183 2 369 181 2 0 0 0 382 199 2 350 206 2 396 217 2 349 233 2 372 215 2 347 257 2 385 254 2 363 255 2 385 292 2 373 294 2 385 325 2 380 323 2"),)),
    (526728, 150296, "2df4f4dd6ddae2b41cbe4847ca020f23024d1f6d221fe1556402ea356623f9f2", 500, 375, 4, "http://farm1.staticflickr.com/48/110025548_2354dbb701_z.jpg", ((494874, "247.13,150.16,72.73,128.40", "274 168 2 276 165 2 270 166 2 283 167 2 266 169 2 300 182 2 258 193 2 309 208 2 253 222 2 308 233 2 257 242 2 298 246 2 268 251 2 301 288 1 274 293 1 0 0 0 0 0 0"), (502702, "195.51,140.73,63.20,130.62", "226 157 2 229 154 2 221 155 2 235 158 2 217 158 2 244 180 2 208 182 2 250 203 2 196 201 1 248 225 2 205 226 2 238 237 2 216 237 2 242 288 1 214 291 1 241 323 1 212 328 1"),)),
    (527215, 249144, "7d403827fdf85b5639fbf53832be1e92129ee259c99d89a30ac70aad7dc1a8f7", 640, 426, 5, "http://farm7.staticflickr.com/6208/6118912613_183765a80e_z.jpg", ((455635, "451.71,207.97,61.03,189.46", "0 0 0 0 0 0 0 0 0 479 228 2 499 232 1 470 241 2 505 248 2 454 228 2 0 0 0 464 214 2 0 0 0 475 301 2 496 304 2 482 344 2 494 347 2 488 380 2 491 380 2"),)),
    (527427, 136728, "c5e3f3f5bbd1990de2cae0d0a81e50a09db5963eb79c609b925959eb84a183ab", 640, 480, 4, "http://farm4.staticflickr.com/3564/3470619108_19b4bd67d1_z.jpg", ((439875, "165.37,148.39,411.27,328.59", "480 214 2 492 208 2 0 0 0 522 226 2 0 0 0 526 288 2 483 252 2 449 358 2 438 314 2 480 272 2 443 264 2 405 393 2 394 348 2 266 396 2 271 371 2 220 465 2 243 473 2"),)),
    (530975, 142832, "5643aadb0049998a1f1e2a418af7be084c40d5ff5ac7eadf17b38cb5c61c0117", 640, 480, 5, "http://farm4.staticflickr.com/3322/3648849691_cddc5e04ec_z.jpg", ((558017, "14.81,146.09,57.57,122.81", "24 160 2 27 157 2 20 157 2 30 158 2 14 159 1 37 176 2 0 0 0 47 199 1 43 213 2 61 202 2 56 256 2 0 0 0 0 0 0 81 212 1 0 0 0 79 254 1 0 0 0"), (2155792, "108.73,119.00,48.30,132.42", "124 133 2 125 129 2 0 0 0 131 128 2 0 0 0 140 139 2 134 139 2 146 157 2 127 157 2 139 172 2 126 144 2 143 181 2 134 181 2 126 211 2 142 214 2 126 240 2 146 246 2"),)),
    (531036, 307853, "79be9c405b3ce4d2eea61fc29b0e1e633504fbeebee6483d64dfb38b1dd844a5", 480, 640, 4, "http://farm8.staticflickr.com/7288/8740762571_0849912a06_z.jpg", ((191770, "381.58,281.03,73.63,184.08", "415 304 2 416 300 2 0 0 0 422 299 2 0 0 0 420 317 2 451 315 2 408 338 2 0 0 0 390 347 2 0 0 0 431 371 2 446 371 2 422 410 2 442 412 2 423 452 2 443 450 2"),)),
    (532690, 110520, "351e9f13f945d031fb6a17669b91c0fddffaaf92b057bd1b25ef094e763e4a27", 528, 640, 4, "http://farm1.staticflickr.com/155/353485795_d364074053_z.jpg", ((494829, "58.97,40.27,441.52,591.10", "303 161 2 310 131 2 272 137 2 0 0 0 205 151 2 350 205 2 169 246 2 398 332 2 133 406 2 447 447 2 84 508 2 367 465 2 215 480 2 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (537991, 121892, "54a0356ad918b76c12aa25e98aadd6720786eade12fe0676f63b802ecddf370c", 640, 480, 4, "http://farm1.staticflickr.com/136/317343650_bcd1ba28ef_z.jpg", ((189976, "95.59,3.77,278.54,469.72", "224 116 2 241 97 2 208 97 2 270 98 2 188 91 2 318 172 2 153 167 2 360 275 2 111 290 2 290 272 2 166 263 2 283 354 2 174 359 2 0 0 0 0 0 0 0 0 0 0 0 0"),)),
    (538364, 213567, "1b05f4958120b8ab11c5a3b00449320455e16c1a9795871e8e42913c1a04be69", 640, 480, 4, "http://farm9.staticflickr.com/8042/7933423348_c30bd9bd4e_z.jpg", ((202541, "593.47,120.87,46.53,218.13", "620 137 2 623 134 2 616 135 2 630 137 2 610 139 2 0 0 0 605 169 2 0 0 0 599 200 2 0 0 0 599 236 1 639 235 2 614 233 2 0 0 0 617 276 2 0 0 0 622 324 2"), (1709632, "0.00,111.85,43.16,160.71", "20 131 2 19 127 2 16 127 2 0 0 0 4 129 2 19 147 2 0 144 1 24 166 2 0 0 0 32 189 2 0 0 0 14 187 2 1 187 1 8 223 2 12 224 2 0 0 0 21 258 2"),)),
    (540414, 263719, "fcde3d391e5ff8ca3df743484effef4c13e1cf5f3aa548af815b777a23adc365", 640, 480, 5, "http://farm9.staticflickr.com/8327/8395092894_bc8081f393_z.jpg", ((1256950, "293.72,185.26,46.62,148.62", "322 207 2 324 204 2 319 204 2 0 0 0 315 200 2 332 212 2 304 210 2 333 237 2 303 229 2 314 240 2 318 233 2 325 248 2 312 247 2 324 290 2 303 286 2 319 317 2 298 322 2"), (2028895, "469.09,296.80,132.76,133.31", "549 317 2 555 314 2 546 313 2 566 318 2 0 0 0 571 346 2 550 335 2 534 362 2 519 350 2 543 332 2 487 362 2 590 402 2 570 394 2 546 374 2 532 360 2 570 414 2 559 398 2"),)),
    (541055, 117304, "abf3a946d2dad79fd000391e46bd094ec6a5ecfb1c2d1eae398aef3025662bcf", 640, 360, 4, "http://farm4.staticflickr.com/3598/3465325303_2a5f83799f_z.jpg", ((439621, "271.50,79.97,64.19,203.81", "305 104 2 309 99 2 300 100 2 0 0 0 0 0 0 317 127 2 285 125 2 0 0 0 282 159 2 325 161 2 315 169 2 314 187 2 299 185 2 308 234 2 297 234 2 301 266 2 290 271 2"), (474852, "201.47,101.90,86.40,222.80", "242 123 2 246 118 2 237 119 2 252 122 2 230 121 2 259 150 2 220 148 2 270 177 2 212 183 2 283 189 2 228 199 2 253 220 2 230 219 2 246 260 2 243 260 2 236 303 2 228 306 2"), (532929, "154.52,78.08,54.25,235.07", "192 102 2 197 96 2 187 96 2 0 0 0 178 99 2 196 124 2 167 124 2 188 158 2 163 165 2 211 168 1 192 174 2 197 189 2 176 189 2 194 239 1 181 240 2 177 286 2 164 292 2"), (534903, "325.21,79.28,163.42,247.55", "420 101 2 425 98 2 414 98 2 431 100 2 0 0 0 444 132 2 400 130 2 446 171 2 385 166 2 416 192 2 395 183 2 428 180 2 399 180 2 431 248 2 384 248 2 430 285 2 392 289 2"),)),
    (544444, 117323, "e208eb88ca5bb9a93ee044f18e775949281ec207153e1acf5f9a5abba705eaf8", 427, 640, 4, "http://farm4.staticflickr.com/3328/4631838153_4c3e8a0dbd_z.jpg", ((499548, "126.09,260.04,236.48,257.35", "224 300 2 228 290 2 216 290 2 244 283 2 208 290 2 271 320 2 209 339 2 327 341 2 194 384 2 317 320 2 153 418 2 299 405 2 264 417 2 311 438 2 252 456 2 350 499 1 285 509 2"),)),
    (547336, 246773, "d89167b4015b2ffb42d4bccabc87fed2de0ca2721f96140fb451b31a52ff9982", 640, 480, 4, "http://farm6.staticflickr.com/5291/5495465828_fb0bf27878_z.jpg", ((495052, "99.83,25.11,371.28,413.32", "346 124 2 379 114 2 341 95 2 417 133 2 0 0 0 416 261 2 318 112 2 367 353 2 278 147 2 292 374 2 222 207 2 355 331 2 306 247 2 264 403 2 205 228 2 223 269 1 169 329 2"),)),
    (548780, 221572, "8d8b075cbc8bfb5a1767d65683ede1bb389ac0e49d79d6dfc7b3f934e1c263df", 640, 427, 5, "http://farm9.staticflickr.com/8451/8030321271_ebc4c4820c_z.jpg", ((218317, "325.52,64.15,205.84,311.15", "457 107 2 468 97 2 449 98 2 479 100 2 445 102 1 499 144 2 435 140 2 521 198 2 404 193 2 486 197 2 433 143 2 466 216 2 417 204 2 398 209 2 409 235 2 354 291 2 392 332 2"), (224905, "236.05,71.97,165.04,298.42", "356 113 2 362 103 2 348 103 2 374 106 2 336 105 1 376 141 2 317 141 2 380 194 2 292 183 2 341 181 2 320 185 2 348 206 2 308 203 2 277 235 2 273 206 2 268 346 2 282 299 2"), (236688, "175.60,0.00,86.36,210.14", "0 0 0 0 0 0 0 0 0 212 8 2 236 7 2 190 36 2 243 34 2 184 78 2 249 73 2 182 113 2 0 0 0 205 99 2 233 96 2 210 171 2 234 169 2 199 231 1 230 230 1"), (237126, "103.86,0.48,79.83,210.62", "0 0 0 0 0 0 0 0 0 125 16 2 147 15 2 120 37 2 159 34 2 116 83 2 173 82 2 113 108 2 180 111 1 130 97 2 160 95 2 130 163 2 168 162 2 118 227 1 170 230 1"),)),
    (549930, 169461, "4b9c8d2bd7df7bb6653c2106eb785dde6cdb39dda55c792e704e0b3cd95bd0e1", 640, 558, 4, "http://farm9.staticflickr.com/8006/7586304304_1a7401a456_z.jpg", ((520591, "80.94,160.29,214.17,397.71", "0 0 0 0 0 0 0 0 0 190 187 2 222 186 2 176 232 2 253 233 2 154 293 2 267 288 1 113 345 2 0 0 0 209 346 2 246 350 2 187 449 2 260 445 2 187 534 2 279 534 2"), (2166072, "261.77,150.72,98.16,407.26", "0 0 0 0 0 0 0 0 0 274 186 2 306 182 2 280 222 2 330 221 2 269 288 2 345 285 2 0 0 0 357 344 1 288 332 2 333 327 2 307 435 2 306 442 2 326 532 2 305 526 2"),)),
    (556000, 175148, "346e603474857f6750d0f946eb80a76b104d600195c40cadca91b9f36f9ad84d", 640, 515, 5, "http://farm4.staticflickr.com/3127/3157863995_c27c0cacfb_z.jpg", ((206924, "223.36,104.16,109.94,408.53", "271 153 2 275 143 2 0 0 0 298 140 2 0 0 0 293 193 2 285 194 2 259 247 2 269 226 2 236 228 2 249 217 2 284 323 2 277 322 2 286 409 2 279 409 2 288 496 2 284 496 2"), (223908, "0.11,126.74,188.87,388.26", "0 0 0 0 0 0 0 0 0 23 173 2 89 180 2 9 237 2 95 233 2 3 317 2 118 308 2 44 319 2 125 354 2 32 380 2 97 381 2 44 505 2 153 503 2 0 0 0 0 0 0"),)),
    (556158, 162703, "1e67bbb26205214fd8e29e4809513dd6b5454ad5ebf3800f2f08251538ee9ade", 480, 640, 4, "http://farm9.staticflickr.com/8491/8308083760_2c42047b3e_z.jpg", ((435295, "147.80,140.80,139.20,322.87", "230 170 2 235 166 2 224 166 2 241 171 2 217 171 2 256 210 2 205 208 2 273 247 2 187 242 2 269 285 2 172 270 2 241 293 2 206 293 2 240 360 2 206 356 2 247 426 2 211 413 2"),)),
    (559348, 119574, "a73654c0617b1c814453ecb7db42adb053b50b521775fe67e54de7fe839062a4", 640, 480, 4, "http://farm1.staticflickr.com/232/444593491_4569474ab4_z.jpg", ((1208626, "34.07,267.04,118.14,190.57", "0 0 0 0 0 0 0 0 0 82 282 2 96 284 2 65 304 2 101 308 2 58 338 2 111 344 2 0 0 0 118 372 2 73 363 2 94 364 2 60 405 2 112 406 2 41 445 2 132 448 2"),)),
    (563349, 121218, "c0cae527104995431e0e90c67a0805089e2ccfeabcc968ac67f0c5290b783792", 427, 640, 4, "http://farm6.staticflickr.com/5171/5588983985_fcf49f0d7b_z.jpg", ((438675, "185.43,247.73,178.93,166.27", "0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 218 290 2 207 297 2 256 280 2 238 294 2 229 265 2 235 262 2 266 358 2 271 343 2 324 351 2 320 320 2 356 387 2 350 342 2"),)),
    (564091, 112448, "4c4e7f97fbf0f9695f38cd52dc58c7a6f8abb227a0b59695894e254c4852a373", 427, 640, 5, "http://farm4.staticflickr.com/3271/2587900333_8f616f59a9_z.jpg", ((186054, "9.78,0.00,314.70,370.79", "197 49 2 220 39 2 195 29 2 251 63 2 0 0 0 294 110 2 145 75 2 281 217 2 117 189 2 232 135 2 154 275 2 230 252 2 134 235 2 297 258 2 32 238 2 113 326 2 193 344 2"),)),
    (566923, 96590, "e51432e0639abe0ab3e226d98babe8b6275729e8c2def6cdfe498f86d9d29bb3", 640, 426, 4, "http://farm1.staticflickr.com/205/515499299_0450056310_z.jpg", ((1243154, "221.14,13.64,112.96,228.80", "286 36 2 293 32 2 0 0 0 0 0 0 0 0 0 284 58 2 332 81 1 259 102 2 0 0 0 230 129 1 0 0 0 277 136 2 309 140 2 234 192 2 274 205 2 229 245 1 261 256 1"), (1287513, "560.02,75.63,79.98,344.63", "607 98 1 615 93 2 0 0 0 625 98 2 0 0 0 604 133 2 0 0 0 587 184 2 0 0 0 559 232 1 0 0 0 583 247 2 610 260 2 535 325 1 578 331 2 556 410 1 565 412 2"), (1300345, "71.64,2.55,80.23,194.85", "0 0 0 0 0 0 0 0 0 132 8 1 149 17 2 116 26 2 152 37 1 100 58 2 0 0 0 84 83 2 0 0 0 109 84 2 133 90 1 93 128 2 111 139 2 91 165 2 101 185 2"),)),
    (571264, 145233, "a310373f200737f37ab6e24eaca6f9cf54045e801eb112194506472901c97fe2", 640, 426, 4, "http://farm9.staticflickr.com/8012/6988704388_144a769946_z.jpg", ((217633, "265.17,29.68,150.30,358.99", "0 0 0 0 0 0 0 0 0 330 57 2 358 54 2 317 90 2 383 95 2 300 141 2 397 159 2 319 157 2 0 0 0 334 191 2 373 191 2 297 276 2 363 283 2 297 366 2 358 370 2"),)),
    (572620, 201272, "e2cf936c71429fe653a8885bb7868f3c28317b78ed5cfbe9142272db21852e04", 640, 427, 4, "http://farm7.staticflickr.com/6041/5915075988_a0ea460639_z.jpg", ((1295263, "277.31,91.16,111.31,191.91", "334 137 1 0 0 0 334 132 2 0 0 0 328 126 2 312 129 2 309 140 2 335 118 2 341 148 2 355 104 2 370 136 2 307 181 2 300 190 2 354 184 2 326 218 2 365 226 2 335 270 2"), (1321152, "437.56,186.82,48.93,141.05", "0 0 0 0 0 0 0 0 0 464 199 2 0 0 0 475 211 2 452 216 2 479 222 2 443 229 2 463 210 2 450 210 2 473 253 2 458 253 2 478 286 2 461 288 2 478 312 2 461 315 2"),)),
    (574520, 251550, "4b4a90e16ea438f390bf42ab4982ec902304d9b2bc54eb6d058940f6c853e391", 640, 399, 4, "http://farm7.staticflickr.com/6219/6276393723_bf8a0d64a3_z.jpg", ((516503, "285.09,77.61,133.61,123.25", "306 151 2 305 148 2 0 0 0 313 144 2 0 0 0 334 127 2 335 169 2 332 94 2 359 177 2 302 88 2 390 191 2 371 173 2 360 195 2 0 0 0 313 179 2 0 0 0 314 207 1"),)),
    (574702, 132648, "43404f8739d1929d5d4e7dc52c28fa0c0730848e82e706783ea46c861947b412", 333, 500, 4, "http://farm3.staticflickr.com/2035/2252476376_bfc3cda192_z.jpg", ((1216228, "216.11,264.53,84.32,123.56", "260 296 2 265 292 2 255 293 2 0 0 0 244 285 2 273 294 2 233 296 2 0 0 0 229 334 1 286 324 2 259 328 1 271 348 1 242 349 1 296 369 2 0 0 0 0 0 0 0 0 0"),)),
    (576566, 83631, "494fbef25a7b5c33413c5adb736c263778e501708a6904644ed40d259edcc9ec", 469, 640, 4, "http://farm4.staticflickr.com/3381/3412884049_337839d219_z.jpg", ((451097, "141.38,17.32,188.61,421.57", "0 0 0 229 183 2 0 0 0 222 165 2 0 0 0 253 146 2 293 184 2 221 80 2 325 190 2 187 41 2 0 0 0 270 222 2 299 243 2 184 240 2 255 319 2 180 308 2 261 386 2"),)),
    (579307, 204401, "4911ff1fd8fb03bea1fd6270003690b394493cfeae7b5527dcd43e7d28ffde6f", 480, 640, 4, "http://farm7.staticflickr.com/6213/6378172019_b4f2463fa0_z.jpg", ((451044, "176.30,168.13,98.01,217.03", "237 209 2 241 206 2 231 205 2 0 0 0 220 195 2 256 227 2 208 224 2 264 264 2 186 261 2 247 290 2 198 267 2 244 317 1 215 316 1 247 365 1 211 368 2 249 425 1 208 423 1"), (500864, "256.00,222.92,123.69,417.08", "302 285 2 311 269 2 290 274 2 342 266 2 0 0 0 361 344 2 288 329 1 361 425 2 0 0 0 347 418 2 0 0 0 321 474 2 295 467 2 328 558 2 305 545 2 0 0 0 302 622 2"),)),
)


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def photo_url(image_id: int) -> str:
    return f"{CORPUS_BASE_URL}{image_id:012d}.jpg"


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[int, bytes]:
    """Return every pinned photograph (bytes keyed by COCO image id) from the cache or the COCO image host; a cached or
    fetched file is refused on any byte-size or SHA-256 mismatch."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    out: dict[int, bytes] = {}
    for image_id, size, digest, *_rest in SAMPLE_IMAGES:
        local = cache / f"{image_id:012d}.jpg"
        data = local.read_bytes() if local.is_file() else b""
        if len(data) != size or _sha256_bytes(data) != digest:
            url = photo_url(image_id)
            if fetcher is not None:
                data = fetcher(url)
            else:
                request = urllib.request.Request(url, headers={"User-Agent": "vitpose-keypoint-pipeline"})
                with urllib.request.urlopen(request, timeout=60) as response:  # noqa: S310 - pinned URL, digest-checked
                    data = response.read()
            if len(data) != size:
                raise ValueError(f"image {image_id}: {len(data)} bytes, pinned {size}")
            got = _sha256_bytes(data)
            if got != digest:
                raise ValueError(f"image {image_id}: sha256 {got} != pinned {digest}")
            local.write_bytes(data)
        out[image_id] = data
    return out


def _parse_person(entry: Sequence[Any]) -> tuple[int, list[float], dict[str, list[float]]]:
    ann_id, box_text, kp_text = entry
    x, y, w, h = (float(v) for v in box_text.split(","))
    values = [int(v) for v in kp_text.split()]
    joints = {}
    for index, name in enumerate(KEYPOINT_NAMES):
        px, py, visible = values[3 * index : 3 * index + 3]
        if visible > 0:
            joints[name] = [float(px), float(py)]
    return int(ann_id), [x, y, x + w, y + h], joints


def category_for(n_labelled: int) -> str:
    """The sample's category: how many of the 17 joints are labelled (`full` 16-17, `partial` 14-15, `sparse` <= 13)."""
    for floor, name in _BUCKETS:
        if n_labelled >= floor:
            return name
    return _BUCKETS[-1][1]


def degrade(
    image: Image.Image,
    box: Sequence[float],
    keypoints: Mapping[str, Sequence[float]],
    *,
    target_height: int = TARGET_HEIGHT,
    quality: int = JPEG_QUALITY,
) -> tuple[Image.Image, list[float], dict[str, list[float]], float]:
    """The stated degradation: downscale the whole image (bicubic) so the person box is `target_height` px tall
    (never upscaled), re-encode as JPEG at `quality`, and scale the box and joints by the same factor. Returns
    `(image, box, keypoints, scale)`; deterministic."""
    if not isinstance(image, Image.Image):
        raise TypeError("image must be a PIL image")
    if not 8 <= int(target_height) <= 1024:
        raise ValueError("target_height must be in 8..1024")
    if not 1 <= int(quality) <= 95:
        raise ValueError("quality must be in 1..95")
    x0, y0, x1, y1 = (float(v) for v in box)
    scale = min(1.0, target_height / max(1.0, y1 - y0))
    rgb = image.convert("RGB")
    if scale < 1.0:
        rgb = rgb.resize((max(16, round(rgb.width * scale)), max(16, round(rgb.height * scale))), Image.BICUBIC)
        scale_x, scale_y = rgb.width / image.width, rgb.height / image.height
    else:
        scale_x = scale_y = 1.0
    buffer = io.BytesIO()
    rgb.save(buffer, format="JPEG", quality=int(quality))
    degraded = Image.open(io.BytesIO(buffer.getvalue())).convert("RGB")
    new_box = [x0 * scale_x, y0 * scale_y, min(x1 * scale_x, degraded.width), min(y1 * scale_y, degraded.height)]
    new_joints = {name: [px * scale_x, py * scale_y] for name, (px, py) in keypoints.items()}
    return degraded, new_box, new_joints, float(scale)


def read_corpus(
    files: Mapping[int, bytes], *, target_height: int | None = TARGET_HEIGHT, quality: int = JPEG_QUALITY
) -> list[dict[str, Any]]:
    """Decode the verified photographs into one record per pinned person: degraded by `degrade` (the default) or,
    with `target_height=None`, the served photograph unchanged (the clean counterpart)."""
    out = []
    for image_id, _size, _digest, width, height, licence, flickr_url, persons in SAMPLE_IMAGES:
        if image_id not in files:
            continue
        image = Image.open(io.BytesIO(files[image_id]))
        image.load()
        if image.size != (width, height):
            raise ValueError(f"image {image_id}: decoded size {image.size} != pinned {(width, height)}")
        image = image.convert("RGB")
        for entry in persons:
            ann_id, box, joints = _parse_person(entry)
            record: dict[str, Any] = {
                "id": f"coco-{image_id}-{ann_id}",
                "image_id": image_id,
                "annotation_id": ann_id,
                "category": category_for(len(joints)),
                "n_labelled": len(joints),
                "license": CORPUS_LICENSES[licence][0],
                "license_url": CORPUS_LICENSES[licence][1],
                "flickr_url": flickr_url,
                "coco_url": photo_url(image_id),
                "source_box": box,
            }
            if target_height is None:
                record.update({"image": image, "box": box, "keypoints": joints, "scale": 1.0})
            else:
                degraded, new_box, new_joints, scale = degrade(image, box, joints, target_height=target_height, quality=quality)
                record.update({"image": degraded, "box": new_box, "keypoints": new_joints, "scale": scale})
            out.append(record)
    return out


def build_sample_dataset(
    records: Sequence[Mapping[str, Any]], *, seed: int = SAMPLE_SEED, sizes: Mapping[str, int] | None = None
) -> dict[str, list[dict[str, Any]]]:
    """Seeded image-level draw: shuffle the pinned images, cut `sizes` (train / validation / test) images in order,
    and give every person to its image's split."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    image_ids = sorted({r["image_id"] for r in records})
    random.Random(seed).shuffle(image_ids)
    needed = sum(sizes.values())
    if len(image_ids) < needed:
        raise ValueError(f"only {len(image_ids)} images available, need {needed}")
    out, cursor = {}, 0
    for name, count in sizes.items():
        chosen = set(image_ids[cursor : cursor + count])
        out[name] = [dict(r) for r in records if r["image_id"] in chosen]
        cursor += count
    return out


def fetch_sample_dataset(*, cache_dir: str | Path | None = None, seed: int = SAMPLE_SEED) -> dict[str, list[dict[str, Any]]]:
    return build_sample_dataset(read_corpus(fetch_corpus(cache_dir=cache_dir)), seed=seed)


def _open(image: Any, where: str) -> Image.Image:
    if isinstance(image, str | Path):
        path = Path(image)
        if not path.is_file():
            raise ValueError(f"{where}: image file not found: {path}")
        image = Image.open(path)
        image.load()
    if not isinstance(image, Image.Image):
        raise ValueError(f"{where}: must be a PIL.Image.Image or a file path")
    return image


def _check_record(record: Any, index: int) -> dict[str, Any]:
    where = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{where} must be a mapping with id/image/box/keypoints")
    for key in ("id", "image", "box", "keypoints"):
        if key not in record:
            raise ValueError(f"{where} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{where}: id must match {_ID_RE.pattern}")
    try:
        image = validate_image(_open(record["image"], f"{where}.image"))
    except TypeError as exc:
        raise ValueError(f"{where}: {exc}") from exc
    try:
        box = _check_boxes([record["box"]], image.width, image.height)[0]
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{where}: {exc}") from exc
    joints = record["keypoints"]
    if not isinstance(joints, Mapping):
        raise ValueError(f"{where}: keypoints must map joint names to [x, y]")
    unknown = sorted(set(joints) - set(KEYPOINT_NAMES))
    if unknown:
        raise ValueError(f"{where}: unknown keypoint names {unknown}")
    if len(joints) < MIN_LABELLED:
        raise ValueError(f"{where}: {len(joints)} labelled joints, MIN_LABELLED is {MIN_LABELLED}")
    clean_joints = {}
    for name, value in joints.items():
        try:
            px, py = (float(v) for v in value)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"{where}: keypoint {name} must be [x, y]") from exc
        if not (0 <= px < image.width and 0 <= py < image.height):
            raise ValueError(f"{where}: keypoint {name} ({px}, {py}) lies outside the {image.width}x{image.height} image")
        clean_joints[name] = [px, py]
    item = {"id": rid, "image": image, "box": box, "keypoints": clean_joints}
    category = record.get("category")
    if category is not None:
        if not isinstance(category, str) or not 1 <= len(category.strip()) <= MAX_CATEGORY_CHARS:
            raise ValueError(f"{where}: category must be a str of 1..{MAX_CATEGORY_CHARS} characters")
        item["category"] = category.strip()
    for key in ("image_id", "annotation_id", "n_labelled", "license", "license_url", "flickr_url", "coco_url", "source_box", "scale"):
        if key in record:
            item[key] = record[key]
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a keypoint dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise ValueError("records must be a list of {id, image, box, keypoints} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked, ids, categories = [], set(), {}
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        if "category" in item:
            categories[item["category"]] = categories.get(item["category"], 0) + 1
        checked.append(item)
    heights = [r["box"][3] - r["box"][1] for r in checked]
    labelled = [len(r["keypoints"]) for r in checked]
    return {
        "records": checked,
        "n_records": len(checked),
        "n_images": len({image_digest(r["image"]) for r in checked}),
        "category_counts": dict(sorted(categories.items())),
        "box_height": {"min": round(min(heights), 1), "max": round(max(heights), 1)},
        "labelled_joints": {"min": min(labelled), "max": max(labelled)},
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def image_digest(image: Image.Image) -> str:
    """SHA-256 of the decoded RGB pixels (size-prefixed) — the identity a split is made disjoint on."""
    rgb = image.convert("RGB")
    return _sha256_bytes(f"{rgb.width}x{rgb.height}:".encode() + rgb.tobytes())


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    """Order-independent SHA-256 over every record's id, image pixels, box and joints."""
    rows = []
    for r in records:
        joints = sorted((k, round(v[0], 2), round(v[1], 2)) for k, v in r["keypoints"].items())
        rows.append(json.dumps([r["id"], image_digest(r["image"]), [round(v, 2) for v in r["box"]], joints], separators=(",", ":")))
    return _sha256_bytes("\n".join(sorted(rows)).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no image (by decoded-pixel digest) appears in two splits; returns the split sizes."""
    seen: dict[str, str] = {}
    for name, part in splits.items():
        for r in part:
            key = image_digest(r["image"])
            if key in seen and seen[key] != name:
                raise ValueError(f"image {r['id']!r} appears in both {seen[key]!r} and {name!r}")
            seen[key] = name
    return {name: len(part) for name, part in splits.items()}


def _split_images(n_images: int, val_fraction: float, test_fraction: float) -> tuple[int, int, int]:
    """(train, validation, test) image counts that `split_dataset` cuts from `n_images` distinct images."""
    n_test = max(1, round(n_images * test_fraction))
    n_val = round(n_images * val_fraction)
    return n_images - n_test - n_val, n_val, n_test


def min_byod_records(*, val_fraction: float = 0.15, test_fraction: float = 0.2) -> dict[str, int]:
    """The smallest BYOD set `split_dataset` always accepts when every image carries the same number of labelled
    persons: `MIN_RECORDS` training persons on the training images and (when `val_fraction` > 0) at least one
    validation image, after at least one test image is held out. With the default fractions that is 12 persons on
    12 single-person images (split 8 / 2 / 2 images); images with several persons need fewer images."""
    best: dict[str, int] | None = None
    for per_image in range(1, MIN_RECORDS + 1):
        for n in range(2, MAX_RECORDS + 1):
            train, val, test = _split_images(n, val_fraction, test_fraction)
            if train >= 1 and train * per_image >= MIN_RECORDS and (val >= 1 or val_fraction == 0):
                if best is None or n * per_image < best["total"]:
                    best = {"total": n * per_image, "images": n, "persons_per_image": per_image,
                            "train_images": train, "validation_images": val, "test_images": test}
                break
    if best is None:
        raise ValueError("no dataset size satisfies these fractions")
    return best


def split_dataset(
    records: Sequence[Mapping[str, Any]], *, val_fraction: float = 0.15, test_fraction: float = 0.2, seed: int = 0
) -> dict[str, list[dict[str, Any]]]:
    """Seeded image-level shuffle of a BYOD dataset into train/validation/test (persons follow their image)."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    by_image: dict[str, list[dict[str, Any]]] = {}
    for record in checked:
        by_image.setdefault(image_digest(record["image"]), []).append(record)
    keys = list(by_image)
    random.Random(seed).shuffle(keys)
    n = len(keys)
    _n_train, n_val, n_test = _split_images(n, val_fraction, test_fraction)
    need = min_byod_records(val_fraction=val_fraction, test_fraction=test_fraction)
    advice = (
        f"supply at least {need['total']} labelled persons on {need['images']} single-person images "
        "(or more persons per image, with at least 8 on the training images)"
    )
    if n - n_test - n_val < 1 or (val_fraction > 0 and n_val < 1):
        raise ValueError(f"{n} distinct images are too few to split into train/validation/test — {advice}")
    groups = {"test": keys[:n_test], "validation": keys[n_test : n_test + n_val], "train": keys[n_test + n_val :]}
    out = {name: [r for key in part for r in by_image[key]] for name, part in groups.items()}
    if len(out["train"]) < MIN_RECORDS:
        raise ValueError(
            f"the train split holds {len(out['train'])} persons on {len(groups['train'])} image(s); at least "
            f"{MIN_RECORDS} are required — {advice}"
        )
    return out


def _box(row: Mapping[str, Any], where: str) -> list[float]:
    try:
        return [float(row["x0"]), float(row["y0"]), float(row["x1"]), float(row["y1"])]
    except (KeyError, TypeError, ValueError):
        raise ValueError(f"{where}: x0, y0, x1 and y1 must be present and numeric") from None


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Records from a directory or zip holding image files plus one `keypoints.csv` with the columns `file`, `person`,
    `x0`, `y0`, `x1`, `y1` and one `<joint>_x` / `<joint>_y` pair per labelled COCO joint (blank = unlabelled); an
    optional `category` column is carried."""
    source = Path(path)
    members: dict[str, bytes] = {}
    if source.is_dir():
        for file in sorted(source.rglob("*")):
            if file.is_file() and not any(part == "__MACOSX" or part.startswith(".") for part in file.parts):
                members[file.name] = file.read_bytes()
    elif zipfile.is_zipfile(source):
        with zipfile.ZipFile(source) as archive:
            for info in archive.infolist():
                parts = Path(info.filename).parts
                if not info.is_dir() and not any(part == "__MACOSX" or part.startswith(".") for part in parts):
                    members[Path(info.filename).name] = archive.read(info)  # flattened; no extractall
    else:
        raise ValueError(f"{source} is neither a directory nor a zip file")
    if "keypoints.csv" not in members:
        raise ValueError("BYOD needs a keypoints.csv beside the images")
    images: dict[str, Image.Image] = {}
    out = []
    rows = list(csv.DictReader(io.StringIO(members["keypoints.csv"].decode("utf-8-sig"))))
    if not rows:
        raise ValueError("keypoints.csv has no data rows: add one row per labelled person")
    for line, row in enumerate(rows, start=2):  # line 1 is the header
        name = str(row.get("file", "")).strip()
        where = f"keypoints.csv line {line} (file {name!r})"
        if name not in members:
            raise ValueError(f"{where}: names an image that is not among the uploaded files")
        if name not in images:
            try:
                image = Image.open(io.BytesIO(members[name]))
                image.load()
            except Exception as exc:  # noqa: BLE001
                raise ValueError(f"{where}: the file is not a decodable image") from exc
            images[name] = image.convert("RGB")
        joints = {}
        for joint in KEYPOINT_NAMES:
            sx, sy = str(row.get(f"{joint}_x", "")).strip(), str(row.get(f"{joint}_y", "")).strip()
            if sx and sy:
                try:
                    joints[joint] = [float(sx), float(sy)]
                except ValueError:
                    raise ValueError(f"{where}: {joint}_x / {joint}_y must be numbers") from None
        person = str(row.get("person", "0")).strip() or "0"
        record: dict[str, Any] = {
            "id": re.sub(r"[^A-Za-z0-9_.:-]", "_", f"{Path(name).stem}-{person}")[:64],
            "image": images[name],
            "box": _box(row, where),
            "keypoints": joints,
            "n_labelled": len(joints),
        }
        category = str(row.get("category", "")).strip()
        record["category"] = category or category_for(len(joints))
        out.append(record)
    return out


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """A summary table (id, category, image size, box, labelled joints, provenance), not the BYOD format."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["id", "category", "width", "height", "x0", "y0", "x1", "y1", "labelled_joints", "scale", "license", "flickr_url"])
        for r in records:
            writer.writerow([r["id"], r.get("category", ""), r["image"].width, r["image"].height, *[round(v, 2) for v in r["box"]], len(r["keypoints"]), r.get("scale", ""), r.get("license", ""), r.get("flickr_url", "")])
    return out

## 3. Pin, stage and verify the model

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `95be2991424e…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `VitPoseKeypointPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, detector_dir=DETECTOR_WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The package also pins a second snapshot `rtdetr-r50vd` (4 files), carried and verified the same way. The effective identity, device and weight source are printed before any inference.

In [ ]:
# @title Infrastructure: stage and digest-verify the pinned snapshot, then load the model
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "vitpose-base",
  "modelId": "usyd-community/vitpose-base",
  "revision": "95be2991424e646950d656bb7fc15ec9be119700",
  "files": [
    {
      "path": "README.md",
      "bytes": 11309,
      "sha256": "6e956c405569ad33e9cce408193376a0104bf610e8928bbfa94759603b541578"
    },
    {
      "path": "config.json",
      "bytes": 1799,
      "sha256": "488377867b16542e0c0025a0eab646abfd066e67f115ab3ec100e3af49872787"
    },
    {
      "path": "model.safetensors",
      "bytes": 360007012,
      "sha256": "cd9a4e6cefc33c51ddcc32dd509fc4114b5845256463a69a10ea2dfde402b2f8"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 363,
      "sha256": "9b11cadc98c30b968a70cc1658ce1fbd74b721b0f21402a2ff8bc1dc9d1474a0"
    }
  ],
  "totalBytes": 360020483
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})

DETECTOR_MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "rtdetr-r50vd",
  "modelId": "PekingU/rtdetr_r50vd",
  "revision": "df939e661d8c52e80608d1ec566561aabd25a4e7",
  "files": [
    {
      "path": "README.md",
      "bytes": 9053,
      "sha256": "4a0c10ddd0dbf6a2c6815cfc1613a5d74f3b0e7c8c77f0252212d3e5365e98cb"
    },
    {
      "path": "config.json",
      "bytes": 5113,
      "sha256": "2ed2a305c51eef46715eb755a02b2a266ecfb752936cc9574bb5714601c2742d"
    },
    {
      "path": "model.safetensors",
      "bytes": 172175856,
      "sha256": "5263d5521eff3e356f6cd8a371fd5dfb891725beda5f713674f79669115cdc64"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 841,
      "sha256": "ffb4b9461a1dad746be8f0f9c8330ed7743a1ba5fba4f75c232cd281b3d4c64a"
    }
  ],
  "totalBytes": 172190863
}

if (DETECTOR_MANIFEST['modelId'], DETECTOR_MANIFEST['revision']) != (DETECTOR_MODEL_ID, DETECTOR_REVISION):
    raise RuntimeError('inline rtdetr-r50vd manifest does not name the identity carried by the pipeline module')
DETECTOR_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(DETECTOR_WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(DETECTOR_MANIFEST, handle, indent=2)
fetched_rtdetr_r50vd = stage_missing_detector_files(DETECTOR_WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(DETECTOR_WEIGHTS_DIR), 'fetched': fetched_rtdetr_r50vd})
_extra = verify_detector_snapshot(DETECTOR_WEIGHTS_DIR)
_extra_files = _extra.get('files', []) if isinstance(_extra, dict) else []
print({'verified_files_rtdetr_r50vd': len(_extra_files) if isinstance(_extra_files, list) else _extra_files})
pipe = VitPoseKeypointPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, detector_dir=DETECTOR_WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

{'model_id': 'usyd-community/vitpose-base', 'revision': '95be2991424e646950d656bb7fc15ec9be119700', 'license': 'apache-2.0', 'files': 4, 'total_bytes': 360020483}


{'weights_dir': '/content/weights/vitpose-base', 'fetched': ['README.md', 'config.json', 'model.safetensors', 'preprocessor_config.json']}


{'verified_files': 4, 'revision': '95be2991424e646950d656bb7fc15ec9be119700'}


{'weights_dir': '/content/weights/rtdetr-r50vd', 'fetched': ['README.md', 'config.json', 'model.safetensors', 'preprocessor_config.json']}


{'verified_files_rtdetr_r50vd': 4}


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


{'device': 'cuda:0', 'source': 'local-snapshot'}


## 4. COCO persons, the small-person degradation and the split

`fetch_corpus` returns the 300 pinned photographs from the cache under `weights/coco-val-persons/` or the COCO image host — every cached file is re-hashed and every fetched file refused on any byte-size or SHA-256 mismatch — and `read_corpus` turns each labelled person into a record: `degrade` downscales the whole photograph (bicubic) so the person's box is `TARGET_HEIGHT` px tall, re-encodes it as JPEG at `JPEG_QUALITY`, and scales the box and the labelled joints by the same factor; the record's `category` says how many of the 17 joints COCO labelled (`full` 16–17, `partial` 14–15, `sparse` 12–13 — a proxy for occlusion and truncation). `build_sample_dataset` draws a seeded image-level split (180 / 45 / 75 photographs; every person follows its photograph), and `read_corpus(..., target_height=None)` builds the **clean counterparts** of the test persons — the same photographs unchanged — for Section 6. `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no photograph (by decoded-pixel digest) is shared, and the training split's summary table is written to `outputs/vitpose_keypoint_train.csv`.

Look for: 300 photographs and 498 persons, boxes about 40 px tall after degradation, three categories, three digests, and four refusal probes — a duplicate id, a joint outside its image, a person with too few labelled joints, and a dataset too small to use — each rejected before the model does anything.

*Evaluation practice.* **Bring your own data (optional):** set `USE_BYOD = True` and either `BYOD_PATH` (a zip or a folder holding `keypoints.csv` and the photographs, as a path in this runtime — this works on Colab, Kaggle and Jupyter) or leave `BYOD_PATH` empty to upload exactly one zip through the Colab dialog; then choose **Run after** from this cell. This cell first puts the model back to the pinned base, so Section 6 and Section 7's epoch 0 read the frozen model. Your persons are not degraded. The effective minimum is 12 persons on 12 single-person images.

**Predict before running:** a photograph can hold several persons. Why must all of them land in the same split?

In [ ]:
import hashlib
import json
import time

import numpy as np
from PIL import Image, ImageDraw

USE_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ''  # @param {type:"string"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
# A re-run after Section 7 (BYOD, or a new split): Section 6 and Section 7's epoch 0 must read the pinned base.
had_adapter = pipe.adapter is not None
restored_tensors = pipe.restore_base()
if had_adapter or restored_tensors:
    print({'restored_pinned_base': len(restored_tensors), 'note': 'the fine-tuned head and blocks were put back to the checkpoint; Sections 5-7 start from it again'})
if USE_BYOD:
    if BYOD_PATH.strip():
        byod_zip = Path(BYOD_PATH.strip()).expanduser()
        if not byod_zip.exists():
            raise FileNotFoundError(f'BYOD_PATH {BYOD_PATH!r} does not exist (relative paths start at {Path.cwd()}): give a .zip or a folder holding keypoints.csv and the photographs.')
        file_name = byod_zip.name
    else:
        try:
            from google.colab import files
        except ImportError:
            raise RuntimeError('USE_BYOD is True but BYOD_PATH is empty, and the upload dialog exists only in Google Colab: on Kaggle or Jupyter put the zip (or folder) in the runtime and set BYOD_PATH to its path.') from None
        uploaded = files.upload() or {}
        if len(uploaded) != 1:
            raise ValueError(f'Upload exactly one .zip file (received {len(uploaded)}; a cancelled dialog sends none): run this cell again.')
        file_name, payload = next(iter(uploaded.items()))
        if not file_name.lower().endswith('.zip'):
            raise ValueError(f'{file_name}: upload one .zip holding keypoints.csv and the photographs.')
        byod_zip = Path('work') / 'byod.zip'
        byod_zip.parent.mkdir(parents=True, exist_ok=True)
        byod_zip.write_bytes(payload)
    records = load_byod_dataset(byod_zip)
    splits = split_dataset(records, seed=SPLIT_SEED)
    clean_test = None
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod': len(records), 'effective_minimum': min_byod_records()['total']}
    if len(splits['test']) < 20:
        print({'caution': f"only {len(splits['test'])} held-out test persons: PCK and OKS move in large steps and carry no dispersion estimate; add persons before reading them"})
else:
    t0 = time.perf_counter()
    corpus_files = fetch_corpus(cache_dir='weights/coco-val-persons')
    corpus = read_corpus(corpus_files)
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    test_images = {r['image_id'] for r in splits['test']}
    clean_test = read_corpus({k: v for k, v in corpus_files.items() if k in test_images}, target_height=None)
    data_source = f'{CORPUS_NAME}: {CORPUS_RELEASE}'
    raw_rows = {'photographs': len(corpus_files), 'bytes': sum(len(v) for v in corpus_files.values()), 'persons': len(corpus), 'seconds': round(time.perf_counter() - t0, 1)}
# The training split must hold MIN_RECORDS; validation and test only need one person each (split_dataset checks that).
dataset_manifests = {name: validate_dataset(part, min_records=MIN_RECORDS if name == 'train' else 1) for name, part in splits.items()}
splits = {name: manifest['records'] for name, manifest in dataset_manifests.items()}
disjoint = check_split_disjoint(splits)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
if clean_test is not None:
    clean_test = validate_dataset(clean_test)['records']
    if [r['id'] for r in clean_test] != [r['id'] for r in test_records]:  # a contract check: clean counterparts align with the test persons
        raise RuntimeError('the clean counterparts do not line up with the test persons')
write_dataset_csv(train_records, 'outputs/vitpose_keypoint_train.csv')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint, 'degradation': {'target_height_px': TARGET_HEIGHT, 'jpeg_quality': JPEG_QUALITY}, 'licence': CORPUS_LICENSE})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'images': manifest['n_images'], 'category_counts': manifest['category_counts'], 'box_height': manifest['box_height'], 'labelled_joints': manifest['labelled_joints'], 'digest': manifest['digest'][:16] + '...'}})
example = train_records[0]
print({'example': {'id': example['id'], 'category': example.get('category'), 'image': list(example['image'].size), 'box': [round(v, 1) for v in example['box']], 'labelled': len(example['keypoints']), 'scale': example.get('scale'), 'flickr': example.get('flickr_url'), 'licence': example.get('license')}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'joint outside its image': [{**train_records[0], 'keypoints': {**train_records[0]['keypoints'], 'Nose': [-5.0, 0.0]}}, *train_records[1:8]],
    'too few labelled joints': [{**train_records[0], 'keypoints': dict(list(train_records[0]['keypoints'].items())[:5])}, *train_records[1:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

{'data_source': 'COCO 2017 validation persons (Flickr CC BY 2.0 / CC BY-SA 2.0 photographs; keypoint annotations CC BY 4.0): COCO 2017 (annotations_trainval2017 person_keypoints_val2017.json; images val2017)', 'raw_rows': {'photographs': 300, 'bytes': 52141813, 'persons': 498, 'seconds': 121.8}, 'splits': {'train': 290, 'validation': 72, 'test': 136}, 'degradation': {'target_height_px': 40, 'jpeg_quality': 30}, 'licence': "photographs: each record's Flickr licence (CC BY 2.0 or CC BY-SA 2.0, ids 4 and 5 of the COCO licence table); annotations: CC BY 4.0 (COCO Consortium)"}
{'train': {'n': 290, 'images': 288, 'category_counts': {'full': 90, 'partial': 93, 'sparse': 107}, 'box_height': {'min': 39.6, 'max': 40.5}, 'labelled_joints': {'min': 12, 'max': 17}, 'digest': '319541ce1215953a...'}}
{'validation': {'n': 72, 'images': 69, 'category_counts': {'full': 20, 'partial': 22, 'sparse': 30}, 'box_height': {'min': 39.6, 'max': 40.4}, 'labelled_joints': {'min': 12, 'max': 17}, 'digest': 'f2f0c

**What to notice:** 300 photographs and 498 persons, boxes about 40 px tall, the three categories, the three digests and the four refusals.

<details><summary>Check your reasoning</summary>Persons in one photograph share its lighting, background and compression, and some overlap; split by person, the test set would contain the same photograph as training and the score would measure memory of that image. Keeping every person with its photograph makes the test a question about unseen photographs.</details>

## 5. Estimate a synthetic drawing through the two-stage inference contract

*Core concept.* A top-down pose estimator needs a person box first and always returns 17 joints for it, whether or not the box contains a person. **Predict before running:** will the photograph-trained detector find the cartoon person?

The inference contract is exercised as the inference-only tutorial exercised it: a 640×640 cartoon person drawn with Pillow from a dictionary of 17 joint coordinates in COCO order (`REFERENCE_JOINTS`), with the drawn figure's box `DRAWN_BOX` — an image family the adaptation never sees, and a figure the adapted model will estimate again in Section 9. `validate_inputs` applies exactly the checks `estimate` applies (image sides, 1..`MAX_PERSONS` boxes inside the image, both thresholds in `[0, 1]`) and returns an input manifest; a box that leaves the image is validated too and its rejection recorded as a finding. `detect_people` runs stage 1 alone: on the cartoon the photograph-trained RT-DETR finds **no** `person` at the default 0.3 (recorded as a finding), so `estimate` receives the drawn box as a caller box (`box_source: caller`) and returns 17 joints with heatmap maxima as scores — a box that contains no person would still yield 17 joints, with low scores as the only signal. The per-image `evaluation_report` against the self-drawn joints is `sample-sanity` (PCK within `PCK_FRACTION` of the box's longest side) — plumbing evidence, not a measurement; whether the model is *good at small photographed persons* is what Section 6 measures.

In [ ]:
CX = 320
REFERENCE_JOINTS = {
    'Nose': (CX, 130), 'L_Eye': (CX + 12, 118), 'R_Eye': (CX - 12, 118), 'L_Ear': (CX + 30, 125), 'R_Ear': (CX - 30, 125),
    'L_Shoulder': (CX + 60, 200), 'R_Shoulder': (CX - 60, 200), 'L_Elbow': (CX + 95, 290), 'R_Elbow': (CX - 95, 290),
    'L_Wrist': (CX + 120, 380), 'R_Wrist': (CX - 120, 380), 'L_Hip': (CX + 40, 350), 'R_Hip': (CX - 40, 350),
    'L_Knee': (CX + 50, 450), 'R_Knee': (CX - 50, 450), 'L_Ankle': (CX + 55, 545), 'R_Ankle': (CX - 55, 545),
}
DRAWN_BOX = [180.0, 80.0, 460.0, 560.0]


def cartoon_person(width=640, height=640):
    """A flat cartoon person drawn from REFERENCE_JOINTS: head, torso, arms, legs on a plain background."""
    img = Image.new('RGB', (width, height), (225, 232, 240))
    d = ImageDraw.Draw(img)
    d.rectangle([0, 520, width, height], fill=(150, 160, 140))
    skin, shirt, pants = (222, 180, 140), (200, 60, 60), (40, 60, 140)
    j = REFERENCE_JOINTS
    d.ellipse([CX - 42, 88, CX + 42, 172], fill=skin, outline=(120, 80, 60), width=3)
    for eye in ('L_Eye', 'R_Eye'):
        d.ellipse([j[eye][0] - 5, j[eye][1] - 5, j[eye][0] + 5, j[eye][1] + 5], fill=(30, 30, 30))
    d.arc([CX - 18, 135, CX + 18, 160], 10, 170, fill=(120, 40, 40), width=3)
    d.polygon([j['R_Shoulder'], j['L_Shoulder'], (j['L_Hip'][0] + 10, j['L_Hip'][1]), (j['R_Hip'][0] - 10, j['R_Hip'][1])], fill=shirt)
    d.line([(CX, 172), (CX, 200)], fill=skin, width=18)
    for side in ('L', 'R'):
        d.line([j[f'{side}_Shoulder'], j[f'{side}_Elbow'], j[f'{side}_Wrist']], fill=shirt, width=26, joint='curve')
        d.line([j[f'{side}_Elbow'], j[f'{side}_Wrist']], fill=skin, width=22, joint='curve')
        d.ellipse([j[f'{side}_Wrist'][0] - 14, j[f'{side}_Wrist'][1] - 14, j[f'{side}_Wrist'][0] + 14, j[f'{side}_Wrist'][1] + 14], fill=skin)
        d.line([j[f'{side}_Hip'], j[f'{side}_Knee'], j[f'{side}_Ankle']], fill=pants, width=34, joint='curve')
        d.ellipse([j[f'{side}_Ankle'][0] - 22, j[f'{side}_Ankle'][1] - 10, j[f'{side}_Ankle'][0] + 22, j[f'{side}_Ankle'][1] + 14], fill=(40, 40, 40))
    return img


scene = cartoon_person()
scene_reference = {k: (float(x), float(y)) for k, (x, y) in REFERENCE_JOINTS.items()}
scene_name = 'synthetic_person_640x640.png'
scene_sha256 = hashlib.sha256(np.asarray(scene).tobytes()).hexdigest()
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_PERSONS': MAX_PERSONS, 'DETECTION_THRESHOLD': DETECTION_THRESHOLD, 'KEYPOINT_THRESHOLD': KEYPOINT_THRESHOLD, 'PCK_FRACTION': PCK_FRACTION, 'MIN_LABELLED': MIN_LABELLED, 'MIN_RECORDS': MIN_RECORDS, 'MAX_RECORDS': MAX_RECORDS, 'device': pipe.device}})
input_manifest = validate_inputs(scene, person_boxes=[DRAWN_BOX], detection_threshold=DETECTION_THRESHOLD, keypoint_threshold=KEYPOINT_THRESHOLD, names=[scene_name])
try:
    validate_inputs(scene, person_boxes=[[0, 0, scene.width + 1, 10]])
except ValueError as exc:
    input_manifest['findings'].append({'input': 'box-outside-image-probe', 'verdict': 'rejected', 'message': str(exc)})
stage1 = pipe.detect_people(scene, threshold=DETECTION_THRESHOLD)
scene_findings = []
if stage1['n_persons'] == 0:
    scene_findings.append('stage 1 found no `person` on the drawing at the default threshold; stage 2 receives the drawn box as a caller box')
with open('outputs/vitpose_keypoint_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'scene': scene_name, 'sha256': scene_sha256[:16] + '...', 'manifest_verdict': input_manifest['verdict'], 'findings': len(input_manifest['findings']), 'stage1_persons': stage1['n_persons'], 'scene_findings': scene_findings})


def estimate_scene(pipeline, label):
    started = time.perf_counter()
    result = pipeline.estimate(scene, person_boxes=[DRAWN_BOX], detection_threshold=DETECTION_THRESHOLD, keypoint_threshold=KEYPOINT_THRESHOLD)
    elapsed = time.perf_counter() - started
    pose = result['poses'][0]
    checks = {
        'one_person': result['n_persons'] == 1 and result['box_source'] == 'caller',
        'seventeen_joints_in_order': [kp['name'] for kp in pose['all_keypoints']] == list(KEYPOINT_NAMES),
        'identity_reported': result['model_id'] == MODEL_ID and result['model_revision'] == MODEL_REVISION and result['detector_revision'] == DETECTOR_REVISION,
    }
    if not all(checks.values()):
        raise RuntimeError(f'estimate output failed a sanity check: {checks}')
    report = evaluation_report(result, [scene_reference], sample_kind='synthetic (authored in this notebook)')
    annotated = scene.copy()
    draw = ImageDraw.Draw(annotated)
    joints = {kp['name']: (kp['x'], kp['y']) for kp in pose['all_keypoints']}
    for a, b in SKELETON_EDGES:
        draw.line([joints[KEYPOINT_NAMES[a]], joints[KEYPOINT_NAMES[b]]], fill=(255, 200, 0), width=3)
    annotated.save(f'outputs/vitpose_keypoint_scene_{label}.png')
    print({label: {'seconds': round(elapsed, 3), 'checks': checks, 'mean_score': round(pose['mean_keypoint_score'], 3), 'pck': round(report['metrics'][0]['value'], 3), 'mean_error_px': round(report['metrics'][0]['mean_error_px'], 1), 'verdict': report['verdict']}})
    return result, report


frozen_scene_result, frozen_scene = estimate_scene(pipe, 'frozen')

{'ceilings': {'MIN_IMAGE_SIDE': 16, 'MAX_IMAGE_SIDE': 4096, 'MAX_PERSONS': 50, 'DETECTION_THRESHOLD': 0.3, 'KEYPOINT_THRESHOLD': 0.3, 'PCK_FRACTION': 0.1, 'MIN_LABELLED': 12, 'MIN_RECORDS': 8, 'MAX_RECORDS': 5000, 'device': 'cuda:0'}}


{'scene': 'synthetic_person_640x640.png', 'sha256': 'b4d22c331cfd5b21...', 'manifest_verdict': 'accepted', 'findings': 1, 'stage1_persons': 0, 'scene_findings': ['stage 1 found no `person` on the drawing at the default threshold; stage 2 receives the drawn box as a caller box']}
{'frozen': {'seconds': 0.074, 'checks': {'one_person': True, 'seventeen_joints_in_order': True, 'identity_reported': True}, 'mean_score': 0.93, 'pck': 1.0, 'mean_error_px': 12.8, 'verdict': 'sample-sanity'}}


**What to notice:** `stage1_persons`, `box_source`, the joints' mean score and the self-drawn PCK.

<details><summary>Check your reasoning</summary>No: the build record's RT-DETR found no `person` on the cartoon at the default 0.3, so `estimate` used the drawn box as a caller box — and ViTPose still returned 17 joints, matching the self-drawn ones (PCK 1.0). A box with no person in it would also get 17 joints; only the low heatmap scores would say so. This is plumbing evidence, not a measurement.</details>

## 6. Baselines, the frozen model on the small persons — and on the clean ones

Two box-only baselines frame the adaptation, each read two ways by `pose_metrics` (carried in `metrics.py`): **PCK** — the share of labelled joints within `PCK_FRACTION` (0.1) of the box's longest side of their label, the package's own `keypoint_pck` convention — and **mean OKS** — the COCO object-keypoint-similarity kernel per joint (COCO's per-joint sigmas, the box area as the object scale) averaged over the labelled joints, the primitive of the COCO keypoint AP without its thresholds — averaged over the persons and per category. The **box-centre** baseline puts every joint at the box centre: the floor. The **mean-pose** baseline puts every joint at its mean position relative to the box over the training split — the layout prior a box alone gives away, and the number a pose estimator must beat to be doing anything. The **frozen model** is scored by `pipe.evaluate`, which runs every record's box through `estimate` and scores all 17 joints. It is scored twice: on the **small persons** the adaptation is about, and on their **clean counterparts** — the same persons in the unchanged photographs — which is where the checkpoint was trained to work. **What to look for:** the frozen model against the mean-pose prior on the small persons and on the clean ones, and the per-category rows.

*Evaluation practice.* The cell records a **verdict** — whether the frozen model beats the box-centre floor — instead of asserting it.

**Predict before running:** how much PCK will a 40-px JPEG person cost the frozen model, compared with the same person at full resolution?

In [ ]:
METRICS = ('pck', 'oks')
baseline_centre = box_centre_baseline(test_records)
baseline_prior = mean_pose_baseline(train_records, test_records)
print({'box_centre_baseline': {k: round(baseline_centre[k], 3) for k in METRICS}, 'n': baseline_centre['n'], 'note': baseline_centre['baseline']})
print({'mean_pose_baseline': {k: round(baseline_prior[k], 3) for k in METRICS}, 'note': baseline_prior['baseline']})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records)
print({'frozen_small_persons': {k: round(frozen_test[k], 3) for k in METRICS}, 'n': frozen_test['n'], 'verdict': frozen_test['verdict'], 'seconds': round(time.perf_counter() - t0, 1)})
frozen_clean = pipe.evaluate(clean_test) if clean_test is not None else None
if frozen_clean is not None:
    print({'frozen_clean_counterparts': {k: round(frozen_clean[k], 3) for k in METRICS}})
print({'definitions': frozen_test['definitions']})
frozen_fields = {c: {'n': v['n'], 'pck': round(v['pck'], 3), 'oks': round(v['oks'], 3)} for c, v in frozen_test['per_category'].items()}
print({'by_category_frozen': frozen_fields})
frozen_verdict = 'frozen above the box-centre floor' if frozen_test['pck'] > baseline_centre['pck'] else 'frozen NOT above the box-centre floor'
print({'frozen_vs_floor': frozen_verdict})

{'box_centre_baseline': {'pck': 0.086, 'oks': 0.165}, 'n': 136, 'note': 'every joint at the box centre'}
{'mean_pose_baseline': {'pck': 0.472, 'oks': 0.44}, 'note': "every joint at the training split's mean position relative to the box"}


{'frozen_small_persons': {'pck': 0.598, 'oks': 0.564}, 'n': 136, 'verdict': 'measured', 'seconds': 3.0}


{'frozen_clean_counterparts': {'pck': 0.957, 'oks': 0.94}}
{'definitions': {'pck': "share of labelled joints whose prediction lies within PCK_FRACTION (0.1) of the person box's longest side of the reference; in 0..1, higher is better", 'oks': 'mean over labelled joints of exp(-d^2 / (2 * area * (2 * sigma_joint)^2)) with the COCO keypoint sigmas and the box area — the COCO OKS kernel per joint; in 0..1, higher is better'}}
{'by_category_frozen': {'full': {'n': 38, 'pck': 0.604, 'oks': 0.579}, 'partial': {'n': 47, 'pck': 0.627, 'oks': 0.569}, 'sparse': {'n': 51, 'pck': 0.568, 'oks': 0.549}}}
{'frozen_vs_floor': 'frozen above the box-centre floor'}


**What to notice:** PCK and OKS for the floor, the mean-pose prior and the frozen model, small and clean, and the per-category rows.

<details><summary>Check your reasoning</summary>About a third. In the Kaggle T4 release run (20 September 2026) the frozen model scored PCK 0.957 on the clean persons and 0.598 on the same persons at 40 px (OKS 0.94 and 0.564), against 0.472 for the mean-pose prior and 0.086 for the box centre. On fully labelled small persons it scored 0.604, only a little above the mean-pose prior there: small, fully visible persons are being read largely from the box layout, not from the pixels.</details>

## 7. Bounded fine-tuning of the heatmap head and the last encoder blocks

`pipe.adapt` trains only the heatmap head (two deconvolutions and the final convolution) and the last `TRAINABLE_BLOCKS` encoder blocks with the final backbone LayerNorm — 18,376,977 of 89,994,513 parameters for two blocks — while the patch embedding, the earlier blocks and the whole detector stay frozen. Every record's box is affine-warped to the processor's 192×256 crop exactly as at inference, its labelled joints become Gaussian targets (sigma `HEATMAP_SIGMA` = 2) on the 64×48 heatmap grid in the same frame, and the head's heatmaps are trained with the **joint-weighted mean-squared error** ViTPose was trained with (unlabelled joints carry no weight). AdamW without weight decay at a fixed learning rate, gradient clipping at 1.0, seeded shuffling, no scheduler, no augmentation; the head's BatchNorm statistics stay frozen so the adapter is parameters-only. Epoch 0 records the frozen model's validation metrics; every epoch is scored on the 72 validation persons, and the epoch with the highest **mean of validation PCK and OKS** is kept.

Watch the training loss fall from about 0.0028 towards 0.0013 while the validation PCK climbs from about 0.66 to 0.74 over eight epochs: the head is learning to read blurred, block-edged limbs it never saw at training resolution, which 290 persons are enough to teach. The build record's counter-examples — the head alone, which moved PCK by a hundredth, and four blocks at 32-px persons, which peaked and then overfit — are why the default is two blocks at 40 px.

*Core concept.* Every call to `pipe.adapt` starts from the **pinned base**: tensors an earlier call (or an artifact) changed are put back first, so epoch 0 is always the frozen model, a head-only re-run really is head only, and re-running Sections 7–9 with a changed field repeats the comparison validly. To compare a change side by side without replacing the default exports, use Section 10.

**Predict before running:** will validation keep the last epoch, or an earlier one?

In [ ]:
EPOCHS = 8  # @param {type:"integer"}
LEARNING_RATE = 5e-5  # @param {type:"number"}
BATCH_SIZE = 8  # @param {type:"integer"}
TRAINABLE_BLOCKS = 2  # @param {type:"integer"}


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 5)}
    if entry.get('val'):
        row.update({'val_' + k: round(entry['val'][k], 3) for k in (*METRICS, 'score')})
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


settings = {'epochs': EPOCHS, 'lr': LEARNING_RATE, 'batch_size': BATCH_SIZE, 'trainable_blocks': TRAINABLE_BLOCKS}
if settings != {'epochs': 8, 'lr': 5e-5, 'batch_size': 8, 'trainable_blocks': 2}:
    print({'note': 'changed settings: this run starts again from the pinned base and replaces the default results of Sections 8-9; Section 10 compares a change side by side instead', 'settings': settings})
t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_blocks=TRAINABLE_BLOCKS, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'preparation_seconds': adapt_result['preparation_seconds'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'loss': adapt_result['loss'], 'started_from': adapt_result['started_from'], 'seconds': adapt_seconds})

{'epoch': 0, 'train_loss': None, 'val_pck': 0.663, 'val_oks': 0.627, 'val_score': 0.645, 'note': 'frozen model'}


{'epoch': 1, 'train_loss': 0.0028, 'val_pck': 0.71, 'val_oks': 0.667, 'val_score': 0.688}


{'epoch': 2, 'train_loss': 0.00243, 'val_pck': 0.723, 'val_oks': 0.676, 'val_score': 0.7}


{'epoch': 3, 'train_loss': 0.00222, 'val_pck': 0.732, 'val_oks': 0.69, 'val_score': 0.711}


{'epoch': 4, 'train_loss': 0.00203, 'val_pck': 0.735, 'val_oks': 0.696, 'val_score': 0.715}


{'epoch': 5, 'train_loss': 0.00183, 'val_pck': 0.732, 'val_oks': 0.7, 'val_score': 0.716}


{'epoch': 6, 'train_loss': 0.00164, 'val_pck': 0.744, 'val_oks': 0.7, 'val_score': 0.722}


{'epoch': 7, 'train_loss': 0.00144, 'val_pck': 0.742, 'val_oks': 0.702, 'val_score': 0.722}


{'epoch': 8, 'train_loss': 0.00125, 'val_pck': 0.743, 'val_oks': 0.702, 'val_score': 0.723}
{'trainable_parameters': 18376977, 'total_parameters': 89994513, 'preparation_seconds': 3.554, 'best_epoch': 8, 'selection': 'highest mean of validation PCK and OKS', 'loss': 'joint-weighted mean-squared error against Gaussian heatmap targets (sigma 2.0) in the 64x48 heatmap frame', 'started_from': 'pinned base', 'seconds': 62.8}


**What to notice:** the validation PCK and OKS per epoch, and `best_epoch`.

<details><summary>Check your reasoning</summary>Whichever epoch has the highest mean of validation PCK and OKS — on the build workstation the validation PCK climbed from about 0.66 to 0.74 over the eight epochs. If no epoch beats the frozen model, epoch 0 is kept and the adapted model *is* the frozen model: that is a valid result, and Section 8 records it as `no gain`.</details>

## 8. Held-out evaluation

The test persons were never used for training or epoch selection, and no photograph appears in two splits. The adapted model is scored exactly as the frozen model was in Section 6 — on the small persons and on their clean counterparts — the systems are put side by side on both measures, and the per-category breakdown is repeated. Read it in this order: **PCK on the small persons** first, then **mean OKS**, then the **clean counterparts** — PCK *and* OKS, because they need not move together — then the per-category rows. The cell records a verdict — `improved`, `no gain` or `worse` — instead of asserting a gain: when validation keeps epoch 0 the adapted model is the frozen model, and on undegraded BYOD persons the frozen model may already be near the ceiling; the export, reload and result still run. A hundred-odd persons from one seeded split under one degradation give **no dispersion estimate**; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a gain at 40-px JPEG persons says nothing about motion blur, unusual viewpoints or your camera's own artefacts until you measure them.

**Predict before running:** will the fine-tune cost the clean persons anything?

In [ ]:
adapted_test = pipe.evaluate(test_records)
adapted_val = pipe.evaluate(val_records)
adapted_clean = pipe.evaluate(clean_test) if clean_test is not None else None
adapted_fields = {c: {'n': v['n'], 'pck': round(v['pck'], 3), 'oks': round(v['oks'], 3)} for c, v in adapted_test['per_category'].items()}
comparison = {metric: {'box_centre': round(baseline_centre[metric], 3), 'mean_pose': round(baseline_prior[metric], 3), 'frozen': round(frozen_test[metric], 3), 'adapted': round(adapted_test[metric], 3)} for metric in METRICS}
if frozen_clean is not None:
    for metric in METRICS:
        comparison[metric].update({'frozen_clean': round(frozen_clean[metric], 3), 'adapted_clean': round(adapted_clean[metric], 3)})
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 3) for metric in METRICS}
def direction(new, old):
    return 'improved' if new > old else ('no gain' if new == old else 'worse')
# Reported verdicts, not assertions: a fine-tune that does not help is a result to record, and export and reload still run.
comparison['verdicts'] = {'frozen_vs_floor': frozen_verdict, 'adapted_vs_frozen_pck': direction(adapted_test['pck'], frozen_test['pck']), 'adapted_vs_frozen_oks': direction(adapted_test['oks'], frozen_test['oks']), 'adapted_above_mean_pose_pck': bool(adapted_test['pck'] > baseline_prior['pck'])}
if adapted_clean is not None:
    comparison['verdicts'].update({'clean_pck': direction(adapted_clean['pck'], frozen_clean['pck']), 'clean_oks': direction(adapted_clean['oks'], frozen_clean['oks'])})
comparison['delta_vs_mean_pose'] = {metric: round(adapted_test[metric] - baseline_prior[metric], 3) for metric in METRICS}
comparison['by_category'] = {c: {'n': frozen_fields[c]['n'], 'frozen_pck': frozen_fields[c]['pck'], 'adapted_pck': adapted_fields[c]['pck'], 'frozen_oks': frozen_fields[c]['oks'], 'adapted_oks': adapted_fields[c]['oks']} for c in frozen_fields}
for key, row in comparison.items():
    print({key: row})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'degradation': {'target_height_px': TARGET_HEIGHT, 'jpeg_quality': JPEG_QUALITY},
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'baselines': {'box_centre': {k: v for k, v in baseline_centre.items() if k != 'per_record'}, 'mean_pose': {k: v for k, v in baseline_prior.items() if k != 'per_record'}},
    'frozen_test': frozen_test,
    'frozen_clean': frozen_clean,
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'clean_metrics': adapted_clean,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/vitpose_keypoint_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
print({'verdicts': comparison['verdicts']})
print({'report': 'outputs/vitpose_keypoint_evaluation_report.json', 'adapted_beats_mean_pose': adapted_test['pck'] > baseline_prior['pck']})

{'pck': {'box_centre': 0.086, 'mean_pose': 0.472, 'frozen': 0.598, 'adapted': 0.718, 'frozen_clean': 0.957, 'adapted_clean': 0.959}}
{'oks': {'box_centre': 0.165, 'mean_pose': 0.44, 'frozen': 0.564, 'adapted': 0.681, 'frozen_clean': 0.94, 'adapted_clean': 0.93}}
{'delta_vs_frozen': {'pck': 0.119, 'oks': 0.117}}
{'verdicts': {'frozen_vs_floor': 'frozen above the box-centre floor', 'adapted_vs_frozen_pck': 'improved', 'adapted_vs_frozen_oks': 'improved', 'adapted_above_mean_pose_pck': True, 'clean_pck': 'improved', 'clean_oks': 'worse'}}
{'delta_vs_mean_pose': {'pck': 0.245, 'oks': 0.241}}
{'by_category': {'full': {'n': 38, 'frozen_pck': 0.604, 'adapted_pck': 0.795, 'frozen_oks': 0.579, 'adapted_oks': 0.721}, 'partial': {'n': 47, 'frozen_pck': 0.627, 'adapted_pck': 0.703, 'frozen_oks': 0.569, 'adapted_oks': 0.677}, 'sparse': {'n': 51, 'frozen_pck': 0.568, 'adapted_pck': 0.673, 'frozen_oks': 0.549, 'adapted_oks': 0.655}}}
{'verdicts': {'frozen_vs_floor': 'frozen above the box-centre floor

**What to notice:** `delta_vs_frozen`, the clean rows (PCK and OKS), the per-category rows and the `verdicts`.

<details><summary>Check your reasoning</summary>A little, on one measure. In the Kaggle T4 release run (20 September 2026) the small persons rose from PCK 0.598 to 0.718 and OKS 0.564 to 0.681 — past the mean-pose prior — with fully labelled persons gaining most (0.604 → 0.795). On the clean persons PCK held (0.957 → 0.959) but OKS fell from 0.94 to 0.93: a small but real cost that PCK alone hides. The build workstation's record (RTX 5070 Ti) was close: 0.721 PCK, OKS 0.565 → 0.684, clean PCK 0.958.</details>

## 9. Look at the skeletons, export the adapter and reload it

The drawing from Section 5 is estimated again by the adapted model — an image family the adaptation never saw, so this is a small look at what the adaptation did *outside* its corpus: the build record kept the self-drawn PCK at 1.0 — and four held-out small persons are written as side-by-side panels (`outputs/vitpose_keypoint_examples/`: the degraded crop enlarged four times with the frozen skeleton, the adapted skeleton, and the labelled joints) so the numbers can be checked by eye: the adapted skeletons should sit on the blurred limbs where the frozen ones drift.

`pipe.save_artifact` writes the trained tensors — the head, the last two blocks and the final LayerNorm, about 74 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `VitPoseKeypointPipeline.from_artifact` re-verifies both base snapshots, checks the artifact manifest, its digest and its exact tensor set **before** deserialising, refuses any tensor outside the head and the recorded blocks, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical joints on eight test persons (VER4) — a contract check, so it stays a hard check.

**Predict before running:** in the panels, where will the adapted skeleton differ most from the frozen one?

In [ ]:
import shutil

adapted_scene_result, adapted_scene = estimate_scene(pipe, 'adapted')
examples_dir = Path('outputs/vitpose_keypoint_examples')
shutil.rmtree(examples_dir, ignore_errors=True)
examples_dir.mkdir(parents=True)
frozen_base = VitPoseKeypointPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, detector_dir=DETECTOR_WEIGHTS_DIR, device=pipe.device)


def skeleton_panel(record, joints, colour, zoom=4):
    x0, y0, x1, y1 = record['box']
    pad = 0.15 * (y1 - y0)
    crop_box = (max(0, int(x0 - pad)), max(0, int(y0 - pad)), min(record['image'].width, int(x1 + pad) + 1), min(record['image'].height, int(y1 + pad) + 1))
    panel = record['image'].crop(crop_box)
    panel = panel.resize((panel.width * zoom, panel.height * zoom), Image.Resampling.NEAREST)
    draw = ImageDraw.Draw(panel)
    at = {k: ((v[0] - crop_box[0]) * zoom, (v[1] - crop_box[1]) * zoom) for k, v in joints.items()}
    for a, b in SKELETON_EDGES:
        na, nb = KEYPOINT_NAMES[a], KEYPOINT_NAMES[b]
        if na in at and nb in at:
            draw.line([at[na], at[nb]], fill=colour, width=2)
    for x, y in at.values():
        draw.ellipse([x - 3, y - 3, x + 3, y + 3], fill=colour)
    return panel


for record in test_records[:4]:
    panels = [skeleton_panel(record, frozen_base.predict_keypoints(record), (0, 120, 255)), skeleton_panel(record, pipe.predict_keypoints(record), (0, 200, 80)), skeleton_panel(record, record['keypoints'], (255, 255, 255))]
    sheet = Image.new('RGB', (sum(p.width for p in panels) + 20, max(p.height for p in panels)), (255, 255, 255))
    x = 0
    for panel in panels:
        sheet.paste(panel, (x, 0))
        x += panel.width + 10
    sheet.save(examples_dir / f"{record['id']}_{record.get('category', 'person')}.png")
print({'examples': sorted(p.name for p in examples_dir.iterdir()), 'panel_order': ['frozen skeleton', 'adapted skeleton', 'labelled joints'], 'zoom': 4})

artifact_dir = Path('outputs/vitpose_keypoint_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'vitpose_keypoint', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = VitPoseKeypointPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, detector_dir=DETECTOR_WEIGHTS_DIR, device=pipe.device)
before = [pipe.predict_keypoints(r) for r in test_records[:8]]
after = [reloaded.predict_keypoints(r) for r in test_records[:8]]
parity = {'identical_persons': sum(all(np.allclose(a[k], b[k]) for k in KEYPOINT_NAMES) for a, b in zip(before, after, strict=True)), 'of': len(before), 'max_abs_difference_px': float(max(abs(np.array(a[k]) - np.array(b[k])).max() for a, b in zip(before, after, strict=True) for k in KEYPOINT_NAMES))}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_persons'] == parity['of']

result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'detector_model_id': DETECTOR_MODEL_ID,
    'detector_revision': DETECTOR_REVISION,
    'detector_license': DETECTOR_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'total_bytes': snapshot.get('total_bytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHT_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': pipe.weight_sha256},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'release': CORPUS_RELEASE, 'license': CORPUS_LICENSE, 'base_url': CORPUS_BASE_URL, 'bytes': CORPUS_BYTES, 'pinned_photographs': CORPUS_IMAGES, 'pinned_persons': CORPUS_PERSONS, 'degradation': {'target_height_px': TARGET_HEIGHT, 'jpeg_quality': JPEG_QUALITY}},
    'inference_contract': {'input_manifest': input_manifest, 'scene': {'name': scene_name, 'sha256': scene_sha256, 'findings': scene_findings}, 'stage1': stage1, 'frozen_report': frozen_scene, 'adapted_report': adapted_scene, 'output_files': ['outputs/vitpose_keypoint_scene_frozen.png', 'outputs/vitpose_keypoint_scene_adapted.png']},
    'comparison': comparison,
    'examples': 'outputs/vitpose_keypoint_examples',
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device, 'dtype': 'float32'},
}
with open('outputs/vitpose_keypoint_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

{'adapted': {'seconds': 0.027, 'checks': {'one_person': True, 'seventeen_joints_in_order': True, 'identity_reported': True}, 'mean_score': 0.963, 'pck': 1.0, 'mean_error_px': 11.5, 'verdict': 'sample-sanity'}}


{'examples': ['coco-1000-1234574_full.png', 'coco-1000-1259139_full.png', 'coco-1000-1269164_full.png', 'coco-1000-1303041_sparse.png'], 'panel_order': ['frozen skeleton', 'adapted skeleton', 'labelled joints'], 'zoom': 4}


{'artifact': 'outputs/vitpose_keypoint_adapter', 'format': 'org.valcorza.vitpose-base.adapter.v1', 'tensors': 42, 'bytes': 73512628, 'sha256': '5baace2ce609cbc7...'}


{'reload_parity': {'identical_persons': 8, 'of': 8, 'max_abs_difference_px': 7.62939453125e-06}, 'reloaded_best_epoch': 8}
['vitpose_keypoint_adapter', 'vitpose_keypoint_evaluation_report.json', 'vitpose_keypoint_examples', 'vitpose_keypoint_input_manifest.json', 'vitpose_keypoint_result.json', 'vitpose_keypoint_scene_adapted.png', 'vitpose_keypoint_scene_frozen.png', 'vitpose_keypoint_train.csv']


**What to notice:** the four panels in `outputs/vitpose_keypoint_examples/`, the drawing's PCK after adaptation, and the reload parity line.

<details><summary>Check your reasoning</summary>On the limbs: wrists, elbows and ankles are the joints a 40-px JPEG blurs most, and that is where the frozen skeleton drifts. The build record kept the self-drawn PCK at 1.0 after adaptation. Reload parity held in the release run: eight of eight persons identical, with a largest difference of 7.6e-06 px.</details>

## 10. Change one thing: the head alone (optional)

*Evaluation practice.* A **Predict → Change one thing → Run → Observe → Explain** activity, off by default so Run all is unaffected. Set `RUN_EXPERIMENT = True`, change **one** field — by default zero encoder blocks, so only the heatmap head trains — and run this cell after Sections 4–9. The experiment loads its **own** pipeline from the verified snapshots, so it starts from the checkpoint and never touches the default `pipe`; it writes only to `outputs/vitpose_keypoint_experiment/`, prints the default and the changed run side by side (epoch 0 must match), saves and reloads its own adapter to check parity, and checks that the default exports are byte-identical afterwards.

**Predict:** how much of the default run's gain will the head alone recover?

In [ ]:
RUN_EXPERIMENT = False  # @param {type:"boolean"}
EXPERIMENT_TRAINABLE_BLOCKS = 0  # @param {type:"integer"}
EXPERIMENT_EPOCHS = 8  # @param {type:"integer"}
EXPERIMENT_LEARNING_RATE = 5e-5  # @param {type:"number"}

if not RUN_EXPERIMENT:
    print({'experiment': 'skipped (RUN_EXPERIMENT = False); the default path above is complete'})
else:
    canonical_files = {'adapter': artifact_dir / 'adapter.safetensors', 'evaluation_report': Path('outputs/vitpose_keypoint_evaluation_report.json'), 'result': Path('outputs/vitpose_keypoint_result.json')}
    canonical = {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in canonical_files.items()}
    experiment_dir = Path('outputs/vitpose_keypoint_experiment')
    shutil.rmtree(experiment_dir, ignore_errors=True)
    experiment_dir.mkdir(parents=True)
    # Its own pipeline from the verified snapshots: the experiment starts from the checkpoint and the default pipe is untouched.
    experiment_pipe = VitPoseKeypointPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, detector_dir=DETECTOR_WEIGHTS_DIR, device=pipe.device)
    experiment_result = experiment_pipe.adapt(train_records, val_records, epochs=EXPERIMENT_EPOCHS, lr=EXPERIMENT_LEARNING_RATE, batch_size=BATCH_SIZE, trainable_blocks=EXPERIMENT_TRAINABLE_BLOCKS, progress=report)
    experiment_test = experiment_pipe.evaluate(test_records)
    experiment_pipe.save_artifact(experiment_dir / 'adapter', metadata={'tutorial': 'vitpose_keypoint', 'experiment': True})
    experiment_reloaded = VitPoseKeypointPipeline.from_artifact(experiment_dir / 'adapter', weights_dir=WEIGHTS_DIR, detector_dir=DETECTOR_WEIGHTS_DIR, device=pipe.device)
    experiment_parity = all(np.allclose(experiment_pipe.predict_keypoints(r)[k], experiment_reloaded.predict_keypoints(r)[k], atol=1e-3) for r in test_records[:4] for k in KEYPOINT_NAMES)
    side_by_side = {
        'settings': {'default': {'trainable_blocks': adapt_result['trainable_blocks'], 'epochs': adapt_result['epochs'], 'lr': adapt_result['lr']}, 'experiment': {'trainable_blocks': EXPERIMENT_TRAINABLE_BLOCKS, 'epochs': EXPERIMENT_EPOCHS, 'lr': EXPERIMENT_LEARNING_RATE}},
        'epoch_0_val_score': {'default': (adapt_result['history'][0]['val'] or {}).get('score'), 'experiment': (experiment_result['history'][0]['val'] or {}).get('score')},
        'best_epoch': {'default': adapt_result['best_epoch'], 'experiment': experiment_result['best_epoch']},
        'test': {metric: {'frozen': round(frozen_test[metric], 3), 'default': round(adapted_test[metric], 3), 'experiment': round(experiment_test[metric], 3)} for metric in METRICS},
        'trainable_parameters': {'default': adapt_result['n_trainable'], 'experiment': experiment_result['n_trainable']},
        'experiment_reload_parity': experiment_parity,
    }
    for key, row in side_by_side.items():
        print({key: row})
    with open(experiment_dir / 'experiment_report.json', 'w', encoding='utf-8') as handle:
        json.dump({'side_by_side': side_by_side, 'history': experiment_result['history']}, handle, indent=2, ensure_ascii=False, default=str)
    unchanged = {name: hashlib.sha256(path.read_bytes()).hexdigest() == canonical[name] for name, path in canonical_files.items()}
    if not all(unchanged.values()):
        raise RuntimeError(f'the experiment changed a default export: {unchanged}')
    print({'default_exports_unchanged': unchanged, 'experiment_outputs': str(experiment_dir)})
    del experiment_pipe, experiment_reloaded

{'experiment': 'skipped (RUN_EXPERIMENT = False); the default path above is complete'}


**Observe → Explain.** Compare `epoch_0_val_score` (it must be equal: both runs start from the checkpoint), the `test` rows and `experiment_reload_parity`.

<details><summary>Check your reasoning</summary>The build record's head-only run moved PCK by about a hundredth, against roughly twelve points for two blocks: the head can only re-read the features it is given, and at 40 px the features themselves are what is wrong — the last encoder blocks have to learn to read blurred, block-edged limbs. No experiment run is recorded on the release runtime; on 136 persons, read a difference of a point or two as noise.</details>

## Interpretation and limits

A pose estimator trained on crops of full-resolution persons loses a third of its PCK when the person is 40 px tall and JPEG-blocked (0.957 → 0.598 in the Kaggle T4 release run of 20 September 2026, against a box-only prior of 0.472), and a bounded fine-tuning of the heatmap head and the last two encoder blocks on 290 such persons recovers a third of that loss (0.718 PCK, 0.681 mean OKS) while holding the clean persons' PCK (0.957 → 0.959) at a small cost in their OKS (0.94 → 0.93), with a 74 MB adapter that reloads joint-for-joint. That is the claim: the adaptation contract works end to end on a real labelled set under a stated degradation, and the numbers it produces are read on two measures, per category, against two box-only baselines, the frozen model and the clean counterparts rather than in isolation.

The test split is 136 persons from one seeded draw of one sample under one degradation, the validation split that picks the epoch is 72, and both measures score joints against COCO's labels at a stated tolerance — PCK at a tenth of the box, OKS with COCO's sigmas — so a gain here says the contract works under 40-px JPEG-30 downscaling, not that the adapted model handles motion blur, low light, unusual viewpoints, crowds or your camera's own artefacts, and not that its heatmap scores are calibrated (they are not, before or after). The categories are COCO's labelling density, a proxy for occlusion, not a property of the person. Fine-tuning on one degradation also shapes what the model expects: the clean counterparts held here, but a different degradation would need its own measurement.

Three things to carry to real data. **Baselines first:** the box-centre floor and the mean-pose prior on *your* labels are the numbers to read before any model number, per subset. **Degradation:** the persons the model learns from define what it learns to read; make your training input the way your deployment makes it (the contract's `degrade` is one stated recipe; BYOD uses your labels as they are). **Leakage:** keep every photograph in one split (the contract does this by decoded pixels and keeps every person with its photograph) and split by camera or session when your images come from few sources.

Successful execution proves that the recorded repository revision's package, carried in this standalone notebook, can acquire and digest-verify both pinned model snapshots, fetch and digest-verify a real labelled photograph set and degrade it under a stated recipe, validate the demonstrated dataset contract without leakage, execute the two-stage inference contract and a bounded fine-tuning, evaluate against two box-only baselines, the frozen model and the clean counterparts on an image-disjoint split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, pose quality under any other degradation, calibration of the heatmap scores, or production fitness.

**Optional experiments (off by default; each names its field and what to run):** Section 10 trains the head alone in its own pipeline and prints it beside the default run — change `EXPERIMENT_TRAINABLE_BLOCKS` (4 peaks early and falls in the build record), `EXPERIMENT_EPOCHS` or `EXPERIMENT_LEARNING_RATE` (for example `1e-4`) there and run that cell again. Changing `EPOCHS`, `LEARNING_RATE` or `TRAINABLE_BLOCKS` and choosing **Run after** from Section 7 also starts from the pinned base — every `adapt` puts it back first — but replaces the default results and exports. BYOD: `USE_BYOD` and `BYOD_PATH` in Section 4, then **Run after** from Section 4, and read the two baselines before the adapted number.

## Troubleshooting

- **Section 1 stops with "This notebook needs a Linux x86_64 runtime"** — you are on Windows, macOS or an ARM machine. Use Google Colab, Kaggle or a Linux x86_64 Jupyter server.
- **The uv wheel fails its size/SHA-256 check, or a download in Section 1 times out** — run Section 1 again; a complete environment is reused, an incomplete one is finished. If it repeats, the network is blocking or altering `files.pythonhosted.org` or `pypi.org`.
- **"The isolated environment's Python process exited"** — usually out of memory. Restart the session and choose **Run all**; leave the optional experiment off on a small runtime.
- **You re-ran Section 1 on its own** — nothing is lost: it keeps the running worker and every variable, so the cells after it keep working. After a session restart, run from the top.
- **Section 3 reports a size or SHA-256 mismatch, or cannot reach the Hub** — the message names the file. Delete it from the snapshot folder Section 3 prints and run Section 3 again; the snapshot comes from `huggingface.co`.
- **Section 4 cannot fetch a photograph, or one fails its digest** — `fetch_corpus` names it; the default path needs `images.cocodataset.org` (plain HTTP; every file is checked against its pinned SHA-256). Run Section 4 again (cached photographs are re-hashed); delete `weights/coco-val-persons/` if a cached file is corrupt.
- **Section 7 takes half an hour** — that is the CPU cost of eight epochs; choose a GPU runtime and **Run all** again.
- **Out of memory** — lower `BATCH_SIZE` in Section 7, or restart the session and choose **Run all**; leave Section 10 off on a small runtime.
- **BYOD: "BYOD_PATH … does not exist"** — the path is relative to the working directory printed in the message.
- **BYOD: "the upload dialog exists only in Google Colab"** — on Kaggle or Jupyter, put the zip in the runtime (or attach it as a dataset) and set `BYOD_PATH`.
- **BYOD: "Upload exactly one .zip file"** — the dialog was cancelled or several files were chosen; run the cell again.
- **BYOD: "keypoints.csv line N (file …): names an image that is not among the uploaded files"** — fix the `file` column of that row, or add the image to the zip.
- **BYOD: "… must be numbers" / "… must be present and numeric"** — a box or joint coordinate on that line is not a number.
- **BYOD: "the train split holds … persons" or "… too few to split"** — add images or persons; the message names the minimum.
## Glossary

- **Top-down pose** — find each person's box first, then estimate that person's joints inside it.
- **Heatmap** — a 64 × 48 map per joint whose maximum is the predicted location; the maximum's value is a ranking score, not a probability.
- **PCK** — the share of labelled joints within a tenth of the box's longest side of their label.
- **OKS / sigma** — COCO's object-keypoint similarity: a per-joint Gaussian kernel whose width (sigma) is set per joint and scaled by the box area, averaged over the labelled joints.
- **Box-centre / mean-pose baseline** — every joint at the box centre; every joint at its mean position in the box over the training split.
- **Clean counterpart** — the same person in the unchanged, full-resolution photograph.
- **Degradation** — the stated recipe: downscale so the box is 40 px tall, then JPEG at quality 30.
- **Epoch selection** — keeping the epoch with the highest mean of validation PCK and OKS; epoch 0 is the frozen model.
- **Adapter / reload parity** — the trained tensors only (safetensors) overlaid on the pinned base; the reloaded pipeline gives identical joints.
- **BYOD** — bring your own data: your labelled persons through the same cells.

## Conclusion (your notes)

Optional — fill in from **your** run, not the recorded one:

- The data was ___ persons on ___ photographs; test split ___ persons.
- Mean-pose prior PCK ___; frozen ___ on the small persons and ___ on the clean ones.
- After fine-tuning (epoch ___ kept): PCK ___, OKS ___; clean PCK ___, clean OKS ___ — a cost of ___.
- What I would need before claiming the fine-tune helps on my camera: ___ (for example my own degradation, more test persons, several seeds).

## References

- Repository README: https://github.com/kurtvalcorza/vitpose-keypoint-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/vitpose-keypoint-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/vitpose-keypoint-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/usyd-community/vitpose-base
- Upstream code: https://github.com/ViTAE-Transformer/ViTPose
- ViTPose: Simple Vision Transformer Baselines for Human Pose Estimation (Xu, Zhang, Zhang, Tao, NeurIPS 2022): https://arxiv.org/abs/2204.12484
- RT-DETR (Zhao et al., CVPR 2024): https://arxiv.org/abs/2304.08069
- Microsoft COCO: Common Objects in Context (Lin et al., ECCV 2014; keypoint annotations CC BY 4.0, photographs under their Flickr licences): https://cocodataset.org/#termsofuse
- DIMER Notebook Specification 2.2 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)